# STEP 0 – Rules of the honest test (read before running anything below)

**Source:** Hyndman & Athanasopoulos, *Forecasting: Principles and Practice*, 2nd ed. (OTexts, 2018), §3.4 and §4.2.
These rules apply to every product in the Product hierarchy (Category2 → Sub_Category → Brand → Product_Code, 188 drinks).

| Week | Monday date | Role |
|---|---|---|
| 1 – 53 | 7 Apr 2025 – **6 Apr 2026** | Training + cross-validation (models may learn from these) |
| 54 – 66 | 13 Apr 2026 – 6 Jul 2026 | **TEST – sealed** (13 weeks = forecast horizon h) |
| 67 – 79 | 13 Jul 2026 – 5 Oct 2026 | Final forecast (refit on weeks 1 – 66 only after the test is scored) |

*Dates come from the `ds` column of `../eabl parsed csv/Y_product.csv` (66 Monday-labelled weeks).*

**(a) The seal is Monday 6 Apr 2026 (week 53).** Nothing used to build, choose or tune a model may use sales after this week.
> FPP2 §3.4: "The accuracy of forecasts can only be determined by considering how well a model performs on new data that were not used when fitting the model."

**(b) Weeks 54 – 66 are never edited or deleted.** Cleaning steps (F6 – F9) may *flag* these weeks but never change `y`, fill zeros, or remove rows. Any cleaning only touches weeks 1 – 53, keeps the original values, and saves to a new file.

**(c) Settings are locked before the test is opened:**
- `F2_H = 13`, `F2_MIN_SALES = 3` (F2 cell, this Prep notebook)
- Cross-validation: `h = 13`, `n_windows = 2`, `step_size = 13` (F1 notebook)

These are not changed after seeing test results.
> FPP2 §4.2: "Formalising and documenting the decision rules and assumptions … can promote consistency, as the same rules can be implemented repeatedly."

**(d) The test is scored ONCE,** at the end. No changing models or settings and re-scoring.

**(e) Judgement forecasts** (products with no sales before 6 Apr 2026) use only information known by 6 Apr 2026. Planners do not see weeks 54 – 66 first.
> FPP2 §4.2: implement "a systematic approach" and "Segregate forecasters and users".

**(f) Existing F1 accuracy numbers are CV scores, NOT test scores.** They were computed with cross-validation on all 66 weeks, so the last CV round used weeks 54 – 66, and the same errors chose each product's model. This applies to the WAPEs in F1 (standard group CV, Tier 1 / Tier 2 overall WAPE) and to `tier1_weekly_vs_cumulative.csv` and `final_forecast_all_skus.csv`.

**(g) Earlier cells that looked at ALL 66 weeks** (flags / diagnostics only – no sales values were changed). Each must be re-checked on weeks 1 – 53 before it is used to clean data or choose a model:
- Cleaning notebook: cell 31 (Category2 taken from the current product master), cell 34 (large-order limit = 5 × each SKU's 99th percentile over all dates), cell 35 ("normal for this customer" compares with any other date, including later ones), cells 40 – 41 (Sunday-spike rule; all 8 rows of `known_spikes.csv` are in the week of 8 Dec 2025, inside training)
- Prep notebook: cell 44 (structural breaks), cells 63 and 65 – 70 (STL seasonal strength)


In [ ]:
# ==========================================================================
# STEP 7 - PART 7a: LOAD THE PREPARED FILES, SEAL WEEKS 54-66, BUILD THE ROUTER
#          (F1 rebuild, leak-free.  No model is fitted in this part.)
#
# WHY (FPP2 2nd ed., Hyndman & Athanasopoulos 2018):
#   3.4 (p.84): "The accuracy of forecasts can only be determined by considering how well a model
#     performs on new data that were not used when fitting the model." -> weeks 54-66 are removed
#     from memory here; only Step 8 may read them back, once.
#   3.4: time series cross-validation - each CV round learns only the weeks before its cut-off.
#   10.1: the hierarchy has a Total at level 0 (Scorecard row 3 files).
#   12.2: intermittent products use Croston-type methods; 12.7: short series -> AICc;
#   4.6: "Judgmental forecasting is usually the only available method for new product forecasting".
#   Scorecard row 4 (Prep cell 83): no 13-week season in weeks 1-53 -> season_length = 1.
#
# THE ROUTER (one lane per product and per set, checked in this order; rules from Prep F2-F5):
#   backtest only:  history "too new" -> "judgement" (first sale after the seal; FPP2 4.6,
#                     scored separately at Step 8)
#   label "not known yet"   -> "not launched"  (no sale before that CV round's cut-off: no forecast)
#   label "stopped"         -> "zero"           (forecast 0; F2 rule)
#   CV rounds only: fewer than 13 weeks on sale by the round's cut-off -> "too short for this round"
#                     (F3_MIN_FIT = 13, Prep cell 30) - not scored in that round
#   everything else -> the family for THAT set: "standard" or "intermittent" (F4 / F5)
#   model choice (backtest): history "full" -> chosen by CV;  "short" -> chosen by AICc (F3, FPP2 12.7)
#
# SETS (cut-offs read from product_routing_labels.csv, checked against the 66-week calendar):
#   cv1:      learn weeks 1-27 (to 6 Oct 2025),  predict 28-40   (both inside weeks 1-53)
#   cv2:      learn weeks 1-40 (to 5 Jan 2026),  predict 41-53
#   backtest: learn weeks 1-53 (to 6 Apr 2026),  predict 54-66   (SEALED - Step 8 only)
#
# WHERE EACH INPUT COMES FROM (files saved by the Prep notebook; nothing in memory is needed):
#   ../eabl parsed csv/Y_product_with_total.csv      -> Prep cell 84 (Scorecard row 3): 283 series x 66 weeks
#   ../eabl parsed csv/S_product_with_total.csv      -> Prep cell 84: summing matrix with the Total row
#   ../eabl parsed csv/tags_product_with_total.json  -> Prep cell 84: level name -> unique_ids
#   ../eabl parsed csv/product_routing_labels.csv    -> Prep cell 65 (F4, re-saved after F5): label_*,
#                                                      family_*, history, first_sale_week, cutoff_* per product
#   ../eabl parsed csv/product_week_flags.csv        -> Prep cell 68 (F9): holiday / spike flags, weeks 1-53
#   pd, np, json -> imported below
#
# WHAT IT CHANGES: nothing on disk. Old F1 cells are not used or changed (do NOT run old cell 4,
#   which overwrites S_product.csv).
# WHAT IT WRITES:  nothing. New tables in memory only (all names start with f1_ / F1_).
# ==========================================================================
import json
import numpy as np
import pandas as pd

F1_DIR = "../eabl parsed csv/"
F1_SEAL = pd.Timestamp("2026-04-06")          # STEP 0 rule (a): week 53, the seal
F1_SEASON_LENGTH = 1                          # Scorecard row 4: no 13-week season -> non-seasonal models
F1_H = 13                                     # STEP 0 rule (c): horizon, locked
F1_MIN_FIT = 13                               # Prep cell 30 (F3): weeks needed before a CV round
F1_SETS = ["cv1", "cv2", "backtest"]          # "final" is used only at Step 9

# ---- 1. Read the prepared files ----
f1_Y_all = pd.read_csv(F1_DIR + "Y_product_with_total.csv", parse_dates=["ds"])
f1_S = pd.read_csv(F1_DIR + "S_product_with_total.csv", index_col=0)
with open(F1_DIR + "tags_product_with_total.json") as f:
    f1_tags = {k: np.array(v, dtype=object) for k, v in json.load(f).items() if k != "_made_by"}
f1_labels = pd.read_csv(F1_DIR + "product_routing_labels.csv", dtype=str)
f1_flags = pd.read_csv(F1_DIR + "product_week_flags.csv", parse_dates=["ds"])

# ---- 2. Checks: the files are the ones Prep made ----
F1_BOTTOM = "Category2/Sub_Category/Brand/Product_Code"
f1_weeks_all = pd.Index(sorted(f1_Y_all["ds"].unique()))
assert len(f1_weeks_all) == 66 and (f1_weeks_all.dayofweek == 0).all(), "expected 66 Monday weeks"
assert f1_Y_all["unique_id"].nunique() == 283 and "Total" in set(f1_Y_all["unique_id"]), "run Prep cell 84 first"
assert len(f1_S) == 283 and f1_S["unique_id"].iloc[0] == "Total", "S_product_with_total.csv is not the 283-row S"
assert list(f1_tags) == ["Total", "Category2", "Category2/Sub_Category", "Category2/Sub_Category/Brand", F1_BOTTOM]
assert len(f1_labels) == 188 and set(f1_labels["unique_id"]) == set(f1_tags[F1_BOTTOM]), \
    "product_routing_labels.csv does not match the 188 bottom-level products"
assert f1_flags["made_by"].str.startswith("F9").all() and f1_flags["ds"].max() <= F1_SEAL, "flags file not from F9"

# Cut-offs: read from the labels file and checked against the calendar (week 27, 40, 53)
F1_CUTOFF = {s: pd.Timestamp(f1_labels[f"cutoff_{s}"].iloc[0]) for s in F1_SETS}
for s in F1_SETS:
    assert f1_labels[f"cutoff_{s}"].nunique() == 1, f"cutoff_{s} differs between products"
assert [f1_weeks_all.get_loc(F1_CUTOFF[s]) + 1 for s in F1_SETS] == [27, 40, 53] and F1_CUTOFF["backtest"] == F1_SEAL

# ---- 3. THE SEAL: keep weeks 1-53 in memory, drop weeks 54-66 ----
f1_Y = f1_Y_all[f1_Y_all["ds"] <= F1_SEAL].reset_index(drop=True)
f1_sealed_rows = len(f1_Y_all) - len(f1_Y)
del f1_Y_all                                   # weeks 54-66 are no longer in memory (Step 8 re-reads the file)
f1_weeks = pd.Index(sorted(f1_Y["ds"].unique()))
assert len(f1_weeks) == 53 and f1_Y["ds"].max() == F1_SEAL

# y = S b still holds on weeks 1-53 (FPP2 10.1). The 7 "too new" products have no week here
# (first sale after the seal), so they count as 0 - reindex fills them with 0.
f1_wide = f1_Y.pivot(index="unique_id", columns="ds", values="y").fillna(0.0)
f1_Smat = f1_S.set_index("unique_id")
f1_wide_all = f1_wide.reindex(f1_Smat.index, fill_value=0.0)
assert np.allclose(f1_Smat.values @ f1_wide_all.loc[list(f1_Smat.columns)].values,
                   f1_wide_all.values), "y = S b does not hold on weeks 1-53"
f1_not_in_learn = sorted(set(f1_Smat.index) - set(f1_wide.index))      # series with no week 1-53

# ---- 4. Training data per set: only weeks up to that set's cut-off ----
f1_train = {s: f1_Y[f1_Y["ds"] <= F1_CUTOFF[s]].reset_index(drop=True) for s in F1_SETS}
f1_flags_train = {s: f1_flags[f1_flags["ds"] <= F1_CUTOFF[s]].reset_index(drop=True) for s in F1_SETS}
for s in F1_SETS:
    assert f1_train[s]["ds"].max() == F1_CUTOFF[s] and f1_flags_train[s]["ds"].max() <= F1_CUTOFF[s]

# ---- 5. The router: one lane per product and set ----
f1_first = pd.to_datetime(f1_labels["first_sale_week"])
f1_route_rows = []
for s in F1_SETS:
    f1_on_sale = ((F1_CUTOFF[s] - f1_first).dt.days // 7 + 1).where(f1_first <= F1_CUTOFF[s], 0)
    for i, r in f1_labels.iterrows():
        if s == "backtest" and r["history"] == "too new":
            lane = "judgement"                 # first sale after the seal: judgement (F3, FPP2 4.6)
        elif r[f"label_{s}"] == "not known yet":
            lane = "not launched"
        elif r[f"label_{s}"] == "stopped":
            lane = "zero"
        elif s != "backtest" and f1_on_sale[i] < F1_MIN_FIT:
            lane = "too short for this round"
        else:
            lane = r[f"family_{s}"]
        if lane in ("standard", "intermittent"):
            choose = "CV" if (s != "backtest" or r["history"] == "full") else "AICc"
        else:
            choose = "-"
        f1_route_rows.append({"set": s, "unique_id": r["unique_id"], "SKU": r["SKU"],
                              "label": r[f"label_{s}"], "weeks_on_sale": int(f1_on_sale[i]),
                              "lane": lane, "choose_by": choose})
f1_routes = pd.DataFrame(f1_route_rows)
assert f1_routes.groupby("set").size().eq(188).all(), "every product needs a lane in every set"
assert f1_routes["lane"].isin(["not launched", "zero", "too short for this round", "judgement",
                               "standard", "intermittent"]).all(), "unknown lane - stop and check"
assert (f1_routes.query("set == 'backtest' and lane == 'judgement'")["weeks_on_sale"] == 0).all()
# a product may not be modelled in a set before it has been launched in that set
assert not ((f1_routes["weeks_on_sale"] == 0) & f1_routes["lane"].isin(["standard", "intermittent"])).any()

# ---- Report ----
print("STEP 7a - F1 inputs loaded, weeks 54-66 sealed, router built (nothing written)\n")
print(f"Hierarchy: {f1_Smat.shape[0]} series in S (Total + 3 + 8 + 83 + 188) x {f1_Smat.shape[1]} products | y = S b - OK")
print(f"  {f1_Y['unique_id'].nunique()} series have sales in weeks 1-53; {len(f1_not_in_learn)} start after the seal "
      f"(7 'too new' products + their 3 brands): {f1_not_in_learn}")
print(f"Weeks kept in memory: 1-53 ({f1_weeks[0].date()} to {f1_weeks[-1].date()}); "
      f"{f1_sealed_rows:,} rows of weeks 54-66 dropped from memory")
print(f"season_length = {F1_SEASON_LENGTH} (Scorecard row 4) | h = {F1_H} | min weeks before a CV round = {F1_MIN_FIT}")
print("\nTraining weeks per set:")
for s in F1_SETS:
    print(f"  {s:9} learns weeks 1-{f1_weeks.get_loc(F1_CUTOFF[s]) + 1} (to {F1_CUTOFF[s].date()}): "
          f"{len(f1_train[s]):,} rows, flags {len(f1_flags_train[s]):,} rows")
print("\nProducts per lane and set:")
display(pd.crosstab(f1_routes["lane"], f1_routes["set"]).reindex(columns=F1_SETS)
        .reindex(["standard", "intermittent", "zero", "judgement", "too short for this round", "not launched"])
        .fillna(0).astype(int))
print("Backtest: how the model will be chosen (FPP2 3.4 CV / 12.7 AICc):")
display(f1_routes[f1_routes["set"] == "backtest"].groupby(["lane", "choose_by"]).size().rename("products").to_frame())


In [ ]:
# ==========================================================================
# STEP 7 - PART 7a SEAL FOR LABELS: REMOVE AFTER-SEAL INFORMATION FROM f1_labels (memory only)
#   7a read the whole product_routing_labels.csv. Some of its columns were made from ALL 66 weeks
#   (for Step 9, the final refit), so they carry information from the sealed test weeks 54-66:
#     label_final, class_final, family_final, cutoff_final -> labels / cut-off from weeks 1-66
#     master_status_INFO_ONLY -> today's product-master status (may have changed after the seal)
#     first_sale_week for the 7 "too new" products -> launch dates inside weeks 54-66
#   7a's lanes did NOT use any of this (only per-round columns and "launched by the cut-off? yes/no").
#   This cell removes it from memory so that no later part (7b-7g) can use it by mistake.
#
# WHY (FPP2 2nd ed., Hyndman & Athanasopoulos 2018):
#   3.4 (p.77): forecasts must be judged on "new data that were not used when fitting the model".
#   4.2 (judgemental forecasting): forecasters must not see information they would not have had
#     (Step 0 rules (a) and (e): nothing after 6 Apr 2026 is used; planners see nothing after it).
#
# WHERE EACH INPUT COMES FROM: f1_labels, f1_routes, F1_SEAL, F1_CUTOFF, F1_SETS -> Part 7a (cell above);
#   pd -> 7a import.   Step 9 (final refit) will read the FULL file from disk again.
# WHAT IT CHANGES: only f1_labels in memory (5 columns dropped, 7 launch dates blanked).
# WHAT IT WRITES: nothing.  New names start with f1s_ / F1S_.
# ==========================================================================
F1S_AFTER_SEAL_COLS = ["label_final", "class_final", "family_final", "cutoff_final", "master_status_INFO_ONLY"]

if "F1S_LABELS_SEALED" not in globals():                 # safe to run twice
    assert set(F1S_AFTER_SEAL_COLS).issubset(f1_labels.columns), "f1_labels is not the 7a table - re-run 7a"
    f1s_first = pd.to_datetime(f1_labels["first_sale_week"])
    f1s_after = f1s_first > F1_SEAL
    # the products launched after the seal must be exactly 7a's judgement products
    f1s_judgement = set(f1_routes.loc[(f1_routes["set"] == "backtest") & (f1_routes["lane"] == "judgement"), "unique_id"])
    assert set(f1_labels.loc[f1s_after, "unique_id"]) == f1s_judgement, "after-seal launches != judgement lane"

    f1s_cols_before = f1_labels.shape[1]
    f1_labels = f1_labels.drop(columns=F1S_AFTER_SEAL_COLS)
    f1_labels["first_sale_week"] = f1_labels["first_sale_week"].where(~f1s_after)    # blank (NaN) after the seal
    F1S_LABELS_SEALED = True

    # PROOF: the lanes 7a built do not change - "weeks on sale" recomputed from the sealed column
    f1s_first_sealed = pd.to_datetime(f1_labels["first_sale_week"])
    for f1s_set in F1_SETS:
        f1s_on_sale = ((F1_CUTOFF[f1s_set] - f1s_first_sealed).dt.days // 7 + 1) \
                          .where(f1s_first_sealed <= F1_CUTOFF[f1s_set], 0).astype(int)
        f1s_routes_set = f1_routes[f1_routes["set"] == f1s_set].set_index("unique_id")["weeks_on_sale"]
        assert (f1s_on_sale.set_axis(f1_labels["unique_id"]).reindex(f1s_routes_set.index) == f1s_routes_set).all(), \
            f"{f1s_set}: weeks on sale would change - stop and check"

    print("7a labels sealed (memory only, nothing written)")
    print(f"  f1_labels columns: {f1s_cols_before} -> {f1_labels.shape[1]}  (dropped: {F1S_AFTER_SEAL_COLS})")
    print(f"  launch dates blanked (after 6 Apr 2026): {int(f1s_after.sum())} products = the {len(f1s_judgement)} judgement products")
    print("  proof: weeks on sale and lanes for cv1 / cv2 / backtest unchanged - OK")
else:
    print("7a labels were already sealed in this session - nothing to do")
print("  columns kept:", list(f1_labels.columns))


In [ ]:
# ==========================================================================
# STEP 7 - PART 7a SPIKE v2 (Roadmap I2 result): REPLACE THE DECEMBER 2025 SPIKES IN THE TRAINING WEEKS (memory only)
#   This is notebook cell 3: after "7a SEAL FOR LABELS" (cell 2), before "7a CHECKS" (cell 4).  After changing it or the
#   switch: Kernel -> Restart Kernel and Run All Cells.
#   SWITCH: F1SP_CLEAN = True  -> December spikes replaced (the version chosen by the I2 tests)
#           F1SP_CLEAN = False -> nothing is replaced; the notebook runs exactly as recorded (with the spikes)
#   Why: the I2 tests on cv2 (learn weeks 1-40, predict 41-53) showed that replacing the December spikes by each
#   product's usual week made the reconciled forecasts 11.9% better (5 of 5 levels) - above the stricter rule fixed
#   before that test (>= 3% and >= 4 of 5 levels).  The 8 Dec week alone gave 2.9%; December has 5 spike weeks.
#   WHAT IS CLEANED: f1_train["cv2"] and f1_train["backtest"] (the sets whose training weeks contain December).
#     cv1 learns weeks 1-27 (no December) - unchanged.  Every product flagged "spike" by F8 in the weeks of 1, 8, 15,
#     22 and 29 Dec 2025 (84 product-weeks) gets its F8 "usual_week" (median of its OWN 13 earlier weeks - no
#     look-ahead); every brand, sub-category, category and the Total is re-added from the products (S matrix).
#     Spikes in other months (bulk orders, promotions, Easter) are NOT touched: they can happen again in the forecast
#     periods, so the models must keep learning from them (FPP2 12.9).
#   WHAT IS NOT CLEANED: f1_Y (the real weeks 1-53).  Every score (cv1, cv2, Step 8) still compares with REAL sales.
#   LIMIT: valid for forecast periods WITHOUT December (backtest Apr-Jul, future Jul-Oct).  Do NOT use this cleaning
#     to forecast a December - run with F1SP_CLEAN = False for that.
#   Safe to run twice, and in any order: it checks THE DATA, not a note (fix of 12 Nov 2026).  Before the fix it
#   skipped whenever the note F1SP_DONE existed - so after cell 1 was re-run on its own (which reloads the raw sales)
#   it skipped although the data was raw again.  Now it compares the Total of the week of 22 Dec 2025 (the biggest
#   spike week) in the training data of cv2 and backtest with the recorded value in f1_Y: lower = already cleaned
#   (nothing to do); equal = raw -> it cleans.  With the switch OFF it removes the note, so no later cell can mistake
#   a with-spikes run for the cleaned one.
#
# WHY (FPP2 2nd ed., Hyndman & Athanasopoulos 2018):
#   12.9 (p.488): outliers may be replaced "if we are willing to assume that the outliers are genuinely errors, or that
#     they won't occur in the forecasting period" - the December surge does not occur in Apr-Jul or Jul-Oct.
#   3.4 (p.77): decided by a test on weeks not used to fit (cv2), not assumed.
#   Limits (recorded): one round of evidence (13 weeks, Jan-Mar, right after December); the surge was real demand.
#
# WHERE EACH INPUT COMES FROM (all already in memory; nothing read from disk):
#   f1_train, f1_flags_train, f1_Smat, F1_CUTOFF, f1_Y (real weeks 1-53, never cleaned) -> 7a (cell 1);  pd, np -> 7a imports
#
# WHAT IT CHANGES: if F1SP_CLEAN = True, f1_train["cv2"] and f1_train["backtest"] in memory (December weeks only).
#   f1_Y and all files are not changed.  WHAT IT WRITES: nothing.  New names start with f1sp_ / F1SP_.
# ==========================================================================
F1SP_CLEAN = True                                                   # True = clean December | False = as recorded
F1SP_WEEKS = list(pd.date_range("2025-12-01", "2025-12-29", freq="W-MON"))   # weeks 35-39: 1, 8, 15, 22, 29 Dec
F1SP_SETS = ["cv2", "backtest"]                                     # sets whose training weeks contain December

F1SP_CHECK_WEEK = pd.Timestamp("2025-12-22")                        # biggest spike week: cleaned Total < recorded Total

def f1sp_total(f1sp_df):
    """Total of the check week in a weekly table (unique_id, ds, y)."""
    return float(f1sp_df.loc[(f1sp_df["unique_id"] == "Total") & (f1sp_df["ds"] == F1SP_CHECK_WEEK), "y"].iloc[0])

f1sp_recorded = f1sp_total(f1_Y)                                    # the real value (f1_Y is never cleaned)
f1sp_cleaned_now = {f1sp_s: f1sp_total(f1_train[f1sp_s]) < f1sp_recorded - 1 for f1sp_s in F1SP_SETS}
assert len(set(f1sp_cleaned_now.values())) == 1, f"cv2 and backtest disagree {f1sp_cleaned_now} - re-run cell 1, then this cell"

if not F1SP_CLEAN:
    globals().pop("F1SP_DONE", None)                                # no 'cleaned' note in a with-spikes run
    assert not any(f1sp_cleaned_now.values()), "switch is OFF but the data is cleaned - Kernel -> Restart & Run All"
    print("7a spike v2: switch is OFF - nothing replaced; the notebook runs with the data exactly as recorded")
elif all(f1sp_cleaned_now.values()):                                # checked in the DATA, not from a note
    F1SP_DONE = True
    print(f"7a spike v2: December already cleaned in the data (Total of {F1SP_CHECK_WEEK.date()}: "
          f"{f1sp_total(f1_train['backtest']):,.0f} vs recorded {f1sp_recorded:,.0f}) - nothing to do")
else:
    assert all(F1_CUTOFF[f1sp_s] >= F1SP_WEEKS[-1] for f1sp_s in F1SP_SETS) and F1_CUTOFF["cv1"] < F1SP_WEEKS[0]
    f1sp_ids = list(f1_Smat.index)
    f1sp_bottom = list(f1_Smat.columns)
    f1sp_fl = f1_flags_train["cv2"]                     # F8 flags up to cv2's cut-off (weeks 1-40, earlier-only rules)
    f1sp_sp = f1sp_fl[f1sp_fl["ds"].isin(F1SP_WEEKS) & (f1sp_fl["product_flag"] == "spike")][["unique_id", "ds", "y", "usual_week"]]
    assert len(f1sp_sp) > 0 and f1sp_sp["usual_week"].notna().all(), "no December spike flags found"

    f1sp_tot_before = f1_train["backtest"][f1_train["backtest"]["unique_id"] == "Total"].set_index("ds")["y"]
    f1sp_aff = set()
    for f1sp_s in F1SP_SETS:
        f1sp_tr = f1_train[f1sp_s].copy()
        for f1sp_wk in F1SP_WEEKS:                                      # one December week at a time
            f1sp_w_sp = f1sp_sp[f1sp_sp["ds"] == f1sp_wk]
            if f1sp_w_sp.empty:
                continue
            f1sp_delta = pd.Series(0.0, index=f1sp_bottom)              # change per product
            f1sp_delta[f1sp_w_sp["unique_id"].values] = (f1sp_w_sp["usual_week"] - f1sp_w_sp["y"]).values
            f1sp_up = pd.Series(f1_Smat.values @ f1sp_delta.values, index=f1sp_ids)   # change per series (S)
            f1sp_up = f1sp_up[f1sp_up.abs() > 1e-9]
            f1sp_rows = (f1sp_tr["ds"] == f1sp_wk) & f1sp_tr["unique_id"].isin(f1sp_up.index)
            assert int(f1sp_rows.sum()) == len(f1sp_up), f"{f1sp_s} {f1sp_wk.date()}: a changed series has no row"
            f1sp_tr.loc[f1sp_rows, "y"] = f1sp_tr.loc[f1sp_rows, "y"] + f1sp_tr.loc[f1sp_rows, "unique_id"].map(f1sp_up)
            f1sp_aff |= set(f1sp_up.index)
            # the hierarchy still adds up in this week
            f1sp_w = f1sp_tr[f1sp_tr["ds"] == f1sp_wk].set_index("unique_id")["y"].reindex(f1sp_ids).fillna(0.0)
            assert np.allclose(f1_Smat.values @ f1sp_w[f1sp_bottom].values, f1sp_w.values), f"{f1sp_s}: y = S b broken"
        assert (f1sp_tr["y"] >= 0).all(), f"{f1sp_s}: cleaning made a negative value"
        f1_train[f1sp_s] = f1sp_tr
    assert all(f1sp_total(f1_train[f1sp_s]) < f1sp_recorded - 1 for f1sp_s in F1SP_SETS), "cleaning did not lower 22 Dec"
    F1SP_DONE = True

    f1sp_tot_after = f1_train["backtest"][f1_train["backtest"]["unique_id"] == "Total"].set_index("ds")["y"]
    print("7a spike v2: December 2025 spikes replaced in the training weeks of cv2 and backtest (memory only; f1_Y unchanged)")
    print(f"  {len(f1sp_sp)} product-weeks replaced by their usual week (F8) | extra cases removed: "
          f"{(f1sp_sp['y'] - f1sp_sp['usual_week']).sum():,.0f} | series changed incl. levels above: {len(f1sp_aff)}")
    display(pd.DataFrame({"Total before": f1sp_tot_before.reindex(F1SP_WEEKS), "Total after": f1sp_tot_after.reindex(F1SP_WEEKS),
                          "spike products": f1sp_sp.groupby("ds").size().reindex(F1SP_WEEKS).fillna(0).astype(int)}).round(0))
    print("  cv1 unchanged (learns weeks 1-27). Other months' spikes kept. Scores still use REAL sales (f1_Y).")
    print("  Valid for forecast periods without December only. NOW: Restart & Run All (the I2 test cell will print skipped).")

In [ ]:
# ==========================================================================
# STEP 7 - PART 7a CHECKS: NO DUPLICATES, NO MISSING VALUES, NO NEGATIVE SALES (weeks 1-53)
#   From OLD F1 cells 14-16 (duplicate (unique_id, ds) pairs) and OLD F1 cells 47-48 (NaN in y),
#   rewritten to use the sealed 7a tables instead of the old Y_product / results dict.
#   Added (not in old F1): negative-sales check - cases sold in a week cannot be below 0.
#   Why: StatsForecast needs exactly one row per (unique_id, ds) and a y with no missing values.
#
# WHERE EACH INPUT COMES FROM: f1_Y, f1_train, F1_SETS -> Part 7a (the cell above); pd, display -> 7a / Jupyter
# WHAT IT CHANGES: nothing.  WHAT IT WRITES: nothing.  New names start with f1c_.
# ==========================================================================
f1c_rows = []
for f1c_name, f1c_df in [("weeks 1-53 (f1_Y)", f1_Y)] + [(f"train {s}", f1_train[s]) for s in F1_SETS]:
    f1c_rows.append({"table": f1c_name, "rows": len(f1c_df),
                     "duplicate (unique_id, ds)": int(f1c_df.duplicated(subset=["unique_id", "ds"]).sum()),
                     "missing y": int(f1c_df["y"].isna().sum()),
                     "negative y": int((f1c_df["y"] < 0).sum())})
f1c_check = pd.DataFrame(f1c_rows).set_index("table")
display(f1c_check)
assert (f1c_check[["duplicate (unique_id, ds)", "missing y", "negative y"]] == 0).all().all(), \
    "a check failed - stop and look at the table above"
print("All checks passed: one row per series and week, no missing y, no negative y.")


In [ ]:
# ==========================================================================
# STEP 7 - PART 7a PICTURE: ADI vs CV2 OF EVERY PRODUCT (backtest weeks 1-53, from each product's launch)
#   From OLD F1 cell 62 (ADI vs CV2 scatter), rewritten so that it matches the classes 7a routes by:
#     old cell 62: whole 66-week calendar, zeros BEFORE launch counted, stale classification_product.csv
#     here:        weeks 1-53 only (f1_Y, sealed), each series starts at its FIRST SALE (Prep F5),
#                  classes compared with class_backtest in product_routing_labels.csv (f1_labels)
#   Formula = Prep cell 60 classify_demand (Syntetos, Boylan & Croston 2005, JORS 56(5) 495-503);
#   launch start = Prep F5 (FPP2 12.2: Croston's a_i is the time BETWEEN two sales):
#     ADI = weeks from launch to the cut-off / weeks with a sale      (cut-off 1.32)
#     CV2 = (std / mean)^2 of the weeks with a sale (pandas std)      (cut-off 0.49)
#   Only a picture + a check: routing is NOT changed (7a's f1_routes stays as it is).
#
# WHERE EACH INPUT COMES FROM: f1_Y, f1_weeks, f1_labels, f1_tags, F1_BOTTOM, F1_CUTOFF -> Part 7a
#   pd, np -> 7a imports;  plt -> NEW import below (matplotlib);  display -> Jupyter
# WHAT IT CHANGES: nothing.  WHAT IT WRITES: nothing (shows 1 plot + 2 tables).  New names start with f1p_.
# ==========================================================================
import matplotlib.pyplot as plt

F1P_ADI_CUT, F1P_CV2_CUT = 1.32, 0.49                      # Prep cell 60 thresholds (unchanged)
assert F1_CUTOFF["backtest"] == f1_weeks[-1], "run Part 7a first"

f1p_products = f1_Y[f1_Y["unique_id"].isin(f1_tags[F1_BOTTOM])]
f1p_rows = []
for f1p_id, f1p_g in f1p_products.groupby("unique_id"):
    f1p_s = f1p_g.set_index("ds")["y"].reindex(f1_weeks)       # weeks 1-53 for this product
    f1p_sold = f1p_s[f1p_s > 0]
    if len(f1p_sold) == 0:
        continue                                                # no sale in weeks 1-53 ("too new")
    f1p_s = f1p_s.loc[f1p_sold.index.min():]                   # start at the first sale (Prep F5)
    f1p_adi = len(f1p_s) / len(f1p_sold)
    f1p_cv2 = (f1p_sold.std() / f1p_sold.mean()) ** 2          # NaN if only one selling week
    if f1p_adi < F1P_ADI_CUT and f1p_cv2 < F1P_CV2_CUT:
        f1p_class = "smooth"
    elif f1p_adi >= F1P_ADI_CUT and f1p_cv2 < F1P_CV2_CUT:
        f1p_class = "intermittent"
    elif f1p_adi < F1P_ADI_CUT and f1p_cv2 >= F1P_CV2_CUT:
        f1p_class = "erratic"
    else:
        f1p_class = "lumpy"                                     # same fall-through as Prep cell 60
    f1p_rows.append({"unique_id": f1p_id, "adi": f1p_adi, "cv2": f1p_cv2, "class_here": f1p_class})
f1p_adi_cv2 = pd.DataFrame(f1p_rows).merge(f1_labels[["unique_id", "SKU", "class_backtest", "label_backtest"]],
                                          on="unique_id", how="left")

# ---- Check: the picture uses the same classes that 7a routes by ----
f1p_diff = f1p_adi_cv2[f1p_adi_cv2["class_here"] != f1p_adi_cv2["class_backtest"]]
print(f"Products launched by {F1_CUTOFF['backtest'].date()}: {len(f1p_adi_cv2)} of {len(f1_labels)} "
      f"(the other {len(f1_labels) - len(f1p_adi_cv2)} are 'too new')")
print(f"Same class as class_backtest in product_routing_labels.csv: {len(f1p_adi_cv2) - len(f1p_diff)} of {len(f1p_adi_cv2)}")
if len(f1p_diff):
    display(f1p_diff)
assert len(f1p_diff) == 0, "class differs from product_routing_labels.csv - stop and check"
display(f1p_adi_cv2["class_here"].value_counts().rename("products").to_frame())

# ---- Plot (one marker shape AND one colour per class, so it reads without colour too) ----
F1P_STYLE = {"smooth": ("#0072B2", "o"), "erratic": ("#E69F00", "s"),
             "intermittent": ("#009E73", "^"), "lumpy": ("#CC79A7", "D")}
f1p_plot = f1p_adi_cv2.dropna(subset=["cv2"])                   # one-sale products have no CV2
fig, ax = plt.subplots(figsize=(9, 6.5))
for f1p_c, (f1p_col, f1p_mk) in F1P_STYLE.items():
    f1p_sub = f1p_plot[f1p_plot["class_here"] == f1p_c]
    ax.scatter(f1p_sub["adi"], f1p_sub["cv2"], c=f1p_col, marker=f1p_mk, s=36, alpha=0.8,
               edgecolors="white", linewidths=0.5, label=f"{f1p_c} ({len(f1p_sub)})")
ax.axvline(F1P_ADI_CUT, color="grey", ls="--", lw=1)
ax.axhline(F1P_CV2_CUT, color="grey", ls="--", lw=1)
ax.set_xscale("log")
ax.set_xlabel("ADI - weeks per sale, from launch (log scale; line at 1.32)")
ax.set_ylabel("CV² - variability of sale size (line at 0.49)")
ax.set_title(f"ADI vs CV² - {len(f1p_plot)} products, weeks 1-53 from launch (backtest set)", loc="left")
ax.legend(title="class (products)")
plt.tight_layout()
plt.show()
f1p_one_sale = f1p_adi_cv2["cv2"].isna().sum()
if f1p_one_sale:
    print(f"Not on the plot: {f1p_one_sale} product(s) with only one selling week (CV2 cannot be computed) - "
          "classed 'lumpy', as in Prep cell 60.")


In [ ]:
# ==========================================================================
# STEP 7 - PART 7b: STANDARD MODELS - NAIVE/MEAN/DRIFT + ETS/ARIMA/THETA/CES, CV ON cv1 & cv2,
#                   AutoETS-AICc FOR SHORT PRODUCTS (WEEKS 1-53)
#   Leakage table Step 7: "re-do model choice ... on weeks 1-53".  Covers:
#     - the STANDARD-lane products (intermittent products -> Part 7c), and
#     - the 95 series ABOVE product level (Total, Category2, Sub_Category, Brand), because
#       reconciliation needs a base forecast for every series (Part 7g; Scorecard row 9 compares
#       bottom-up and MinT inside the same CV windows).
#   Errors and winners are NOT computed here: that is the next cell (from old F1 cell 88:
#   MASE picks the winner, MAE and WAPE shown).  Negative forecasts are counted, NOT clipped.
#
# WHY (FPP2 2nd ed., Hyndman & Athanasopoulos 2018):
#   3.1 (p.57): the average (mean), naive and drift methods are used "as benchmarks throughout this
#     book" (seasonal naive is left out: season_length = 1, so it would equal naive - Scorecard row 4).
#   3.4 (p.77): "It is important to evaluate forecast accuracy using genuine forecasts" -> each CV
#     round learns only the weeks before its cut-off and predicts the next 13 weeks, whose real sales
#     are known (weeks 28-40 and 41-53, all before the seal).  Weeks 54-66 are not in memory (7a).
#   10.6 (p.414): "Suppose we forecast all series independently ... We call these the base forecasts."
#   10.7 (p.418): optimal reconciliation (MinT) uses "all the information available within a
#     hierarchical ... structure" -> base forecasts are needed at every level, not only products.
#   7.6-7.7 / Ch 8: ETS and ARIMA models chosen automatically inside their own family.
#   8.10 (p.346): AICc compares models "in the same class" only - never ETS against ARIMA.
#   12.7 (p.479): for short series "The AICc is particularly useful here" -> the backtest products
#     marked "AICc" by 7a get AutoETS (the ETS form with the lowest AICc).
#   statsforecast AutoETS refuses series with 6 weeks or fewer ("tiny datasets": n <= 2 + 4).
#   For those, simple exponential smoothing (the method behind ETS(A,N,N), FPP2 7.5 p.267) with
#   alpha chosen by minimising the sum of squared errors (FPP2 7.1 p.244) is used instead.
#   12.3 (p.469): "To impose a positivity constraint, simply work on the log scale" -> negative
#     forecasts are only COUNTED here; the log version (cell from old F1 cells 104-105) deals with them.
#
# WHERE EACH INPUT COMES FROM (all made by Part 7a, the first code cell; nothing read from disk):
#   f1_routes   -> 7a step 5: one lane per product and set ("standard", "intermittent", ...) + choose_by
#   f1_train    -> 7a step 4: {"cv1": weeks 1-27, "cv2": weeks 1-40, "backtest": weeks 1-53}, all 283 series
#   f1_Y        -> 7a step 3: weeks 1-53 (real sales of the CV prediction weeks 28-40 and 41-53)
#   f1_tags, F1_BOTTOM -> 7a steps 1-2: level name -> series ids (Total ... Product)
#   f1_weeks    -> 7a step 3: the 53 Monday weeks;  F1_CUTOFF -> 7a step 2: cut-off date per set
#   f1_labels   -> 7a step 1 (sealed by the cell under 7a): SKU names
#   F1_H (13), F1_SEASON_LENGTH (1), F1_MIN_FIT (13) -> 7a settings;  pd -> 7a import;  display -> Jupyter
#   StatsForecast + the models -> NEW imports below (statsforecast 2.0.3, the EABL venv)
#
# WHAT IT CHANGES: nothing.  WHAT IT WRITES: nothing (tables in memory only).
# All new names start with f1b_ / F1B_ (the loop variable is f1b_set, so 7a's names are untouched).
# ==========================================================================
import time
from statsforecast import StatsForecast
from statsforecast.models import (Naive, HistoricAverage, RandomWalkWithDrift, AutoETS, AutoARIMA,
                                  AutoTheta, AutoCES, SimpleExponentialSmoothingOptimized)

F1B_CV_SETS = ["cv1", "cv2"]                                  # the two practice rounds (backtest = Step 8)
F1B_BENCHMARKS = ["Naive", "Mean", "Drift"]                   # FPP2 3.1
F1B_AUTO = ["AutoETS", "AutoARIMA", "AutoTheta", "CES"]       # CES = the column name AutoCES produces
F1B_MODEL_COLS = F1B_BENCHMARKS + F1B_AUTO
F1B_ETS_MIN_WEEKS = 7                                         # AutoETS needs at least 7 weeks (see WHY)
F1B_LEVELS = {"Total": "Total", "Category2": "Category2", "Category2/Sub_Category": "Sub_Category",
              "Category2/Sub_Category/Brand": "Brand", F1_BOTTOM: "Product"}     # f1_tags key -> short name
assert list(F1B_LEVELS) == list(f1_tags), "f1_tags levels are not the 7a ones - re-run 7a"
f1b_level_of = {f1b_uid: F1B_LEVELS[f1b_key] for f1b_key, f1b_uids in f1_tags.items() for f1b_uid in f1b_uids}
f1b_upper_all = [f1b_uid for f1b_key, f1b_uids in f1_tags.items() if f1b_key != F1_BOTTOM for f1b_uid in f1b_uids]
assert len(f1b_upper_all) == 95                               # 1 Total + 3 + 8 + 83 brands

def f1b_models():
    """The 7 standard models, all non-seasonal (season_length = F1_SEASON_LENGTH = 1, from 7a)."""
    return [Naive(),                                          # last week, repeated
            HistoricAverage(alias="Mean"),                    # average of all past weeks
            RandomWalkWithDrift(alias="Drift"),               # last week + average weekly change
            AutoETS(season_length=F1_SEASON_LENGTH),          # exponential smoothing, form chosen by AICc
            AutoARIMA(season_length=F1_SEASON_LENGTH),        # ARIMA, orders chosen by AICc
            AutoTheta(season_length=F1_SEASON_LENGTH),        # theta method
            AutoCES(season_length=F1_SEASON_LENGTH)]          # complex exponential smoothing

assert F1_SEASON_LENGTH == 1 and F1_H == 13 and F1_MIN_FIT == 13, "run Part 7a first"

# ---- 1. Practice rounds: learn up to each cut-off, predict the next 13 weeks ----
f1b_parts, f1b_run_rows = [], []
for f1b_set in F1B_CV_SETS:
    # standard-lane products of THIS round (7a router)
    f1b_prod_ids = f1_routes.loc[(f1_routes["set"] == f1b_set) & (f1_routes["lane"] == "standard"), "unique_id"].tolist()
    # series above product level with at least F1_MIN_FIT weeks before this cut-off (same rule as products)
    f1b_upper_weeks = f1_train[f1b_set][f1_train[f1b_set]["unique_id"].isin(f1b_upper_all)].groupby("unique_id").size()
    f1b_upper_ids = f1b_upper_weeks[f1b_upper_weeks >= F1_MIN_FIT].index.tolist()
    f1b_ids = f1b_prod_ids + f1b_upper_ids
    f1b_train_set = f1_train[f1b_set][f1_train[f1b_set]["unique_id"].isin(f1b_ids)]
    # leak + alignment checks: every series ends exactly at this round's cut-off (so all forecasts
    # start the same week - the old F1 cell 135 fault) and nothing after the cut-off is used
    f1b_last = f1b_train_set.groupby("unique_id")["ds"].max()
    assert len(f1b_last) == len(f1b_ids) and (f1b_last == F1_CUTOFF[f1b_set]).all(), \
        f"{f1b_set}: a series does not end at the cut-off - stop and check"
    f1b_pos = f1_weeks.get_loc(F1_CUTOFF[f1b_set])
    f1b_pred_weeks = f1_weeks[f1b_pos + 1: f1b_pos + 1 + F1_H]        # weeks 28-40 or 41-53
    assert len(f1b_pred_weeks) == F1_H and f1b_pred_weeks.max() <= F1_CUTOFF["backtest"]

    f1b_t0 = time.time()
    f1b_sf = StatsForecast(models=f1b_models(), freq="W-MON", n_jobs=-1)
    f1b_fc = f1b_sf.forecast(df=f1b_train_set, h=F1_H)
    f1b_secs = time.time() - f1b_t0

    assert set(f1b_fc["ds"]) == set(f1b_pred_weeks), f"{f1b_set}: forecast weeks are not the 13 after the cut-off"
    assert len(f1b_fc) == len(f1b_ids) * F1_H and f1b_fc[F1B_MODEL_COLS].notna().all().all()
    # real sales of the predicted weeks (inside weeks 1-53, so allowed) for scoring in the next cell
    f1b_fc = f1b_fc.merge(f1_Y[["unique_id", "ds", "y"]], on=["unique_id", "ds"], how="left", validate="one_to_one")
    assert f1b_fc["y"].notna().all(), f"{f1b_set}: a predicted week has no real sales row"
    f1b_fc.insert(0, "set", f1b_set)
    f1b_fc.insert(1, "level", f1b_fc["unique_id"].map(f1b_level_of))
    f1b_fc.insert(4, "cutoff", F1_CUTOFF[f1b_set])
    f1b_parts.append(f1b_fc[["set", "level", "unique_id", "ds", "cutoff", "y"] + F1B_MODEL_COLS])
    f1b_run_rows.append({"set": f1b_set, "learns weeks": f"1-{f1b_pos + 1}",
                         "predicts weeks": f"{f1b_pos + 2}-{f1b_pos + 1 + F1_H}",
                         "products (standard)": len(f1b_prod_ids), "higher-level series": len(f1b_upper_ids),
                         "higher-level left out (<13 weeks or no sales yet)": len(f1b_upper_all) - len(f1b_upper_ids),
                         "forecast rows": len(f1b_fc), "seconds": round(f1b_secs, 1)})
f1b_cv = pd.concat(f1b_parts, ignore_index=True)        # one row per set x series x predicted week
f1b_runs = pd.DataFrame(f1b_run_rows).set_index("set")

# ---- 2. Short backtest products (7a choose_by == "AICc"): AutoETS, form chosen by AICc ----
f1b_aicc_ids = f1_routes.loc[(f1_routes["set"] == "backtest") & (f1_routes["lane"] == "standard")
                             & (f1_routes["choose_by"] == "AICc"), "unique_id"].tolist()
f1b_train_bt = f1_train["backtest"][f1_train["backtest"]["unique_id"].isin(f1b_aicc_ids)]
f1b_weeks_bt = f1b_train_bt.groupby("unique_id").size()
assert len(f1b_weeks_bt) == len(f1b_aicc_ids) and \
       (f1b_train_bt.groupby("unique_id")["ds"].max() == F1_CUTOFF["backtest"]).all()
f1b_ets_ids = f1b_weeks_bt[f1b_weeks_bt >= F1B_ETS_MIN_WEEKS].index.tolist()
f1b_ses_ids = f1b_weeks_bt[f1b_weeks_bt < F1B_ETS_MIN_WEEKS].index.tolist()

f1b_aicc_rows = []
if f1b_ets_ids:
    f1b_sf_ets = StatsForecast(models=[AutoETS(season_length=F1_SEASON_LENGTH)], freq="W-MON", n_jobs=1)
    f1b_sf_ets.fit(df=f1b_train_bt[f1b_train_bt["unique_id"].isin(f1b_ets_ids)])
    for f1b_i, f1b_uid in enumerate(f1b_sf_ets.uids):          # fitted_ rows follow sf.uids order
        f1b_m = f1b_sf_ets.fitted_[f1b_i, 0].model_
        f1b_aicc_rows.append({"unique_id": f1b_uid, "model": "AutoETS", "chosen form (by AICc)": f1b_m["method"],
                              "AICc": round(float(f1b_m["aicc"]), 2)})
if f1b_ses_ids:
    # fitted here only to prove it works on these short series; its forecasts are made in Part 7g
    f1b_sf_ses = StatsForecast(models=[SimpleExponentialSmoothingOptimized(alias="SES")], freq="W-MON", n_jobs=1)
    f1b_sf_ses.fit(df=f1b_train_bt[f1b_train_bt["unique_id"].isin(f1b_ses_ids)])
    for f1b_i, f1b_uid in enumerate(f1b_sf_ses.uids):
        assert len(f1b_sf_ses.fitted_[f1b_i, 0].model_["fitted"]) == f1b_weeks_bt[f1b_uid]
        f1b_aicc_rows.append({"unique_id": f1b_uid, "model": "SES (too few weeks for AutoETS)",
                              "chosen form (by AICc)": "ETS(A,N,N) method - no AICc search possible", "AICc": None})
f1b_aicc = (pd.DataFrame(f1b_aicc_rows)
            .merge(f1_labels[["unique_id", "SKU"]], on="unique_id", how="left")
            .assign(weeks_of_history=lambda d: d["unique_id"].map(f1b_weeks_bt)))
f1b_aicc = f1b_aicc[["unique_id", "SKU", "weeks_of_history", "model", "chosen form (by AICc)", "AICc"]]
assert len(f1b_aicc) == len(f1b_aicc_ids)

# ---- 3. Negative forecasts: counted only (sales cannot be negative; FPP2 12.3 -> log version) ----
f1b_neg = (f1b_cv[F1B_MODEL_COLS] < 0).groupby(f1b_cv["level"]).sum().reindex(list(F1B_LEVELS.values())).fillna(0).astype(int)
f1b_neg.loc["All"] = f1b_neg.sum()

# ---- Report ----
print("STEP 7b - standard models fitted on weeks before each cut-off (nothing written; weeks 54-66 never used)\n")
print(f"Models ({len(F1B_MODEL_COLS)}): benchmarks {F1B_BENCHMARKS} + automatic {F1B_AUTO}, season_length = {F1_SEASON_LENGTH}")
print("\nPractice rounds (cross-validation, FPP2 3.4):")
display(f1b_runs)
print(f"f1b_cv: {len(f1b_cv):,} rows = series x 13 weeks, with real sales y for scoring | "
      f"missing forecasts: {int(f1b_cv[F1B_MODEL_COLS].isna().sum().sum())}")
display(f1b_cv.groupby(["set", "level"]).agg(series=("unique_id", "nunique"), rows=("y", "size"))
        .unstack("set").reindex(list(F1B_LEVELS.values())))
display(f1b_cv.head(3))
print("\nNegative forecasts (rows below 0; NOT clipped - handled by the log version, FPP2 12.3):")
display(f1b_neg)
print(f"\nShort backtest products ({len(f1b_aicc)}): model chosen by AICc inside the ETS family (FPP2 8.10, 12.7):")
display(f1b_aicc)
print("NEXT: errors and winners (MASE picks; MAE and WAPE shown) - the cell from old F1 cell 88.")


## Step 7 – Part 7b: the models, the columns they produce, and how they are scored
*(Rewritten from old F1 cell 74: SeasonalNaive removed (`season_length = 1`), Mean and Drift added, no interval columns here – intervals are made in Part 7f.)*

**The 7 standard models (7b) – one column each in `f1b_cv`**

| Column | Model | Best guess for every future week | Type |
|---|---|---|---|
| `Naive` | Naive | the last known week, repeated | Benchmark (FPP2 §3.1) |
| `Mean` | HistoricAverage | the average of all past weeks | Benchmark (FPP2 §3.1) |
| `Drift` | RandomWalkWithDrift | last week + the average weekly change so far | Benchmark (FPP2 §3.1) |
| `AutoETS` | AutoETS | a smoothed level (and trend if it helps); form chosen by AICc | Automatic (FPP2 Ch 7) |
| `AutoARIMA` | AutoARIMA | uses how recent weeks relate to each other; orders chosen by AICc | Automatic (FPP2 Ch 8) |
| `AutoTheta` | AutoTheta | a blend of a straight trend line and a smoothed curve | Automatic |
| `CES` | AutoCES | a more flexible smoothing curve | Automatic |

**The 5 log versions (cell below, from old F1 cells 104–105) – one column each in `f1l_all`**

`Drift_log`, `AutoETS_log`, `AutoARIMA_log`, `AutoTheta_log`, `CES_log`: the same model fitted on log(1 + sales), then turned back into cases and **bias-adjusted** so the forecast is an average (FPP2 §3.2: "medians do not add up, whereas means do"). This stops forecasts going far below 0 (FPP2 §12.3). Naive and Mean get no log version: they never forecast below 0.

**The other columns**

| Column | Meaning |
|---|---|
| `set` | practice round: `cv1` learns weeks 1–27 and predicts 28–40; `cv2` learns 1–40 and predicts 41–53 |
| `level` | Total / Category2 / Sub_Category / Brand / Product |
| `unique_id` | the series (a product code, brand, category … or `Total`) |
| `ds` | the predicted week (Monday) |
| `cutoff` | the last week the models were allowed to see |
| `y` | the real sales of that week (known – all inside weeks 1–53; weeks 54–66 stay sealed) |

**Short backtest products (`f1b_aicc`)**: too few weeks for practice rounds, so the ETS form is chosen by **AICc** (FPP2 §8.10, §12.7). Two 5-week products get SES (the ETS(A,N,N) method), because AutoETS needs at least 7 weeks.

**How the guesses are scored (next cells)**

| Score | Meaning | Use |
|---|---|---|
| **MASE** | average error ÷ the series' average week-to-week change in its training weeks (FPP2 §3.4, Hyndman & Koehler 2006). No units, so big and small series compare fairly | **picks the winner** |
| MAE | average error in cases per week | shown |
| WAPE | total error ÷ total real sales (blank when real sales are 0) | shown |

Lower is better for all three. MASE above 1 does **not** by itself mean "worse than Naive": the scale is a one-week naive error, our forecasts go 13 weeks ahead – compare models with each other.


In [ ]:
# ==========================================================================
# STEP 7 - PART 7b, CELL #11: LOG VERSION OF THE MODELS THAT CAN GO NEGATIVE (cv1 & cv2)
#   From OLD F1 cells 104 + 105 (log1p -> fit -> expm1), rewritten:
#     old: 6 models with SeasonalNaive / season_length = 13, fitted on ALL weeks (test weeks included),
#          back-transformed WITHOUT bias adjustment, intervals of the old run
#     here: the 5 models that produced negatives in 7b (Drift, AutoETS, AutoARIMA, AutoTheta, CES),
#          season_length = 1, the SAME practice rounds and the SAME series as 7b (f1b_cv),
#          back-transformed AND bias-adjusted (mean, not median), so the forecasts add up in 7g.
#   Naive and Mean get no log version: they never forecast below 0 (Decision C).
#   Runs BEFORE the winners cell, so raw and log versions compete in one pick (Decision A).
#
# WHY (FPP2 2nd ed., Hyndman & Athanasopoulos 2018):
#   12.3 (p.469): "To impose a positivity constraint, simply work on the log scale".
#   3.2 (p.66): zero sales -> "observations are adjusted by adding a constant" -> log(1 + y) = np.log1p.
#   3.2 (p.66): the back-transformed forecast "will usually be the median of the forecast distribution".
#   3.2 (p.67): "medians do not add up, whereas means do" -> bias adjustment, equation (3.2) with
#     lambda = 0:  mean = exp(w) * (1 + sigma_h^2 / 2);  minus 1 here because we added 1 (log1p).
#     sigma_h (the h-step forecast standard deviation on the log scale) is read from each model's
#     95% interval: sigma_h = (hi95 - lo95) / (2 * 1.96)  (normal intervals, FPP2 3.5).
#   The 95% interval is used ONLY for this adjustment; real intervals are made in Part 7f (Decision D).
#
# WHERE EACH INPUT COMES FROM:
#   f1b_cv, F1B_CV_SETS, F1B_MODEL_COLS -> Part 7b (cell above): the 7b forecasts + real sales y
#   f1_train, F1_CUTOFF, F1_H, F1_SEASON_LENGTH -> Part 7a;  pd, np -> 7a imports
#   StatsForecast, RandomWalkWithDrift, AutoETS, AutoARIMA, AutoTheta, AutoCES, time -> 7b imports
#   display -> Jupyter
# WHAT IT CHANGES: nothing (f1b_cv is NOT modified).  WHAT IT WRITES: nothing (memory only).
# All new names start with f1l_ / F1L_.
# ==========================================================================
F1L_BASE = ["Drift", "AutoETS", "AutoARIMA", "AutoTheta", "CES"]       # the 7b models that went below 0
F1L_LOG_COLS = [f"{f1l_m}_log" for f1l_m in F1L_BASE]                  # their log-version column names
F1L_ALL_MODELS = F1B_MODEL_COLS + F1L_LOG_COLS                         # 7 raw + 5 log = 12 candidates
F1L_Z95 = 1.959963984540054                                            # normal 97.5% point (95% interval)
assert set(F1L_BASE) <= set(F1B_MODEL_COLS), "7b model names changed - re-run 7b"

def f1l_models():
    """The same 5 models as 7b (non-seasonal), with '_log' column names."""
    return [RandomWalkWithDrift(alias="Drift_log"),
            AutoETS(season_length=F1_SEASON_LENGTH, alias="AutoETS_log"),
            AutoARIMA(season_length=F1_SEASON_LENGTH, alias="AutoARIMA_log"),
            AutoTheta(season_length=F1_SEASON_LENGTH, alias="AutoTheta_log"),
            AutoCES(season_length=F1_SEASON_LENGTH, alias="CES_log")]

f1l_parts, f1l_run_rows, f1l_adj_rows = [], [], []
for f1l_set in F1B_CV_SETS:
    # exactly the series 7b forecast in this round (same products + higher levels)
    f1l_ids = f1b_cv.loc[f1b_cv["set"] == f1l_set, "unique_id"].unique()
    f1l_train = f1_train[f1l_set][f1_train[f1l_set]["unique_id"].isin(f1l_ids)].copy()
    assert f1l_train["unique_id"].nunique() == len(f1l_ids)
    assert (f1l_train.groupby("unique_id")["ds"].max() == F1_CUTOFF[f1l_set]).all(), "series does not end at the cut-off"
    assert (f1l_train["y"] >= 0).all(), "negative sales in training - log(1 + y) not allowed"
    f1l_train["y"] = np.log1p(f1l_train["y"])                         # log(1 + sales)

    f1l_t0 = time.time()
    f1l_sf = StatsForecast(models=f1l_models(), freq="W-MON", n_jobs=-1)
    f1l_fc = f1l_sf.forecast(df=f1l_train, h=F1_H, level=[95])          # log scale, with 95% interval
    f1l_secs = time.time() - f1l_t0
    assert len(f1l_fc) == len(f1l_ids) * F1_H

    f1l_out = f1l_fc[["unique_id", "ds"]].copy()
    for f1l_c in F1L_LOG_COLS:
        f1l_w = f1l_fc[f1l_c]                                           # point forecast on the log scale
        f1l_sig = (f1l_fc[f"{f1l_c}-hi-95"] - f1l_fc[f"{f1l_c}-lo-95"]) / (2 * F1L_Z95)
        assert f1l_w.notna().all() and f1l_sig.notna().all() and (f1l_sig >= 0).all(), f"{f1l_set} {f1l_c}: bad output"
        f1l_out[f1l_c] = np.exp(f1l_w) * (1 + f1l_sig ** 2 / 2) - 1    # bias-adjusted mean, in cases
        f1l_adj_rows.append({"set": f1l_set, "model": f1l_c,
                             "median (no adjustment), total cases": round(float(np.expm1(f1l_w).sum()), 1),
                             "mean (bias-adjusted), total cases": round(float(f1l_out[f1l_c].sum()), 1)})
    f1l_parts.append(f1l_out.assign(set=f1l_set))
    f1l_run_rows.append({"set": f1l_set, "series": len(f1l_ids), "forecast rows": len(f1l_out),
                         "seconds": round(f1l_secs, 1)})

# one table with the 7b columns + the 5 log columns (f1b_cv itself is left unchanged)
f1l_log_cv = pd.concat(f1l_parts, ignore_index=True)
f1l_all = f1b_cv.merge(f1l_log_cv, on=["set", "unique_id", "ds"], how="left", validate="one_to_one")
assert len(f1l_all) == len(f1b_cv) and f1l_all[F1L_ALL_MODELS].notna().all().all(), "log rows do not match 7b rows"

# ---- Report ----
f1l_runs = pd.DataFrame(f1l_run_rows).set_index("set")
f1l_adj = pd.DataFrame(f1l_adj_rows)
f1l_adj["adjustment %"] = (100 * (f1l_adj["mean (bias-adjusted), total cases"]
                                  / f1l_adj["median (no adjustment), total cases"] - 1)).round(1)
f1l_neg = pd.DataFrame({"raw (7b)": (f1l_all[F1L_BASE] < 0).sum().values,
                        "log version": (f1l_all[F1L_LOG_COLS] < 0).sum().values,
                        "lowest log forecast (cases)": f1l_all[F1L_LOG_COLS].min().round(2).values},
                       index=F1L_BASE)
f1l_neg.loc["All"] = [f1l_neg["raw (7b)"].sum(), f1l_neg["log version"].sum(), f1l_neg["lowest log forecast (cases)"].min()]

print("STEP 7b #11 - log versions of 5 models, same rounds and series as 7b (nothing written; weeks 54-66 never used)\n")
display(f1l_runs)
print("Bias adjustment (FPP2 3.2): sum of all forecasts, median vs mean - the mean is what adds up in 7g:")
display(f1l_adj)
print("Negative forecasts, raw vs log version (a log forecast can never be below -1 case):")
display(f1l_neg)
print(f"f1l_all: {len(f1l_all):,} rows x {len(F1L_ALL_MODELS)} candidate models {F1L_ALL_MODELS}")
print("NEXT: #6 errors and winners - MASE picks from these 12 columns; MAE and WAPE shown.")


In [ ]:
# ==========================================================================
# STEP 7 - PART 7b, CELL #6: ERRORS AND WINNERS - MASE PICKS, MAE AND WAPE SHOWN (cv1 & cv2)
#   From OLD F1 cell 88 (WAPE per model / per product, idxmin = winner), rewritten:
#     old: WAPE picked the winner, errors came from CV windows inside the test weeks (leak),
#          products only, 6 models
#     here: MASE picks (WAPE only shown), errors from the two practice rounds inside weeks 1-53,
#          all 12 candidates (7 raw + 5 log, Decision A), every level gets a winner (Decision B)
#   Score per series and round, then the average over the rounds the series has (1 or 2);
#   lowest average MASE wins.  Ties: the first model in F1L_ALL_MODELS order (simpler model first).
#   If a series had a MASE scale of 0 (flat training weeks), WAPE would decide instead (none expected).
#
# WHY (FPP2 2nd ed., Hyndman & Athanasopoulos 2018):
#   3.4 (p.77): accuracy "can only be determined by considering how well a model performs on new data
#     that were not used when fitting the model" -> scored on the 13 weeks after each cut-off.
#   3.4 (pp.80-81): scaled errors, "for a non-seasonal time series ... uses naive forecasts":
#     scale = mean |y_t - y_(t-1)| over the TRAINING weeks of that round;  MASE = mean(|error|) / scale.
#     "an alternative to using percentage errors when comparing forecast accuracy across series with
#     different units" (Hyndman & Koehler 2006, Int. J. Forecasting).
#   10.6 (p.414): base forecasts are needed for every series of the hierarchy -> winners at all levels.
#   WAPE = sum|error| / sum(y): a business measure (not in FPP2) - shown only, blank when sum(y) = 0.
#
# WHERE EACH INPUT COMES FROM:
#   f1l_all, F1L_ALL_MODELS -> cell #11 (7b forecasts + 5 log versions, with real sales y)
#   F1B_CV_SETS, F1B_LEVELS, f1b_upper_all -> Part 7b;  f1_train, f1_routes, f1_labels -> Part 7a (labels sealed)
#   pd, np -> 7a imports;  display -> Jupyter
# WHAT IT CHANGES: nothing.  WHAT IT WRITES: nothing (memory only).  New names start with f1e_ / F1E_.
# ==========================================================================
F1E_LEVEL_ORDER = list(F1B_LEVELS.values())                     # Total ... Product

# ---- 1. MASE scale per series and round: mean week-to-week change in that round's training weeks ----
f1e_scale_parts = []
for f1e_set in F1B_CV_SETS:
    f1e_tr = f1_train[f1e_set].sort_values(["unique_id", "ds"])
    f1e_sc = f1e_tr.groupby("unique_id")["y"].apply(lambda f1e_y: f1e_y.diff().abs().mean())
    f1e_scale_parts.append(f1e_sc.rename("scale").reset_index().assign(set=f1e_set))
f1e_scale = pd.concat(f1e_scale_parts, ignore_index=True)

# ---- 2. Errors -> MAE, MASE, WAPE per set x series x model ----
f1e_err = f1l_all[["set", "level", "unique_id", "ds", "y"]].copy()
for f1e_m in F1L_ALL_MODELS:
    f1e_err[f1e_m] = (f1l_all["y"] - f1l_all[f1e_m]).abs()          # absolute error per week
f1e_keys = ["set", "level", "unique_id"]
f1e_mae = f1e_err.groupby(f1e_keys)[F1L_ALL_MODELS].mean()
f1e_ysum = f1e_err.groupby(f1e_keys)["y"].sum()
f1e_wape = f1e_err.groupby(f1e_keys)[F1L_ALL_MODELS].sum().div(f1e_ysum.where(f1e_ysum > 0), axis=0)
f1e_sc_ix = f1e_mae.reset_index()[f1e_keys].merge(f1e_scale, on=["set", "unique_id"], how="left", validate="one_to_one")
assert f1e_sc_ix["scale"].notna().all(), "a scored series has no training weeks"
f1e_scale_vals = f1e_sc_ix["scale"].where(f1e_sc_ix["scale"] > 0).values   # 0 -> NaN (WAPE decides)
f1e_mase = f1e_mae.div(f1e_scale_vals, axis=0)

# long table: one row per set x series x model, with the 3 scores
f1e_scores = (pd.concat({"MAE": f1e_mae, "MASE": f1e_mase, "WAPE": f1e_wape}, axis=1)
              .stack(level=1, future_stack=True).rename_axis(f1e_keys + ["model"]).reset_index())

# ---- 3. Winner per series: lowest MASE averaged over the rounds it has ----
f1e_avg_mase = f1e_mase.groupby(level="unique_id").mean()[F1L_ALL_MODELS]
f1e_avg_wape = f1e_wape.groupby(level="unique_id").mean()[F1L_ALL_MODELS]
f1e_by_wape = f1e_avg_mase.isna().all(axis=1)                    # MASE scale 0 in every round
f1e_pick = f1e_avg_mase.copy()                                   # the score that decides
f1e_pick.loc[f1e_by_wape] = f1e_avg_wape.loc[f1e_by_wape]        # WAPE only where MASE is impossible
assert f1e_pick.notna().any(axis=1).all(), "a series has neither MASE nor WAPE - stop and check"
f1e_win = pd.DataFrame({
    "level": f1e_mae.reset_index().drop_duplicates("unique_id").set_index("unique_id")["level"],
    "rounds": f1e_mase.groupby(level="unique_id").size(),
    "picked_by": np.where(f1e_by_wape, "WAPE", "MASE"),
    "winner": f1e_pick.idxmin(axis=1)})
f1e_win["winner_avg_MASE"] = [f1e_avg_mase.at[f1e_u, f1e_w] for f1e_u, f1e_w in f1e_win["winner"].items()]
f1e_win["Naive_avg_MASE"] = f1e_avg_mase["Naive"]

# which winners are used later for the backtest forecast (7g): the backtest standard products chosen
# by CV (7a choose_by == "CV") + every series above product level
f1e_bt_cv = set(f1_routes.loc[(f1_routes["set"] == "backtest") & (f1_routes["lane"] == "standard")
                              & (f1_routes["choose_by"] == "CV"), "unique_id"])
assert f1e_bt_cv <= set(f1e_win.index), "a backtest CV product has no practice-round score"
f1e_win["used_in_backtest"] = f1e_win.index.isin(f1e_bt_cv) | (f1e_win["level"] != "Product")
f1e_win = f1e_win.join(f1_labels.set_index("unique_id")["SKU"]).reset_index()
f1e_win = f1e_win[["level", "unique_id", "SKU", "rounds", "picked_by", "winner", "winner_avg_MASE",
                   "Naive_avg_MASE", "used_in_backtest"]]

# ---- 4. Overall picture per model (old cell 88's wape_overall, now per level, with MASE) ----
f1e_overall = f1e_scores.groupby(["level", "model"]).agg(mean_MASE=("MASE", "mean"), median_MASE=("MASE", "median"),
                                                         MAE=("MAE", "mean"))
f1e_wsum = (f1e_err.groupby("level")[F1L_ALL_MODELS].sum().div(f1e_err.groupby("level")["y"].sum(), axis=0)
            .stack(future_stack=True).rename("WAPE (all weeks)"))
f1e_overall = f1e_overall.join(f1e_wsum.rename_axis(["level", "model"]))
f1e_wins = (f1e_win[f1e_win["used_in_backtest"]].groupby(["level", "winner"]).size()
            .rename("series won").rename_axis(["level", "model"]))
f1e_overall = f1e_overall.join(f1e_wins).fillna({"series won": 0}).astype({"series won": int})

# ---- Report ----
print("STEP 7b #6 - errors and winners on the practice rounds (weeks 28-40 and 41-53; nothing written)\n")
print(f"Scored: {f1e_err.groupby(['set', 'unique_id']).ngroups} series-rounds, {len(F1L_ALL_MODELS)} models | "
      f"MASE scale 0 (WAPE decides): {int(f1e_by_wape.sum())} | "
      f"WAPE blank (real sales 0): {int(f1e_ysum.eq(0).sum())} series-round(s)")
print("\nWinners per level (series that get a forecast in the backtest, 7g):")
display(f1e_win[f1e_win["used_in_backtest"]].groupby("level")["winner"].value_counts().unstack(fill_value=0)
        .reindex(F1E_LEVEL_ORDER))
print(f"Products: {int((f1e_win['used_in_backtest'] & (f1e_win['level'] == 'Product')).sum())} backtest CV products "
      f"+ {int((~f1e_win['used_in_backtest']).sum())} scored only (not standard-by-CV in the backtest)")
f1e_no_win = [f1e_u for f1e_u in f1b_upper_all if f1e_u not in set(f1e_win["unique_id"])]
print(f"Higher-level series with no winner (fewer than 13 weeks in both rounds - Part 7g decides): {len(f1e_no_win)}")
print("\nProducts - every model's score (lower is better; 'series won' counts the backtest products):")
display(f1e_overall.loc["Product"].sort_values("mean_MASE").round(3))
print("\nWinners (first 10 rows of f1e_win):")
display(f1e_win.head(10))
print("NEXT: #7 honest check - pick on cv1, score on cv2.")


In [ ]:
# ==========================================================================
# STEP 7 - PART 7b, CELL #7: HONEST CHECK - PICK THE WINNER ON cv1, SCORE IT ON cv2
#   Replaces OLD F1 cell 100 ("blended WAPE"): it picked each product's winner and reported the score
#   on the SAME weeks it was picked on, so the number always looked better than reality.
#   Here: each series' winner is picked on cv1 only, then scored on cv2 (weeks 41-53), which played no
#   part in that pick.  It is compared on cv2 with simple rules that need no picking per series:
#     "always AutoETS", "always Naive", and "one model for all" (the model with the best average
#     cv1 MASE at that level).  Only series scored in BOTH rounds are used.
#   This is a check only: the winners used later stay the ones from cell #6 (both rounds).
#
# WHY (FPP2 2nd ed., Hyndman & Athanasopoulos 2018):
#   3.4 (p.77): accuracy "can only be determined by considering how well a model performs on new data
#     that were not used when fitting the model" -> picking a winner is part of fitting, so the pick
#     is judged on a round it did not see.
#   3.4 (pp.80-81): MASE (same scores as cell #6).
#
# WHERE EACH INPUT COMES FROM:
#   f1e_mase (MASE per set x level x series x model), F1E_LEVEL_ORDER -> cell #6
#   F1L_ALL_MODELS -> cell #11;  pd, np -> 7a imports;  display -> Jupyter
# WHAT IT CHANGES: nothing.  WHAT IT WRITES: nothing (memory only).  New names start with f1h_ / F1H_.
# ==========================================================================
f1h_cv1 = f1e_mase.xs("cv1", level="set")                      # index: (level, unique_id)
f1h_cv2 = f1e_mase.xs("cv2", level="set")
f1h_both = f1h_cv1.index.intersection(f1h_cv2.index)           # series scored in both rounds
f1h_cv1, f1h_cv2 = f1h_cv1.loc[f1h_both], f1h_cv2.loc[f1h_both]
assert f1h_cv1.notna().any(axis=1).all() and f1h_cv2.notna().any(axis=1).all(), "MASE missing in a round"

f1h_pick = f1h_cv1.fillna(np.inf).idxmin(axis=1)               # winner chosen on cv1 only
f1h_cols = f1h_cv2.columns.get_indexer(f1h_pick)
f1h_detail = pd.DataFrame({
    "picked on cv1": f1h_pick,
    "its MASE on cv1 (where it was picked)": f1h_cv1.to_numpy()[np.arange(len(f1h_cv1)), f1h_cols],
    "its MASE on cv2 (honest)": f1h_cv2.to_numpy()[np.arange(len(f1h_cv2)), f1h_cols],
    "always AutoETS on cv2": f1h_cv2["AutoETS"],
    "always Naive on cv2": f1h_cv2["Naive"]}, index=f1h_both)

f1h_rows = []
for f1h_lev in F1E_LEVEL_ORDER:
    if f1h_lev not in f1h_detail.index.get_level_values("level"):
        continue
    f1h_d = f1h_detail.xs(f1h_lev, level="level")
    f1h_one = f1h_cv1.xs(f1h_lev, level="level").mean().idxmin()        # one model for all, chosen on cv1
    f1h_rows.append({"level": f1h_lev, "series": len(f1h_d),
                     "per-series pick: MASE on cv1 (looks)": f1h_d["its MASE on cv1 (where it was picked)"].mean(),
                     "per-series pick: MASE on cv2 (honest)": f1h_d["its MASE on cv2 (honest)"].mean(),
                     "always AutoETS (cv2)": f1h_d["always AutoETS on cv2"].mean(),
                     "always Naive (cv2)": f1h_d["always Naive on cv2"].mean(),
                     "one model for all (cv2)": f1h_cv2.xs(f1h_lev, level="level")[f1h_one].mean(),
                     "that one model": f1h_one,
                     "% series where pick beats AutoETS (cv2)":
                         100 * (f1h_d["its MASE on cv2 (honest)"] < f1h_d["always AutoETS on cv2"]).mean()})
f1h_summary = pd.DataFrame(f1h_rows).set_index("level")

# ---- Report ----
print("STEP 7b #7 - honest check: winner picked on cv1, scored on cv2 (mean MASE, lower is better; nothing written)\n")
display(f1h_summary.round(3))
f1h_p = f1h_summary.loc["Product"] if "Product" in f1h_summary.index else None
if f1h_p is not None:
    print(f"Products: the pick LOOKED like MASE {f1h_p['per-series pick: MASE on cv1 (looks)']:.3f} on the round it was "
          f"picked on, but scored {f1h_p['per-series pick: MASE on cv2 (honest)']:.3f} on the next round.")
    print(f"  Compare on cv2: always AutoETS {f1h_p['always AutoETS (cv2)']:.3f} | always Naive "
          f"{f1h_p['always Naive (cv2)']:.3f} | one model for all ({f1h_p['that one model']}) "
          f"{f1h_p['one model for all (cv2)']:.3f}")
print("A check only - the winners from cell #6 (both rounds) are kept.  NEXT: #8 chosen forecast per series.")


In [ ]:
# ==========================================================================
# STEP 7 - PART 7b, CELL #8: THE CHOSEN MODEL'S FORECAST PER SERIES (cv1 & cv2)
#   From OLD F1 cell 96 (final_forecast["chosen_model_forecast"] = the winner's column), rewritten:
#     old: took the winner's column from the old FUTURE forecast (old data, test weeks)
#     here: takes it from the practice rounds only (f1l_all, weeks 28-40 and 41-53), for every series
#          with a winner in cell #6 (all levels).  The forecasts for weeks 54-66 are made in Part 7g.
#   Used by 7e (combinations) and 7g (bottom-up vs MinT inside the same CV windows, Scorecard row 9).
#   Note: the winner was picked on these same rounds, so this table is for COMPARING methods with each
#   other (both start from it) - not a final accuracy figure (that is cell #7 and Step 8).
#
# WHY (FPP2 2nd ed.): 10.6 (p.414) "Suppose we forecast all series independently ... We call these the
#   base forecasts" -> one chosen base forecast per series, at every level.
#
# WHERE EACH INPUT COMES FROM:
#   f1l_all, F1L_ALL_MODELS -> cell #11;  f1e_win (winner per series) -> cell #6
#   pd, np -> 7a imports;  display -> Jupyter
# WHAT IT CHANGES: nothing.  WHAT IT WRITES: nothing (memory only).  New names start with f1w_.
# ==========================================================================
f1w_cv = f1l_all.merge(f1e_win[["unique_id", "winner", "used_in_backtest"]], on="unique_id",
                       how="inner", validate="many_to_one")
assert f1w_cv["unique_id"].nunique() == len(f1e_win), "a winner has no practice-round rows"
f1w_cols = pd.Index(F1L_ALL_MODELS).get_indexer(f1w_cv["winner"])
assert (f1w_cols >= 0).all(), "a winner is not one of the 12 model columns"
f1w_cv["chosen"] = f1w_cv[F1L_ALL_MODELS].to_numpy()[np.arange(len(f1w_cv)), f1w_cols]
f1w_cv = f1w_cv.rename(columns={"winner": "chosen_model"})[
    ["set", "level", "unique_id", "ds", "cutoff", "y", "chosen_model", "chosen", "used_in_backtest"]]
assert f1w_cv["chosen"].notna().all()

# ---- Report ----
print("STEP 7b #8 - the chosen model's forecast per series, practice rounds only (nothing written)\n")
print(f"f1w_cv: {len(f1w_cv):,} rows, {f1w_cv['unique_id'].nunique()} series | "
      f"chosen forecasts below 0: {int((f1w_cv['chosen'] < 0).sum())} "
      f"(lowest {f1w_cv['chosen'].min():.2f} cases)")
display(f1w_cv.groupby(["set", "level"]).agg(series=("unique_id", "nunique"), rows=("chosen", "size"),
                                             real_sales=("y", "sum"), chosen_forecast=("chosen", "sum"))
        .round(0).unstack("set").reindex(F1E_LEVEL_ORDER))
display(f1w_cv.head(3))
print("NEXT: #9 chart - error per model.")


In [ ]:
# ==========================================================================
# STEP 7 - PART 7b, CELL #9: CHART - AVERAGE MASE PER MODEL (backtest products)
#   From OLD F1 cell 89 (WAPE bar per model), rewritten:
#     old: WAPE bars, best model shown by colour only
#     here: MASE (the score that picks), WAPE written next to each bar, best model marked in words,
#          raw and log versions in two colours + a legend, line at MASE = 1.
#   Products = the backtest standard products chosen by CV (the ones whose winner is used in 7g).
#   MASE = 1 is the one-week naive error in training; forecasts here go 13 weeks ahead, so compare
#   bars with each other (Naive's own bar is the benchmark), not with the line alone.
#   A bar longer than the axis is cut and its value written at the end of the axis.
#
# WHY (FPP2 2nd ed.): 3.1 (p.57) benchmarks (Naive, Mean, Drift) are the bar to beat;
#   3.4 (pp.80-81) MASE.
#
# WHERE EACH INPUT COMES FROM:
#   f1e_avg_mase (average MASE per series x model), f1e_win, f1e_err (absolute errors) -> cell #6
#   F1L_ALL_MODELS, F1L_LOG_COLS -> cell #11;  pd, np -> 7a imports;  plt -> Cell B import (matplotlib)
# WHAT IT CHANGES: nothing.  WHAT IT WRITES: nothing (shows 1 chart).  New names start with f1v_ / F1V_.
# ==========================================================================
F1V_RAW, F1V_LOG = "#0072B2", "#E69F00"                          # same blue / orange as Cell B
f1v_ids = f1e_win.loc[f1e_win["used_in_backtest"] & (f1e_win["level"] == "Product"), "unique_id"]
f1v_mase = f1e_avg_mase.loc[f1v_ids, F1L_ALL_MODELS].mean().sort_values(ascending=False)   # best at the top
f1v_err = f1e_err[f1e_err["unique_id"].isin(f1v_ids)]                 # the same products as the bars
f1v_wape = f1v_err[F1L_ALL_MODELS].sum() / f1v_err["y"].sum()          # WAPE over all their weeks
f1v_xmax = max(2.0, f1v_mase.iloc[1] * 1.25)                     # axis ends after the 2nd-worst bar

fig, ax = plt.subplots(figsize=(9, 6.5))
f1v_colors = [F1V_LOG if f1v_m in F1L_LOG_COLS else F1V_RAW for f1v_m in f1v_mase.index]
ax.barh(f1v_mase.index, f1v_mase.clip(upper=f1v_xmax).values, color=f1v_colors, height=0.7, edgecolor="white")
ax.axvline(1.0, color="grey", ls="--", lw=1)
ax.text(1.0, len(f1v_mase) - 0.4, " MASE = 1 (one-week naive error)", color="grey", fontsize=8, va="bottom")
for f1v_i, (f1v_m, f1v_v) in enumerate(f1v_mase.items()):
    f1v_txt = f"{f1v_v:.2f}   (WAPE {f1v_wape[f1v_m]:.0%})" + ("   best" if f1v_v == f1v_mase.min() else "")
    if f1v_v > f1v_xmax:
        f1v_txt = f"{f1v_v:.2f} - bar cut   (WAPE {f1v_wape[f1v_m]:.0%})"
    ax.text(min(f1v_v, f1v_xmax) + 0.02, f1v_i, f1v_txt, va="center", fontsize=9)
ax.set_xlim(0, f1v_xmax * 1.45)
ax.set_xlabel("average MASE over cv1 and cv2 (lower is better)")
ax.set_title(f"Standard products ({len(f1v_ids)}) - average error per model, practice rounds", loc="left")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=F1V_RAW), plt.Rectangle((0, 0), 1, 1, color=F1V_LOG)],
          labels=["raw (7b)", "log version, bias-adjusted (#11)"], loc="center right")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()


In [ ]:
# ==========================================================================
# STEP 7 - PART 7b, CELL #10: CHART - HOW MANY PRODUCTS EACH MODEL WON
#   From OLD F1 cell 134 (bar of best_model_per_product_2w.value_counts(), "out of 25"), rewritten:
#     old: old data, hard-coded "out of 25" title, models that won nothing were missing
#     here: the winners from cell #6 for the backtest standard products, all 12 models shown
#          (0 included), the count in the title comes from the data.
#   Cell #9 shows how good each model is ON AVERAGE; this shows how OFTEN each one is the best.
#
# WHERE EACH INPUT COMES FROM:
#   f1e_win -> cell #6;  F1L_ALL_MODELS, F1L_LOG_COLS -> cell #11;  F1V_RAW, F1V_LOG -> cell #9
#   pd -> 7a import;  plt -> Cell B import (matplotlib)
# WHAT IT CHANGES: nothing.  WHAT IT WRITES: nothing (shows 1 chart).  New names start with f1n_.
# ==========================================================================
f1n_prod = f1e_win[f1e_win["used_in_backtest"] & (f1e_win["level"] == "Product")]
f1n_counts = f1n_prod["winner"].value_counts().reindex(F1L_ALL_MODELS, fill_value=0).sort_values()
assert f1n_counts.sum() == len(f1n_prod)

fig, ax = plt.subplots(figsize=(9, 6.5))
ax.barh(f1n_counts.index, f1n_counts.values, height=0.7, edgecolor="white",
        color=[F1V_LOG if f1n_m in F1L_LOG_COLS else F1V_RAW for f1n_m in f1n_counts.index])
for f1n_i, f1n_v in enumerate(f1n_counts.values):
    ax.text(f1n_v + 0.3, f1n_i, str(f1n_v), va="center", fontsize=9)
ax.set_xlabel("number of products won (lowest average MASE)")
ax.set_title(f"Which model wins how many products (out of {len(f1n_prod)})", loc="left")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=F1V_RAW), plt.Rectangle((0, 0), 1, 1, color=F1V_LOG)],
          labels=["raw (7b)", "log version, bias-adjusted (#11)"], loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()
print(f"Raw models won {int(f1n_counts.drop(F1L_LOG_COLS).sum())} products, log versions {int(f1n_counts[F1L_LOG_COLS].sum())}.")


In [ ]:
# ==========================================================================
# STEP 7 - PART 7c, CELL 12: INTERMITTENT MODELS - CROSTON / SBA / OPTIMIZED / ADIDA / IMAPA + MEAN,
#                            CV ON cv1 & cv2 (WEEKS 1-53), ONE-SALE PRODUCTS -> JUDGEMENT RULE
#   From OLD F1 cell 125 (import + StatsForecast of the 6 Croston-type models), rewritten:
#     old: TSB fixed at 0.3 (chosen on the test weeks - a leak), fitted on old data, no practice rounds
#     here: TSB is left out (cell 13 tests its settings inside cv1/cv2); Mean is added as a candidate;
#           every model learns only the weeks before each cut-off and predicts the next 13 weeks.
#   Which series (your decisions 1, 5 and problem 4):
#     - products whose BACKTEST lane is "intermittent" (7a router), in EVERY practice round where they
#       have at least F1_MIN_FIT (13) weeks of history - whatever their lane was in that round;
#     - series ABOVE product level that are sparse: more than 50% zero weeks in weeks 1-53 (decision 5).
#       The 7a ADI rule (ADI >= 1.32) is counted next to it for comparison only.
#     - products with only ONE selling week in weeks 1-53 get a judgement rate instead of a model
#       (decision 1): total sold / weeks since launch, flagged "please review".
#   Scores and the choice of method are NOT made here: that is cell 13a (RMSSE picks, decision 4).
#   "Zero" (always 0) is added as a guard column only - it can never be chosen (decision 4).
#   The forecasts and real sales are kept together, so 7f can build ranges from these errors.
#
# WHY (FPP2 2nd ed., Hyndman & Athanasopoulos 2018, unless named):
#   12.2 (p.467): Croston's method - forecasts the size of a sale and the gap between sales; "There
#     are no algebraic results allowing us to compute prediction intervals for this method" -> 7f.
#   3.1 (p.60): "Sometimes one of these simple methods will be the best forecasting method available"
#     -> Mean (HistoricAverage) is a candidate.
#   3.4 (p.77): accuracy "can only be determined by considering how well a model performs on new data
#     that were not used when fitting the model" -> learn up to each cut-off, predict the next 13 weeks.
#   10.6 (p.414): base forecasts for every series -> sparse higher-level series get these models too.
#   Ch 4 (p.101): "judgmental forecasting is the only option, such as when there is a complete lack of
#     historical data" -> one-sale products.
#   Not in FPP2 (named, peer-reviewed): CrostonSBA - Syntetos & Boylan 2005 (Int. J. Forecasting);
#     ADIDA - Nikolopoulos et al. 2011 (J. Oper. Res. Soc.); IMAPA - Petropoulos & Kourentzes 2015
#     (J. Oper. Res. Soc.).
#
# WHERE EACH INPUT COMES FROM (all already in memory; nothing read from disk):
#   f1_routes   -> 7a step 5: lane + weeks_on_sale per product and set ("backtest" lane decides)
#   f1_train    -> 7a step 4: {"cv1": weeks 1-27, "cv2": weeks 1-40, "backtest": weeks 1-53}, all series
#   f1_Y        -> 7a step 3: weeks 1-53 (real sales of the predicted weeks 28-40 and 41-53)
#   f1_weeks, F1_CUTOFF, F1_H (13), F1_MIN_FIT (13) -> 7a;  f1_labels -> 7a (sealed): SKU names
#   F1B_CV_SETS (["cv1", "cv2"]), f1b_upper_all (95 higher-level ids), f1b_level_of (id -> level),
#   F1B_LEVELS -> Part 7b;  StatsForecast, HistoricAverage -> Part 7b imports;  pd, np -> 7a imports
#   CrostonClassic, CrostonOptimized, CrostonSBA, ADIDA, IMAPA -> NEW import below (statsforecast 2.0.3)
#   display -> Jupyter built-in
#
# WHAT IT CHANGES: nothing (7a / 7b tables are only read).  WHAT IT WRITES: nothing (memory only).
# All new names start with f1i_ / F1I_ (f1c_ is already used by Cell A, so it is not reused).
# ==========================================================================
from statsforecast.models import CrostonClassic, CrostonOptimized, CrostonSBA, ADIDA, IMAPA

F1I_MODEL_COLS = ["CrostonClassic", "CrostonOptimized", "CrostonSBA", "ADIDA", "IMAPA", "Mean"]  # can be chosen
F1I_GUARD = "Zero"                                   # always 0 - shown for comparison, never chosen
F1I_SPARSE_SHARE = 0.5                               # decision 5: more than 50% zero weeks = sparse
F1I_ADI_CUT = 1.32                                   # 7a / Prep F4 ADI line - comparison only

def f1i_models():
    """The 5 Croston-type models + Mean; no settings to tune (TSB, which needs them, is cell 13)."""
    return [CrostonClassic(),                        # sale size / gap, alpha = 0.1 fixed (FPP2 12.2)
            CrostonOptimized(),                      # the same, alpha chosen from the data
            CrostonSBA(),                            # Croston x 0.95 (Syntetos & Boylan 2005)
            ADIDA(),                                 # weeks added into blocks, then split back
            IMAPA(),                                 # ADIDA at several block sizes, averaged
            HistoricAverage(alias="Mean")]           # average of all past weeks (FPP2 3.1)

assert F1_H == 13 and F1_MIN_FIT == 13 and F1B_CV_SETS == ["cv1", "cv2"], "run Parts 7a and 7b first"

# ---- 1. Which products: backtest lane "intermittent" (7a router) ----
f1i_bt = f1_routes[f1_routes["set"] == "backtest"].set_index("unique_id")
f1i_int_ids = f1i_bt.index[f1i_bt["lane"] == "intermittent"].tolist()
f1i_tr_bt = f1_train["backtest"]                     # weeks 1-53 only (sealed in 7a)
f1i_hist = (f1i_tr_bt[f1i_tr_bt["unique_id"].isin(f1i_int_ids)].groupby("unique_id")["y"]
            .agg(weeks="size", selling_weeks=lambda v: int((v > 0).sum()), total_sold="sum"))
assert len(f1i_hist) == len(f1i_int_ids), "an intermittent product has no weeks 1-53"
assert (f1i_hist["weeks"] == f1i_bt.loc[f1i_hist.index, "weeks_on_sale"]).all(), "weeks since launch != 7a"

# ---- 2. One-sale products -> judgement rate (decision 1), weeks 1-53 only ----
f1i_one_ids = f1i_hist.index[f1i_hist["selling_weeks"] <= 1].tolist()
f1i_last_sale = (f1i_tr_bt[f1i_tr_bt["unique_id"].isin(f1i_one_ids) & (f1i_tr_bt["y"] > 0)]
                 .groupby("unique_id")["ds"].max())
f1i_judgement = f1i_hist.loc[f1i_one_ids].assign(
    choose_by=f1i_bt.loc[f1i_one_ids, "choose_by"],
    last_sale_week=f1i_last_sale.reindex(f1i_one_ids).dt.date,
    weeks_since_last_sale=((F1_CUTOFF["backtest"] - f1i_last_sale.reindex(f1i_one_ids)).dt.days // 7),
    rate_per_week=lambda d: d["total_sold"] / d["weeks"],         # total sold / weeks since launch
    next_13_weeks=lambda d: d["rate_per_week"] * F1_H,
    flag="judgement - please review")
f1i_judgement = (f1i_judgement.join(f1_labels.set_index("unique_id")["SKU"])
                 .reset_index()[["unique_id", "SKU", "choose_by", "weeks", "selling_weeks", "total_sold",
                                 "last_sale_week", "weeks_since_last_sale", "rate_per_week", "next_13_weeks", "flag"]])

# ---- 3. Which higher-level series are sparse (decision 5), weeks 1-53 only ----
f1i_up = (f1i_tr_bt[f1i_tr_bt["unique_id"].isin(f1b_upper_all)].groupby("unique_id")["y"]
          .agg(weeks="size", selling_weeks=lambda v: int((v > 0).sum()), zero_share=lambda v: float((v == 0).mean())))
f1i_up["ADI"] = f1i_up["weeks"] / f1i_up["selling_weeks"]
f1i_up["level"] = f1i_up.index.map(f1b_level_of)
f1i_sparse_ids = f1i_up.index[f1i_up["zero_share"] > F1I_SPARSE_SHARE].tolist()      # used
f1i_adi_ids = f1i_up.index[f1i_up["ADI"] >= F1I_ADI_CUT].tolist()                     # comparison only

# ---- 4. Practice rounds: learn up to each cut-off, predict the next 13 weeks ----
f1i_parts, f1i_run_rows, f1i_ids_by_set = [], [], {}
for f1i_set in F1B_CV_SETS:
    f1i_tr = f1_train[f1i_set]
    f1i_w = f1i_tr[f1i_tr["unique_id"].isin(f1i_int_ids + f1i_sparse_ids)].groupby("unique_id").size()
    f1i_ok = f1i_w[f1i_w >= F1_MIN_FIT].index                         # at least 13 weeks before the cut-off
    f1i_prod_ids = [u for u in f1i_int_ids if u in f1i_ok]
    f1i_upper_ids = [u for u in f1i_sparse_ids if u in f1i_ok]
    f1i_ids = f1i_prod_ids + f1i_upper_ids
    f1i_ids_by_set[f1i_set] = f1i_ids                                 # cell 13 (TSB) uses the same series
    f1i_train_set = f1i_tr[f1i_tr["unique_id"].isin(f1i_ids)]
    # leak + alignment checks (same as 7b): every series ends at this round's cut-off
    f1i_last = f1i_train_set.groupby("unique_id")["ds"].max()
    assert len(f1i_last) == len(f1i_ids) and (f1i_last == F1_CUTOFF[f1i_set]).all(), \
        f"{f1i_set}: a series does not end at the cut-off - stop and check"
    f1i_pos = f1_weeks.get_loc(F1_CUTOFF[f1i_set])
    f1i_pred_weeks = f1_weeks[f1i_pos + 1: f1i_pos + 1 + F1_H]        # weeks 28-40 or 41-53
    assert len(f1i_pred_weeks) == F1_H and f1i_pred_weeks.max() <= F1_CUTOFF["backtest"]

    f1i_fc = StatsForecast(models=f1i_models(), freq="W-MON", n_jobs=-1).forecast(df=f1i_train_set, h=F1_H)
    assert set(f1i_fc["ds"]) == set(f1i_pred_weeks), f"{f1i_set}: forecast weeks are not the 13 after the cut-off"
    assert len(f1i_fc) == len(f1i_ids) * F1_H and f1i_fc[F1I_MODEL_COLS].notna().all().all()
    f1i_fc[F1I_GUARD] = 0.0
    f1i_fc = f1i_fc.merge(f1_Y[["unique_id", "ds", "y"]], on=["unique_id", "ds"], how="left", validate="one_to_one")
    assert f1i_fc["y"].notna().all(), f"{f1i_set}: a predicted week has no real sales row"
    f1i_fc.insert(0, "set", f1i_set)
    f1i_fc.insert(1, "level", f1i_fc["unique_id"].map(f1b_level_of))
    f1i_fc.insert(4, "cutoff", F1_CUTOFF[f1i_set])
    f1i_fc["lane_this_round"] = f1i_fc["unique_id"].map(
        f1_routes[f1_routes["set"] == f1i_set].set_index("unique_id")["lane"]).fillna("higher level")
    f1i_parts.append(f1i_fc[["set", "level", "unique_id", "ds", "cutoff", "y", "lane_this_round"]
                            + F1I_MODEL_COLS + [F1I_GUARD]])
    f1i_lanes = f1_routes[(f1_routes["set"] == f1i_set) & f1_routes["unique_id"].isin(f1i_prod_ids)]["lane"]
    f1i_run_rows.append({"set": f1i_set, "learns weeks": f"1-{f1i_pos + 1}",
                         "predicts weeks": f"{f1i_pos + 2}-{f1i_pos + 1 + F1_H}",
                         "products (backtest intermittent, >=13 weeks)": len(f1i_prod_ids),
                         "  of which intermittent this round": int((f1i_lanes == "intermittent").sum()),
                         "  of which standard this round": int((f1i_lanes == "standard").sum()),
                         "  of which other lane this round": int((~f1i_lanes.isin(["intermittent", "standard"])).sum()),
                         "sparse higher-level series": len(f1i_upper_ids), "forecast rows": len(f1i_fc)})
f1i_cv = pd.concat(f1i_parts, ignore_index=True)         # one row per set x series x predicted week
f1i_runs = pd.DataFrame(f1i_run_rows).set_index("set")
assert (f1i_cv[F1I_MODEL_COLS] >= 0).all().all(), "an intermittent forecast is negative - stop and check"

# ---- Report ----
print("STEP 7c cell 12 - intermittent models fitted on weeks before each cut-off (nothing written; weeks 54-66 never used)\n")
print(f"Candidates ({len(F1I_MODEL_COLS)}): {F1I_MODEL_COLS} | guard (never chosen): {F1I_GUARD} | TSB -> cell 13")
print(f"Backtest-intermittent products: {len(f1i_int_ids)} "
      f"({int((f1i_bt.loc[f1i_int_ids, 'choose_by'] == 'CV').sum())} chosen by CV, "
      f"{int((f1i_bt.loc[f1i_int_ids, 'choose_by'] == 'AICc').sum())} short / AICc)")
print("\nPractice rounds (cross-validation, FPP2 3.4):")
display(f1i_runs.T)
print(f"f1i_cv: {len(f1i_cv):,} rows, {f1i_cv['unique_id'].nunique()} series | negative forecasts: 0 | "
      f"missing: {int(f1i_cv[F1I_MODEL_COLS].isna().sum().sum())}")

print(f"\nOne-sale products -> judgement rate, no model (decision 1): {len(f1i_judgement)}")
display(f1i_judgement.round(2))

print(f"\nSparse higher-level series (weeks 1-53): rule used = more than {F1I_SPARSE_SHARE:.0%} zero weeks -> "
      f"{len(f1i_sparse_ids)} series | 7a ADI rule (ADI >= {F1I_ADI_CUT}) would give {len(f1i_adi_ids)} (comparison only)")
display(pd.DataFrame({"zero weeks > 50% (used)": f1i_up.loc[f1i_sparse_ids, "level"].value_counts(),
                      "ADI >= 1.32 (comparison)": f1i_up.loc[f1i_adi_ids, "level"].value_counts()})
        .reindex(list(F1B_LEVELS.values())).fillna(0).astype(int))
f1i_only_adi = sorted(set(f1i_adi_ids) - set(f1i_sparse_ids))
print(f"In the ADI rule but not the 50% rule: {len(f1i_only_adi)} series; their zero share in weeks 1-53: "
      f"{f1i_up.loc[f1i_only_adi, 'zero_share'].min():.0%} to {f1i_up.loc[f1i_only_adi, 'zero_share'].max():.0%}"
      if f1i_only_adi else "In the ADI rule but not the 50% rule: none")
display(f1i_cv.head(3))
print("NEXT: cell 13 - TSB settings 0.1 / 0.2 / 0.3 / 0.5 on the same series and rounds.")


In [ ]:
# ==========================================================================
# STEP 7 - PART 7c, CELL 13: TSB SETTINGS TEST - 0.1 / 0.2 / 0.3 / 0.5, PICKED ON cv1, CHECKED ON cv2
#   From OLD F1 cells 121-122 (tsb_variants + WAPE per setting), rewritten:
#     old: cross_validation(n_windows=2) on ALL weeks -> the 2 windows fell in the test weeks (a leak),
#          old data, WAPE picked the setting, and the winner (0.3) was then used in old cell 125
#     here: the same 4 settings, inside the SAME practice rounds and series as cell 12 (f1i_ids_by_set),
#           RMSSE picks (decision 4), the setting is chosen on cv1 only and then checked on cv2 (decision 2).
#   TSB has two learning speeds: alpha_d (size of a sale) and alpha_p (chance of a sale in a week).
#   As in the old cells, both get the same value.  statsforecast cannot tune them itself.
#   The setting is picked on the PRODUCTS (the intermittent lane); sparse brands are shown for information
#   and use the same setting.  Only the chosen setting goes on to cell 13a, as the column "TSB".
#
# RMSSE (per series and round; decision 4):
#   sqrt( mean over the 13 predicted weeks of (y - forecast)^2  /  mean over that round's TRAINING weeks
#         of (y_t - y_(t-1))^2 ).  Squared errors -> rewards the mean, not 0 (FPP2 3.4 p.79).
#   A series whose training weeks never change (scale 0) cannot be scaled -> left out and counted.
#
# WHY (FPP2 2nd ed., Hyndman & Athanasopoulos 2018, unless named):
#   3.4 (p.77): accuracy "can only be determined by considering how well a model performs on new data
#     that were not used when fitting the model" -> choosing a setting is part of fitting, so it is
#     chosen on cv1 and judged on cv2, and the test weeks 54-66 are never used.
#   3.4 (p.79): "A forecast method that minimises the MAE will lead to forecasts of the median, while
#     minimising the RMSE will lead to forecasts of the mean."
#   Not in FPP2 (named, peer-reviewed): TSB - Teunter, Syntetos & Babai 2011 (Eur. J. Oper. Res.);
#     RMSSE - the M5 competition measure, Makridakis, Spiliotis & Assimakopoulos 2022 (Int. J. Forecasting).
#
# WHERE EACH INPUT COMES FROM (all already in memory; nothing read from disk):
#   f1i_ids_by_set (series per round), f1i_cv (cell 12 table: set / level / unique_id / ds / y) -> cell 12
#   f1_train, f1_weeks, F1_CUTOFF, F1_H (13) -> 7a;  F1B_CV_SETS, f1b_level_of -> 7b
#   StatsForecast -> 7b import;  pd, np -> 7a imports;  display -> Jupyter built-in
#   TSB -> NEW import below (statsforecast 2.0.3)
#
# WHAT IT CHANGES: nothing (f1i_cv and all 7a / 7b tables are only read).  WHAT IT WRITES: nothing.
# All new names start with f1t_ / F1T_.
# ==========================================================================
from statsforecast.models import TSB

F1T_ALPHAS = [0.1, 0.2, 0.3, 0.5]                                     # the 4 settings of old cell 121
F1T_COLS = [f"TSB_{f1t_a:.1f}" for f1t_a in F1T_ALPHAS]               # "TSB_0.1" ... "TSB_0.5"

def f1t_models():
    """TSB at the 4 settings; alpha_d = alpha_p, as in old cell 121."""
    return [TSB(alpha_d=f1t_a, alpha_p=f1t_a, alias=f1t_c) for f1t_a, f1t_c in zip(F1T_ALPHAS, F1T_COLS)]

def f1t_rmsse(f1t_df, f1t_cols, f1t_scale):
    """RMSSE per set x series for each column in f1t_cols. f1t_scale: index (set, unique_id) -> mean sq. change."""
    f1t_sq = f1t_df[["set", "unique_id"]].copy()
    for f1t_c in f1t_cols:
        f1t_sq[f1t_c] = (f1t_df["y"] - f1t_df[f1t_c]) ** 2
    f1t_mse = f1t_sq.groupby(["set", "unique_id"])[f1t_cols].mean()
    return np.sqrt(f1t_mse.div(f1t_scale.reindex(f1t_mse.index).where(lambda s: s > 0), axis=0))

assert set(f1i_ids_by_set) == set(F1B_CV_SETS), "run cell 12 first"

# ---- 1. Forecast with the 4 settings, same rounds and series as cell 12 ----
f1t_parts, f1t_scale_parts = [], []
for f1t_set in F1B_CV_SETS:
    f1t_train_set = f1_train[f1t_set][f1_train[f1t_set]["unique_id"].isin(f1i_ids_by_set[f1t_set])]
    assert (f1t_train_set.groupby("unique_id")["ds"].max() == F1_CUTOFF[f1t_set]).all(), \
        f"{f1t_set}: a series does not end at the cut-off - stop and check"
    f1t_fc = StatsForecast(models=f1t_models(), freq="W-MON", n_jobs=-1).forecast(df=f1t_train_set, h=F1_H)
    f1t_pos = f1_weeks.get_loc(F1_CUTOFF[f1t_set])
    assert set(f1t_fc["ds"]) == set(f1_weeks[f1t_pos + 1: f1t_pos + 1 + F1_H]), f"{f1t_set}: wrong forecast weeks"
    f1t_fc.insert(0, "set", f1t_set)
    f1t_parts.append(f1t_fc)
    # RMSSE scale: mean squared week-to-week change in THIS round's training weeks
    f1t_sc = (f1t_train_set.sort_values(["unique_id", "ds"]).groupby("unique_id")["y"]
              .apply(lambda f1t_y: (f1t_y.diff() ** 2).mean()))
    f1t_scale_parts.append(f1t_sc.rename("scale").reset_index().assign(set=f1t_set))
f1t_cv = (pd.concat(f1t_parts, ignore_index=True)
          .merge(f1i_cv[["set", "level", "unique_id", "ds", "y"]], on=["set", "unique_id", "ds"],
                 how="inner", validate="one_to_one"))
assert len(f1t_cv) == len(f1i_cv), "TSB rows do not match cell 12 rows - stop and check"
assert f1t_cv[F1T_COLS].notna().all().all() and (f1t_cv[F1T_COLS] >= 0).all().all()
f1t_scale = pd.concat(f1t_scale_parts, ignore_index=True).set_index(["set", "unique_id"])["scale"]

# ---- 2. Scores per setting and round: RMSSE (picks), bias (forecast total / real total) ----
f1t_rm = f1t_rmsse(f1t_cv, F1T_COLS, f1t_scale)
f1t_rm["level"] = f1t_rm.index.get_level_values("unique_id").map(f1b_level_of)
f1t_no_scale = int(f1t_rm[F1T_COLS].isna().all(axis=1).sum())        # scale 0 -> left out
f1t_rows = []
for f1t_set in F1B_CV_SETS:
    for f1t_lev in ["Product", "Brand"]:
        f1t_r = f1t_rm.xs(f1t_set, level="set")
        f1t_r = f1t_r[f1t_r["level"] == f1t_lev]
        f1t_d = f1t_cv[(f1t_cv["set"] == f1t_set) & (f1t_cv["level"] == f1t_lev)]
        for f1t_c in F1T_COLS:
            f1t_rows.append({"set": f1t_set, "level": f1t_lev, "setting": f1t_c,
                             "series scored": int(f1t_r[f1t_c].notna().sum()),
                             "mean RMSSE": f1t_r[f1t_c].mean(),
                             "bias (forecast total / real total)": f1t_d[f1t_c].sum() / f1t_d["y"].sum()})
f1t_scores = pd.DataFrame(f1t_rows)

# ---- 3. Pick on cv1 (products), check on cv2 ----
f1t_prod = f1t_scores[f1t_scores["level"] == "Product"].pivot(index="setting", columns="set", values="mean RMSSE")
F1T_BEST = f1t_prod["cv1"].idxmin()                                   # chosen on cv1 only
f1t_cv2_rank = int(f1t_prod["cv2"].rank().loc[F1T_BEST])               # 1 = also best on cv2
f1t_cv2_best = f1t_prod["cv2"].idxmin()

# ---- Report ----
print("STEP 7c cell 13 - TSB settings test inside cv1 / cv2 (nothing written; weeks 54-66 never used)\n")
print(f"Settings: {F1T_COLS} (alpha_d = alpha_p) | f1t_cv: {len(f1t_cv):,} rows = the same rows as cell 12 | "
      f"series with scale 0 (left out of RMSSE): {f1t_no_scale}")
print("\nMean RMSSE per setting (lower is better) and bias (1.00 = right total):")
display(f1t_scores.pivot_table(index="setting", columns=["level", "set"],
                               values=["mean RMSSE", "bias (forecast total / real total)"])
        .reindex(columns=["Product", "Brand"], level="level").round(3))
print(f"Chosen on cv1 (products): {F1T_BEST}  (mean RMSSE {f1t_prod.loc[F1T_BEST, 'cv1']:.3f})")
print(f"Check on cv2: {F1T_BEST} ranks {f1t_cv2_rank} of {len(F1T_COLS)} (RMSSE {f1t_prod.loc[F1T_BEST, 'cv2']:.3f}); "
      f"best on cv2 was {f1t_cv2_best} ({f1t_prod.loc[f1t_cv2_best, 'cv2']:.3f})")
print(f"-> {F1T_BEST} goes to cell 13a as the column 'TSB' (the cv1 choice is kept, as decided).")
print("NEXT: cell 13a - scores for all intermittent models, one method per lane, sparse brands re-picked.")


In [ ]:
# ==========================================================================
# STEP 7 - PART 7c, CELL 13a: SCORES + CHOICES - ONE METHOD FOR THE INTERMITTENT LANE (cv1 -> cv2),
#                             SPARSE BRANDS RE-PICKED FROM ALL 18 MODELS, LANE CHECK (option B)
#   New cell (no old F1 cell; it replaces the old habit of picking per product on the test weeks).
#   Your decisions 3, 4, 5 and 6:
#     3  ONE method for all intermittent products: the candidate with the lowest mean RMSSE over the lane
#        products in cv1.  Check on cv2: it must still beat Mean and Zero there.  If it does not, it is
#        shown to you to decide (nothing is changed automatically).
#     4  RMSSE picks; MASE shown only; bias = forecast total / real total; Zero is a guard, never chosen.
#     5  each sparse brand gets a new winner from ALL 18 models (12 from 7b + 6 new from 7c; Mean is in
#        both and is the same model, so it is counted once), lowest RMSSE
#        averaged over the rounds it has.  It replaces the 7b winner in later steps; 7b is not changed.
#     6  option B: for products that were "standard" in a round, the lane method is compared with their
#        best standard model on the same weeks.  Flagged only if standard is better by 20% or more in
#        BOTH rounds and its bias is between 0.75 and 1.25.  You decide; nothing switches automatically.
#
# WHY (FPP2 2nd ed., Hyndman & Athanasopoulos 2018, unless named):
#   3.4 (p.77): accuracy "can only be determined by considering how well a model performs on new data
#     that were not used when fitting the model" -> the lane method is chosen on cv1 and judged on cv2.
#   3.4 (p.79): "minimising the RMSE will lead to forecasts of the mean" -> RMSSE picks (zero trap).
#   3.4 (pp.80-81): MASE (shown, as in 7b).   3.1 (p.60): simple methods are the bar to beat (Mean).
#   10.6 (p.414): base forecasts for every series -> the sparse brands keep a base forecast, re-chosen.
#   Not in FPP2 (named, peer-reviewed): RMSSE - M5 competition, Makridakis, Spiliotis & Assimakopoulos
#     2022 (Int. J. Forecasting); one method for many series - Fildes & Petropoulos 2015 (J. Business Research).
#   The 20% margin and the 0.75-1.25 bias band (decision 6) are my suggestions, not textbook rules.
#
# WHERE EACH INPUT COMES FROM (all already in memory; nothing read from disk):
#   f1i_cv, F1I_MODEL_COLS, F1I_GUARD, f1i_sparse_ids, f1i_judgement -> cell 12
#   f1t_cv, F1T_BEST, f1t_scale (RMSSE scale per set x series), f1t_rmsse() -> cell 13
#   f1l_all, F1L_ALL_MODELS (12 standard + log columns, practice rounds) -> 7b cell #11
#   f1e_scale (MASE scale per set x series), f1e_win (7b winners) -> 7b cell #6
#   F1B_CV_SETS -> 7b;  f1_labels (SKU names) -> 7a;  pd, np -> 7a imports;  display -> Jupyter
#
# WHAT IT CHANGES: nothing (all earlier tables are only read).  WHAT IT WRITES: nothing (memory only).
# All new names start with f1k_ / F1K_.
# ==========================================================================
F1K_INT = F1I_MODEL_COLS + ["TSB"]                    # the 7 intermittent candidates (TSB = F1T_BEST)
F1K_ALL = F1L_ALL_MODELS + [f1k_c for f1k_c in F1K_INT if f1k_c not in F1L_ALL_MODELS]  # 18 (Mean once)
F1K_MARGIN = 0.20                                     # decision 6: standard must be >= 20% better
F1K_BIAS_BAND = (0.75, 1.25)                          # decision 6: "sensible" bias for the standard model
assert F1T_BEST in f1t_cv.columns and len(F1K_ALL) == len(set(F1K_ALL)) == 18, "run cells 12 and 13 first"

def f1k_mase(f1k_df, f1k_cols):
    """MASE per set x series (shown only): mean |error| / mean |weekly change| in training (7b f1e_scale)."""
    f1k_ab = f1k_df[["set", "unique_id"]].copy()
    for f1k_c in f1k_cols:
        f1k_ab[f1k_c] = (f1k_df["y"] - f1k_df[f1k_c]).abs()
    f1k_mae = f1k_ab.groupby(["set", "unique_id"])[f1k_cols].mean()
    f1k_sc = f1e_scale.set_index(["set", "unique_id"])["scale"].reindex(f1k_mae.index)
    return f1k_mae.div(f1k_sc.where(f1k_sc > 0), axis=0)

# ---- 1. One table: the 6 cell-12 candidates + Zero + the chosen TSB setting ----
f1k_cv = f1i_cv.merge(f1t_cv[["set", "unique_id", "ds", F1T_BEST]].rename(columns={F1T_BEST: "TSB"}),
                      on=["set", "unique_id", "ds"], how="inner", validate="one_to_one")
assert len(f1k_cv) == len(f1i_cv) and f1k_cv["TSB"].notna().all()
f1k_rm = f1t_rmsse(f1k_cv, F1K_INT + [F1I_GUARD], f1t_scale)          # RMSSE per set x series
f1k_ms = f1k_mase(f1k_cv, F1K_INT + [F1I_GUARD])                       # MASE per set x series
f1k_level = f1k_cv.drop_duplicates(["set", "unique_id"]).set_index(["set", "unique_id"])["level"]

# ---- 2. Decision 3: one method for the intermittent lane (products), picked on cv1, checked on cv2 ----
f1k_lane_rows = []
for f1k_set in F1B_CV_SETS:
    f1k_ix = f1k_level[(f1k_level == "Product") & (f1k_level.index.get_level_values("set") == f1k_set)].index
    f1k_d = f1k_cv[(f1k_cv["set"] == f1k_set) & (f1k_cv["level"] == "Product")]
    for f1k_c in F1K_INT + [F1I_GUARD]:
        f1k_lane_rows.append({"set": f1k_set, "model": f1k_c, "products": len(f1k_ix),
                              "mean RMSSE (picks)": f1k_rm.loc[f1k_ix, f1k_c].mean(),
                              "mean MASE (shown)": f1k_ms.loc[f1k_ix, f1k_c].mean(),
                              "bias (forecast total / real total)": f1k_d[f1k_c].sum() / f1k_d["y"].sum()})
f1k_lane = pd.DataFrame(f1k_lane_rows)
f1k_lane_rm = f1k_lane.pivot(index="model", columns="set", values="mean RMSSE (picks)")
F1K_LANE_METHOD = f1k_lane_rm.loc[F1K_INT, "cv1"].idxmin()            # chosen on cv1 only; Zero excluded
f1k_cv2 = f1k_lane_rm["cv2"]
f1k_beats = {f1k_r: bool(f1k_cv2[F1K_LANE_METHOD] < f1k_cv2[f1k_r])
             for f1k_r in ["Mean", F1I_GUARD] if f1k_r != F1K_LANE_METHOD}
F1K_LANE_CONFIRMED = all(f1k_beats.values())                         # True -> used; False -> your decision

# ---- 3. Decision 5: sparse brands re-picked from all 18 models (RMSSE averaged over their rounds) ----
f1k_new = [f1k_c for f1k_c in F1K_INT if f1k_c not in F1L_ALL_MODELS]          # the 6 not already in 7b
f1k_sp = f1k_cv[f1k_cv["unique_id"].isin(f1i_sparse_ids)][["set", "unique_id", "ds", "y"] + f1k_new]
f1k_sp = f1k_sp.merge(f1l_all[["set", "unique_id", "ds"] + F1L_ALL_MODELS], on=["set", "unique_id", "ds"],
                      how="inner", validate="one_to_one")
assert len(f1k_sp) == int(f1k_cv["unique_id"].isin(f1i_sparse_ids).sum()), "a sparse brand has no 7b forecast"
# Mean is the same model in 7b and 7c: check the two columns agree before using the 7b one
assert np.allclose(f1k_sp["Mean"].values, f1k_cv[f1k_cv["unique_id"].isin(f1i_sparse_ids)]
                   .merge(f1k_sp[["set", "unique_id", "ds"]], on=["set", "unique_id", "ds"])["Mean"].values)
f1k_sp_rm = f1t_rmsse(f1k_sp, F1K_ALL, f1t_scale)
f1k_sp_avg = f1k_sp_rm.groupby(level="unique_id").mean()
f1k_sp_tot = f1k_sp.groupby("unique_id")[["y"] + F1K_ALL].sum()                   # 13-week totals, both rounds
f1k_old = f1e_win.set_index("unique_id")["winner"]
f1k_sparse_win = pd.DataFrame({"rounds": f1k_sp_rm.groupby(level="unique_id").size(),
                               "old winner (7b, MASE)": f1k_old.reindex(f1k_sp_avg.index),
                               "new winner (RMSSE, 18 models)": f1k_sp_avg.idxmin(axis=1)})
f1k_sparse_win["new winner RMSSE"] = [f1k_sp_avg.at[f1k_u, f1k_w] for f1k_u, f1k_w in f1k_sparse_win["new winner (RMSSE, 18 models)"].items()]
f1k_sparse_win["old winner RMSSE"] = [f1k_sp_avg.at[f1k_u, f1k_w] for f1k_u, f1k_w in f1k_sparse_win["old winner (7b, MASE)"].items()]
f1k_ysum = f1k_sp_tot["y"].where(f1k_sp_tot["y"] > 0)                            # blank if the brand sold 0
f1k_sparse_win["old bias"] = [f1k_sp_tot.at[f1k_u, f1k_w] / f1k_ysum[f1k_u] for f1k_u, f1k_w in f1k_sparse_win["old winner (7b, MASE)"].items()]
f1k_sparse_win["new bias"] = [f1k_sp_tot.at[f1k_u, f1k_w] / f1k_ysum[f1k_u] for f1k_u, f1k_w in f1k_sparse_win["new winner (RMSSE, 18 models)"].items()]
f1k_sparse_win["real sales (both rounds)"] = f1k_sp_tot["y"].reindex(f1k_sparse_win.index).values
f1k_sparse_win = f1k_sparse_win.rename_axis("unique_id").reset_index()

# ---- 4. Decision 6 (option B): lane check for products that were "standard" in a round ----
F1K_LANE_COL = "lane method"                          # the lane method's column, renamed so it cannot clash with 7b
f1k_std = (f1k_cv[f1k_cv["lane_this_round"] == "standard"][["set", "unique_id", "ds", "y", F1K_LANE_METHOD]]
           .rename(columns={F1K_LANE_METHOD: F1K_LANE_COL}))
f1k_std = f1k_std.merge(f1l_all[["set", "unique_id", "ds"] + F1L_ALL_MODELS], on=["set", "unique_id", "ds"],
                        how="inner", validate="one_to_one")
f1k_std_rm = f1t_rmsse(f1k_std, [F1K_LANE_COL] + F1L_ALL_MODELS, f1t_scale)
f1k_best_std = f1k_std_rm[F1L_ALL_MODELS].groupby(level="unique_id").mean().idxmin(axis=1)   # best standard model
f1k_check_rows = []
for (f1k_set, f1k_u), f1k_r in f1k_std_rm.iterrows():
    f1k_m = f1k_best_std[f1k_u]
    f1k_w = f1k_std[(f1k_std["set"] == f1k_set) & (f1k_std["unique_id"] == f1k_u)]
    f1k_check_rows.append({"unique_id": f1k_u, "set": f1k_set, "best standard model": f1k_m,
                           f"RMSSE lane method ({F1K_LANE_METHOD})": f1k_r[F1K_LANE_COL],
                           "RMSSE best standard": f1k_r[f1k_m],
                           "standard better by 20%+": bool(f1k_r[f1k_m] <= (1 - F1K_MARGIN) * f1k_r[F1K_LANE_COL]),
                           "standard bias": f1k_w[f1k_m].sum() / f1k_w["y"].sum() if f1k_w["y"].sum() > 0 else np.nan})
f1k_lane_check = pd.DataFrame(f1k_check_rows)
f1k_flag = f1k_lane_check.groupby("unique_id").agg(
    rounds=("set", "size"), better_in=("standard better by 20%+", "sum"),
    bias_ok=("standard bias", lambda b: bool(b.between(*F1K_BIAS_BAND).all())))
f1k_flag["flag (you decide)"] = (f1k_flag["rounds"] == 2) & (f1k_flag["better_in"] == 2) & f1k_flag["bias_ok"]
f1k_flag = f1k_flag.join(f1_labels.set_index("unique_id")["SKU"]).reset_index()

# ---- Report ----
print("STEP 7c cell 13a - scores and choices on the practice rounds (nothing written; weeks 54-66 never used)\n")
print(f"Candidates for the intermittent lane ({len(F1K_INT)}): {F1K_INT}  (TSB = {F1T_BEST}) | guard: {F1I_GUARD}")
print("\nDecision 3 + 4 - intermittent products, per model and round (RMSSE picks; lower is better; bias 1.00 = right total):")
display(f1k_lane.pivot_table(index="model", columns="set",
                             values=["mean RMSSE (picks)", "mean MASE (shown)", "bias (forecast total / real total)"])
        .loc[F1K_INT + [F1I_GUARD]].round(3))
print(f"Chosen on cv1: {F1K_LANE_METHOD} (mean RMSSE {f1k_lane_rm.loc[F1K_LANE_METHOD, 'cv1']:.3f})")
print(f"Check on cv2: RMSSE {f1k_cv2[F1K_LANE_METHOD]:.3f} | " +
      " | ".join(f"beats {f1k_r} ({f1k_cv2[f1k_r]:.3f}): {'yes' if f1k_b else 'NO'}" for f1k_r, f1k_b in f1k_beats.items()))
print("-> CONFIRMED: " + F1K_LANE_METHOD + " is the method for all intermittent products (7g)." if F1K_LANE_CONFIRMED else
      "-> NOT CONFIRMED on cv2 - your decision is needed before 7g (see the table above).")
print(f"   ({len(f1i_judgement)} one-sale products keep their judgement rate from cell 12.)")

print(f"\nDecision 5 - sparse brands ({len(f1k_sparse_win)}): new winner from {len(F1K_ALL)} models by RMSSE "
      f"(replaces the 7b winner later; 7b unchanged):")
display(f1k_sparse_win.round(3))
print(f"Brands that sold nothing in their scored weeks (bias blank; any forecast of 0 is exact there): "
      f"{int((f1k_sparse_win['real sales (both rounds)'] == 0).sum())}")
print(f"Winner changed: {int((f1k_sparse_win['new winner (RMSSE, 18 models)'] != f1k_sparse_win['old winner (7b, MASE)']).sum())} "
      f"of {len(f1k_sparse_win)} | "
      f"median bias old {f1k_sparse_win['old bias'].median():.2f} -> new {f1k_sparse_win['new bias'].median():.2f}")

print(f"\nDecision 6 (option B) - products that were 'standard' in a round ({f1k_flag.shape[0]}): "
      f"flag = standard better by {F1K_MARGIN:.0%}+ in BOTH rounds and bias {F1K_BIAS_BAND[0]}-{F1K_BIAS_BAND[1]}")
display(f1k_lane_check.round(3))
display(f1k_flag)
print(f"Flagged for your decision: {int(f1k_flag['flag (you decide)'].sum())} (nothing is switched automatically)")
print("NEXT: cell 13b - the chosen forecast for every 7c series, ready for 7e and 7g.")


In [ ]:
# ==========================================================================
# STEP 7 - PART 7c, CELL 13b: THE CHOSEN FORECAST FOR EVERY 7c SERIES (cv1 & cv2) + THE PLAN FOR 7g
#   New cell - the 7c twin of 7b cell #8 (f1w_cv).  It puts one chosen forecast per series and week
#   into one table, for 7e (combinations) and 7g (backtest + reconciliation inside the same CV windows).
#   Your decision 3 (made after cell 13a): KEEP TSB for all intermittent products, although the cv2 check
#   against Zero failed (TSB 0.676, Zero 0.619).  TSB + ADIDA and TSB + ADIDA + IMAPA are tested in 7e;
#   one of them replaces TSB only if it wins on cv1 and holds on cv2.
#   Who gets what (the same rules in the practice rounds and in 7g):
#     - intermittent products (backtest lane)          -> TSB (setting F1T_BEST from cell 13)
#     - the one-sale products (decision 1)              -> judgement rate = total sold / weeks since launch.
#       In each practice round the rate uses ONLY that round's training weeks (no peeking); in 7g it is
#       the cell-12 rate from weeks 1-53.
#     - the sparse brands (decision 5)                  -> their own winner from cell 13a (18 models)
#   Products with fewer than 13 weeks of history in BOTH practice rounds have no practice-round rows; they
#   are in the 7g plan only (same rules).
#
# WHY (FPP2 2nd ed., Hyndman & Athanasopoulos 2018):
#   10.6 (p.414): "Suppose we forecast all series independently ... We call these the base forecasts"
#     -> one chosen base forecast per series.
#   3.4 (p.77): the practice-round forecasts use only the weeks before each cut-off (also for the rate).
#   Ch 4 (p.101): judgmental forecasting when there is "a complete lack of historical data".
#   12.4 (p.472): "combining multiple forecasts leads to increased forecast accuracy" -> tested in 7e.
#
# WHERE EACH INPUT COMES FROM (all already in memory; nothing read from disk):
#   f1k_cv (7 candidates + y per set x series x week), f1k_sp (sparse brands, 18 models), f1k_sparse_win,
#   F1K_LANE_METHOD, F1K_LANE_CONFIRMED -> cell 13a
#   F1T_BEST -> cell 13;  f1i_int_ids, f1i_judgement, f1i_bt (backtest routes) -> cell 12
#   f1_train (training weeks per round) -> 7a;  f1b_level_of, F1B_CV_SETS -> 7b
#   pd, np -> 7a imports;  display -> Jupyter built-in
#
# WHAT IT CHANGES: nothing (all earlier tables are only read).  WHAT IT WRITES: nothing (memory only).
# All new names start with f1q_ / F1Q_.
# ==========================================================================
F1Q_LANE_METHOD = "TSB"                                # your decision 3: keep TSB (setting F1T_BEST)
F1Q_TEST_IN_7E = ["TSB + ADIDA", "TSB + ADIDA + IMAPA"]  # your decision 3: combinations to test in 7e
assert F1K_LANE_METHOD == F1Q_LANE_METHOD and not F1K_LANE_CONFIRMED, "cell 13a result changed - stop and check"

f1q_one = set(f1i_judgement["unique_id"])                                      # 8 one-sale products
f1q_win = f1k_sparse_win.set_index("unique_id")["new winner (RMSSE, 18 models)"]  # 15 sparse brands

# ---- 1. Practice rounds: one chosen forecast per series and week ----
f1q_parts = []
# (a) intermittent products except the one-sale ones -> TSB
f1q_a = f1k_cv[(f1k_cv["level"] == "Product") & ~f1k_cv["unique_id"].isin(f1q_one)].copy()
f1q_a["chosen_model"], f1q_a["chosen"] = F1Q_LANE_METHOD + f" ({F1T_BEST})", f1q_a["TSB"]
f1q_parts.append(f1q_a)
# (b) one-sale products -> judgement rate from THIS round's training weeks only
f1q_b = f1k_cv[f1k_cv["unique_id"].isin(f1q_one)].copy()
f1q_rate = pd.concat([f1_train[f1q_s][f1_train[f1q_s]["unique_id"].isin(f1q_one)].groupby("unique_id")["y"]
                      .mean().rename("rate").reset_index().assign(set=f1q_s) for f1q_s in F1B_CV_SETS])
f1q_b = f1q_b.merge(f1q_rate, on=["set", "unique_id"], how="left", validate="many_to_one")
assert f1q_b["rate"].notna().all()
f1q_b["chosen_model"], f1q_b["chosen"] = "judgement rate", f1q_b["rate"]
f1q_parts.append(f1q_b.drop(columns="rate"))
# (c) sparse brands -> their cell-13a winner (any of the 18 models)
f1q_c = f1k_sp.merge(f1k_cv[["set", "unique_id", "ds", "level", "cutoff"]], on=["set", "unique_id", "ds"],
                     how="left", validate="one_to_one")
f1q_c["chosen_model"] = f1q_c["unique_id"].map(f1q_win)
f1q_c["chosen"] = [f1q_r[f1q_m] for f1q_m, (_, f1q_r) in zip(f1q_c["chosen_model"], f1q_c.iterrows())]
f1q_parts.append(f1q_c)

f1q_cols = ["set", "level", "unique_id", "ds", "cutoff", "y", "chosen_model", "chosen"]
f1q_cv = pd.concat([f1q_p[f1q_cols] for f1q_p in f1q_parts], ignore_index=True)
assert len(f1q_cv) == len(f1k_cv), "a 7c row is missing or counted twice"
assert not f1q_cv.duplicated(["set", "unique_id", "ds"]).any()
# log versions (7b #11) can dip a little below 0 but never below -1 case: counted, NOT clipped (as in 7b)
assert f1q_cv["chosen"].notna().all() and (f1q_cv["chosen"] > -1).all(), "a chosen forecast is blank or below -1"
f1q_neg = f1q_cv[f1q_cv["chosen"] < 0]
f1q_cv["source"] = np.where(f1q_cv["level"] == "Product",
                            np.where(f1q_cv["unique_id"].isin(f1q_one), "7c judgement rate", "7c lane method"),
                            "7c sparse-brand winner")

# ---- 2. The plan for 7g (backtest: learn weeks 1-53, forecast 54-66) ----
f1q_plan = pd.concat([
    pd.DataFrame({"unique_id": [u for u in f1i_int_ids if u not in f1q_one], "method in 7g": F1Q_LANE_METHOD,
                  "setting / rate": F1T_BEST}),
    f1i_judgement[["unique_id", "rate_per_week"]].rename(columns={"rate_per_week": "setting / rate"})
        .assign(**{"method in 7g": "judgement rate (please review)"}),
    pd.DataFrame({"unique_id": f1q_win.index, "method in 7g": f1q_win.values, "setting / rate": "-"})],
    ignore_index=True)
f1q_plan["level"] = f1q_plan["unique_id"].map(f1b_level_of)
f1q_plan["choose_by (7a)"] = f1q_plan["unique_id"].map(f1i_bt["choose_by"]).fillna("-")
f1q_plan["in practice rounds"] = f1q_plan["unique_id"].isin(set(f1q_cv["unique_id"]))
f1q_plan = f1q_plan[["level", "unique_id", "choose_by (7a)", "method in 7g", "setting / rate", "in practice rounds"]]
assert f1q_plan["unique_id"].is_unique and len(f1q_plan) == len(f1i_int_ids) + len(f1q_win)

# ---- Report ----
print("STEP 7c cell 13b - the chosen forecast per 7c series, practice rounds only (nothing written)\n")
print(f"Decision 3: {F1Q_LANE_METHOD} ({F1T_BEST}) kept for the intermittent lane | to test in 7e: {F1Q_TEST_IN_7E}")
print(f"f1q_cv: {len(f1q_cv):,} rows, {f1q_cv['unique_id'].nunique()} series | blank: 0 | "
      f"chosen below 0: {len(f1q_neg)} rows (lowest {f1q_cv['chosen'].min():.2f} cases; "
      f"{', '.join(sorted(set(f1q_neg['unique_id'] + ' / ' + f1q_neg['chosen_model']))) or 'none'}) - not clipped, as in 7b")
display(f1q_cv.groupby(["set", "source"]).agg(series=("unique_id", "nunique"), real_sales=("y", "sum"),
                                              chosen_forecast=("chosen", "sum"))
        .assign(**{"bias (forecast / real)": lambda d: (d["chosen_forecast"] / d["real_sales"].where(d["real_sales"] > 0)).round(2)})
        .round(1).unstack("set"))
print(f"\nPlan for 7g ({len(f1q_plan)} series): who gets which method")
display(f1q_plan.groupby(["level", "method in 7g"]).size().rename("series").to_frame())
print(f"  {int((~f1q_plan['in practice rounds']).sum())} of them have no practice-round rows "
      f"(fewer than 13 weeks in both rounds) - they follow the same plan in 7g.")
print("  (bias is blank where real sales were 0 - e.g. the one-sale products sold nothing in those weeks)")
display(f1q_cv.head(3))
print("NEXT: Part 7d - holidays.")


In [ ]:
# ==========================================================================
# STEP 7 - PART 7c, CELL 13b: THE CHOSEN FORECAST FOR EVERY 7c SERIES (cv1 & cv2) + THE PLAN FOR 7g
#   New cell - the 7c twin of 7b cell #8 (f1w_cv).  It puts one chosen forecast per series and week
#   into one table, for 7e (combinations) and 7g (backtest + reconciliation inside the same CV windows).
#   Your decision 3 (made after cell 13a): KEEP TSB for all intermittent products, although the cv2 check
#   against Zero failed (TSB 0.676, Zero 0.619).  TSB + ADIDA and TSB + ADIDA + IMAPA are tested in 7e;
#   one of them replaces TSB only if it wins on cv1 and holds on cv2.
#   Who gets what (the same rules in the practice rounds and in 7g):
#     - intermittent products (backtest lane)          -> TSB (setting F1T_BEST from cell 13)
#     - the one-sale products (decision 1)              -> judgement rate = total sold / weeks since launch.
#       In each practice round the rate uses ONLY that round's training weeks (no peeking); in 7g it is
#       the cell-12 rate from weeks 1-53.
#     - the sparse brands (decision 5)                  -> their own winner from cell 13a (18 models)
#   Products with fewer than 13 weeks of history in BOTH practice rounds have no practice-round rows; they
#   are in the 7g plan only (same rules).
#
# WHY (FPP2 2nd ed., Hyndman & Athanasopoulos 2018):
#   10.6 (p.414): "Suppose we forecast all series independently ... We call these the base forecasts"
#     -> one chosen base forecast per series.
#   3.4 (p.77): the practice-round forecasts use only the weeks before each cut-off (also for the rate).
#   Ch 4 (p.101): judgmental forecasting when there is "a complete lack of historical data".
#   12.4 (p.472): "combining multiple forecasts leads to increased forecast accuracy" -> tested in 7e.
#
# WHERE EACH INPUT COMES FROM (all already in memory; nothing read from disk):
#   f1k_cv (7 candidates + y per set x series x week), f1k_sp (sparse brands, 18 models), f1k_sparse_win,
#   F1K_LANE_METHOD, F1K_LANE_CONFIRMED -> cell 13a
#   F1T_BEST -> cell 13;  f1i_int_ids, f1i_judgement, f1i_bt (backtest routes) -> cell 12
#   f1_train (training weeks per round) -> 7a;  f1b_level_of, F1B_CV_SETS -> 7b
#   pd, np -> 7a imports;  display -> Jupyter built-in
#
# WHAT IT CHANGES: nothing (all earlier tables are only read).  WHAT IT WRITES: nothing (memory only).
# All new names start with f1q_ / F1Q_.
# ==========================================================================
F1Q_LANE_METHOD = "TSB"                                # your decision 3: keep TSB (setting F1T_BEST)
F1Q_TEST_IN_7E = ["TSB + ADIDA", "TSB + ADIDA + IMAPA"]  # your decision 3: combinations to test in 7e
assert F1K_LANE_METHOD == F1Q_LANE_METHOD and not F1K_LANE_CONFIRMED, "cell 13a result changed - stop and check"

f1q_one = set(f1i_judgement["unique_id"])                                      # 8 one-sale products
f1q_win = f1k_sparse_win.set_index("unique_id")["new winner (RMSSE, 18 models)"]  # 15 sparse brands

# ---- 1. Practice rounds: one chosen forecast per series and week ----
f1q_parts = []
# (a) intermittent products except the one-sale ones -> TSB
f1q_a = f1k_cv[(f1k_cv["level"] == "Product") & ~f1k_cv["unique_id"].isin(f1q_one)].copy()
f1q_a["chosen_model"], f1q_a["chosen"] = F1Q_LANE_METHOD + f" ({F1T_BEST})", f1q_a["TSB"]
f1q_parts.append(f1q_a)
# (b) one-sale products -> judgement rate from THIS round's training weeks only
f1q_b = f1k_cv[f1k_cv["unique_id"].isin(f1q_one)].copy()
f1q_rate = pd.concat([f1_train[f1q_s][f1_train[f1q_s]["unique_id"].isin(f1q_one)].groupby("unique_id")["y"]
                      .mean().rename("rate").reset_index().assign(set=f1q_s) for f1q_s in F1B_CV_SETS])
f1q_b = f1q_b.merge(f1q_rate, on=["set", "unique_id"], how="left", validate="many_to_one")
assert f1q_b["rate"].notna().all()
f1q_b["chosen_model"], f1q_b["chosen"] = "judgement rate", f1q_b["rate"]
f1q_parts.append(f1q_b.drop(columns="rate"))
# (c) sparse brands -> their cell-13a winner (any of the 18 models)
f1q_c = f1k_sp.merge(f1k_cv[["set", "unique_id", "ds", "level", "cutoff"]], on=["set", "unique_id", "ds"],
                     how="left", validate="one_to_one")
f1q_c["chosen_model"] = f1q_c["unique_id"].map(f1q_win)
f1q_c["chosen"] = [f1q_r[f1q_m] for f1q_m, (_, f1q_r) in zip(f1q_c["chosen_model"], f1q_c.iterrows())]
f1q_parts.append(f1q_c)

f1q_cols = ["set", "level", "unique_id", "ds", "cutoff", "y", "chosen_model", "chosen"]
f1q_cv = pd.concat([f1q_p[f1q_cols] for f1q_p in f1q_parts], ignore_index=True)
assert len(f1q_cv) == len(f1k_cv), "a 7c row is missing or counted twice"
assert not f1q_cv.duplicated(["set", "unique_id", "ds"]).any()
# log versions (7b #11) can dip a little below 0 but never below -1 case: counted, NOT clipped (as in 7b)
assert f1q_cv["chosen"].notna().all() and (f1q_cv["chosen"] > -1).all(), "a chosen forecast is blank or below -1"
f1q_neg = f1q_cv[f1q_cv["chosen"] < 0]
f1q_cv["source"] = np.where(f1q_cv["level"] == "Product",
                            np.where(f1q_cv["unique_id"].isin(f1q_one), "7c judgement rate", "7c lane method"),
                            "7c sparse-brand winner")

# ---- 2. The plan for 7g (backtest: learn weeks 1-53, forecast 54-66) ----
f1q_plan = pd.concat([
    pd.DataFrame({"unique_id": [u for u in f1i_int_ids if u not in f1q_one], "method in 7g": F1Q_LANE_METHOD,
                  "setting / rate": F1T_BEST}),
    f1i_judgement[["unique_id", "rate_per_week"]].rename(columns={"rate_per_week": "setting / rate"})
        .assign(**{"method in 7g": "judgement rate (please review)"}),
    pd.DataFrame({"unique_id": f1q_win.index, "method in 7g": f1q_win.values, "setting / rate": "-"})],
    ignore_index=True)
f1q_plan["level"] = f1q_plan["unique_id"].map(f1b_level_of)
f1q_plan["choose_by (7a)"] = f1q_plan["unique_id"].map(f1i_bt["choose_by"]).fillna("-")
f1q_plan["in practice rounds"] = f1q_plan["unique_id"].isin(set(f1q_cv["unique_id"]))
f1q_plan = f1q_plan[["level", "unique_id", "choose_by (7a)", "method in 7g", "setting / rate", "in practice rounds"]]
assert f1q_plan["unique_id"].is_unique and len(f1q_plan) == len(f1i_int_ids) + len(f1q_win)

# ---- Report ----
print("STEP 7c cell 13b - the chosen forecast per 7c series, practice rounds only (nothing written)\n")
print(f"Decision 3: {F1Q_LANE_METHOD} ({F1T_BEST}) kept for the intermittent lane | to test in 7e: {F1Q_TEST_IN_7E}")
print(f"f1q_cv: {len(f1q_cv):,} rows, {f1q_cv['unique_id'].nunique()} series | blank: 0 | "
      f"chosen below 0: {len(f1q_neg)} rows (lowest {f1q_cv['chosen'].min():.2f} cases; "
      f"{', '.join(sorted(set(f1q_neg['unique_id'] + ' / ' + f1q_neg['chosen_model']))) or 'none'}) - not clipped, as in 7b")
display(f1q_cv.groupby(["set", "source"]).agg(series=("unique_id", "nunique"), real_sales=("y", "sum"),
                                              chosen_forecast=("chosen", "sum"))
        .assign(**{"bias (forecast / real)": lambda d: (d["chosen_forecast"] / d["real_sales"].where(d["real_sales"] > 0)).round(2)})
        .round(1).unstack("set"))
print(f"\nPlan for 7g ({len(f1q_plan)} series): who gets which method")
display(f1q_plan.groupby(["level", "method in 7g"]).size().rename("series").to_frame())
print(f"  {int((~f1q_plan['in practice rounds']).sum())} of them have no practice-round rows "
      f"(fewer than 13 weeks in both rounds) - they follow the same plan in 7g.")
print("  (bias is blank where real sales were 0 - e.g. the one-sale products sold nothing in those weeks)")
display(f1q_cv.head(3))
print("NEXT: Part 7d - holidays.")


In [ ]:
# ==========================================================================
# STEP 7 - PART 7d, CELL 14: HOLIDAYS - AutoARIMA WITH 2 HOLIDAY FLAGS (option A1), TESTED ON cv1 & cv2
#   New main code for 7d (the old F1 had no holiday model; old cell 81's idea is rebuilt in cell 15).
#   Option A1: ONE "holiday week" flag (1 = the week contains a Kenyan public holiday) and ONE
#   "week after a holiday" flag, the same for every series.  AutoARIMA learns, for each series, how much
#   these weeks add or take away (regression with ARIMA errors).  It is a NEW CANDIDATE: it is scored on
#   cv1 and cv2 against plain AutoARIMA and against the series' 7b winner; it is used later only where it wins.
#   Which series: the same standard series as 7b (standard products + higher levels with >= 13 weeks),
#   minus the 15 sparse brands of 7c.  Intermittent products get no holiday flags (too few sales; TSB
#   cannot take flags - FPP2 p.465).
#   The flags come from the Prep F9 file (calendar holidays, weeks 1-53, loaded in 7a).  Only the CALENDAR
#   columns are used (is_holiday_week, after_holiday_week); columns worked out from sales are not.
#   Known limit: one average effect for all holidays (Christmas is bigger, some holidays smaller).
#
# WHY (FPP2 2nd ed., Hyndman & Athanasopoulos 2018):
#   12.1 (p.465): "The best way to deal with moving holiday effects is to use dummy variables. However,
#     neither STL, ETS nor TBATS models allow for covariates ... the only choice is a dynamic regression model"
#   5.4 (p.152): a holiday is a "dummy variable" which "takes value 1 corresponding to 'yes' and 0 ... 'no'"
#   5.6 (p.170): holiday indicators "are all known in advance, as they are based on calendar variables"
#     -> using the flags of the predicted weeks is not a leak.
#   9.1-9.2 (p.366): regression with ARIMA errors.   3.4 (p.77, pp.80-81): CV rounds, MASE (as 7b picks).
#   12.7 (p.479): "we need more observations than there are parameters" -> 2 flags, not 1 per holiday.
#
# WHERE EACH INPUT COMES FROM (all already in memory; nothing read from disk):
#   f1_flags (Prep F9 flags, weeks 1-53, per product x week) -> 7a step 1
#   f1_train, f1_weeks, F1_CUTOFF, F1_H (13), F1_SEASON_LENGTH (1) -> 7a
#   f1l_all, F1L_ALL_MODELS (practice-round forecasts of the 12 standard candidates) -> 7b cell #11
#   f1e_scale (MASE scale), f1e_avg_mase, f1e_win (7b winners) -> 7b cell #6
#   F1B_CV_SETS, F1E_LEVEL_ORDER, StatsForecast, AutoARIMA -> 7b;  f1i_sparse_ids -> 7c cell 12
#   pd, np -> 7a imports;  display -> Jupyter built-in
#
# WHAT IT CHANGES: nothing (all earlier tables are only read).  WHAT IT WRITES: nothing (memory only).
# All new names start with f1d_ / F1D_.
# ==========================================================================
F1D_FLAGS = ["is_holiday_week", "after_holiday_week"]          # calendar columns of the Prep F9 file
F1D_COL = "AutoARIMA_hol"                                      # the new candidate's column name

# ---- 1. One calendar row per week (weeks 1-53): the same flags for every product ----
f1d_cal = f1_flags.groupby("ds")[F1D_FLAGS].agg(["min", "max"])
assert all((f1d_cal[(f1d_c, "min")] == f1d_cal[(f1d_c, "max")]).all() for f1d_c in F1D_FLAGS), \
    "a calendar flag differs between products in the same week - stop and check"
f1d_cal = f1d_cal.xs("max", axis=1, level=1).astype(float).reindex(f1_weeks)
assert f1d_cal.notna().all().all() and len(f1d_cal) == 53, "the calendar does not cover weeks 1-53"
f1d_cal = f1d_cal.rename_axis("ds").reset_index()

# ---- 2. Fit AutoARIMA + flags per practice round, on the 7b series minus the sparse brands ----
f1d_parts, f1d_run_rows = [], []
for f1d_set in F1B_CV_SETS:
    f1d_ids = sorted(set(f1l_all.loc[f1l_all["set"] == f1d_set, "unique_id"]) - set(f1i_sparse_ids))
    f1d_tr = (f1_train[f1d_set][f1_train[f1d_set]["unique_id"].isin(f1d_ids)]
              .merge(f1d_cal, on="ds", how="left", validate="many_to_one"))
    assert f1d_tr[F1D_FLAGS].notna().all().all() and (f1d_tr.groupby("unique_id")["ds"].max() == F1_CUTOFF[f1d_set]).all()
    f1d_pos = f1_weeks.get_loc(F1_CUTOFF[f1d_set])
    f1d_fut_weeks = f1_weeks[f1d_pos + 1: f1d_pos + 1 + F1_H]               # weeks 28-40 or 41-53
    # flags of the predicted weeks: calendar only (known in advance, FPP2 5.6)
    f1d_X = (pd.MultiIndex.from_product([f1d_ids, f1d_fut_weeks], names=["unique_id", "ds"]).to_frame(index=False)
             .merge(f1d_cal, on="ds", how="left", validate="many_to_one"))
    # if a series cannot be fitted with the flags (e.g. no holiday in its training weeks), plain AutoARIMA is used
    f1d_sf = StatsForecast(models=[AutoARIMA(season_length=F1_SEASON_LENGTH, alias=F1D_COL)], freq="W-MON",
                           n_jobs=-1, fallback_model=AutoARIMA(season_length=F1_SEASON_LENGTH, alias=F1D_COL))
    f1d_fc = f1d_sf.forecast(df=f1d_tr[["unique_id", "ds", "y"] + F1D_FLAGS], h=F1_H, X_df=f1d_X)
    assert set(f1d_fc["ds"]) == set(f1d_fut_weeks) and len(f1d_fc) == len(f1d_ids) * F1_H
    f1d_fc = (f1d_fc.merge(f1l_all.loc[f1l_all["set"] == f1d_set, ["unique_id", "ds", "level", "y", "AutoARIMA"]],
                           on=["unique_id", "ds"], how="left", validate="one_to_one")
              .merge(f1d_cal, on="ds", how="left", validate="many_to_one"))
    f1d_fc.insert(0, "set", f1d_set)
    f1d_parts.append(f1d_fc)
    # training weeks with each flag: a flag that is always 0 for a series cannot be learned for it
    f1d_seen = f1d_tr.groupby("unique_id")[F1D_FLAGS].max().min(axis=1)
    f1d_run_rows.append({"set": f1d_set, "series": len(f1d_ids),
                         "holiday weeks in training (company)": int(f1d_cal.loc[f1d_cal["ds"] <= F1_CUTOFF[f1d_set], "is_holiday_week"].sum()),
                         "holiday weeks in predicted weeks": int(f1d_cal[f1d_cal["ds"].isin(f1d_fut_weeks)]["is_holiday_week"].sum()),
                         "series that saw both flags in training": int((f1d_seen == 1).sum())})
f1d_cv = pd.concat(f1d_parts, ignore_index=True)
assert f1d_cv[[F1D_COL, "AutoARIMA", "y"]].notna().all().all(), "a forecast or real-sales value is missing"
f1d_runs = pd.DataFrame(f1d_run_rows).set_index("set")

# ---- 3. Scores: MASE (as 7b picks), overall and in holiday / after-holiday weeks only ----
f1d_err = f1d_cv[["set", "level", "unique_id", "y"] + F1D_FLAGS].copy()
for f1d_c in [F1D_COL, "AutoARIMA"]:
    f1d_err[f1d_c] = (f1d_cv["y"] - f1d_cv[f1d_c]).abs()
f1d_sc = f1e_scale.set_index(["set", "unique_id"])["scale"]
f1d_mase = (f1d_err.groupby(["set", "level", "unique_id"])[[F1D_COL, "AutoARIMA"]].mean()
            .div(f1d_sc.reindex(f1d_err.groupby(["set", "level", "unique_id"]).size().index.droplevel("level"))
                 .where(lambda s: s > 0).values, axis=0))
f1d_lvl = f1d_mase.groupby(["level", "set"]).mean().unstack("set").reindex(F1E_LEVEL_ORDER)
f1d_wk = (f1d_err.assign(week_type=np.select([f1d_err["is_holiday_week"] == 1, f1d_err["after_holiday_week"] == 1],
                                             ["holiday week", "after-holiday week"], "normal week"))
          .groupby(["week_type", "set"])[[F1D_COL, "AutoARIMA"]].sum().unstack("set"))
f1d_wk = f1d_wk.div(f1d_cv.assign(week_type=np.select([f1d_cv["is_holiday_week"] == 1, f1d_cv["after_holiday_week"] == 1],
                                                     ["holiday week", "after-holiday week"], "normal week"))
                    .groupby(["week_type", "set"])["y"].sum().unstack("set"), axis=1, level=1)   # WAPE

# ---- 4. Where does the holiday model beat the series' 7b winner? (average MASE over its rounds) ----
f1d_avg = f1d_mase.groupby(level="unique_id").mean()
f1d_win = (f1e_win.set_index("unique_id").loc[f1d_avg.index, ["level", "winner", "winner_avg_MASE"]]
           .assign(**{f"{F1D_COL}_avg_MASE": f1d_avg[F1D_COL]}))
f1d_win["holiday model wins"] = f1d_win[f"{F1D_COL}_avg_MASE"] < f1d_win["winner_avg_MASE"]
# honest check (as 7b #7): where it beats plain AutoARIMA on cv1, does it also beat it on cv2?
f1d_both = f1d_mase.reset_index().pivot(index="unique_id", columns="set", values=[F1D_COL, "AutoARIMA"]).dropna()
f1d_b1 = f1d_both[(F1D_COL, "cv1")] < f1d_both[("AutoARIMA", "cv1")]
f1d_b2 = f1d_both[(F1D_COL, "cv2")] < f1d_both[("AutoARIMA", "cv2")]

# ---- Report ----
print("STEP 7d cell 14 - holidays: AutoARIMA with 2 calendar flags (option A1), practice rounds (nothing written)\n")
print("Calendar (Prep F9 flags, weeks 1-53): holiday weeks", int(f1d_cal["is_holiday_week"].sum()),
      "| after-holiday weeks", int(f1d_cal["after_holiday_week"].sum()))
display(f1d_runs)
print("Mean MASE per level (lower is better): with flags vs plain AutoARIMA")
display(f1d_lvl.round(3))
print("WAPE by week type (all series together): does the error fall in holiday weeks?")
display(f1d_wk.round(3))
print(f"Series where {F1D_COL} beats its 7b winner (average MASE): {int(f1d_win['holiday model wins'].sum())} of {len(f1d_win)}")
display(f1d_win.groupby("level")["holiday model wins"].agg(["sum", "size"]).reindex(F1E_LEVEL_ORDER)
        .rename(columns={"sum": "holiday model wins", "size": "series"}))
print(f"Honest check: beats plain AutoARIMA on cv1 in {int(f1d_b1.sum())} of {len(f1d_both)} series; "
      f"of those, also on cv2: {int((f1d_b1 & f1d_b2).sum())} ({(f1d_b1 & f1d_b2).sum() / max(f1d_b1.sum(), 1):.0%})")
print("NEXT: cell 15 - the holiday effect measured (old cell 81 idea, rebuilt leak-free).")


Step 7 – Part 7d: holidays (tested, not used). The cell above tested holiday flags the way FPP2 recommends for weekly data (§12.1, p.465: holiday dummy variables in a dynamic regression, here AutoARIMA with a "holiday week" and an "after-holiday week" flag), on cv1 and cv2. The holiday model was worse than plain AutoARIMA at every level and in both rounds, even in holiday weeks, and its few wins did not repeat between rounds. Holiday effects in weeks 1–53 differ widely between holidays (about −18% to +77% vs nearby normal weeks), and most holidays occur only once in the training weeks. A "same holiday last year" rule could not be relied on either: the only holiday seen twice (Good Friday) went from +5% in 2025 to +29% in 2026. Following FPP2 §12.7 (p.479), "we need more observations than there are parameters", holidays are therefore not added to the forecasts. The backtest weeks (Labour Day, Madaraka Day) and the future weeks (Mazingira Day) contain only holidays with small past effects (+4% to +12%). To revisit when 2+ years of data are available.



In [ ]:
# ==========================================================================
# STEP 7 - PART 7e, CELL 16: COMBINATIONS - DO AVERAGES OF MODELS BEAT ONE MODEL? (cv1 & cv2)
#   New main code for 7e.  An average (simple mean) of several models' forecasts competes with the
#   current single choice.  Ways tested (all "simple average"; the weighted blend of old cells 155/157 is cell 17):
#     STANDARD series (7b: standard products + Total, Category2, Sub_Category, Brand; NOT the 15 sparse brands)
#       now   single winner per series, picked on cv1          (the honest version of the 7b winner, as 7b #7)
#       (a)   top-2 per series: the series' 2 best models on cv1, averaged
#       (b)   fixed trio for all: AutoETS + AutoARIMA + AutoTheta (decided before seeing scores)
#       (d)   all 11 standard models averaged (Drift_log left out of every average - see below)
#       (f)   best pair or trio PER LEVEL: all 220 pairs/trios of the 11 tried on cv1, the best one kept
#   Drift_log is NOT used in any average (version 2 of this cell, after the first 7e run): it can explode on spiky
#   series (7b #11: 3,270 -> 163,506, from the log bias adjustment exp(w)(1 + s^2/2) with s up to 9.9),
#   and leaving it out cost almost nothing in the first run (Category2 0.933 -> 0.953, Sub_Category better).
#   It stays a candidate for the single winner ("now"), exactly as in 7b.
#     INTERMITTENT products (7c lane, without the 8 one-sale products)
#       now   TSB (your decision 3)
#       (c)   TSB + ADIDA, and TSB + ADIDA + IMAPA (your request)
#       (e)   all 7 intermittent models averaged
#       (f)   best pair or trio of the 7, tried on cv1 (56), the best one kept
#   HOW THE WINNER IS DECIDED (honest - no option is marked on the weeks it was chosen on):
#     - every choice inside an option (best model, top 2, best combo) is made on cv1 only;
#     - so every option is compared on cv2 (weeks 41-53), which none of them used;
#     - standard series: the lowest cv2 wins.  cv1 is NOT used to compare, because "now" (the single
#       winner) was itself picked on cv1, so its cv1 score is flattered;
#     - intermittent: your rule for TSB + ADIDA - "wins on cv1 and holds on cv2" - for the fixed
#       options (c, e); (f) chose on cv1, so it is judged on cv2 only;
#     - an option replaces the current choice only if it is better; otherwise nothing changes.
#   Scores: MASE for standard series (as 7b), RMSSE for intermittent products (as 7c).
#
# WHY (FPP2 2nd ed., Hyndman & Athanasopoulos 2018):
#   12.4 (p.472): "An easy way to improve forecast accuracy is to use several different methods on the same
#     time series, and to average the resulting forecasts" ... "combining multiple forecasts leads to
#     increased forecast accuracy"; Clemen (1989): "using a simple average has proven hard to beat".
#     Bates & Granger (1969) - the original study (both cited in FPP2 p.472).
#   3.4 (p.77): judged on new data not used for the choice -> cv1 chooses, cv2 judges.
#   3.4 (pp.80-81) MASE;  3.4 (p.79) RMSE -> mean (RMSSE for intermittent, M5 - Makridakis et al. 2022, IJF).
#
# WHERE EACH INPUT COMES FROM (all already in memory; nothing read from disk):
#   f1l_all, F1L_ALL_MODELS (practice-round forecasts of the 12 standard models + y) -> 7b cell #11
#   f1e_scale (MASE scale per set x series), f1e_win (7b winners) -> 7b cell #6;  F1E_LEVEL_ORDER -> 7b #6
#   f1k_cv, F1K_INT (the 7 intermittent candidates + y), f1t_rmsse(), f1t_scale -> cells 13a / 13
#   f1i_sparse_ids (15 sparse brands) -> cell 12;  f1q_one (8 one-sale products) -> cell 13b
#   pd -> 7a import;  itertools -> NEW import (Python standard library)
#   display -> Jupyter built-in
#
# WHAT IT CHANGES: nothing (all earlier tables are only read).  WHAT IT WRITES: nothing (memory only).
# All new names start with f1m_ / F1M_.
# ==========================================================================
import itertools

F1M_TRIO = ["AutoETS", "AutoARIMA", "AutoTheta"]                       # way (b), fixed in advance
F1M_POOL = [f1m_m for f1m_m in F1L_ALL_MODELS if f1m_m != "Drift_log"]   # models allowed in averages (11)
F1M_INT_COMBOS = {"TSB + ADIDA": ["TSB", "ADIDA"], "TSB + ADIDA + IMAPA": ["TSB", "ADIDA", "IMAPA"]}  # way (c)
assert set(F1M_TRIO) <= set(F1L_ALL_MODELS) and len(F1L_ALL_MODELS) == 12 and {"TSB", "ADIDA", "IMAPA"} <= set(F1K_INT), "run 7b and 7c first"

def f1m_mase(f1m_df, f1m_fc):
    """MASE per set x series of forecast column f1m_fc: mean |y - forecast| / 7b MASE scale (f1e_scale)."""
    f1m_mae = (f1m_df["y"] - f1m_df[f1m_fc]).abs().groupby([f1m_df["set"], f1m_df["unique_id"]]).mean()
    f1m_s = f1e_scale.set_index(["set", "unique_id"])["scale"].reindex(f1m_mae.index)
    return f1m_mae / f1m_s.where(f1m_s > 0)

# ==== PART 1: STANDARD SERIES ====
f1m_ids = set(f1e_win.loc[f1e_win["used_in_backtest"], "unique_id"]) - set(f1i_sparse_ids)
f1m_st = f1l_all[f1l_all["unique_id"].isin(f1m_ids)].copy()
f1m_both = f1m_st.groupby("unique_id")["set"].nunique().pipe(lambda s: s[s == 2].index)   # in cv1 AND cv2
f1m_st = f1m_st[f1m_st["unique_id"].isin(f1m_both)].copy()
f1m_level = f1m_st.drop_duplicates("unique_id").set_index("unique_id")["level"]

# MASE of every single model per set x series; ranks on cv1 decide "now" (best 1) and (a) (best 2)
f1m_single = pd.DataFrame({f1m_m: f1m_mase(f1m_st, f1m_m) for f1m_m in F1L_ALL_MODELS})
f1m_rank1 = f1m_single.xs("cv1", level="set")
f1m_best1 = f1m_rank1.idxmin(axis=1)                                          # single winner, picked on cv1
f1m_best2 = f1m_rank1[F1M_POOL].apply(lambda f1m_r: list(f1m_r.nsmallest(2).index), axis=1)   # top 2 (no Drift_log), cv1

# forecasts of each option (one column each)
f1m_st["now (single winner)"] = [f1m_r[f1m_best1[f1m_u]] for f1m_u, (_, f1m_r) in zip(f1m_st["unique_id"], f1m_st.iterrows())]
f1m_st["(a) top-2 per series"] = [f1m_r[f1m_best2[f1m_u]].mean() for f1m_u, (_, f1m_r) in zip(f1m_st["unique_id"], f1m_st.iterrows())]
f1m_st["(b) fixed trio"] = f1m_st[F1M_TRIO].mean(axis=1)
f1m_st["(d) all 11"] = f1m_st[F1M_POOL].mean(axis=1)

# (f) best pair/trio PER LEVEL, tried on cv1: mean cv1 MASE over the level's series
f1m_combos = [list(f1m_c) for f1m_k in (2, 3) for f1m_c in itertools.combinations(F1M_POOL, f1m_k)]
f1m_cv1 = f1m_st[f1m_st["set"] == "cv1"]
f1m_best_combo = {}
for f1m_lev in F1E_LEVEL_ORDER:
    f1m_d = f1m_cv1[f1m_cv1["level"] == f1m_lev]
    if f1m_d.empty:
        continue
    f1m_sc = {" + ".join(f1m_c): f1m_mase(f1m_d.assign(_c=f1m_d[f1m_c].mean(axis=1)), "_c").mean() for f1m_c in f1m_combos}
    f1m_best_combo[f1m_lev] = min(f1m_sc, key=f1m_sc.get)
f1m_st["(f) best pair/trio per level"] = [f1m_r[f1m_best_combo[f1m_l].split(" + ")].mean()
                                          for f1m_l, (_, f1m_r) in zip(f1m_st["level"], f1m_st.iterrows())]

F1M_STD_OPTS = ["now (single winner)", "(a) top-2 per series", "(b) fixed trio", "(d) all 11", "(f) best pair/trio per level"]
f1m_opt = pd.DataFrame({f1m_o: f1m_mase(f1m_st, f1m_o) for f1m_o in F1M_STD_OPTS})
f1m_opt["level"] = f1m_opt.index.get_level_values("unique_id").map(f1m_level)
f1m_std = f1m_opt.groupby(["level", "set"])[F1M_STD_OPTS].mean().unstack("set").reindex(F1E_LEVEL_ORDER)

# decision per level: the lowest cv2 (every option chose on cv1 only); it must beat "now" on cv2
f1m_std_dec = []
for f1m_lev in f1m_std.index:
    f1m_now2 = f1m_std.loc[f1m_lev, ("now (single winner)", "cv2")]
    f1m_ok = {f1m_o: f1m_std.loc[f1m_lev, (f1m_o, "cv2")] for f1m_o in F1M_STD_OPTS[1:]
              if f1m_std.loc[f1m_lev, (f1m_o, "cv2")] < f1m_now2}
    f1m_pick = min(f1m_ok, key=f1m_ok.get) if f1m_ok else "now (single winner)"
    f1m_std_dec.append({"level": f1m_lev, "series": int((f1m_level == f1m_lev).sum()),
                        "now cv2 MASE": f1m_now2, "chosen": f1m_pick,
                        "chosen cv2 MASE": f1m_std.loc[f1m_lev, (f1m_pick, "cv2")],
                        "(f) combo for this level": f1m_best_combo.get(f1m_lev, "-")})
f1m_std_dec = pd.DataFrame(f1m_std_dec).set_index("level")

# ==== PART 2: INTERMITTENT PRODUCTS (TSB lane) ====
f1m_in = f1k_cv[(f1k_cv["level"] == "Product") & ~f1k_cv["unique_id"].isin(f1q_one)].copy()
for f1m_n, f1m_c in F1M_INT_COMBOS.items():
    f1m_in[f1m_n] = f1m_in[f1m_c].mean(axis=1)
f1m_in["(e) all 7"] = f1m_in[F1K_INT].mean(axis=1)
f1m_icombos = [list(f1m_c) for f1m_k in (2, 3) for f1m_c in itertools.combinations(F1K_INT, f1m_k)]
f1m_i1 = f1m_in[f1m_in["set"] == "cv1"]
f1m_isc = {" + ".join(f1m_c): f1t_rmsse(f1m_i1.assign(_c=f1m_i1[f1m_c].mean(axis=1)), ["_c"], f1t_scale)["_c"].mean()
           for f1m_c in f1m_icombos}
F1M_INT_F = min(f1m_isc, key=f1m_isc.get)                             # (f) chosen on cv1
f1m_in["(f) best pair/trio"] = f1m_in[F1M_INT_F.split(" + ")].mean(axis=1)
F1M_INT_OPTS = ["TSB"] + list(F1M_INT_COMBOS) + ["(e) all 7", "(f) best pair/trio"]
F1M_INT_FIXED = list(F1M_INT_COMBOS) + ["(e) all 7"]
f1m_int = f1t_rmsse(f1m_in, F1M_INT_OPTS, f1t_scale).groupby(level="set").mean().T
f1m_int["bias cv1"] = [f1m_in.loc[f1m_in["set"] == "cv1", f1m_o].sum() / f1m_in.loc[f1m_in["set"] == "cv1", "y"].sum() for f1m_o in F1M_INT_OPTS]
f1m_int["bias cv2"] = [f1m_in.loc[f1m_in["set"] == "cv2", f1m_o].sum() / f1m_in.loc[f1m_in["set"] == "cv2", "y"].sum() for f1m_o in F1M_INT_OPTS]
f1m_iok = {f1m_o: f1m_int.loc[f1m_o, "cv2"] for f1m_o in F1M_INT_OPTS[1:]
           if f1m_int.loc[f1m_o, "cv2"] < f1m_int.loc["TSB", "cv2"]
           and (f1m_o not in F1M_INT_FIXED or f1m_int.loc[f1m_o, "cv1"] < f1m_int.loc["TSB", "cv1"])}
F1M_INT_CHOICE = min(f1m_iok, key=f1m_iok.get) if f1m_iok else "TSB"

# ---- Report ----
print("STEP 7e cell 16 - combinations (simple averages) vs the current single choice, practice rounds (nothing written)\n")
print(f"STANDARD series scored in both rounds: {len(f1m_both)} (sparse brands excluded) | MASE, lower is better")
print("cv2 is the fair column for every option (all of them chose on cv1 only); cv1 is shown for information")
display(f1m_std.round(3))
print("Decision per level (the lowest cv2; an option replaces 'now' only if better on cv2):")
display(f1m_std_dec.round(3))
print(f"\nINTERMITTENT products (TSB lane, {f1m_in['unique_id'].nunique()} products) | RMSSE, lower is better; bias 1.00 = right total")
print(f"(f) best pair/trio of the 7, chosen on cv1: {F1M_INT_F}")
display(f1m_int.round(3))
print(f"Decision: {F1M_INT_CHOICE}" + ("  (TSB stays)" if F1M_INT_CHOICE == "TSB" else "  (replaces TSB in 7g)"))
print("NEXT: cell 17 - the weighted top-2 blend of old cells 155/157, rebuilt and compared.")

In [ ]:
# ==========================================================================
# STEP 7 - PART 7f, CELL 18: RESIDUALS + FORECAST RANGES (80% / 95%) FROM PAST ERRORS, TESTED ON cv2
#   New main code for 7f.  For EVERY series that gets a forecast in 7g, it builds the CHOSEN practice
#   forecast (the 7e / 7c rules), measures its past errors (residuals), and turns them into
#   "between X and Y" ranges that 7g will put around the backtest forecast (weeks 54-66).
#   Why ranges are made from past errors: TSB, the judgement rate and the averages of 7e have no formula
#   for ranges (FPP2 p.467: "There are no algebraic results allowing us to compute prediction intervals"
#   for Croston-type methods), so one method is used for ALL series: the spread of past 13-week-ahead errors.
#   How:
#     1. chosen practice forecast per series: standard series -> the 7e rule of its level (Drift_log never
#        used in averages); intermittent / one-sale / sparse brands -> 7c (cell 13b, f1q_cv)
#     2. error per week, scaled by the series' MASE scale (so big and small series can be pooled):
#        r = (real - forecast) / scale
#     3. per GROUP (Total, Category2, Sub_Category, Brand, Product, intermittent, one-sale, sparse brand):
#        the 10th/90th (80% range) and 2.5th/97.5th (95% range) percentiles of r
#     4. range = forecast + percentile x scale; the lower end is never below 0 (sales cannot be negative)
#        and the upper end never below the lower end.  One-sale products get NO data-based range (they
#        sold nothing in the practice weeks, so their errors say nothing) - the planner sets it (cell 19 flag).
#        The same for any series whose training weeks hardly moved (scale below F1R_MIN_SCALE = 0.05 cases).
#     5. HONEST TEST: percentiles from cv1 only, applied to cv2 -> how often real sales fall inside?
#        (80% should hold about 80% of weeks, 95% about 95%)
#     6. the percentiles from BOTH rounds are kept for 7g (f1r_q)
#   Also shown (diagnostics, no forecast changes): bias per group and error by weeks ahead (1-13).
#
# WHY (FPP2 2nd ed., Hyndman & Athanasopoulos 2018):
#   3.5 (p.88): "a prediction interval gives an interval within which we expect y to lie with a specified
#     probability ... In this book we usually calculate 80% intervals and 95% intervals".
#   3.5 (p.92): "Assuming future errors will be similar to past errors ... compute prediction intervals by
#     calculating percentiles" (bootstrapped intervals) -> percentiles of past forecast errors.
#   12.2 (p.467): no formula for Croston-type intervals -> the same error-based method for all series.
#   3.4 (p.77): the coverage is checked on cv2, which the cv1 percentiles did not use.
#
# WHERE EACH INPUT COMES FROM (all already in memory; nothing read from disk):
#   f1l_all (practice forecasts of the 12 standard models + y) -> 7b #11;  f1e_win, f1e_scale, F1E_LEVEL_ORDER -> 7b #6
#   f1m_std_dec (chosen rule per level), f1m_best_combo, f1m_best1, f1m_best2, F1M_TRIO, F1M_POOL -> 7e cell 16
#   f1q_cv (7c chosen forecasts, with "source") -> cell 13b;  f1i_sparse_ids -> cell 12
#   pd -> 7a import;  display -> Jupyter built-in
#
# WHAT IT CHANGES: nothing (all earlier tables are only read).  WHAT IT WRITES: nothing (memory only).
# All new names start with f1r_ / F1R_.
# ==========================================================================
F1R_Q = {"80%": (0.10, 0.90), "95%": (0.025, 0.975)}            # FPP2 3.5: 80% and 95% ranges
F1R_7C_GROUP = {"7c lane method": "Product (intermittent, TSB)", "7c judgement rate": "Product (one-sale, judgement)",
                "7c sparse-brand winner": "Brand (sparse)"}
# one-sale products sold nothing in the practice weeks, so their errors cannot give a range: planner sets it
F1R_NO_RANGE = ["Product (one-sale, judgement)"]
# a series whose training weeks hardly moved (scale below 0.05 cases, e.g. a brand that sold once) cannot be
# scaled: one 0.08-case miss / 0.004 = -20 would distort its group's percentiles -> left out, planner sets it
F1R_MIN_SCALE = 0.05
assert "chosen" in f1m_std_dec.columns and "source" in f1q_cv.columns, "run 7e cell 16 and 7c cell 13b first"

# ---- 1. Chosen practice forecast for every standard series (7e rule of its level) ----
f1r_ids = set(f1e_win.loc[f1e_win["used_in_backtest"], "unique_id"]) - set(f1i_sparse_ids)
f1r_st = f1l_all[f1l_all["unique_id"].isin(f1r_ids)].copy()
f1r_win7b = f1e_win.set_index("unique_id")["winner"]                 # fallback for series not in both rounds

def f1r_chosen(f1r_row):
    """The forecast of one row under its level's 7e rule (Drift_log is never inside an average)."""
    f1r_rule = f1m_std_dec.at[f1r_row["level"], "chosen"]
    if f1r_rule == "(b) fixed trio":
        return f1r_row[F1M_TRIO].mean()
    if f1r_rule == "(d) all 11":
        return f1r_row[F1M_POOL].mean()
    if f1r_rule == "(f) best pair/trio per level":
        return f1r_row[f1m_best_combo[f1r_row["level"]].split(" + ")].mean()
    if f1r_rule == "(a) top-2 per series" and f1r_row["unique_id"] in f1m_best2.index:
        return f1r_row[f1m_best2[f1r_row["unique_id"]]].mean()
    if f1r_rule == "now (single winner)" and f1r_row["unique_id"] in f1m_best1.index:
        return f1r_row[f1m_best1[f1r_row["unique_id"]]]
    return f1r_row[f1r_win7b[f1r_row["unique_id"]]]                 # series in only one round: 7b winner

f1r_st["chosen"] = f1r_st.apply(f1r_chosen, axis=1)
f1r_st["group"] = f1r_st["level"]
f1r_st["rule"] = f1r_st["level"].map(f1m_std_dec["chosen"])

# ---- 2. Add the 7c series (intermittent, one-sale, sparse brands) and compute scaled errors ----
f1r_7c = f1q_cv.assign(group=f1q_cv["source"].map(F1R_7C_GROUP), rule=f1q_cv["chosen_model"])
f1r_cols = ["set", "level", "unique_id", "ds", "y", "chosen", "group", "rule"]
f1r_cv = pd.concat([f1r_st[f1r_cols], f1r_7c[f1r_cols]], ignore_index=True)
assert not f1r_cv.duplicated(["set", "unique_id", "ds"]).any(), "a series is in both 7e and 7c - stop and check"
assert f1r_cv[["chosen", "group"]].notna().all().all()
f1r_cv = f1r_cv.merge(f1e_scale, on=["set", "unique_id"], how="left", validate="many_to_one")
f1r_cv["h"] = f1r_cv.groupby(["set", "unique_id"])["ds"].rank().astype(int)      # weeks ahead 1-13
f1r_cv["r"] = (f1r_cv["y"] - f1r_cv["chosen"]) / f1r_cv["scale"].where(f1r_cv["scale"] >= F1R_MIN_SCALE)
f1r_no_scale = int(f1r_cv["r"].isna().sum())                       # (almost) flat training weeks: cannot be scaled
f1r_tiny = sorted(f1r_cv.loc[f1r_cv["r"].isna(), "unique_id"].unique())   # no data-based range -> planner sets it

# ---- 3-5. Percentiles per group from cv1 -> ranges on cv2 -> coverage (honest test) ----
def f1r_pct(f1r_d):
    f1r_d = f1r_d[~f1r_d["group"].isin(F1R_NO_RANGE)]
    return f1r_d.groupby("group")["r"].quantile([f1r_p for f1r_lh in F1R_Q.values() for f1r_p in f1r_lh]).unstack()

f1r_q1 = f1r_pct(f1r_cv[f1r_cv["set"] == "cv1"])
f1r_t = f1r_cv[(f1r_cv["set"] == "cv2") & f1r_cv["r"].notna() & f1r_cv["group"].isin(f1r_q1.index)].copy()
f1r_cov_rows = []
for f1r_g, f1r_d in f1r_t.groupby("group"):
    f1r_row = {"group": f1r_g, "cv2 weeks": len(f1r_d)}
    for f1r_name, (f1r_lo, f1r_hi) in F1R_Q.items():
        f1r_low_raw = f1r_d["chosen"] + f1r_q1.at[f1r_g, f1r_lo] * f1r_d["scale"]
        f1r_low = f1r_low_raw.clip(lower=0)                                   # sales cannot be negative
        f1r_high = (f1r_d["chosen"] + f1r_q1.at[f1r_g, f1r_hi] * f1r_d["scale"]).clip(lower=f1r_low)   # never below the lower end
        f1r_row[f"{f1r_name} coverage"] = ((f1r_d["y"] >= f1r_low) & (f1r_d["y"] <= f1r_high)).mean()
        f1r_row[f"{f1r_name} lower ends below 0 (set to 0)"] = int((f1r_low_raw < 0).sum())
    f1r_cov_rows.append(f1r_row)
f1r_cov = pd.DataFrame(f1r_cov_rows).set_index("group")

# ---- 6. Percentiles from BOTH rounds, kept for 7g ----
f1r_q = f1r_pct(f1r_cv).join(f1r_cv.groupby("group")["r"].count().rename("errors used"), how="left")

# ---- Diagnostics (no forecast changes): bias per group and round; error by weeks ahead ----
f1r_bias = (f1r_cv.groupby(["group", "set"])[["chosen", "y"]].sum()
            .pipe(lambda d: (d["chosen"] / d["y"].where(d["y"] > 0))).unstack("set").rename(columns=lambda c: f"bias {c}"))
f1r_hz = (f1r_cv[f1r_cv["set"] == "cv2"].assign(abs_r=lambda d: d["r"].abs())
          .groupby(["group", "h"])["abs_r"].mean().unstack("h"))                # mean |scaled error| by week ahead

# ---- Report ----
F1R_ORDER = list(F1E_LEVEL_ORDER) + list(F1R_7C_GROUP.values())
print("STEP 7f cell 18 - residuals and forecast ranges from past errors (nothing written; weeks 54-66 never used)\n")
print(f"f1r_cv: {len(f1r_cv):,} rows, {f1r_cv['unique_id'].nunique()} series | errors that could not be scaled "
      f"(scale below {F1R_MIN_SCALE} cases): {f1r_no_scale} rows, {len(f1r_tiny)} series")
print("\nRules used (7e for standard series, 7c for the others):")
display(f1r_cv.groupby(["group", "rule"])["unique_id"].nunique().rename("series").to_frame().reindex(F1R_ORDER, level=0))
print("HONEST TEST - ranges built from cv1 errors only, checked on cv2 (80% should be near 0.80, 95% near 0.95):")
display(f1r_cov.reindex([g for g in F1R_ORDER if g in f1r_cov.index]).round(2))
print(f"No data-based range (planner sets it): {F1R_NO_RANGE} + series with scale below {F1R_MIN_SCALE} cases: {f1r_tiny}")
print("Percentiles of scaled errors kept for 7g (both rounds; range = forecast + percentile x scale):")
display(f1r_q.reindex([g for g in F1R_ORDER if g in f1r_q.index]).round(2))
print("Bias (forecast total / real total; 1.00 = right) - diagnostic only:")
display(f1r_bias.reindex([g for g in F1R_ORDER if g in f1r_bias.index]).round(2))
print("Mean |scaled error| by weeks ahead (cv2) - does accuracy fall from week 1 to week 13?")
display(f1r_hz.reindex([g for g in F1R_ORDER if g in f1r_hz.index]).round(2))
print("NEXT: cell 19 - weekly vs 13-week error, low-confidence rule, risky products (old cells 139, 162, 73, 147).")


In [ ]:
# ==========================================================================
# STEP 7 - PART 7f, CELL 18b: CHART - HOW THE 80% AND 95% RANGES ARE CUT FROM THE PAST MISTAKES
#   Shows, for one group (default: Product), every scaled mistake of the practice rounds (cv1 + cv2) lined up
#   as a histogram, and where cell 18 cut it: the lowest 10% and highest 10% are left out -> the middle 80%
#   is the 80% range; the lowest 2.5% and highest 2.5% are left out -> the middle 95% is the 95% range.
#   The cut points are read from f1r_q (cell 18) - the SAME numbers 7g will use.  Counts under the chart prove
#   that about 10% / 80% / 10% (and 2.5% / 95% / 2.5%) of the mistakes fall in each part.
#   The x-axis stops at the 0.5th and 99.5th percentiles so the shape is readable; the few mistakes beyond
#   that are not drawn but are counted in the title (they lie in the 2.5% tails; the ranges and the proof
#   table below use ALL mistakes).
#
# WHY (FPP2 2nd ed.): 3.5 (p.88) 80% and 95% intervals;  3.5 (p.92) "calculating percentiles" of past errors.
#
# WHERE EACH INPUT COMES FROM: f1r_cv (scaled mistakes "r" + "group"), f1r_q (percentiles) -> cell 18
#   F1V_RAW, F1V_LOG (blue / orange) -> 7b cell #9;  plt -> Cell B import;  pd -> 7a import;  display -> Jupyter
# WHAT IT CHANGES: nothing.  WHAT IT WRITES: nothing (shows 1 chart).  New names start with f1g_ / F1G_.
# ==========================================================================
F1G_GROUP = "Product"                                              # any row of f1r_q, e.g. "Brand"
f1g_r = f1r_cv.loc[(f1r_cv["group"] == F1G_GROUP) & f1r_cv["r"].notna(), "r"]
f1g_q = f1r_q.loc[F1G_GROUP]
f1g_lo80, f1g_hi80, f1g_lo95, f1g_hi95 = f1g_q[0.1], f1g_q[0.9], f1g_q[0.025], f1g_q[0.975]
f1g_xmin, f1g_xmax = f1g_r.quantile(0.005), f1g_r.quantile(0.995)  # readable x-axis

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.axvspan(f1g_lo80, f1g_hi80, color=F1V_RAW, alpha=0.18, label="80% range (middle 80%)")
ax.axvspan(f1g_lo95, f1g_lo80, color=F1V_LOG, alpha=0.15, label="extra part of the 95% range")
ax.axvspan(f1g_hi80, f1g_hi95, color=F1V_LOG, alpha=0.15)
ax.hist(f1g_r[f1g_r.between(f1g_xmin, f1g_xmax)], bins=60, color="#555555", edgecolor="white", linewidth=0.5)
for f1g_x, f1g_c in [(f1g_lo80, F1V_RAW), (f1g_hi80, F1V_RAW), (f1g_lo95, F1V_LOG), (f1g_hi95, F1V_LOG)]:
    ax.axvline(f1g_x, color=f1g_c, lw=2)
ax.axvline(0, color="black", lw=1, ls=":")
f1g_top = ax.get_ylim()[1]
ax.text(0, f1g_top * 0.97, " forecast exactly right (0)", fontsize=8, va="top")
ax.text(f1g_lo80, f1g_top * 0.88, f"{f1g_lo80:+.2f}\n10th pct ", ha="right", va="top", fontsize=9, color=F1V_RAW)
ax.text(f1g_hi80, f1g_top * 0.88, f" {f1g_hi80:+.2f}\n 90th pct", ha="left", va="top", fontsize=9, color=F1V_RAW)
ax.text(f1g_lo95, f1g_top * 0.70, f"{f1g_lo95:+.2f}\n2.5th pct ", ha="right", va="top", fontsize=9, color=F1V_LOG)
ax.text(f1g_hi95, f1g_top * 0.70, f" {f1g_hi95:+.2f}\n 97.5th pct", ha="left", va="top", fontsize=9, color=F1V_LOG)
ax.text((f1g_xmin + f1g_lo95) / 2, f1g_top * 0.45, "lowest\n2.5%\ncut off", ha="center", fontsize=8, color="#444444")
ax.text((f1g_hi95 + f1g_xmax) / 2, f1g_top * 0.45, "highest\n2.5%\ncut off", ha="center", fontsize=8, color="#444444")
f1g_out = int(((f1g_r < f1g_xmin) | (f1g_r > f1g_xmax)).sum())
ax.set_xlim(f1g_xmin, f1g_xmax)
ax.set_xlabel("scaled mistake = (real sales - forecast) / usual weekly change   (left = forecast too high, right = too low)")
ax.set_ylabel("number of weeks")
ax.set_title(f"{F1G_GROUP}: {len(f1g_r):,} past mistakes (cv1 + cv2) and where the ranges are cut"
             f"  ({f1g_out} extreme ones beyond the axis)", loc="left", fontsize=11)
ax.legend(loc="upper right", fontsize=9, frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

# proof: share of mistakes in each part (should be about 10 / 80 / 10 and 2.5 / 95 / 2.5)
f1g_parts = pd.DataFrame({
    "80% range": [(f1g_r < f1g_lo80).mean(), f1g_r.between(f1g_lo80, f1g_hi80).mean(), (f1g_r > f1g_hi80).mean()],
    "95% range": [(f1g_r < f1g_lo95).mean(), f1g_r.between(f1g_lo95, f1g_hi95).mean(), (f1g_r > f1g_hi95).mean()]},
    index=["below the range (cut off)", "inside the range", "above the range (cut off)"])
print(f"{F1G_GROUP}: share of the {len(f1g_r):,} past mistakes in each part")
display((f1g_parts * 100).round(1).astype(str) + "%")


In [ ]:
# ==========================================================================
# STEP 7 - PART 7f, CELL 19 (from OLD F1 cells 139, 162, 73, 147): WEEKLY vs 13-WEEK ERROR, LOW-CONFIDENCE
#   RULE, NEGATIVE LOWER ENDS, RISKY PRODUCTS -> ONE REVIEW LIST FOR PLANNERS
#   The old cells, rewritten:
#     139 compare_weekly_vs_cumulative: weekly WAPE vs error of the 13-week total, per product.
#         old: leaky CV windows, sparse products only, wrote tier1_weekly_vs_cumulative.csv
#         here: the CHOSEN forecast of every series, on cv2 only (honest), nothing written
#     162 unpredictable_threshold = 1.0: products whose best WAPE > 100% got a min/max stock buffer.
#         old: best of 6 models on leaky windows; buffer = mean / 90th percentile of ALL 66 weeks (a leak)
#         here: the same 100% rule, on the chosen forecast in cv2; the buffer is replaced by the 80% / 95%
#               ranges of cell 18 (built from errors in weeks 1-53 only)
#     73  negative lower bounds clipped to 0: counted in cell 18 (lower ends below 0 are set to 0)
#     147 tier1_flagged: a hand-typed list of 5 risky products.
#         here: the list is built by RULES (below); the 5 old products are looked up to show where they are now
#   Flags (a series can have several; nothing is changed, it is a list for review):
#     low confidence          weekly WAPE on cv2 > 100% (old cell 162 rule)
#     13-week total off       the 13-week total on cv2 is off by more than 50% (planners order per quarter)
#     judgement (one-sale)    the 8 one-sale products (cell 12) - no data-based range, planner sets it
#     no data-based range     training weeks hardly moved (scale below 0.05 cases, cell 18) - planner sets it
#     judgement (new)         launched after the seal (7a router) - no history at all
#     short history           chosen by AICc in 7a (fewer than 13 weeks in a practice round)
#     SES (5 weeks)           AutoETS refused; simple exponential smoothing used (decision 7, 7b)
#     stopped                 7a lane "zero": forecast 0 in 7g
#   Plus: the holiday weeks inside weeks 54-66, as a calendar reminder only (7d: holidays not modelled).
#
# WHY (FPP2 2nd ed., Hyndman & Athanasopoulos 2018):
#   3.4 (p.77): judged on data not used to choose -> cv2 only.   3.5 (p.88): ranges from cell 18.
#   Ch 4 (p.101): judgmental forecasting when "there is a complete lack of historical data".
#   7.1 (p.239): SES "is suitable for forecasting data with no clear trend or seasonal pattern".
#   5.6 (p.170): holidays are "known in advance" (calendar) -> the reminder uses the calendar only.
#
# WHERE EACH INPUT COMES FROM (all already in memory; holidays from the calendar package, no sales):
#   f1r_cv (chosen practice forecasts + group), f1r_tiny (series with no data-based range) -> cell 18;  f1i_judgement -> cell 12;  f1b_ses_ids -> 7b
#   f1_routes, f1_labels, F1_CUTOFF, F1_H -> 7a;  pd -> 7a import;  display -> Jupyter
#   holidays -> NEW import: the same "holidays" package the Prep notebook uses (holidays.Kenya)
#
# WHAT IT CHANGES: nothing (all earlier tables are only read).  WHAT IT WRITES: nothing (memory only).
# All new names start with f1z_ / F1Z_.
# ==========================================================================
import holidays

F1Z_WAPE_LIMIT = 1.0          # old cell 162: weekly WAPE above 100% = low confidence
F1Z_TOTAL_LIMIT = 0.5         # 13-week total off by more than 50%
F1Z_OLD_147 = ["Spirits/Mainstream/Kenya Cane/717336", "Spirits/Mainstream/Kenya Cane/717383",
               "Spirits/Premium/Gordons Flavours/768493", "Spirits/Premium/JW Red Label/763938",
               "Spirits/Reserve/Singleton/734970"]                    # the hand-typed list of old cell 147

# ---- 1. Old cell 139: weekly WAPE vs 13-week total error, chosen forecast, cv2 only ----
f1z_c2 = f1r_cv[f1r_cv["set"] == "cv2"].assign(abs_err=lambda d: (d["y"] - d["chosen"]).abs())
f1z_err = f1z_c2.groupby(["unique_id", "level", "group"]).agg(real_13wk=("y", "sum"), forecast_13wk=("chosen", "sum"),
                                                               abs_err=("abs_err", "sum")).reset_index()
f1z_err["weekly WAPE"] = f1z_err["abs_err"] / f1z_err["real_13wk"].where(f1z_err["real_13wk"] > 0)
f1z_err["13-week total error"] = ((f1z_err["forecast_13wk"] - f1z_err["real_13wk"]).abs()
                                  / f1z_err["real_13wk"].where(f1z_err["real_13wk"] > 0))
f1z_g = (f1z_err.assign(tot_err=(f1z_err["forecast_13wk"] - f1z_err["real_13wk"]).abs())
         .groupby("group")[["abs_err", "tot_err", "real_13wk"]].sum())
f1z_real = f1z_g["real_13wk"].where(f1z_g["real_13wk"] > 0)                 # blank where the group sold nothing
f1z_err_lvl = pd.DataFrame({"weekly WAPE": f1z_g["abs_err"] / f1z_real,
                            "13-week total error": f1z_g["tot_err"] / f1z_real, "real sales (cv2)": f1z_g["real_13wk"]})

# ---- 2. Flags per series (old 162 rule + structural flags) ----
f1z_bt = f1_routes[f1_routes["set"] == "backtest"].set_index("unique_id")
f1z_flags = {
    "low confidence (weekly WAPE > 100%)": set(f1z_err.loc[f1z_err["weekly WAPE"] > F1Z_WAPE_LIMIT, "unique_id"]),
    "13-week total off by > 50%": set(f1z_err.loc[f1z_err["13-week total error"] > F1Z_TOTAL_LIMIT, "unique_id"]),
    "judgement (one-sale) - planner sets range": set(f1i_judgement["unique_id"]),
    "no data-based range (training hardly moved)": set(f1r_tiny),
    "judgement (launched after the seal)": set(f1z_bt.index[f1z_bt["lane"] == "judgement"]),
    "short history (AICc)": set(f1z_bt.index[f1z_bt["choose_by"] == "AICc"]),
    "SES (5 weeks of history)": set(f1b_ses_ids),
    "stopped (forecast 0)": set(f1z_bt.index[f1z_bt["lane"] == "zero"]),
}
f1z_all = sorted(set().union(*f1z_flags.values()))
f1z_risk = pd.DataFrame({"unique_id": f1z_all})
for f1z_name, f1z_ids in f1z_flags.items():
    f1z_risk[f1z_name] = f1z_risk["unique_id"].isin(f1z_ids)
f1z_risk["flags"] = f1z_risk[list(f1z_flags)].sum(axis=1)
f1z_risk["flag list"] = f1z_risk[list(f1z_flags)].apply(lambda f1z_r: "; ".join(f1z_r.index[f1z_r]), axis=1)
f1z_risk = (f1z_risk.merge(f1_labels[["unique_id", "SKU"]], on="unique_id", how="left")
            .merge(f1z_err[["unique_id", "level", "weekly WAPE", "13-week total error"]], on="unique_id", how="left")
            .sort_values(["flags", "unique_id"], ascending=[False, True]).reset_index(drop=True))
f1z_risk["level"] = f1z_risk["level"].fillna("Product")            # series without practice rows are products

# ---- 3. Old cell 147: where are the 5 hand-picked risky products now? ----
f1z_old = (pd.DataFrame({"unique_id": F1Z_OLD_147})
           .merge(f1z_bt[["lane", "choose_by"]].rename(columns={"lane": "backtest lane"}), left_on="unique_id",
                  right_index=True, how="left")
           .merge(f1z_err[["unique_id", "group", "weekly WAPE", "13-week total error"]], on="unique_id", how="left")
           .merge(f1z_risk[["unique_id", "flags"]], on="unique_id", how="left").fillna({"flags": 0}))

# ---- 4. Holiday weeks inside weeks 54-66 (calendar only; 7d: not modelled) ----
f1z_test_weeks = pd.date_range(F1_CUTOFF["backtest"] + pd.Timedelta(weeks=1), periods=F1_H, freq="W-MON")
f1z_hol = [{"week starting": (pd.Timestamp(f1z_d) - pd.Timedelta(days=pd.Timestamp(f1z_d).weekday())).date(),
            "holiday": f1z_n, "date": f1z_d}
           for f1z_d, f1z_n in sorted(holidays.Kenya(years=sorted({f1z_w.year for f1z_w in f1z_test_weeks})).items())
           if f1z_test_weeks[0] <= pd.Timestamp(f1z_d) <= f1z_test_weeks[-1] + pd.Timedelta(days=6)]
f1z_hol = pd.DataFrame(f1z_hol)

# ---- Report ----
print("STEP 7f cell 19 - weekly vs 13-week error, low-confidence rule, risky products (nothing written)\n")
print("Old cell 139 - weekly WAPE vs 13-week total error, chosen forecast, cv2 (all series in each group):")
display(f1z_err_lvl.round(2))
print("Old cell 73 - negative lower ends: counted in cell 18 and set to 0 (sales cannot be negative)")
print(f"\nOne review list for planners: {len(f1z_risk)} series with at least one flag (nothing is changed)")
display(pd.Series({f1z_n: len(f1z_i) for f1z_n, f1z_i in f1z_flags.items()}, name="series").to_frame())
display(f1z_risk[["unique_id", "SKU", "level", "flags", "flag list", "weekly WAPE", "13-week total error"]].head(15).round(2))
print("(the full list is the table f1z_risk in memory - one row per flagged series)")
print("Old cell 147 - the 5 hand-typed risky products, now:")
display(f1z_old.round(2))
print("Holiday weeks inside weeks 54-66 (calendar reminder only - 7d: holidays are not added to forecasts):")
display(f1z_hol)
print("NEXT: Part 7g - backtest forecasts (learn weeks 1-53, forecast 54-66) and reconciliation.")


In [ ]:
# ==========================================================================
# STEP 7 - PART 7f, CELL 20: RESIDUAL DIAGNOSTICS - ARE THE LEFTOVER MISTAKES RANDOM? (FPP2 3.3)
#   The chosen forecast (7e rule) of the 12 big, steady series - Total, the 3 categories and the 8
#   sub-categories - is fitted on weeks 1-53 (the backtest training weeks; 54-66 stay sealed) and its
#   one-step-ahead fitted values are compared with real sales.  The leftovers ("residuals") are checked
#   for the two ESSENTIAL properties of FPP2 3.3:
#     1. uncorrelated  -> Ljung-Box test, lag h = 10 (FPP2: h = 10 for non-seasonal data; T = 53, so h <= T/5)
#                         p-value above 0.05 = no pattern left = good;  ACF chart with +/- 2/sqrt(T) lines
#     2. zero mean     -> mean residual as % of mean sales (far from 0 = biased forecasts)
#   A check only: nothing is changed.  The chosen forecasts were already judged on UNSEEN weeks (cv1, cv2),
#   which is the stronger test (FPP2 3.4); this adds the in-sample check FPP2 recommends.
#   Products and brands are not checked: too noisy and too many to act on series by series.
#   Notes: an average's fitted value = the average of its models' fitted values; log models are fitted on
#   log(1 + sales) and their fitted values turned back with exp - 1 (no bias adjustment - fine for a check).
#   Ljung-Box degrees of freedom: model_df = 0 (the averages have no single parameter count).
#
# WHY (FPP2 2nd ed., Hyndman & Athanasopoulos 2018):
#   3.3 (pp.69-70): "A good forecasting method will yield residuals with the following properties:
#     1. The residuals are uncorrelated ... 2. The residuals have zero mean."  "Any forecasting method that
#     does not satisfy these properties can be improved."
#   3.3 (p.74): the Ljung-Box test; "We suggest using h = 10 for non-seasonal data".
#   2.9 (p.51): for white noise, 95% of autocorrelations lie within +/- 2/sqrt(T).
#
# WHERE EACH INPUT COMES FROM (all already in memory; nothing read from disk):
#   f1_train["backtest"] (weeks 1-53), F1_SEASON_LENGTH, f1_tags -> 7a;  F1B_LEVELS, StatsForecast,
#   AutoETS, AutoARIMA, AutoTheta, AutoCES, Naive, HistoricAverage, RandomWalkWithDrift -> 7b imports
#   f1m_std_dec, f1m_best_combo, f1m_best2, F1M_TRIO, F1M_POOL -> 7e cell 16
#   F1V_RAW, F1V_LOG -> 7b #9;  plt -> Cell B;  pd, np -> 7a imports;  display -> Jupyter
#   acorr_ljungbox, acf -> NEW import from statsmodels (already installed; the Prep notebook uses it)
#
# WHAT IT CHANGES: nothing.  WHAT IT WRITES: nothing (memory only; shows 1 chart).  New names: f1x_ / F1X_.
# ==========================================================================
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import acf

F1X_LAG = 10                                            # FPP2 3.3: h = 10 for non-seasonal data
F1X_LEVELS = ["Total", "Category2", "Sub_Category"]
F1X_BUILD = {"Naive": lambda: Naive(), "Mean": lambda: HistoricAverage(alias="Mean"),
             "Drift": lambda: RandomWalkWithDrift(alias="Drift"),
             "AutoETS": lambda: AutoETS(season_length=F1_SEASON_LENGTH),
             "AutoARIMA": lambda: AutoARIMA(season_length=F1_SEASON_LENGTH),
             "AutoTheta": lambda: AutoTheta(season_length=F1_SEASON_LENGTH),
             "CES": lambda: AutoCES(season_length=F1_SEASON_LENGTH)}

def f1x_parts(f1x_level, f1x_uid):
    """The models inside this series' chosen 7e forecast."""
    f1x_rule = f1m_std_dec.at[f1x_level, "chosen"]
    if f1x_rule == "(a) top-2 per series":
        return list(f1m_best2[f1x_uid])
    if f1x_rule == "(b) fixed trio":
        return list(F1M_TRIO)
    if f1x_rule == "(d) all 11":
        return list(F1M_POOL)
    if f1x_rule == "(f) best pair/trio per level":
        return f1m_best_combo[f1x_level].split(" + ")
    raise ValueError(f"no average for {f1x_level} ({f1x_rule}) - stop and check")

# ---- 1. Fitted values of every needed model on weeks 1-53 ----
f1x_key = {f1x_k: f1x_l for f1x_k, f1x_l in F1B_LEVELS.items()}
f1x_ids = {f1x_u: f1x_l for f1x_k, f1x_l in f1x_key.items() if f1x_l in F1X_LEVELS for f1x_u in f1_tags[f1x_k]}
f1x_need = {f1x_u: f1x_parts(f1x_l, f1x_u) for f1x_u, f1x_l in f1x_ids.items()}
f1x_raw = sorted({f1x_m for f1x_ms in f1x_need.values() for f1x_m in f1x_ms if not f1x_m.endswith("_log")})
f1x_log = sorted({f1x_m[:-4] for f1x_ms in f1x_need.values() for f1x_m in f1x_ms if f1x_m.endswith("_log")})
f1x_tr = f1_train["backtest"][f1_train["backtest"]["unique_id"].isin(f1x_ids)][["unique_id", "ds", "y"]]
assert f1x_tr["ds"].max() == F1_CUTOFF["backtest"], "training weeks go past week 53 - stop and check"

f1x_fit = f1x_tr.copy()
if f1x_raw:
    f1x_sf = StatsForecast(models=[F1X_BUILD[f1x_m]() for f1x_m in f1x_raw], freq="W-MON", n_jobs=1)
    f1x_sf.forecast(df=f1x_tr, h=1, fitted=True)
    f1x_fit = f1x_fit.merge(f1x_sf.forecast_fitted_values().drop(columns="y"), on=["unique_id", "ds"], how="left")
if f1x_log:
    f1x_sf_l = StatsForecast(models=[F1X_BUILD[f1x_m]() for f1x_m in f1x_log], freq="W-MON", n_jobs=1)
    f1x_sf_l.forecast(df=f1x_tr.assign(y=np.log1p(f1x_tr["y"])), h=1, fitted=True)
    f1x_fl = f1x_sf_l.forecast_fitted_values().drop(columns="y")
    f1x_fl[f1x_log] = np.expm1(f1x_fl[f1x_log])                      # back to cases
    f1x_fit = f1x_fit.merge(f1x_fl.rename(columns={f1x_m: f1x_m + "_log" for f1x_m in f1x_log}),
                            on=["unique_id", "ds"], how="left")

# ---- 2. Residuals of the chosen average, Ljung-Box and mean per series ----
f1x_rows, f1x_res = [], {}
for f1x_u, f1x_ms in f1x_need.items():
    f1x_d = f1x_fit[f1x_fit["unique_id"] == f1x_u].sort_values("ds")
    f1x_e = (f1x_d["y"] - f1x_d[f1x_ms].mean(axis=1, skipna=False)).dropna()   # first week(s) have no fitted value
    f1x_res[f1x_u] = f1x_e
    f1x_lb = acorr_ljungbox(f1x_e, lags=[F1X_LAG], model_df=0)
    f1x_rows.append({"level": f1x_ids[f1x_u], "series": f1x_u, "models averaged": " + ".join(f1x_ms),
                     "weeks": len(f1x_e), "mean residual (% of mean sales)": 100 * f1x_e.mean() / f1x_d["y"].mean(),
                     f"Ljung-Box Q* (h={F1X_LAG})": float(f1x_lb["lb_stat"].iloc[0]),
                     "p-value": float(f1x_lb["lb_pvalue"].iloc[0])})
f1x_diag = pd.DataFrame(f1x_rows)
f1x_diag["pattern left?"] = np.where(f1x_diag["p-value"] < 0.05, "yes - some pattern not captured", "no - looks random")

# ---- Report ----
print("STEP 7f cell 20 - residual diagnostics of the chosen forecasts, weeks 1-53 (nothing written; 54-66 sealed)\n")
print(f"Checked: {len(f1x_diag)} series (Total, 3 categories, 8 sub-categories) | Ljung-Box lag {F1X_LAG} | "
      f"p-value < 0.05 means a pattern is left in the mistakes")
display(f1x_diag.round(3))
print(f"Looks random (p >= 0.05): {int((f1x_diag['p-value'] >= 0.05).sum())} of {len(f1x_diag)}")

# ACF of the residuals for the Total and the 3 categories (FPP2 2.9: +/- 2/sqrt(T) lines)
f1x_show = [f1x_u for f1x_u in f1x_need if f1x_ids[f1x_u] in ("Total", "Category2")]
fig, f1x_axes = plt.subplots(1, len(f1x_show), figsize=(4 * len(f1x_show), 3.2), sharey=True)
for f1x_ax, f1x_u in zip(np.atleast_1d(f1x_axes), f1x_show):
    f1x_e = f1x_res[f1x_u]
    f1x_r = acf(f1x_e, nlags=F1X_LAG, fft=False)[1:]
    f1x_b = 2 / np.sqrt(len(f1x_e))
    f1x_ax.bar(range(1, F1X_LAG + 1), f1x_r, color=F1V_RAW, width=0.6)
    f1x_ax.axhline(f1x_b, color=F1V_LOG, ls="--", lw=1); f1x_ax.axhline(-f1x_b, color=F1V_LOG, ls="--", lw=1)
    f1x_ax.axhline(0, color="black", lw=0.8)
    f1x_p = f1x_diag.loc[f1x_diag["series"] == f1x_u, "p-value"].iloc[0]
    f1x_ax.set_title(f"{f1x_u}  (Ljung-Box p = {f1x_p:.2f})", fontsize=10, loc="left")
    f1x_ax.set_xlabel("lag (weeks)")
    f1x_ax.spines[["top", "right"]].set_visible(False)
np.atleast_1d(f1x_axes)[0].set_ylabel("autocorrelation of mistakes")
fig.suptitle("Residual ACF: bars inside the dashed lines = no pattern left (FPP2 2.9, 3.3)", x=0.01, ha="left", fontsize=11)
plt.tight_layout()
plt.show()
print("NEXT: Part 7g - backtest forecasts (learn weeks 1-53, forecast 54-66) and reconciliation.")


### 7f summary — What the forecast ranges are, and why they can be trusted

**1. What a range is**
Every forecast comes with two ranges:

| | Example (one product, one week) | Meaning |
|---|---|---|
| **Forecast** | 12,000 cases | Our best single guess |
| **80% range** | 11,500 – 12,350 | Real sales should land here about 8 weeks in 10 |
| **95% range** | 11,100 – 12,900 | Real sales should land here about 19 weeks in 20 |

No forecast is ever exactly right. The range tells the reader **what to expect from the forecast**: how far real sales could realistically be from it.

**2. How the range is built (before the real sales are seen)**
- In two practice rounds (cv1: learn weeks 1–27, forecast 28–40; cv2: learn weeks 1–40, forecast 41–53) we recorded every **mistake** (real − forecast).
- Mistakes are pooled per group (Total, Category2, Sub_Category, Brand, Product, intermittent products, sparse brands). Each mistake is measured in its series' own "wobble" (its typical week-to-week change), so big and small series can be pooled fairly.
- The **middle 80%** of mistakes gives the 80% range, and the **middle 95%** gives the 95% range.
- Range = **forecast + typical mistakes**, scaled back to the size of each series. The lower end is never below 0.
- The real sales of the weeks being forecast are **never used** to build their own range.

**3. How the range is checked**
- The range is a promise about where **real sales** will land, so it is checked against **real sales**.
- It is **not** checked against the forecast: the forecast is the centre of the range, so it is always inside and that check would prove nothing.
- Forecast **accuracy** is measured separately, by the error (real − forecast) and MASE.
- Test design: ranges built from **cv1** mistakes only, checked on **cv2** real sales (weeks they never saw):

| Group | Inside 80% range (target 80%) | Inside 95% range (target 95%) |
|---|---|---|
| Total | 92% (only 13 weeks) | 100% |
| Category2 | 79% | 97% |
| Sub_Category | 81% | 92% |
| Brand | 74% | 89% |
| Product | 77% | 92% |
| Intermittent products (TSB) | 79% | 89% |
| Sparse brands | 75% | 86% |

**Conclusion:** the 80% ranges are honest (74–81%). The 95% ranges are slightly narrow (86–92% in most groups), so treat them as roughly 90% ranges.

**4. What happens next**
- **7g:** ranges (built from cv1 + cv2 mistakes) are placed around the forecasts for weeks 54–66, which stay sealed.
- **Step 8:** weeks 54–66 are opened **once** and we count how many real weeks fall inside the ranges. This is the final check.
- **Step 9:** ranges are rebuilt from all three rounds and placed around the real future forecast (weeks 67–79).

**5. Why this matters to EABL**
Two products can share the same forecast but carry very different risk:

| | Forecast | 80% range | What the planner does |
|---|---|---|---|
| Product A | 12,000 | 11,500 – 12,350 | Predictable: order close to the forecast, small safety stock |
| Product B | 12,000 | 6,000 – 20,000 | Uncertain: larger safety stock or smaller batches; review weekly |

The range does **not** prove a forecast is true. It tells planners **how much to protect themselves** (safety stock, production, cash), and the tests above show that this warning is reliable.

Series with **no range** (new launches, stopped products, one-sale products, near-flat series) are flagged for the planner to set by judgement.

*Sources: Hyndman & Athanasopoulos, Forecasting: Principles and Practice, 2nd ed. (2018) — §3.4 (p.77) test on unseen data; §3.5 (pp.88, 92) 80%/95% ranges from past errors; Ch 4 (p.101) judgement where there is no history.*


In [ ]:
# ==========================================================================
# STEP 7 - PART 7g, CELL 21: BACKTEST BASE FORECASTS - EVERY SERIES LEARNS WEEKS 1-53, FORECASTS WEEKS 54-66
#   New main code for 7g (part 1 of 3).  For ALL 283 series of the hierarchy it makes the "base" forecast
#   (FPP2 10.6: each series forecast on its own) with the method chosen in 7b-7e, then adds the 7f ranges
#   and a 3x safety check.  Weeks 54-66 are NOT in memory (sealed in 7a): nothing here can see them.
#   Method per series (the "plan"):
#     Total                -> 7e (a): average of its 2 best models (f1m_best2)
#     Category2, Sub_Cat.  -> 7e (f): the level's best pair/trio (f1m_best_combo; never Drift_log in averages)
#     Brand                -> 7e (f) for normal brands; the 15 sparse brands -> their own 7c winner (f1q_win)
#     Product, standard    -> 7e (f) if chosen by CV; short products (AICc lane, 7b): AutoETS, or SES if < 7 weeks
#     Product, intermittent-> TSB (setting F1T_BEST, cell 13); the 8 one-sale products -> judgement rate (cell 12)
#     Product, stopped     -> 0 (7a lane "zero")
#     launched after the seal (7 products + their 3 brands) -> 0: nothing was known on 6 Apr 2026 (FPP2 Ch 4:
#                             a planner sets these; they are reported separately in Step 8)
#   A model that refuses a very short series (fewer than 7 weeks, as AutoETS did in 7b) falls back to the
#   average of past weeks (statsforecast fallback_model = HistoricAverage); those series are listed.
#   Log models (name ending _log) are fitted on log(1 + sales) and turned back with the 7b #11 bias
#   adjustment exp(w) x (1 + sigma^2 / 2) - 1, sigma from the 95% interval - exactly as in the practice rounds.
#   Ranges (7f): forecast + percentile x scale per group (f1r_q); scale = mean |weekly change| in weeks 1-53;
#     lower end never below 0.  Series without a data-based range (one-sale, near-flat, stopped, new): blank.
#   3x check: a forecast week above 3 x the series' highest week in weeks 1-53 is FLAGGED (not changed).
#
# WHY (FPP2 2nd ed., Hyndman & Athanasopoulos 2018):
#   10.6 (p.414): "Suppose we forecast all series independently ... We call these the base forecasts".
#   3.4 (p.77): the methods were chosen on cv1/cv2 only; weeks 54-66 are the honest test (Step 8).
#   12.4 (p.472) combinations;  12.2 Croston-type / TSB;  7.1 (p.239) SES;  3.2 + 12.3 log and bias adjustment;
#   3.5 (pp.88, 92) ranges from past errors;  Ch 4 (p.101) judgement where there is no history.
#
# WHERE EACH INPUT COMES FROM (all already in memory; nothing read from disk):
#   f1_train["backtest"] (weeks 1-53), f1_routes, f1_tags, F1_BOTTOM, f1_weeks, F1_CUTOFF, F1_H, F1_SEASON_LENGTH -> 7a
#   F1B_LEVELS, f1b_ses_ids, StatsForecast + model classes (Naive, HistoricAverage, RandomWalkWithDrift, AutoETS,
#     AutoARIMA, AutoTheta, AutoCES, SimpleExponentialSmoothingOptimized) -> 7b;  F1L_Z95 -> 7b #11
#   CrostonClassic, CrostonOptimized, CrostonSBA, ADIDA, IMAPA, TSB -> 7c imports;  F1T_BEST -> cell 13
#   f1i_judgement (one-sale rates) -> cell 12;  f1q_win (sparse-brand winners) -> cell 13b
#   f1m_std_dec, f1m_best_combo, f1m_best2 -> 7e cell 16;  f1r_q, F1R_MIN_SCALE -> 7f cell 18
#   pd, np -> 7a imports;  display -> Jupyter built-in
#
# WHAT IT CHANGES: nothing.  WHAT IT WRITES: nothing (memory only).  New names start with f1y_ / F1Y_.
# ==========================================================================
F1Y_3X = 3.0                                                    # 3x safety check
F1Y_TSB_ALPHA = float(F1T_BEST.split("_")[1])                   # "TSB_0.5" -> 0.5 (cell 13)
F1Y_RAW = {"Naive": lambda: Naive(), "Mean": lambda: HistoricAverage(alias="Mean"),
           "Drift": lambda: RandomWalkWithDrift(alias="Drift"),
           "AutoETS": lambda: AutoETS(season_length=F1_SEASON_LENGTH),
           "AutoARIMA": lambda: AutoARIMA(season_length=F1_SEASON_LENGTH),
           "AutoTheta": lambda: AutoTheta(season_length=F1_SEASON_LENGTH),
           "CES": lambda: AutoCES(season_length=F1_SEASON_LENGTH),
           "SES": lambda: SimpleExponentialSmoothingOptimized(alias="SES"),
           "CrostonClassic": lambda: CrostonClassic(), "CrostonOptimized": lambda: CrostonOptimized(),
           "CrostonSBA": lambda: CrostonSBA(), "ADIDA": lambda: ADIDA(), "IMAPA": lambda: IMAPA(),
           "TSB": lambda: TSB(alpha_d=F1Y_TSB_ALPHA, alpha_p=F1Y_TSB_ALPHA, alias="TSB")}
F1Y_LOG = {"Drift": lambda: RandomWalkWithDrift(alias="Drift_log"),
           "AutoETS": lambda: AutoETS(season_length=F1_SEASON_LENGTH, alias="AutoETS_log"),
           "AutoARIMA": lambda: AutoARIMA(season_length=F1_SEASON_LENGTH, alias="AutoARIMA_log"),
           "AutoTheta": lambda: AutoTheta(season_length=F1_SEASON_LENGTH, alias="AutoTheta_log"),
           "CES": lambda: AutoCES(season_length=F1_SEASON_LENGTH, alias="CES_log")}

# ---- 1. The plan: which method (list of models to average, or a fixed rule) for each of the 283 series ----
f1y_bt = f1_routes[f1_routes["set"] == "backtest"].set_index("unique_id")
f1y_one = f1i_judgement.set_index("unique_id")["rate_per_week"]
f1y_tr = f1_train["backtest"]
f1y_weeks_seen = f1y_tr.groupby("unique_id").size()
f1y_rows = []
for f1y_key, f1y_lev in F1B_LEVELS.items():
    for f1y_u in f1_tags[f1y_key]:
        if f1y_u not in f1y_weeks_seen.index:                          # no sale before the seal
            f1y_rows.append((f1y_u, f1y_lev, "zero (launched after the seal)", [])); continue
        if f1y_lev == "Product":
            f1y_r = f1y_bt.loc[f1y_u]
            if f1y_r["lane"] == "zero":
                f1y_rows.append((f1y_u, f1y_lev, "zero (stopped)", []))
            elif f1y_u in f1y_one.index:
                f1y_rows.append((f1y_u, f1y_lev, "judgement rate (one-sale)", []))
            elif f1y_r["lane"] == "intermittent":
                f1y_rows.append((f1y_u, f1y_lev, f"TSB ({F1T_BEST})", ["TSB"]))
            elif f1y_r["choose_by"] == "AICc":
                f1y_m = "SES" if f1y_u in f1b_ses_ids else "AutoETS"
                f1y_rows.append((f1y_u, f1y_lev, f"{f1y_m} (short history, AICc)", [f1y_m]))
            else:
                f1y_ms = f1m_best_combo["Product"].split(" + ")
                f1y_rows.append((f1y_u, f1y_lev, " + ".join(f1y_ms), f1y_ms))
        elif f1y_u in f1q_win.index:                                    # sparse brand: its own 7c winner
            f1y_rows.append((f1y_u, f1y_lev, f"{f1q_win[f1y_u]} (sparse brand)", [f1q_win[f1y_u]]))
        elif f1m_std_dec.at[f1y_lev, "chosen"] == "(a) top-2 per series":
            f1y_ms = list(f1m_best2[f1y_u])
            f1y_rows.append((f1y_u, f1y_lev, " + ".join(f1y_ms), f1y_ms))
        else:
            f1y_ms = f1m_best_combo[f1y_lev].split(" + ")
            f1y_rows.append((f1y_u, f1y_lev, " + ".join(f1y_ms), f1y_ms))
f1y_plan = pd.DataFrame(f1y_rows, columns=["unique_id", "level", "method", "models"])
assert len(f1y_plan) == 283 and f1y_plan["unique_id"].is_unique, "the plan must cover all 283 series once"

# ---- 2. Fit the needed models on weeks 1-53 and forecast weeks 54-66 ----
f1y_need = f1y_plan[f1y_plan["models"].str.len() > 0]
f1y_raw_m = sorted({f1y_m for f1y_ms in f1y_need["models"] for f1y_m in f1y_ms if not f1y_m.endswith("_log")})
f1y_log_m = sorted({f1y_m[:-4] for f1y_ms in f1y_need["models"] for f1y_m in f1y_ms if f1y_m.endswith("_log")})
f1y_fut = pd.date_range(F1_CUTOFF["backtest"] + pd.Timedelta(weeks=1), periods=F1_H, freq="W-MON")   # weeks 54-66
f1y_fc = pd.MultiIndex.from_product([f1y_plan["unique_id"], f1y_fut], names=["unique_id", "ds"]).to_frame(index=False)

for f1y_ms, f1y_build, f1y_is_log in [(f1y_raw_m, F1Y_RAW, False), (f1y_log_m, F1Y_LOG, True)]:
    for f1y_m in f1y_ms:                                               # one model at a time, only where needed
        f1y_col = f1y_m + ("_log" if f1y_is_log else "")
        f1y_ids = f1y_need.loc[f1y_need["models"].apply(lambda f1y_l: f1y_col in f1y_l), "unique_id"]
        f1y_d = f1y_tr[f1y_tr["unique_id"].isin(f1y_ids)][["unique_id", "ds", "y"]]
        assert (f1y_d.groupby("unique_id")["ds"].max() == F1_CUTOFF["backtest"]).all(), "training must end at week 53"
        if f1y_is_log:
            f1y_out = StatsForecast(models=[f1y_build[f1y_m]()], freq="W-MON", n_jobs=-1,
                                    fallback_model=HistoricAverage(alias=f1y_col)).forecast(
                df=f1y_d.assign(y=np.log1p(f1y_d["y"])), h=F1_H, level=[95])
            f1y_sig = (f1y_out[f"{f1y_col}-hi-95"] - f1y_out[f"{f1y_col}-lo-95"]) / (2 * F1L_Z95)
            f1y_out[f1y_col] = np.exp(f1y_out[f1y_col]) * (1 + f1y_sig ** 2 / 2) - 1      # 7b #11 bias adjustment
        else:
            f1y_out = StatsForecast(models=[f1y_build[f1y_m]()], freq="W-MON", n_jobs=-1,
                                    fallback_model=HistoricAverage(alias=f1y_col)).forecast(df=f1y_d, h=F1_H)
        assert set(f1y_out["ds"]) == set(f1y_fut), f"{f1y_col}: forecast weeks are not 54-66"
        f1y_fc = f1y_fc.merge(f1y_out[["unique_id", "ds", f1y_col]], on=["unique_id", "ds"], how="left")

f1y_short = f1y_need[f1y_need["unique_id"].map(f1y_weeks_seen) < 7]          # AutoETS / AutoARIMA may refuse these
# ---- 3. The base forecast per series and week ----
f1y_fc = f1y_fc.merge(f1y_plan[["unique_id", "level", "method", "models"]], on="unique_id", how="left")
f1y_fc["base"] = [f1y_r[f1y_ms].mean() if f1y_ms else 0.0
                  for f1y_ms, (_, f1y_r) in zip(f1y_fc["models"], f1y_fc.iterrows())]
f1y_j = f1y_fc["method"] == "judgement rate (one-sale)"
f1y_fc.loc[f1y_j, "base"] = f1y_fc.loc[f1y_j, "unique_id"].map(f1y_one)          # flat judgement rate per week
assert f1y_fc["base"].notna().all() and len(f1y_fc) == 283 * F1_H, "a base forecast is missing"

# ---- 4. 7f ranges around the base forecast + the 3x safety check ----
f1y_scale = f1y_tr.sort_values(["unique_id", "ds"]).groupby("unique_id")["y"].apply(lambda f1y_s: f1y_s.diff().abs().mean())
f1y_max = f1y_tr.groupby("unique_id")["y"].max()
f1y_group = f1y_plan.set_index("unique_id").apply(
    lambda f1y_r: ("Product (intermittent, TSB)" if f1y_r["method"].startswith("TSB")
                   else "Brand (sparse)" if f1y_r["method"].endswith("(sparse brand)")
                   else None if (f1y_r["method"].startswith(("zero", "judgement")))
                   else f1y_r["level"]), axis=1)
f1y_fc["group"] = f1y_fc["unique_id"].map(f1y_group)
f1y_fc["scale"] = f1y_fc["unique_id"].map(f1y_scale)
f1y_ok = f1y_fc["group"].isin(f1r_q.index) & (f1y_fc["scale"] >= F1R_MIN_SCALE)
for f1y_name, f1y_lo, f1y_hi in [("80", 0.10, 0.90), ("95", 0.025, 0.975)]:
    f1y_l = (f1y_fc["base"] + f1y_fc["group"].map(f1r_q[f1y_lo]) * f1y_fc["scale"]).clip(lower=0)
    f1y_h = (f1y_fc["base"] + f1y_fc["group"].map(f1r_q[f1y_hi]) * f1y_fc["scale"]).clip(lower=f1y_l)
    f1y_fc[f"lo{f1y_name}"], f1y_fc[f"hi{f1y_name}"] = f1y_l.where(f1y_ok), f1y_h.where(f1y_ok)
f1y_fc["over 3x past max"] = f1y_fc["base"] > F1Y_3X * f1y_fc["unique_id"].map(f1y_max)
f1y_fc = f1y_fc[["level", "unique_id", "ds", "method", "base", "lo80", "hi80", "lo95", "hi95", "group", "scale",
                 "over 3x past max"] + sorted(set(f1y_raw_m) | {f1y_m + "_log" for f1y_m in f1y_log_m})]

# ---- Report ----
print("STEP 7g cell 21 - backtest BASE forecasts, weeks 54-66, learned from weeks 1-53 (nothing written; 54-66 sealed)\n")
print(f"f1y_fc: {len(f1y_fc):,} rows = 283 series x {F1_H} weeks ({f1y_fut[0].date()} to {f1y_fut[-1].date()}) | "
      f"models fitted: {f1y_raw_m} + log {f1y_log_m}")
display(f1y_plan.groupby(["level", "method"]).size().rename("series").to_frame().reindex(list(F1B_LEVELS.values()), level=0))
f1y_tot = f1y_fc.groupby("level")[["base"]].sum().reindex(list(F1B_LEVELS.values()))
f1y_tot["share with a range"] = f1y_fc.groupby("level")["lo80"].apply(lambda f1y_s: f1y_s.notna().mean()).round(2)
print("13-week base forecast per level (levels do NOT add up yet - that is cell 22, reconciliation):")
display(f1y_tot.round({"base": 0}))
print(f"Series with fewer than 7 weeks of history: {len(f1y_short)} - a model that refuses them falls back to the "
      f"average of past weeks (statsforecast fallback_model): {list(f1y_short['unique_id'])}")
print(f"Below 0: {int((f1y_fc['base'] < 0).sum())} rows (lowest {f1y_fc['base'].min():.2f}) | "
      f"3x check - weeks above 3x the series' highest past week: {int(f1y_fc['over 3x past max'].sum())} "
      f"in {f1y_fc.loc[f1y_fc['over 3x past max'], 'unique_id'].nunique()} series")
display(f1y_fc.loc[f1y_fc["over 3x past max"]].groupby(["unique_id", "method"])["base"].max().rename("highest week").to_frame().head(10))
display(f1y_fc.head(3))
print("NEXT: cell 22 - reconciliation (bottom-up, OLS, WLS structural, MinT shrink), chosen on cv2.")


In [ ]:
# ==========================================================================
# STEP 7 - PART 7g, CELL 22 (v2): RECONCILIATION - MAKE ALL LEVELS ADD UP (BOTTOM-UP, OLS, 2 x WLS, MinT), CHOSEN ON cv2
#   New main code for 7g (part 2 of 3).  Base forecasts (cell 21) do not add up: products, brands, ... Total
#   were forecast separately.  Reconciliation turns them into ONE coherent set (FPP2 10.2-10.7):
#       reconciled = S x G x base,   G = (S' W^-1 S)^-1 S' W^-1          (FPP2 10.6-10.7, equation 10.8)
#   Methods compared (all from FPP2):
#     Bottom-up        (10.3)  add up the 188 product forecasts; upper-level forecasts are ignored
#     OLS              (10.7)  W = identity: every series equally trusted
#     WLS structural   (10.7)  W = diag(S x 1): weight by how many products sit under a series; "particularly
#                              useful in cases where residuals are not available" (judgement forecasts - we have them)
#     WLS variance     (10.7)  W = diag(error variances): each series weighted by its OWN past error size, links between
#                              series ignored (v2: added - FPP2 lists it, and Oliveira & Ramos 2019 found it often beat MinT shrink)
#     MinT shrink      (10.7)  W = shrunk covariance of past forecast errors (Schafer & Strimmer shrinkage, as FPP2
#                              "covariance = shr"); errors = the 13-week-ahead practice errors of the chosen forecasts
#   HOW THE METHOD IS CHOSEN (honest, as everywhere in Step 7):
#     - each method reconciles the chosen PRACTICE forecasts of cv1 and cv2 (the same forecasts as 7f, f1r_cv);
#       series with no practice forecast in a round (not launched / too short / stopped) get 0 there
#     - MinT's W is learned from cv1 errors only, so all methods are judged on cv2, which none of them used
#     - score = MASE per series (7b scale), averaged per level, then over the 5 levels (each level counts equally)
#     - the lowest cv2 score wins, compared with the unreconciled base as a reference
#   DEVIATION FROM THE TEXTBOOK (disclosed): FPP2/FPP3 estimate W from the base models' residuals (one-step fit over
#     the training weeks) and assume W_h is proportional to W_1.  Here W comes from the 13-week-ahead PRACTICE errors of
#     the chosen forecasts (cv1, cv2), because the averages, TSB and judgement forecasts have no single set of residuals,
#     and these errors match the real 13-week horizon.  Cost: fewer rows (13 per round), hence shrinkage.
#   Then the winner (and bottom-up, for comparison) reconciles the BACKTEST base forecasts of weeks 54-66;
#   MinT's W for the backtest uses the errors of BOTH practice rounds.  Saved in memory, not scored (Step 8).
#
# WHY (FPP2 2nd ed., Hyndman & Athanasopoulos 2018):
#   10.2-10.3 (p.407): bottom-up "involves first generating forecasts for each series at the bottom-level, and
#     then summing these".   10.6 (p.414): base forecasts and the mapping matrix.
#   10.7 (pp.416-418): MinT and its four simplifying approximations (OLS, WLS variance, WLS structural, shrink).
#   p.421: "the optimal reconciliation approach ... generates more accurate coherent forecasts than the other
#     traditional alternatives" - tested here on cv2 rather than assumed.
#   Not in FPP2 (named, peer-reviewed): Wickramasuriya, Athanasopoulos & Hyndman (2019, JASA) - MinT;
#     Schafer & Strimmer (2005, Stat. Appl. Genet. Mol. Biol.) - the shrinkage estimator;
#     Oliveira & Ramos (2019, Entropy 21:436) - weekly retail hierarchy: gains from reconciliation were small (2-4%) and
#     the best variant depended on the base models (MinT shrink with ARIMA; WLS variance often better with ETS).
#
# WHERE EACH INPUT COMES FROM (all already in memory; nothing read from disk):
#   f1_Smat (S matrix: 283 series x 188 products), f1_Y (weeks 1-53), f1_weeks, F1_CUTOFF, F1_H -> 7a
#   f1b_level_of, F1B_CV_SETS -> 7b;  f1e_scale (MASE scale per round), F1E_LEVEL_ORDER -> 7b #6
#   f1r_cv (chosen practice forecasts of 249 series) -> 7f cell 18;  f1y_fc (backtest base forecasts) -> cell 21
#   pd, np -> 7a imports;  display -> Jupyter built-in
#
# WHAT IT CHANGES: nothing.  WHAT IT WRITES: nothing (memory only).  New names start with f1u_ / F1U_.
# ==========================================================================
F1U_IDS = list(f1_Smat.index)                                   # the 283 series, in S order
F1U_BOTTOM = list(f1_Smat.columns)                              # the 188 products
F1U_S = f1_Smat.values.astype(float)                            # 283 x 188
F1U_METHODS = ["Bottom-up", "OLS", "WLS structural", "WLS variance", "MinT shrink"]
assert F1U_S.shape == (283, 188) and set(f1y_fc["unique_id"]) == set(F1U_IDS), "run 7a and cell 21 first"

def f1u_G(f1u_W):
    """G = (S' W^-1 S)^-1 S' W^-1 (FPP2 10.7); W is a 283 x 283 matrix."""
    f1u_WiS = np.linalg.solve(f1u_W, F1U_S)                     # W^-1 S
    return np.linalg.solve(F1U_S.T @ f1u_WiS, f1u_WiS.T)        # (S' W^-1 S)^-1 S' W^-1

def f1u_shrink(f1u_E):
    """Schafer-Strimmer shrinkage of the error covariance towards its diagonal. f1u_E: weeks x 283 errors."""
    f1u_T = f1u_E.shape[0]
    f1u_Ec = f1u_E - f1u_E.mean(axis=0)
    f1u_cov = f1u_Ec.T @ f1u_Ec / (f1u_T - 1)
    f1u_var = np.diag(f1u_cov).copy()
    f1u_var[f1u_var <= 0] = f1u_var[f1u_var > 0].min()          # series with no error spread: smallest variance
    f1u_Z = f1u_Ec / np.sqrt(f1u_var)                            # standardised errors
    f1u_R = f1u_Z.T @ f1u_Z / (f1u_T - 1)                        # correlations
    f1u_w = f1u_Z[:, :, None] * f1u_Z[:, None, :]                # T x n x n
    f1u_varR = f1u_T / (f1u_T - 1) ** 3 * ((f1u_w - f1u_w.mean(axis=0)) ** 2).sum(axis=0)
    f1u_off = ~np.eye(len(f1u_var), dtype=bool)
    f1u_lam = float(np.clip(f1u_varR[f1u_off].sum() / (f1u_R[f1u_off] ** 2).sum(), 0, 1))
    f1u_Rs = (1 - f1u_lam) * f1u_R; np.fill_diagonal(f1u_Rs, 1.0)
    f1u_W = np.sqrt(f1u_var)[:, None] * f1u_Rs * np.sqrt(f1u_var)[None, :]
    return f1u_W + 1e-8 * np.eye(len(f1u_var)) * f1u_var.mean(), f1u_lam

def f1u_reconcile(f1u_base, f1u_W_mint):
    """All methods for one 283 x 13 base matrix -> dict method -> 283 x 13 coherent matrix.
    WLS variance uses only the diagonal (each series' own error variance) of the same error matrix as MinT."""
    f1u_b = f1u_base[[F1U_IDS.index(f1u_p) for f1u_p in F1U_BOTTOM], :]
    f1u_out = {"Bottom-up": F1U_S @ f1u_b}
    for f1u_m, f1u_W in [("OLS", np.eye(283)), ("WLS structural", np.diag(F1U_S.sum(axis=1))),
                         ("WLS variance", np.diag(np.diag(f1u_W_mint))), ("MinT shrink", f1u_W_mint)]:
        f1u_out[f1u_m] = F1U_S @ (f1u_G(f1u_W) @ f1u_base)
    return f1u_out

# ---- 1. Practice rounds as 283 x 13 matrices: chosen base forecast and real sales (0 where a series had none) ----
f1u_base_cv, f1u_y_cv = {}, {}
for f1u_set in F1B_CV_SETS:
    f1u_pos = f1_weeks.get_loc(F1_CUTOFF[f1u_set])
    f1u_wk = f1_weeks[f1u_pos + 1: f1u_pos + 1 + F1_H]
    f1u_c = f1r_cv[f1r_cv["set"] == f1u_set].pivot(index="unique_id", columns="ds", values="chosen")
    f1u_base_cv[f1u_set] = f1u_c.reindex(index=F1U_IDS, columns=f1u_wk).fillna(0.0).values
    f1u_y_cv[f1u_set] = (f1_Y[f1_Y["ds"].isin(f1u_wk)].pivot(index="unique_id", columns="ds", values="y")
                         .reindex(index=F1U_IDS, columns=f1u_wk).fillna(0.0).values)
f1u_err = {f1u_s: (f1u_y_cv[f1u_s] - f1u_base_cv[f1u_s]).T for f1u_s in F1B_CV_SETS}   # weeks x 283

# ---- 2. Score every method on cv2 (MinT's W from cv1 errors only) ----
f1u_W_cv1, f1u_lam_cv1 = f1u_shrink(f1u_err["cv1"])
f1u_rec2 = f1u_reconcile(f1u_base_cv["cv2"], f1u_W_cv1)
f1u_rec2 = {"Base (not reconciled)": f1u_base_cv["cv2"], **f1u_rec2}
f1u_sc2 = f1e_scale[f1e_scale["set"] == "cv2"].set_index("unique_id")["scale"].reindex(F1U_IDS)
f1u_ok = (f1u_sc2 > 0).values
f1u_lev = pd.Series([f1b_level_of[f1u_u] for f1u_u in F1U_IDS], index=F1U_IDS)
f1u_rows = []
for f1u_m, f1u_mat in f1u_rec2.items():
    f1u_mase = pd.Series(np.abs(f1u_y_cv["cv2"] - f1u_mat).mean(axis=1) / f1u_sc2.values, index=F1U_IDS)[f1u_ok]
    f1u_by = f1u_mase.groupby(f1u_lev[f1u_ok]).mean().reindex(F1E_LEVEL_ORDER)
    f1u_coh = float(np.abs(F1U_S @ f1u_mat[[F1U_IDS.index(f1u_p) for f1u_p in F1U_BOTTOM], :] - f1u_mat).max())
    f1u_rows.append({"method": f1u_m, **f1u_by.to_dict(), "mean of the 5 levels": f1u_by.mean(),
                     "adds up? (largest gap, cases)": round(f1u_coh, 6)})
f1u_score = pd.DataFrame(f1u_rows).set_index("method")
F1U_CHOICE = f1u_score.loc[F1U_METHODS, "mean of the 5 levels"].idxmin()           # lowest cv2 score wins

# ---- 3. Reconcile the BACKTEST base forecasts (weeks 54-66); MinT's W from both practice rounds ----
f1u_fut = sorted(f1y_fc["ds"].unique())
f1u_base_bt = f1y_fc.pivot(index="unique_id", columns="ds", values="base").reindex(index=F1U_IDS, columns=f1u_fut).values
f1u_W_both, f1u_lam_both = f1u_shrink(np.vstack([f1u_err["cv1"], f1u_err["cv2"]]))
f1u_rec_bt = f1u_reconcile(f1u_base_bt, f1u_W_both)
f1u_bt = f1y_fc[["level", "unique_id", "ds", "method", "base"]].copy()
for f1u_m, f1u_mat in f1u_rec_bt.items():
    f1u_long = pd.DataFrame(f1u_mat, index=F1U_IDS, columns=f1u_fut).stack().rename(f1u_m).reset_index()
    f1u_long.columns = ["unique_id", "ds", f1u_m]
    f1u_bt = f1u_bt.merge(f1u_long, on=["unique_id", "ds"], how="left", validate="one_to_one")
f1u_bt["reconciled (chosen)"] = f1u_bt[F1U_CHOICE]
assert f1u_bt[F1U_METHODS].notna().all().all() and len(f1u_bt) == 283 * F1_H

# ---- Report ----
print("STEP 7g cell 22 - reconciliation: methods judged on cv2, the winner applied to weeks 54-66 (nothing written)\n")
print(f"MinT shrinkage intensity (0 = sample covariance, 1 = diagonal only): from cv1 {f1u_lam_cv1:.2f} | "
      f"from cv1 + cv2 {f1u_lam_both:.2f}")
print("Mean MASE per level on cv2 (lower is better; every method judged on weeks none of them used):")
display(f1u_score.round(3))
print(f"Chosen reconciliation: {F1U_CHOICE}  (mean of the 5 levels {f1u_score.at[F1U_CHOICE, 'mean of the 5 levels']:.3f}"
      f" vs bottom-up {f1u_score.at['Bottom-up', 'mean of the 5 levels']:.3f}, base {f1u_score.at['Base (not reconciled)', 'mean of the 5 levels']:.3f})")
f1u_tot = f1u_bt.groupby("level")[["base", "Bottom-up", "reconciled (chosen)"]].sum().reindex(F1E_LEVEL_ORDER)
print("\nBacktest weeks 54-66, 13-week totals per level: base (does not add up) vs reconciled (adds up at every level):")
display(f1u_tot.round(0))
print(f"Reconciled forecasts below 0: {int((f1u_bt['reconciled (chosen)'] < 0).sum())} rows "
      f"(lowest {f1u_bt['reconciled (chosen)'].min():.2f} cases)")
print("NEXT: cell 23 - the final backtest table (old cells 168-172): ranges, flags, tested flag, row check.")

In [ ]:
# ==========================================================================
# STEP 7 - PART 7g, CELL 23 (v2) (from OLD F1 cells 168-172): THE FINAL BACKTEST TABLE - FORECAST TO USE, RANGES,
#   "TESTED" FLAG, REVIEW FLAGS, ROW CHECK (and an optional save, OFF by default)
#   The old cells, rewritten:
#     168 (md) tested=False for products too short to test -> here "tested" = the series had a practice-round
#              forecast in cv1/cv2 (f1r_cv), so its method has evidence behind it
#     169      CrostonSBA as a safe default for short products -> not copied: short products already have
#              their lane rule (7c: TSB or the judgement rate; 7b: AutoETS / SES)
#     170      hard-coded row counts (351, 208, 494) -> replaced by checks computed from the data (283 x 13)
#     171-172  how the final table is joined -> one table for all 283 series; no "222" check, no Returnable,
#              and NOT the old file name final_forecast_all_skus.csv (that old file is leaky - do not use it)
#   Forecast to use (v2: SET-NEGATIVE-TO-ZERO, then add up - Di Fonzo & Girolimetto 2023):
#     1. take the reconciled forecast (cell 22, F1U_CHOICE) of the 188 PRODUCTS and set values below 0 to 0
#        (sales cannot be negative; a negative here means "nothing will sell")
#     2. rebuild every brand, sub-category, category and the Total by adding up those products (S matrix),
#        so all levels still add up EXACTLY (v1 clipped every level separately and left gaps of up to ~3 cases)
#     3. products whose reconciled forecast went to -0.1 case or lower in some week get a review flag
#        "set to 0 by reconciliation - review" (F1O_NEG_REVIEW): MinT cut them by more than their own forecast
#   Girolimetto (2025, Forecasting) reports this simple heuristic performs near-optimally compared with
#   fully constrained non-negative reconciliation.
#   Ranges (7f) are put around the forecast to use: forecast + percentile x scale, lower end never below 0.
#   SAVE: set F1O_SAVE = True only if you want the table written to a NEW folder "F1 outputs" next to the
#   project folders (never into "eabl parsed csv"); an existing file is never overwritten.
#
# WHY (FPP2 2nd ed., Hyndman & Athanasopoulos 2018):
#   Ch 10 introduction (pp.392-393): coherent forecasts "add up in a manner that is consistent with the aggregation structure".
#   3.5 (p.88): 80% and 95% ranges.   Ch 4 (p.101): judgement where there is no history (flagged).
#   3.4 (p.77): weeks 54-66 are scored once, in Step 8 - this table is not scored here.
#   Not in FPP2 (named, peer-reviewed): Di Fonzo & Girolimetto (2023) and Girolimetto (2025) - set-negative-to-zero.
#
# WHERE EACH INPUT COMES FROM (all already in memory; nothing read from disk):
#   f1u_bt (base, bottom-up and reconciled forecasts), F1U_CHOICE, F1U_IDS, F1U_BOTTOM, F1U_S -> cell 22
#   f1y_fc (method, group, scale, 3x flag) -> cell 21;  f1r_q, f1r_cv, F1R_MIN_SCALE -> 7f cell 18;  f1z_risk -> 7f cell 19
#   f1b_level_of -> 7b
#   f1_labels (SKU names), f1_weeks, F1_H -> 7a;  F1E_LEVEL_ORDER -> 7b #6;  pd, np -> 7a imports
#   os -> NEW import (Python standard library), used only if F1O_SAVE = True;  display -> Jupyter built-in
#
# WHAT IT CHANGES: nothing.  WHAT IT WRITES: nothing unless F1O_SAVE = True.  New names start with f1o_ / F1O_.
# ==========================================================================
import os

F1O_SAVE = False                                                       # True -> write the CSV (see SAVE above)
F1O_FILE = "../F1 outputs/F1_backtest_forecast_weeks54-66.csv"         # new folder, outside "eabl parsed csv"
F1O_NEG_REVIEW = -0.1                                                  # product went this low or lower -> review flag

# ---- 1. Forecast to use: set negative PRODUCTS to 0, then add up every level again (v2) ----
f1o_rec = f1u_bt.pivot(index="unique_id", columns="ds", values="reconciled (chosen)").reindex(F1U_IDS)
f1o_prod = f1o_rec.loc[F1U_BOTTOM].clip(lower=0)                         # 188 products, sales cannot be negative
f1o_fc = pd.DataFrame(F1U_S @ f1o_prod.values, index=F1U_IDS, columns=f1o_rec.columns)   # every level re-added
f1o_fc = f1o_fc.stack().rename("forecast").reset_index()
f1o_fc.columns = ["unique_id", "ds", "forecast"]
f1o_review = sorted(f1o_rec.loc[F1U_BOTTOM].index[f1o_rec.loc[F1U_BOTTOM].min(axis=1) <= F1O_NEG_REVIEW])

# ---- 2. One table: forecast to use + ranges + flags ----
f1o = (f1u_bt.merge(f1y_fc[["unique_id", "ds", "group", "scale", "over 3x past max"]], on=["unique_id", "ds"],
                    how="left", validate="one_to_one")
             .merge(f1o_fc, on=["unique_id", "ds"], how="left", validate="one_to_one"))
f1o["set to 0 (was below 0)"] = f1o["unique_id"].isin(F1U_BOTTOM) & (f1o["reconciled (chosen)"] < 0)
f1o["change from set-to-0 (cases)"] = f1o["forecast"] - f1o["reconciled (chosen)"]
f1o_ok = f1o["group"].isin(f1r_q.index) & (f1o["scale"] >= F1R_MIN_SCALE)
for f1o_n, f1o_lo, f1o_hi in [("80", 0.10, 0.90), ("95", 0.025, 0.975)]:
    f1o_l = (f1o["forecast"] + f1o["group"].map(f1r_q[f1o_lo]) * f1o["scale"]).clip(lower=0)
    f1o_h = (f1o["forecast"] + f1o["group"].map(f1r_q[f1o_hi]) * f1o["scale"]).clip(lower=f1o_l)
    f1o[f"lo{f1o_n}"], f1o[f"hi{f1o_n}"] = f1o_l.where(f1o_ok), f1o_h.where(f1o_ok)
f1o["week"] = f1o["ds"].map(lambda f1o_d: int(len(f1_weeks) + 1 + (f1o_d - f1_weeks[-1]).days // 7 - 1))   # 54-66
f1o["tested"] = f1o["unique_id"].isin(set(f1r_cv["unique_id"]))          # old cell 168 idea
f1o = (f1o.merge(f1_labels[["unique_id", "SKU"]], on="unique_id", how="left")
          .merge(f1z_risk[["unique_id", "flag list"]], on="unique_id", how="left"))
f1o["flag list"] = f1o["flag list"].fillna("")
f1o_rv = f1o["unique_id"].isin(f1o_review)                               # v2: products cut below 0 by reconciliation
f1o.loc[f1o_rv, "flag list"] = (f1o.loc[f1o_rv, "flag list"] + "; set to 0 by reconciliation - review").str.lstrip("; ")
f1o = f1o.rename(columns={"method": "base method"})[
    ["level", "unique_id", "SKU", "ds", "week", "base method", "tested", "base", "Bottom-up", F1U_CHOICE, "forecast",
     "lo80", "hi80", "lo95", "hi95", "set to 0 (was below 0)", "change from set-to-0 (cases)", "over 3x past max",
     "flag list"]]

# ---- 3. Checks (old cell 170, computed from the data - no hard-coded counts) ----
assert len(f1o) == len(F1U_IDS) * F1_H and f1o["unique_id"].nunique() == 283, "rows must be 283 series x 13 weeks"
assert (f1o.groupby("unique_id")["week"].apply(lambda f1o_w: sorted(f1o_w) == list(range(54, 67)))).all(), "weeks 54-66"
assert not f1o["unique_id"].str.contains("Returnable", case=False).any(), "Returnable must not be in the forecast"
assert f1o["forecast"].notna().all() and (f1o["forecast"] >= 0).all()
f1o_w = f1o.pivot(index="unique_id", columns="ds", values="forecast").reindex(F1U_IDS)
f1o_gap = np.abs(F1U_S @ f1o_w.loc[F1U_BOTTOM].values - f1o_w.values)    # does the forecast to use still add up?
f1o_gap_lvl = pd.Series(f1o_gap.max(axis=1), index=F1U_IDS).groupby(f1o_w.index.map(f1b_level_of)).max()
assert f1o_gap.max() < 1e-6, "v2: every level must add up exactly after set-negative-to-zero"

# ---- 4. Optional save (never into the dataset folder, never over an existing file) ----
if F1O_SAVE:
    assert "eabl parsed csv" not in F1O_FILE, "never write into the dataset folder"
    assert not os.path.exists(F1O_FILE), f"{F1O_FILE} already exists - choose a new name; nothing was overwritten"
    os.makedirs(os.path.dirname(F1O_FILE), exist_ok=True)
    f1o.to_csv(F1O_FILE, index=False)

# ---- Report ----
print("STEP 7g cell 23 (v2) - final backtest table, weeks 54-66 (not scored - Step 8 does that once)\n")
print(f"f1o: {len(f1o):,} rows = {f1o['unique_id'].nunique()} series x {F1_H} weeks | reconciliation: {F1U_CHOICE} | "
      f"tested series: {int(f1o.drop_duplicates('unique_id')['tested'].sum())} of 283")
print(f"Set-negative-to-zero: {int(f1o['set to 0 (was below 0)'].sum())} product rows "
      f"({f1o.loc[f1o['set to 0 (was below 0)'], 'unique_id'].nunique()} products) were below 0 and set to 0 | "
      f"cases added back: {f1o.loc[f1o['level'] == 'Product', 'change from set-to-0 (cases)'].sum():.1f} | "
      f"largest gap between levels: {f1o_gap.max():.2f} cases (must be 0)")
print(f"Products flagged 'set to 0 by reconciliation - review' (went to {F1O_NEG_REVIEW} case or lower): "
      f"{len(f1o_review)}")
display(f1o[f1o["unique_id"].isin(f1o_review)].groupby(["unique_id", "SKU", "base method"])
        .agg(base_per_week=("base", "mean"), reconciled_lowest=(F1U_CHOICE, "min"), weeks_set_to_0=("set to 0 (was below 0)", "sum"))
        .round(2))
display(f1o_gap_lvl.rename("largest gap (cases)").reindex(F1E_LEVEL_ORDER).round(2).to_frame())
f1o_sum = f1o.groupby("level")[["base", "Bottom-up", F1U_CHOICE, "forecast"]].sum().reindex(F1E_LEVEL_ORDER)
f1o_sum["with a range (share of rows)"] = f1o.groupby("level")["lo80"].apply(lambda f1o_s: f1o_s.notna().mean()).round(2)
print("13-week totals per level:")
display(f1o_sum.round({"base": 0, "Bottom-up": 0, F1U_CHOICE: 0, "forecast": 0}))
print(f"Series with review flags (7f): {int((f1o.drop_duplicates('unique_id')['flag list'] != '').sum())} | "
      f"3x safety check: {int(f1o['over 3x past max'].sum())} weeks flagged")
display(f1o[f1o["level"] == "Category2"].head(4))
print(("SAVED to " + F1O_FILE) if F1O_SAVE else "Not saved (F1O_SAVE = False) - the table is in memory as f1o.")
print("NEXT: Step 8 - open weeks 54-66 ONCE and score these forecasts and ranges.")


In [ ]:
# ==========================================================================
# CHART CELL: THE PRODUCT HIERARCHY - (1) ITS STRUCTURE, (2) WEEKLY SALES AT EACH LEVEL (weeks 1-53 only)
#   Chart 1 - the family tree: Total -> 3 categories -> 8 sub-categories, with the number of brands and
#             products under each sub-category (83 brands and 188 products are too many to draw one by one).
#   Chart 2 - weekly sales at the top 3 levels, like FPP2 Figure 10.2 / Hyndman's lecture slides:
#             Total; the 3 categories; the 8 sub-categories (coloured by their category, so you can see which
#             lines add up to which).  Weeks 54-66 are sealed (7a) and are NOT in f1_Y, so they cannot appear.
#
# WHY (FPP2 2nd ed., Hyndman & Athanasopoulos 2018):
#   10.1 (pp.394-397): a hierarchical time series and its tree diagram (Figure 10.1); plotting each level
#   (Figure 10.2) to see how patterns differ between levels - smooth at the top, noisier lower down.
#
# WHERE EACH INPUT COMES FROM (all already in memory; nothing read from disk):
#   f1_Y (weekly sales, weeks 1-53), f1_tags (series per level), F1_BOTTOM, F1_SEAL -> 7a
#   F1B_LEVELS (tag key -> level name) -> 7b;  plt -> Cell B;  pd, np -> 7a imports
#
# WHAT IT CHANGES: nothing.  WHAT IT WRITES: nothing (shows 2 charts).  New names start with f1h_ / F1H_.
# ==========================================================================
F1H_CAT_COL = {"Beer": "#E69F00", "RTD": "#009E73", "Spirits": "#0072B2"}     # colour-blind-safe (Okabe-Ito)
F1H_STYLES = ["-", "--", ":", "-."]                                            # sub-categories within a category
f1h_key = {f1h_v: f1h_k for f1h_k, f1h_v in F1B_LEVELS.items()}               # level name -> f1_tags key
f1h_cats = list(f1_tags[f1h_key["Category2"]])
f1h_subs = list(f1_tags[f1h_key["Sub_Category"]])
f1h_brands = pd.Series(list(f1_tags[f1h_key["Brand"]]))
f1h_prods = pd.Series(list(f1_tags[F1_BOTTOM]))
assert f1_Y["ds"].max() <= F1_SEAL, "weeks after the seal must not be plotted"

# ---- Chart 1: the family tree ----
f1h_fig, f1h_ax = plt.subplots(figsize=(15, 6.5))
f1h_ax.axis("off")
f1h_y = {"Total": 3.0, "Category2": 2.0, "Sub_Category": 1.0, "counts": 0.0}
f1h_xs = {f1h_s: f1h_i for f1h_i, f1h_s in enumerate(f1h_subs)}                # sub-categories spread evenly
f1h_xc = {f1h_c: np.mean([f1h_xs[f1h_s] for f1h_s in f1h_subs if f1h_s.split("/")[0] == f1h_c]) for f1h_c in f1h_cats}
f1h_xt = np.mean(list(f1h_xc.values()))

def f1h_box(f1h_x, f1h_yy, f1h_txt, f1h_col):
    f1h_ax.text(f1h_x, f1h_yy, f1h_txt, ha="center", va="center", fontsize=10, color="black",
                bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor=f1h_col, linewidth=2))

def f1h_line(f1h_x0, f1h_y0, f1h_x1, f1h_y1):
    f1h_ax.plot([f1h_x0, f1h_x1], [f1h_y0 - 0.18, f1h_y1 + 0.18], color="#9E9E9E", linewidth=1, zorder=0)

f1h_box(f1h_xt, f1h_y["Total"], "Total EABL\n(1 series)", "#000000")
for f1h_c in f1h_cats:
    f1h_line(f1h_xt, f1h_y["Total"], f1h_xc[f1h_c], f1h_y["Category2"])
    f1h_box(f1h_xc[f1h_c], f1h_y["Category2"], f1h_c, F1H_CAT_COL.get(f1h_c, "#000000"))
for f1h_s in f1h_subs:
    f1h_c = f1h_s.split("/")[0]
    f1h_nb = int(f1h_brands.str.startswith(f1h_s + "/").sum())
    f1h_np = int(f1h_prods.str.startswith(f1h_s + "/").sum())
    f1h_line(f1h_xc[f1h_c], f1h_y["Category2"], f1h_xs[f1h_s], f1h_y["Sub_Category"])
    f1h_box(f1h_xs[f1h_s], f1h_y["Sub_Category"], f1h_s.split("/")[1], F1H_CAT_COL.get(f1h_c, "#000000"))
    f1h_line(f1h_xs[f1h_s], f1h_y["Sub_Category"], f1h_xs[f1h_s], f1h_y["counts"])
    f1h_ax.text(f1h_xs[f1h_s], f1h_y["counts"], f"{f1h_nb} brand{'s' * (f1h_nb != 1)}\n{f1h_np} product{'s' * (f1h_np != 1)}", ha="center", va="center",
                fontsize=9, color="#424242")
for f1h_lv, f1h_txt in [("Total", "Level 1: Total (1)"), ("Category2", f"Level 2: Category2 ({len(f1h_cats)})"),
                        ("Sub_Category", f"Level 3: Sub_Category ({len(f1h_subs)})"),
                        ("counts", f"Levels 4-5: Brand ({len(f1h_brands)}) / Product ({len(f1h_prods)})")]:
    f1h_ax.text(-1.0, f1h_y[f1h_lv], f1h_txt, ha="right", va="center", fontsize=10, color="#424242")
f1h_ax.set_xlim(-3.6, len(f1h_subs) - 0.4); f1h_ax.set_ylim(-0.5, 3.4)
f1h_ax.set_title(f"EABL product hierarchy: {1 + len(f1h_cats) + len(f1h_subs) + len(f1h_brands) + len(f1h_prods)} "
                 "series that must add up (FPP2 10.1)", loc="left", fontsize=12)
plt.tight_layout(); plt.show()

# ---- Chart 2: weekly sales at the top 3 levels (weeks 1-53) ----
f1h_w = f1_Y.pivot(index="ds", columns="unique_id", values="y")
f1h_fig, f1h_axes = plt.subplots(1, 3, figsize=(16, 4.6))
f1h_axes[0].plot(f1h_w.index, f1h_w["Total"], color="#000000", linewidth=1.8)
f1h_axes[0].set_title("Total", loc="left")
for f1h_c in f1h_cats:
    f1h_axes[1].plot(f1h_w.index, f1h_w[f1h_c], color=F1H_CAT_COL.get(f1h_c), linewidth=1.8, label=f1h_c)
f1h_axes[1].set_title("Category2 (3)", loc="left")
f1h_axes[1].legend(frameon=False, fontsize=9)
for f1h_c in f1h_cats:
    for f1h_i, f1h_s in enumerate([f1h_s for f1h_s in f1h_subs if f1h_s.split("/")[0] == f1h_c]):
        f1h_axes[2].plot(f1h_w.index, f1h_w[f1h_s], color=F1H_CAT_COL.get(f1h_c),
                         linestyle=F1H_STYLES[f1h_i % len(F1H_STYLES)], linewidth=1.6, label=f1h_s)
f1h_axes[2].set_title("Sub_Category (8) - colour = its category", loc="left")
f1h_axes[2].legend(frameon=False, fontsize=8, ncol=2)
for f1h_a in f1h_axes:
    f1h_a.set_xlabel("week starting")
    f1h_a.grid(axis="y", color="#E0E0E0", linewidth=0.8)
    f1h_a.spines[["top", "right"]].set_visible(False)
    f1h_a.tick_params(axis="x", labelrotation=30, labelsize=8)
f1h_axes[0].set_ylabel("cases per week")
f1h_fig.suptitle(f"Weekly sales by level, {f1h_w.index.min().date()} to {f1h_w.index.max().date()} "
                 "(weeks 1-53; 54-66 sealed)", x=0.01, ha="left", fontsize=12)
plt.tight_layout(); plt.show()
print("Each line in the middle chart = the sum of its sub-categories on the right; the 3 categories add up to Total.")


In [ ]:
# ==========================================================================
# STEP 7 - PART 7h, CELL 24 (Roadmap I2): THE 8 DEC 2025 SPIKE - DO FORECASTS GET BETTER WITHOUT IT? (cv2 only)
#   The week of 8 Dec 2025 (week 36, contains Jamhuri Day) had unusually high sales.  F8 (Prep cell 67) flagged
#   17 products as "spike" that week, each judged against its OWN 13 earlier weeks ("usual_week").
#   This cell asks: if those 17 spikes are replaced by their usual week, do the forecasts improve?
#     WITH    = the real training weeks (as used everywhere in Step 7)
#     WITHOUT = the same weeks, but each of the 17 spike values replaced by its usual_week (earlier weeks only,
#               no look-ahead); brands, sub-categories, categories and Total are re-added from the products (S),
#               so the hierarchy still adds up.  In MEMORY only - no file is changed.
#   Only cv2 can judge it: cv2 learns weeks 1-40 (the spike is inside) and predicts 41-53; cv1 learns weeks 1-27
#   (no spike).  Both versions use the SAME methods (the 7g plan, f1y_plan) and the SAME MinT weights (from cv1
#   errors, f1u_W_cv1), so the only difference is the spike.  Only series whose numbers change are refitted.
#   Scored on cv2 real sales (weeks 41-53) with MASE per level, as in 7b-7g.
#   DECISION RULE (set BEFORE running, F1J_MIN_GAIN): WITHOUT is adopted only if the reconciled (MinT) mean of the
#     5 levels improves by at least 2% AND at least 3 of the 5 levels improve.  Otherwise the spike stays (no change).
#   Limits (stated honestly): one round of 13 weeks only; the spike was REAL Christmas demand, not a data error.
#
# WHY (FPP2 2nd ed., Hyndman & Athanasopoulos 2018):
#   12.9: outliers - replacing them is an option, but only if they are errors or one-off events; test, do not assume.
#   5.4 (p.157): special events may need special treatment (7d tested holiday flags - not used).
#   3.4 (p.77): judged on weeks not used to fit (cv2 weeks 41-53); weeks 54-66 stay sealed.
#
# WHERE EACH INPUT COMES FROM (all already in memory; nothing read from disk):
#   f1_train["cv2"] (weeks 1-40), f1_flags_train["cv2"] (F8 flags, weeks 1-40), f1_weeks, F1_CUTOFF, F1_H -> 7a
#   StatsForecast, HistoricAverage -> 7b imports;  F1L_Z95 -> 7b #11
#   f1y_plan, F1Y_RAW, F1Y_LOG -> 7g cell 21;  F1U_IDS, F1U_BOTTOM, F1U_S, f1u_reconcile, f1u_W_cv1, f1u_y_cv,
#   f1u_sc2, f1u_ok, f1u_lev -> 7g cell 22 v2;  F1E_LEVEL_ORDER -> 7b #6;  pd, np -> 7a imports;  display -> Jupyter
#
# RUN IT ONCE, BEFORE the 7a spike cell exists.  After the cleaning cell has run, this cell only prints "skipped".
# WHAT IT CHANGES: nothing.  WHAT IT WRITES: nothing (memory only).  New names start with f1j_ / F1J_.
# ==========================================================================
if "F1SP_DONE" in globals():          # the cleaning cell (7a spike) already ran: this test is decided, do not repeat
    print("I2 test skipped: the 8 Dec spike cleaning (7a spike cell) is already applied in this session.")
    print("The test was decided before the cleaning (result: ADOPT WITHOUT); re-running it now would clean twice.")
else:
    F1J_WEEK = pd.Timestamp("2025-12-08")                  # week 36, the spike week (Jamhuri Day, 12 Dec)
    F1J_MIN_GAIN = 0.02                                    # WITHOUT must improve the MinT mean of 5 levels by >= 2%
    F1J_MIN_LEVELS = 3                                     # ... and improve at least 3 of the 5 levels

    # ---- 1. The 17 spikes and their usual week (F8, earlier weeks only) ----
    f1j_fl = f1_flags_train["cv2"]
    f1j_sp = f1j_fl[(f1j_fl["ds"] == F1J_WEEK) & (f1j_fl["product_flag"] == "spike")][["unique_id", "SKU", "y", "usual_week"]]
    assert len(f1j_sp) > 0 and f1j_sp["usual_week"].notna().all(), "no spike flags found for 8 Dec 2025"

    # ---- 2. WITHOUT: replace the spikes, re-add every level above (the hierarchy still adds up) ----
    f1j_delta = pd.Series(0.0, index=F1U_BOTTOM)
    f1j_delta[f1j_sp["unique_id"].values] = (f1j_sp["usual_week"] - f1j_sp["y"]).values     # change per product
    f1j_up = pd.Series(F1U_S @ f1j_delta.values, index=F1U_IDS)                              # change per series (S)
    f1j_aff = list(f1j_up.index[f1j_up.abs() > 1e-9])                                        # series that change
    f1j_tr = f1_train["cv2"].copy()
    f1j_rows = (f1j_tr["ds"] == F1J_WEEK) & f1j_tr["unique_id"].isin(f1j_aff)
    f1j_tr.loc[f1j_rows, "y"] = f1j_tr.loc[f1j_rows, "y"] + f1j_tr.loc[f1j_rows, "unique_id"].map(f1j_up)
    assert np.isclose(f1j_tr.loc[(f1j_tr["ds"] == F1J_WEEK) & (f1j_tr["unique_id"] == "Total"), "y"].iloc[0],
                      f1_train["cv2"].loc[(f1_train["cv2"]["ds"] == F1J_WEEK) & (f1_train["cv2"]["unique_id"] == "Total"), "y"].iloc[0]
                      + f1j_delta.sum()), "Total did not drop by the removed spike"

    # ---- 3. Forecast cv2 weeks 41-53 with the 7g plan (same methods for WITH and WITHOUT) ----
    f1j_pos = f1_weeks.get_loc(F1_CUTOFF["cv2"])
    f1j_wk = list(f1_weeks[f1j_pos + 1: f1j_pos + 1 + F1_H])

    def f1j_base(f1j_train, f1j_ids):
        """Base forecast (mean of the plan's models) for the given series, learned on f1j_train; 0 for no-model lanes."""
        f1j_plan = f1y_plan[f1y_plan["unique_id"].isin(f1j_ids) & (f1y_plan["models"].str.len() > 0)
                            & f1y_plan["unique_id"].isin(set(f1j_train["unique_id"]))]
        f1j_out = pd.DataFrame(0.0, index=f1j_ids, columns=f1j_wk)
        f1j_cols = {}
        for f1j_ms in f1j_plan["models"]:
            for f1j_m in f1j_ms:
                f1j_cols.setdefault(f1j_m, None)
        for f1j_col in f1j_cols:
            f1j_is_log = f1j_col.endswith("_log")
            f1j_ids_m = f1j_plan.loc[f1j_plan["models"].apply(lambda f1j_l: f1j_col in f1j_l), "unique_id"]
            f1j_d = f1j_train[f1j_train["unique_id"].isin(f1j_ids_m)][["unique_id", "ds", "y"]]
            f1j_model = (F1Y_LOG[f1j_col[:-4]] if f1j_is_log else F1Y_RAW[f1j_col])()
            f1j_sf = StatsForecast(models=[f1j_model], freq="W-MON", n_jobs=-1, fallback_model=HistoricAverage(alias=f1j_col))
            if f1j_is_log:
                f1j_f = f1j_sf.forecast(df=f1j_d.assign(y=np.log1p(f1j_d["y"])), h=F1_H, level=[95])
                f1j_sig = (f1j_f[f"{f1j_col}-hi-95"] - f1j_f[f"{f1j_col}-lo-95"]) / (2 * F1L_Z95)
                f1j_f[f1j_col] = np.exp(f1j_f[f1j_col]) * (1 + f1j_sig ** 2 / 2) - 1           # 7b #11 bias adjustment
            else:
                f1j_f = f1j_sf.forecast(df=f1j_d, h=F1_H)
            f1j_cols[f1j_col] = f1j_f.pivot(index="unique_id", columns="ds", values=f1j_col).reindex(columns=f1j_wk)
        for f1j_u, f1j_ms in zip(f1j_plan["unique_id"], f1j_plan["models"]):
            f1j_out.loc[f1j_u] = np.mean([f1j_cols[f1j_m].loc[f1j_u].values for f1j_m in f1j_ms], axis=0)
        return f1j_out

    f1j_with = f1j_base(f1_train["cv2"], F1U_IDS)                          # all 283 series, real weeks
    f1j_without = f1j_with.copy()
    f1j_without.loc[f1j_aff] = f1j_base(f1j_tr, f1j_aff).values            # only the changed series are refitted

    # ---- 4. Score both on cv2 real sales: base and MinT (same W, from cv1 errors) ----
    def f1j_score(f1j_mat):
        f1j_m = pd.Series(np.abs(f1u_y_cv["cv2"] - f1j_mat).mean(axis=1) / f1u_sc2.values, index=F1U_IDS)[f1u_ok]
        f1j_by = f1j_m.groupby(f1u_lev[f1u_ok]).mean().reindex(F1E_LEVEL_ORDER)
        return pd.concat([f1j_by, pd.Series({"mean of the 5 levels": f1j_by.mean()})])

    f1j_res = pd.DataFrame({
        "base WITH": f1j_score(f1j_with.values), "base WITHOUT": f1j_score(f1j_without.values),
        "MinT WITH": f1j_score(f1u_reconcile(f1j_with.values, f1u_W_cv1)["MinT shrink"]),
        "MinT WITHOUT": f1j_score(f1u_reconcile(f1j_without.values, f1u_W_cv1)["MinT shrink"])})
    f1j_res["MinT change (%)"] = (f1j_res["MinT WITHOUT"] / f1j_res["MinT WITH"] - 1) * 100

    # ---- 5. The decision rule (set before running) ----
    f1j_gain = -f1j_res.at["mean of the 5 levels", "MinT change (%)"] / 100
    f1j_levels_better = int((f1j_res.loc[F1E_LEVEL_ORDER, "MinT WITHOUT"] < f1j_res.loc[F1E_LEVEL_ORDER, "MinT WITH"]).sum())
    F1J_ADOPT_WITHOUT = bool(f1j_gain >= F1J_MIN_GAIN and f1j_levels_better >= F1J_MIN_LEVELS)

    # ---- Report ----
    print("STEP 7h cell 24 (I2) - 8 Dec 2025 spike: WITH vs WITHOUT, judged on cv2 (nothing written; weeks 54-66 sealed)\n")
    print(f"Spike products (F8, week {F1J_WEEK.date()}): {len(f1j_sp)} | sales that week {f1j_sp['y'].sum():,.0f} cases -> "
          f"usual {f1j_sp['usual_week'].sum():,.0f} (removed {-f1j_delta.sum():,.0f}) | series refitted: {len(f1j_aff)} "
          f"of 283 (the 17 products + the brands, sub-categories, categories and Total above them)")
    display(f1j_sp.assign(removed=lambda d: d["y"] - d["usual_week"]).sort_values("removed", ascending=False)
            .round(1).reset_index(drop=True))
    print("Mean MASE per level on cv2 weeks 41-53 (lower is better):")
    display(f1j_res.round(3))
    print(f"Decision rule (set before running): adopt WITHOUT only if the MinT mean improves by >= {F1J_MIN_GAIN:.0%} "
          f"AND >= {F1J_MIN_LEVELS} of 5 levels improve.")
    print(f"  MinT mean change: {-f1j_gain:+.1%} | levels better without the spike: {f1j_levels_better} of 5")
    print(f"  RESULT: {'ADOPT WITHOUT - tell Claude; 7a-7g must be redone with the cleaned week before Step 8' if F1J_ADOPT_WITHOUT else 'KEEP THE SPIKE - nothing changes; go to Step 8'}")
    print("Limits: one round (13 weeks); the spike was real Christmas demand.")


In [ ]:
# ==========================================================================
# STEP 7 - PART 7g, SAVE CELL: WRITE THE FINAL BACKTEST TABLE (f1o) TO ITS OWN FILE - ONE FILE PER VERSION
#   Paste this cell right AFTER 7g cell 23 v2.  Run it once per version:
#     1st run - data as recorded (WITH the December spikes; cell "7a SPIKE v2" absent or switch OFF)
#               -> ../F1 outputs/F1_backtest_with_spikes_weeks54-66.csv
#     2nd run - after "7a SPIKE v2" with F1SP_CLEAN = True and Restart & Run All (December spikes replaced)
#               -> ../F1 outputs/F1_backtest_dec_cleaned_weeks54-66.csv
#   The version is detected automatically (F1SP_DONE exists only when the December cleaning ran), so the two
#   files can never be mixed up.  Step 8 will read both files and score both.
#   SAFETY: writes only into a NEW folder "F1 outputs" next to the project folders - never into "eabl parsed csv";
#   never overwrites: if the file already exists it prints "NOT saved" and moves on (so Run All never stops here).
#
# WHY: two versions of the backtest are kept (user decision, 11 Nov 2026) - restarting the kernel clears memory,
#   so each version's table must be saved before the other is run.  Weeks 54-66 are still sealed: f1o holds only
#   the forecasts made from weeks 1-53, no real sales of weeks 54-66.
#
# WHERE EACH INPUT COMES FROM (memory): f1o, F1U_CHOICE -> 7g cell 23 v2 / cell 22 v2;  F1SP_DONE -> "7a SPIKE v2"
#   (only if the December cleaning ran);  pd -> 7a import;  os -> Python standard library (imported in cell 23 v2)
#
# WHAT IT CHANGES: nothing in memory.  WHAT IT WRITES: one CSV per run (3,679 rows).  New names start with f1sv_ / F1SV_.
# ==========================================================================
import os

F1SV_DIR = "../F1 outputs/"                                                 # new folder, outside "eabl parsed csv"
F1SV_VERSION = "dec_cleaned" if globals().get("F1SP_DONE", False) else "with_spikes"
F1SV_FILE = F1SV_DIR + f"F1_backtest_{F1SV_VERSION}_weeks54-66.csv"

assert "f1o" in globals() and len(f1o) == 283 * 13, "run 7g cells 21 -> 22 v2 -> 23 v2 first"
assert f1o["week"].min() == 54 and f1o["week"].max() == 66, "f1o must hold weeks 54-66 only"
assert "eabl parsed csv" not in F1SV_FILE, "never write into the dataset folder"
if os.path.exists(F1SV_FILE):                                              # never overwrite
    print(f"NOT saved: {F1SV_FILE} already exists (nothing was overwritten). Rename or delete it yourself to re-save.")
else:
    os.makedirs(F1SV_DIR, exist_ok=True)
    f1sv_out = f1o.assign(version=F1SV_VERSION, reconciliation=F1U_CHOICE)
    f1sv_out.to_csv(F1SV_FILE, index=False)
    f1sv_back = pd.read_csv(F1SV_FILE)                                       # read back and check
    assert len(f1sv_back) == len(f1sv_out) and abs(f1sv_back["forecast"].sum() - f1o["forecast"].sum()) < 1e-6
    print(f"SAVED version '{F1SV_VERSION}': {F1SV_FILE}")
    print(f"  {len(f1sv_back):,} rows (283 series x 13 weeks) | forecast total, weeks 54-66: "
          f"{f1o.loc[f1o['level'] == 'Total', 'forecast'].sum():,.0f} cases | reconciliation: {F1U_CHOICE}")
print("  Files in the folder now:", sorted(f for f in os.listdir(F1SV_DIR) if f.endswith(".csv")) if os.path.isdir(F1SV_DIR) else [])


In [ ]:
f1view = pd.read_csv("../F1 outputs/F1_backtest_with_spikes_weeks54-66.csv")
display(f1view.head(7000))


In [ ]:
# ==========================================================================
# STEP 8: OPEN WEEKS 54-66 ONCE AND SCORE BOTH SAVED BACKTEST VERSIONS (OFFICIAL = December cleaned)
#   Decided BEFORE this cell was run (11 Nov 2026, Option 1):
#     OFFICIAL   = "dec_cleaned"  - models learned from weeks 1-53 with the 84 December spikes replaced (7a SPIKE v2)
#     COMPARISON = "with_spikes"  - models learned from weeks 1-53 exactly as recorded
#   Both were frozen as files by the 7g save cell before any real sales of weeks 54-66 were seen.  This cell reads
#   BOTH files (not memory), opens the real sales of weeks 54-66 ONCE from the data file, and scores both.
#   Nothing in either file is changed.  The OFFICIAL score is the honest test result (Step 0 rule (d)); the
#   comparison only shows what the December cleaning did.  Rule R (approved before Step 8) is checked on the OFFICIAL.
#   What is scored:
#     A. accuracy per level: MASE of both forecasts + the official's MinT (before set-to-0), bottom-up, base, and the
#        NAIVE benchmark (week 53 repeated)
#     B. 13-week totals (real vs both) and bias per level (median over series of forecast / real)
#     C. official products by lane (standard average, TSB, short history, one-sale, stopped, new launches)
#     D. tested vs untested series (official)
#     E. 80% / 95% range coverage per group (official), overall coverage of both, Rule R trigger (official)
#     F. review flags (official) and the products set to 0 by reconciliation - what really sold
#     G. the series where the two versions differ most (e.g. festive spirits) - real sales vs both forecasts
#   MASE (FPP2 3.4): mean |real - forecast| over weeks 54-66 / mean |weekly change| of the REAL weeks 1-53 - the same
#     scale for both versions (not the cleaned weeks), so the comparison is fair.  Series with a scale below 0.05 cases
#     or no weeks before the seal get no MASE (shown in C with cases).
#
# WHY (FPP2 2nd ed., Hyndman & Athanasopoulos 2018):
#   3.4 (p.77): "The accuracy of forecasts can only be determined by considering how well a model performs on new
#     data that were not used when fitting the model"; the test set should be "at least as large as the maximum
#     forecast horizon" (13 = 13).   3.4 (pp.80-81): scaled errors (Hyndman & Koehler 2006).
#   3.1 (p.57): simple methods as benchmarks (naive).   10.7 (pp.421-422, Table 10.1): accuracy for every level.
#   3.5 (p.88): an 80% / 95% interval should contain the real value about 80% / 95% of the time.
#   Not in FPP2 (named, peer-reviewed): Hyndman & Koehler (2006, Int. J. Forecasting 22:679-688) - MASE.
#
# WHERE EACH INPUT COMES FROM:
#   FILES READ (nothing written):
#     ../F1 outputs/F1_backtest_dec_cleaned_weeks54-66.csv  -> 7g save cell, run 2 (official)
#     ../F1 outputs/F1_backtest_with_spikes_weeks54-66.csv  -> 7g save cell, run 1 (comparison)
#     ../eabl parsed csv/Y_product_with_total.csv           -> Prep cell 84: real weeks 1-53 (scale, naive) and
#                                                              weeks 54-66 (opened here, once)
#     ../eabl parsed csv/S_product_with_total.csv           -> Prep cell 84: check that real sales add up (y = S b)
#   MEMORY: F1_DIR, F1_SEAL -> 7a;  pd, np -> 7a imports;  display -> Jupyter
#   hashlib -> NEW import (Python standard library): fingerprints proving the two files were not changed
#
# WHAT IT CHANGES: nothing.  WHAT IT WRITES: nothing (memory only).  New names start with f8_ / F8_.
# ==========================================================================
import hashlib

F8_OUT_DIR = "../F1 outputs/"
F8_VERSIONS = {"official": "dec_cleaned", "comparison": "with_spikes"}     # decided before Step 8 (Option 1)
F8_MIN_SCALE = 0.05                                                       # same threshold as 7f (F1R_MIN_SCALE)
F8_RULE_R_LIMIT = 0.70                                                    # Rule R: 80% range holds in < 70% of weeks
F8_OFF, F8_CMP = F8_VERSIONS["official"], F8_VERSIONS["comparison"]

# ---- 0. Read the two frozen files + fingerprints ----
f8_fc, f8_fp = {}, {}
for f8_v in F8_VERSIONS.values():
    f8_path = F8_OUT_DIR + f"F1_backtest_{f8_v}_weeks54-66.csv"
    with open(f8_path, "rb") as f8_f:
        f8_fp[f8_v] = hashlib.sha1(f8_f.read()).hexdigest()[:12]
    f8_fc[f8_v] = pd.read_csv(f8_path, parse_dates=["ds"])
    f8_fc[f8_v]["flag list"] = f8_fc[f8_v]["flag list"].fillna("")
    assert (f8_fc[f8_v]["version"] == f8_v).all() and len(f8_fc[f8_v]) == 283 * 13, f"{f8_path} is not the expected file"
assert f8_fc[F8_OFF].set_index(["unique_id", "ds"]).index.equals(f8_fc[F8_CMP].set_index(["unique_id", "ds"]).index), \
    "the two files do not cover the same series and weeks"
f8_ids = list(f8_fc[F8_OFF]["unique_id"].drop_duplicates())
f8_weeks = sorted(f8_fc[F8_OFF]["ds"].unique())
f8_level = f8_fc[F8_OFF].drop_duplicates("unique_id").set_index("unique_id")["level"]
F8_LEVEL_ORDER = ["Total", "Category2", "Sub_Category", "Brand", "Product"]

# ---- 1. Open the seal: real sales of weeks 54-66 (once); real weeks 1-53 for scale and naive ----
f8_file = pd.read_csv(F1_DIR + "Y_product_with_total.csv", parse_dates=["ds"])
f8_learn = f8_file[f8_file["ds"] <= F1_SEAL]                              # real weeks 1-53 (never cleaned)
f8_test = f8_file.loc[f8_file["ds"] > F1_SEAL, ["unique_id", "ds", "y"]]
del f8_file
assert sorted(f8_test["ds"].unique()) == f8_weeks, "the data file's weeks 54-66 do not match the forecast weeks"
f8_y = f8_test.pivot(index="unique_id", columns="ds", values="y").reindex(index=f8_ids, columns=f8_weeks).fillna(0.0)
f8_S = pd.read_csv(F1_DIR + "S_product_with_total.csv", index_col=0).set_index("unique_id").reindex(f8_ids)
f8_bottom = list(f8_S.columns)
assert np.allclose(f8_S.values.astype(float) @ f8_y.loc[f8_bottom].values, f8_y.values), "real sales do not add up (y = S b)"
f8_act = f8_y.stack().rename("actual").reset_index()
f8_act.columns = ["unique_id", "ds", "actual"]

f8_scale = f8_learn.sort_values(["unique_id", "ds"]).groupby("unique_id")["y"].apply(lambda s: s.diff().abs().mean()).reindex(f8_ids)
f8_naive = f8_learn[f8_learn["ds"] == F1_SEAL].set_index("unique_id")["y"].reindex(f8_ids).fillna(0.0)
f8_ok = f8_scale >= F8_MIN_SCALE                                          # MASE possible for this series

# ---- 2. One scoring table per version ----
for f8_v in F8_VERSIONS.values():
    f8_fc[f8_v] = f8_fc[f8_v].merge(f8_act, on=["unique_id", "ds"], how="left", validate="one_to_one")
    f8_fc[f8_v]["Naive"] = f8_fc[f8_v]["unique_id"].map(f8_naive)

def f8_mase(f8_df, f8_col):
    """MASE per series: mean |real - forecast| over weeks 54-66 / mean |weekly change| of real weeks 1-53."""
    f8_mae = (f8_df["actual"] - f8_df[f8_col]).abs().groupby(f8_df["unique_id"]).mean().reindex(f8_ids)
    return (f8_mae / f8_scale)[f8_ok]

f8_o, f8_c = f8_fc[F8_OFF], f8_fc[F8_CMP]
f8_m = pd.DataFrame({f"OFFICIAL {F8_OFF}": f8_mase(f8_o, "forecast"), f"comparison {F8_CMP}": f8_mase(f8_c, "forecast"),
                     "official MinT (before set-to-0)": f8_mase(f8_o, "MinT shrink"),
                     "official bottom-up": f8_mase(f8_o, "Bottom-up"), "official base": f8_mase(f8_o, "base"),
                     "NAIVE (week 53)": f8_mase(f8_o, "Naive")})
f8_m["level"] = f8_m.index.map(f8_level)

# ---- 3. A: accuracy per level ----
f8_lvl = f8_m.groupby("level").mean(numeric_only=True).reindex(F8_LEVEL_ORDER)
f8_lvl.loc["mean of the 5 levels"] = f8_lvl.mean()
f8_lvl["series scored"] = f8_m.groupby("level").size().reindex(F8_LEVEL_ORDER)

# ---- 4. B: 13-week totals and bias ----
f8_tot = pd.Series({"REAL sales": f8_o.loc[f8_o["unique_id"] == "Total", "actual"].sum(),
                    f"OFFICIAL {F8_OFF}": f8_o.loc[f8_o["unique_id"] == "Total", "forecast"].sum(),
                    f"comparison {F8_CMP}": f8_c.loc[f8_c["unique_id"] == "Total", "forecast"].sum(),
                    "NAIVE (week 53)": f8_o.loc[f8_o["unique_id"] == "Total", "Naive"].sum()})
def f8_bias(f8_df):
    f8_s = f8_df.groupby("unique_id")[["actual", "forecast"]].sum()
    f8_s = f8_s[f8_s["actual"] > 0]
    return (f8_s["forecast"] / f8_s["actual"]).groupby(f8_s.index.map(f8_level)).median().reindex(F8_LEVEL_ORDER)
f8_bias_tab = pd.DataFrame({f"OFFICIAL {F8_OFF}": f8_bias(f8_o), f"comparison {F8_CMP}": f8_bias(f8_c)})

# ---- 5. C: official products by lane; D: tested vs untested ----
F8_LANES = [("launched after the seal (forecast 0)", "zero (launched after the seal)"), ("stopped (forecast 0)", "zero (stopped)"),
            ("one-sale (judgement rate)", "judgement rate"), ("intermittent (TSB)", "TSB"), ("short history (AICc)", "(short history")]
def f8_lane(f8_meth):
    for f8_name, f8_key in F8_LANES:
        if f8_key in f8_meth:
            return f8_name
    return "standard (average of 3)"
f8_p = f8_o[f8_o["level"] == "Product"].copy()
f8_p["lane"] = f8_p["base method"].map(f8_lane)
f8_plane = f8_p.drop_duplicates("unique_id").set_index("unique_id")["lane"]
f8_lane_tab = f8_p.groupby("lane").agg(products=("unique_id", "nunique"), real_13wk=("actual", "sum"),
                                        forecast_13wk=("forecast", "sum"),
                                        weeks_really_0=("actual", lambda f8_a: (f8_a == 0).mean()))
f8_mp = f8_m[f8_m["level"] == "Product"]
f8_lane_tab["mean MASE (official)"] = f8_mp[f"OFFICIAL {F8_OFF}"].groupby(f8_plane).mean()
f8_lane_tab["mean MASE (naive)"] = f8_mp["NAIVE (week 53)"].groupby(f8_plane).mean()
f8_one = f8_o.drop_duplicates("unique_id").set_index("unique_id")
f8_td = f8_m.join(f8_one[["tested"]]).groupby(["level", "tested"])[f"OFFICIAL {F8_OFF}"].agg(["mean", "size"]).round(3)

# ---- 6. E: range coverage (rows with a range) + Rule R (official) ----
def f8_group(f8_df):
    f8_g = f8_df["level"].copy()
    f8_g[f8_df["base method"].str.contains("TSB")] = f8_df["level"] + " (intermittent, TSB)"
    f8_g[f8_df["base method"].str.contains("sparse brand")] = "Brand (sparse)"
    return f8_g
def f8_cover(f8_df):
    f8_r = f8_df[f8_df["lo80"].notna()].copy()
    for f8_n in ["80", "95"]:
        f8_r[f"inside {f8_n}%"] = f8_r["actual"].between(f8_r[f"lo{f8_n}"], f8_r[f"hi{f8_n}"])
        f8_r[f"above {f8_n}%"] = f8_r["actual"] > f8_r[f"hi{f8_n}"]
        f8_r[f"below {f8_n}%"] = f8_r["actual"] < f8_r[f"lo{f8_n}"]
    return f8_r
F8_COV_COLS = [f"{f8_k} {f8_n}%" for f8_n in ["80", "95"] for f8_k in ["inside", "above", "below"]]
f8_ro, f8_rc = f8_cover(f8_o), f8_cover(f8_c)
f8_cov = f8_ro.groupby(f8_group(f8_ro))[F8_COV_COLS].mean()
f8_cov.insert(0, "weeks", f8_ro.groupby(f8_group(f8_ro)).size())
f8_cov.loc["ALL - OFFICIAL"] = [len(f8_ro)] + list(f8_ro[F8_COV_COLS].mean())
f8_cov.loc["ALL - comparison"] = [len(f8_rc)] + list(f8_rc[F8_COV_COLS].mean())
f8_cov80 = float(f8_ro["inside 80%"].mean())

# ---- 7. F: review flags and products set to 0; G: biggest differences between the versions ----
f8_fl = f8_m.join((f8_one["flag list"] != "").rename("review flag")).groupby(["level", "review flag"])[f"OFFICIAL {F8_OFF}"] \
            .agg(["mean", "size"]).round(3)
f8_rev_ids = f8_one.index[f8_one["flag list"].str.contains("set to 0 by reconciliation")]
f8_rev = f8_o[f8_o["unique_id"].isin(f8_rev_ids)].groupby(["unique_id", "SKU"])[["actual", "forecast"]].sum().round(2)
f8_13 = pd.DataFrame({"REAL": f8_o.groupby("unique_id")["actual"].sum(), F8_OFF: f8_o.groupby("unique_id")["forecast"].sum(),
                      F8_CMP: f8_c.groupby("unique_id")["forecast"].sum()})
f8_13["level"] = f8_13.index.map(f8_level)
f8_13["gap between versions"] = (f8_13[F8_OFF] - f8_13[F8_CMP]).abs()
f8_13["closer"] = np.where((f8_13[F8_OFF] - f8_13["REAL"]).abs() <= (f8_13[F8_CMP] - f8_13["REAL"]).abs(), F8_OFF, F8_CMP)
f8_big = f8_13[f8_13["level"].isin(["Brand", "Product"])].sort_values("gap between versions", ascending=False).head(10)

# ---- Report ----
print("STEP 8 - weeks 54-66 opened ONCE; both frozen versions scored (nothing written; neither file changed)\n")
print(f"OFFICIAL = {F8_OFF} (file fingerprint {f8_fp[F8_OFF]}) | comparison = {F8_CMP} (fingerprint {f8_fp[F8_CMP]})")
print(f"Real sales read: {len(f8_test):,} rows, weeks {pd.Timestamp(f8_weeks[0]).date()} to {pd.Timestamp(f8_weeks[-1]).date()} "
      f"| real sales add up (y = S b) - OK | MASE scale = REAL weeks 1-53 for both versions")
print("\nA. Mean MASE per level, weeks 54-66 (lower is better; below 1 = better than a typical week-to-week change):")
display(f8_lvl.round(3))
print("B. 13-week Total, cases:")
display(f8_tot.round(0).to_frame("13-week total").T)
print("   Bias per level (median over series of forecast / real; 1.00 = right, above 1 = too high):")
display(f8_bias_tab.round(2))
print("C. OFFICIAL products by lane (cases over 13 weeks; MASE only where the training scale is usable):")
display(f8_lane_tab.round(3))
print("D. OFFICIAL tested (practice-round evidence) vs untested - mean MASE:")
display(f8_td)
print("E. Range coverage, OFFICIAL by group + both overall (80% should be near 0.80, 95% near 0.95):")
display(f8_cov.round(2))
print(f"   Rule R (approved before Step 8) on the OFFICIAL: 80% range holds in {f8_cov80:.2f} of weeks -> "
      f"{'TRIGGERED: in Step 9 rebuild the ranges from cv1 + cv2 + test errors (95% range as fallback)' if f8_cov80 < F8_RULE_R_LIMIT else 'not triggered (>= 0.70): ranges kept as they are'}")
print("F. OFFICIAL review flags - mean MASE, flagged vs not flagged:")
display(f8_fl)
print("   Products set to 0 by reconciliation - what really sold in weeks 54-66:")
display(f8_rev)
print("G. Brands/products where the two versions differ most - real 13-week sales vs both forecasts:")
display(f8_big.round(1))
print(f"   Of these 10, the OFFICIAL version was closer for {int((f8_big['closer'] == F8_OFF).sum())}.")
print("These results are FINAL (Step 0 rule (d)). The OFFICIAL score is the honest test result; any later switch to the")
print("comparison version must be labelled 'chosen using Step 8' (honest check: weeks 67-79 when they arrive).")
print("NEXT: Step 9 - refit on all 66 weeks with the same choices and forecast weeks 67-79.")


In [ ]:
# ==========================================================================
# STEP 8 - SAVE CELL: WRITE THE SCORED WEEKS 54-66 TO TWO SEPARATE FILES (one per version)
#   Paste this cell right AFTER the Step 8 cell and run it once (Step 8 must have run in this session).
#   Writes:
#     ../F1 outputs/F1_step8_scored_dec_cleaned_weeks54-66.csv   (OFFICIAL)
#     ../F1 outputs/F1_step8_scored_with_spikes_weeks54-66.csv   (comparison)
#   Each file = that version's 3,679 forecast rows (283 series x weeks 54-66) + the REAL sales of weeks 54-66 +
#   the scores Step 8 used: error, absolute error, inside / above / below the 80% and 95% ranges, the training scale
#   (real weeks 1-53) and the series' MASE.  So each version's Step 8 result can be re-checked later without
#   re-running anything, and the two versions never mix.
#   SAFETY: writes only into "F1 outputs" (never into "eabl parsed csv"); never overwrites (prints "NOT saved").
#   Note: these files contain the real sales of weeks 54-66 - fine now that Step 8 has opened them once.
#
# WHERE EACH INPUT COMES FROM (memory, all from the Step 8 cell): f8_fc (both versions with real sales and naive),
#   F8_VERSIONS (official / comparison), f8_scale, f8_ok, f8_mase -> Step 8;  pd, np -> 7a imports;  os -> standard library
#
# WHAT IT CHANGES: nothing in memory.  WHAT IT WRITES: two CSV files.  New names start with f8s_ / F8S_.
# ==========================================================================
import os

F8S_DIR = "../F1 outputs/"
assert "f8_fc" in globals() and "F8_VERSIONS" in globals(), "run the Step 8 cell first"
assert "eabl parsed csv" not in F8S_DIR, "never write into the dataset folder"

for f8s_role, f8s_v in F8_VERSIONS.items():
    f8s_file = F8S_DIR + f"F1_step8_scored_{f8s_v}_weeks54-66.csv"
    if os.path.exists(f8s_file):                                           # never overwrite
        print(f"NOT saved ({f8s_role}): {f8s_file} already exists - nothing was overwritten")
        continue
    f8s = f8_fc[f8s_v].copy()
    f8s.insert(0, "role", f8s_role)                                        # official / comparison
    f8s["error (real - forecast)"] = f8s["actual"] - f8s["forecast"]
    f8s["abs error"] = f8s["error (real - forecast)"].abs()
    f8s_has = f8s["lo80"].notna()                                          # rows that have a range
    for f8s_n in ["80", "95"]:
        f8s[f"inside {f8s_n}%"] = f8s["actual"].between(f8s[f"lo{f8s_n}"], f8s[f"hi{f8s_n}"]).where(f8s_has)
        f8s[f"above {f8s_n}%"] = (f8s["actual"] > f8s[f"hi{f8s_n}"]).where(f8s_has)
        f8s[f"below {f8s_n}%"] = (f8s["actual"] < f8s[f"lo{f8s_n}"]).where(f8s_has)
    f8s["scale (real weeks 1-53)"] = f8s["unique_id"].map(f8_scale)
    f8s["series MASE (weeks 54-66)"] = f8s["unique_id"].map(f8_mase(f8_fc[f8s_v], "forecast"))   # blank if not scorable
    f8s.to_csv(f8s_file, index=False)
    f8s_back = pd.read_csv(f8s_file)                                       # read back and check
    assert len(f8s_back) == len(f8s) == 283 * 13 and np.isclose(f8s_back["actual"].sum(), f8s["actual"].sum())
    print(f"SAVED ({f8s_role}) {f8s_v}: {f8s_file}")
    print(f"  {len(f8s_back):,} rows | real 13-week total {f8s.loc[f8s['unique_id'] == 'Total', 'actual'].sum():,.0f} | "
          f"forecast {f8s.loc[f8s['unique_id'] == 'Total', 'forecast'].sum():,.0f} | "
          f"80% inside {f8s.loc[f8s_has, 'inside 80%'].mean():.2f} | 95% inside {f8s.loc[f8s_has, 'inside 95%'].mean():.2f}")
print("Files in the folder now:", sorted(f for f in os.listdir(F8S_DIR) if f.endswith(".csv")))


In [ ]:
# ==========================================================================
# STEP 9 - CELL 9-0 (CHECK BEFORE 9a): ARE F1, F2, F3, F4 AND F5 DONE ON WEEKS 1-66? - DETAILED REPORT   (read-only)
#   Before the final forecast, this cell PROVES, step by step, which Prep categories already cover all 66 weeks.
#   For every step it prints:  STATUS (DONE / PARTLY DONE / NOT DONE)  ->  what was done and HOW (the rule)  ->
#   what it looks like on weeks 1-66 (tables)  ->  for anything not fully done: which parts are done and what is
#   still to do.  It re-creates F2, F3 and F5 here from the sales file with the SAME rules as the Prep notebook and
#   compares them with what Prep saved in product_routing_labels.csv:
#     F1  weekly data      -> Prep cells 21-25 + 84: 66 Monday weeks, zero-filled from launch, 283 series, y = S b
#     F2  status           -> Prep cell 28 rules (active / late launch / stopped / unclear), F2_H = 13, F2_MIN_SALES = 3
#     F3  history          -> Prep cell 30 rule (full / short / too new), saved AS AT THE SEAL (6 Apr 2026) only;
#                             on weeks 1-66 the same rule moved to the July cut-off: full = at least 39 weeks on sale
#     F4  the saved file   -> Prep cell 65: one row per product, columns per set (cv1, cv2, backtest, final)
#     F5  type and family  -> Prep cells 60-61 rule (Syntetos-Boylan ADI / CV2, counted from each product's launch)
#   COVERAGE TO WEEK 66: the S matrix, the tags file and the labels file have no week columns, so their dates cannot
#     be read directly.  Proof instead: they must contain the products that FIRST sold after week 53 - above all the
#     one whose first sale was in week 66 (the last week).  A file built from fewer weeks could not know them.
#     The flags file is shown too: it covers weeks 1-53 only, on purpose (F9, no test-week information).
#   STEP 7 SETTINGS CARRIED INTO STEP 9: the methods and settings chosen in the practice rounds live in memory (made
#     by the Step 7 cells in THIS session).  This section confirms each one exists, prints its value, says whether
#     Step 9 keeps it (decisions D1 = A, D3 = keep) or recalculates it on weeks 1-66, and shows which December version
#     is active (official = December cleaned; a warning is printed if the comparison version is active).  The version
#     is checked FROM THE DATA (Total of 22 Dec 2025 in Step 7's training data vs the file), not only from the flag:
#     if cell 1 was re-run without a kernel restart, cell 3 skips the cleaning and this cell says "NOT CLEANED".
#
# WHY (FPP2 2nd ed., Hyndman & Athanasopoulos 2018):
#   4.2 "Document and justify": the same rules applied again must give the same result - checked here.
#   10.1 the hierarchy adds up (y = S b);  12.2 (p.466) Croston / Syntetos-Boylan intervals (F5);  12.7 short series (F3).
#   Not in FPP2 (named, peer-reviewed): Syntetos, Boylan & Croston (2005, JORS 56:495-503) - ADI 1.32 / CV2 0.49.
#
# WHERE EACH INPUT COMES FROM:
#   FILES READ (nothing written): ../eabl parsed csv/Y_product_with_total.csv, S_product_with_total.csv,
#     tags_product_with_total.json, product_routing_labels.csv  (all saved by the Prep notebook, cells 65 and 84),
#     product_week_flags.csv (Prep cell 68, F9) - only its date range is read
#   MEMORY (Step name -> NOTEBOOK cell number in " F1 Forecasting.ipynb", checked 12 Nov 2026):
#     F1_DIR, F1_SEAL, F1_H, F1_MIN_FIT, F1_BOTTOM, F1_SEASON_LENGTH, f1_train -> 7a (cell 1; f1_train is cleaned by
#     cell 3);  pd, np, json -> imports in cell 1
#     display -> Jupyter.   Step 7 settings (only READ, looked up by name so a missing one is reported, not crashed on):
#       F1SP_CLEAN, F1SP_DONE, f1sp_sp -> 7a SPIKE v2 (cell 3);  F1B_ETS_MIN_WEEKS -> 7b (cell 6);  F1L_Z95 -> 7b #11 (cell 8)
#       F1T_BEST -> 7c cell 13 (cell 15);  f1q_win -> 7c cell 13b (cells 17 and 18 - the same cell twice, same result)
#       f1m_std_dec, f1m_best_combo, f1m_best2 -> 7e cell 16 (cell 22);  f1r_q, F1R_MIN_SCALE, f1r_cv -> 7f cell 18 (cell 23)
#       f1z_risk -> 7f cell 19 (cell 25);  F1Y_RAW, F1Y_LOG, F1Y_3X -> 7g cell 21 (cell 28)
#       F1U_CHOICE, f1u_W_both, f1u_lam_both -> 7g cell 22 v2 (cell 29);  F1O_NEG_REVIEW -> 7g cell 23 v2 (cell 30)
#   Each of these names is assigned in exactly ONE notebook cell (13b appears twice, identical) - nothing reassigns them.
#   SAFE TO RUN TWICE: it only reads.
#
# WHAT IT CHANGES: nothing.  WHAT IT WRITES: nothing.  New names start with f9k_ / F9K_ (check cells of Step 9).
# ==========================================================================
assert all(f9k_n in globals() for f9k_n in ["F1_DIR", "F1_SEAL", "F1_H", "F1_MIN_FIT", "F1_BOTTOM", "pd", "np", "json"]), \
    "cell 1 (Step 7a) has not run in this session - run Kernel -> Restart & Run All (or cell 1) first"
F9K_F2_H, F9K_F2_MIN = 13, 3                                   # Prep cell 28: F2_H and F2_MIN_SALES
F9K_F3_FULL = F1_MIN_FIT + 2 * F1_H                            # 39 weeks on sale = F3 "full" (Prep cell 30 rule)
F9K_FAMILY = {"smooth": "standard", "erratic": "standard", "intermittent": "intermittent", "lumpy": "intermittent"}
F9K_SETS = ["cv1", "cv2", "backtest", "final"]

def f9k_title(f9k_name, f9k_status):
    """Prints a section heading with the step's status."""
    print("\n" + "=" * 110 + f"\n{f9k_name}   ->   STATUS: {f9k_status}\n" + "=" * 110)

# ---- Read the files ----
f9k_Y = pd.read_csv(F1_DIR + "Y_product_with_total.csv", parse_dates=["ds"])
f9k_S = pd.read_csv(F1_DIR + "S_product_with_total.csv", index_col=0).set_index("unique_id")
with open(F1_DIR + "tags_product_with_total.json") as f9k_f:
    f9k_tags = {k: v for k, v in json.load(f9k_f).items() if k != "_made_by"}
f9k_prod_ids = f9k_tags[F1_BOTTOM]
f9k_lab = pd.read_csv(F1_DIR + "product_routing_labels.csv", dtype=str).set_index("unique_id")
f9k_weeks = pd.DatetimeIndex(sorted(f9k_Y["ds"].unique()))
f9k_wide = f9k_Y.pivot(index="unique_id", columns="ds", values="y").reindex(f9k_S.index).fillna(0.0)

# ---- The Prep rules, copied exactly ----
def f9k_f2_label(f9k_y):
    """Prep cell 28 rule. f9k_y: 66 weekly values (0 before the first sale). Returns (label, why)."""
    f9k_sold = np.flatnonzero(f9k_y > 0)
    f9k_first, f9k_last, f9k_n, f9k_T = f9k_sold[0], f9k_sold[-1], len(f9k_sold), len(f9k_y)
    f9k_lead, f9k_tail = int(f9k_first), int(f9k_T - 1 - f9k_last)
    f9k_run = f9k_gap = 0
    for f9k_v in f9k_y[f9k_first:f9k_last + 1]:
        f9k_run = f9k_run + 1 if f9k_v <= 0 else 0
        f9k_gap = max(f9k_gap, f9k_run)
    if f9k_lead >= F9K_F2_H and f9k_tail >= F9K_F2_H:
        return "unclear", "started late AND no sale in the last 13+ weeks"
    if f9k_n < F9K_F2_MIN and f9k_lead < f9k_T - F9K_F2_H:
        return "unclear", f"sold in only {f9k_n} week(s) - too few to judge"
    if f9k_tail >= F9K_F2_H and f9k_tail > f9k_gap:
        return "stopped", f"no sale for the last {f9k_tail} weeks (longest earlier pause {f9k_gap})"
    if f9k_lead >= F9K_F2_H and f9k_lead > f9k_gap:
        return "late launch", f"first sale in week {f9k_lead + 1} of {f9k_T} (longest later pause {f9k_gap})"
    if f9k_lead >= F9K_F2_H or f9k_tail >= F9K_F2_H:
        return "unclear", f"quiet 13+ weeks at the start/end, but paused {f9k_gap} weeks before"
    return "active", "sells from the start to the end of the data"

def f9k_f5_class(f9k_y):
    """Prep cells 60-61 rule: from the first sale on, ADI = weeks / selling weeks, CV2 of the sale sizes.
    Returns (class, ADI, CV2)."""
    f9k_s = f9k_y[np.flatnonzero(f9k_y > 0)[0]:]
    f9k_nz = pd.Series(f9k_s[f9k_s > 0])
    f9k_adi, f9k_cv2 = len(f9k_s) / len(f9k_nz), (f9k_nz.std() / f9k_nz.mean()) ** 2
    if f9k_adi < 1.32 and f9k_cv2 < 0.49:            # the same order of tests as Prep cell 60, so a product with
        f9k_c = "smooth"                               # one sale (CV2 cannot be computed) also ends as "lumpy"
    elif f9k_adi >= 1.32 and f9k_cv2 < 0.49:
        f9k_c = "intermittent"
    elif f9k_adi < 1.32 and f9k_cv2 >= 0.49:
        f9k_c = "erratic"
    else:
        f9k_c = "lumpy"
    return f9k_c, f9k_adi, f9k_cv2

# ---- Re-create F2, F3 and F5 for every product on all 66 weeks ----
f9k_rows = []
for f9k_u in f9k_prod_ids:
    f9k_y = f9k_wide.loc[f9k_u].values
    f9k_first = f9k_weeks[np.flatnonzero(f9k_y > 0)[0]]
    f9k_on_sale_final = int((f9k_weeks[-1] - f9k_first).days // 7 + 1)
    f9k_on_sale_seal = max(int((F1_SEAL - f9k_first).days // 7 + 1), 0)
    f9k_lbl, f9k_why = f9k_f2_label(f9k_y)
    f9k_cls, f9k_adi, f9k_cv2 = f9k_f5_class(f9k_y)
    f9k_rows.append({"unique_id": f9k_u, "SKU": f9k_lab.at[f9k_u, "SKU"], "first sale": f9k_first.date(),
                     "total cases": round(float(f9k_y.sum()), 1), "selling weeks": int((f9k_y > 0).sum()),
                     "F2 re-created": f9k_lbl, "F2 why": f9k_why, "F2 saved (label_final)": f9k_lab.at[f9k_u, "label_final"],
                     "ADI": round(f9k_adi, 2), "CV2": round(f9k_cv2, 2) if pd.notna(f9k_cv2) else np.nan,
                     "F5 re-created": f9k_cls, "F5 saved (class_final)": f9k_lab.at[f9k_u, "class_final"],
                     "family re-created": F9K_FAMILY[f9k_cls], "family saved": f9k_lab.at[f9k_u, "family_final"],
                     "weeks on sale by 6 Apr": f9k_on_sale_seal, "weeks on sale by 6 Jul": f9k_on_sale_final,
                     "F3 saved (at the seal)": f9k_lab.at[f9k_u, "history"],
                     "F3 on weeks 1-66": "full" if f9k_on_sale_final >= F9K_F3_FULL else "short"})
f9k_chk = pd.DataFrame(f9k_rows)
f9k_f2_same = int((f9k_chk["F2 re-created"] == f9k_chk["F2 saved (label_final)"]).sum())
f9k_f5_same = int((f9k_chk["F5 re-created"] == f9k_chk["F5 saved (class_final)"]).sum())
f9k_fam_same = int((f9k_chk["family re-created"] == f9k_chk["family saved"]).sum())
f9k_f3_changed = f9k_chk[f9k_chk["F3 saved (at the seal)"] != f9k_chk["F3 on weeks 1-66"]].copy()
f9k_f3_changed["method changes in Step 9?"] = np.select(
    [f9k_f3_changed["F3 saved (at the seal)"] == "too new",
     f9k_f3_changed["family saved"] == "intermittent",
     (f9k_f3_changed["F2 saved (label_final)"] != "stopped") & (f9k_f3_changed["F3 on weeks 1-66"] == "full")],
    ["no - new launch, judgement rate", "no - intermittent, TSB either way", "YES: AutoETS -> Step 7 average"],
    default="no")

# ---- Coverage to week 66 for the files that have no week columns ----
f9k_last_wk = f9k_weeks[-1]                                          # week 66 = 6 Jul 2026
f9k_gap_week = np.abs(f9k_S.values @ f9k_wide.loc[list(f9k_S.columns)].values - f9k_wide.values).max(axis=0)  # per week
f9k_first_all = f9k_Y[f9k_Y["y"] > 0].groupby("unique_id")["ds"].min()   # first selling week of every series
f9k_late = f9k_first_all[f9k_first_all > F1_SEAL].sort_values()      # series that first sold after week 53
f9k_late_prod = [f9k_u for f9k_u in f9k_late.index if f9k_u in f9k_prod_ids]
f9k_lastwk_prod = [f9k_u for f9k_u in f9k_late_prod if f9k_late[f9k_u] == f9k_last_wk]   # first sale IN week 66
f9k_all_tagged = {f9k_u for f9k_v in f9k_tags.values() for f9k_u in f9k_v}
f9k_flags_ds = pd.read_csv(F1_DIR + "product_week_flags.csv", usecols=["ds"], parse_dates=["ds"])["ds"]
f9k_cov = pd.DataFrame([
    ["Y_product_with_total.csv", "weekly sales: read the dates directly",
     f"{f9k_weeks[0].date()} to {f9k_last_wk.date()} ({len(f9k_weeks)} weeks)", f9k_last_wk == f9k_weeks.max()],
    ["S_product_with_total.csv", f"has a column for all {len(f9k_late_prod)} products that first sold after week 53 "
     f"(incl. {len(f9k_lastwk_prod)} first sold IN week 66) and a row for all {len(f9k_late)} such series",
     f"products found: {sum(f9k_u in f9k_S.columns for f9k_u in f9k_late_prod)} of {len(f9k_late_prod)}; "
     f"y = S b gap in week 66: {f9k_gap_week[-1]:.1e} cases",
     all(f9k_u in f9k_S.columns for f9k_u in f9k_late_prod) and all(f9k_u in f9k_S.index for f9k_u in f9k_late.index)
     and f9k_gap_week[-1] < 1e-6],
    ["tags_product_with_total.json", f"lists all {len(f9k_late)} series that first sold after week 53",
     f"found: {sum(f9k_u in f9k_all_tagged for f9k_u in f9k_late.index)} of {len(f9k_late)}",
     all(f9k_u in f9k_all_tagged for f9k_u in f9k_late.index)],
    ["product_routing_labels.csv", "late products have a row; label_final knows them while label_backtest (weeks 1-53) "
     "says 'not known yet'; latest first_sale_week and cutoff_final = week 66",
     f"rows: {sum(f9k_u in f9k_lab.index for f9k_u in f9k_late_prod)} of {len(f9k_late_prod)}; latest first sale "
     f"{pd.to_datetime(f9k_lab['first_sale_week']).max().date()}; cutoff_final {f9k_lab['cutoff_final'].iloc[0]}",
     all(f9k_u in f9k_lab.index for f9k_u in f9k_late_prod)
     and (f9k_lab.loc[f9k_late_prod, "label_final"] != "not known yet").all()
     and (f9k_lab.loc[f9k_late_prod, "label_backtest"] == "not known yet").all()
     and pd.to_datetime(f9k_lab["first_sale_week"]).max() == f9k_last_wk
     and f9k_lab["cutoff_final"].iloc[0] == str(f9k_last_wk.date())],
    ["product_week_flags.csv", "weekly flags: read the dates directly (weeks 1-53 ON PURPOSE - F9 used no test week)",
     f"{f9k_flags_ds.min().date()} to {f9k_flags_ds.max().date()} ({f9k_flags_ds.nunique()} weeks)",
     "not needed: Step 9 uses only its December 2025 spikes (weeks 35-39)"]],
    columns=["file", "test", "what was found", "reaches week 66?"])
f9k_cov_ok = f9k_cov.loc[f9k_cov["file"] != "product_week_flags.csv", "reaches week 66?"].astype(bool).all()

# ---- Step 7 settings carried into Step 9 (looked up by name: a missing one is reported, not crashed on) ----
def f9k_get(f9k_name):
    """The value of a Step 7 name in this session, or None if that Step 7 cell has not run."""
    return globals().get(f9k_name, None)

def f9k_show(f9k_v, f9k_fmt):
    """Text for the settings table; 'MISSING' if the name does not exist."""
    return "MISSING - run Step 7 first" if f9k_v is None else f9k_fmt(f9k_v)

f9k_q, f9k_win, f9k_dec = f9k_get("f1r_q"), f9k_get("f1q_win"), f9k_get("f1m_std_dec")
f9k_combo, f9k_b2, f9k_W = f9k_get("f1m_best_combo"), f9k_get("f1m_best2"), f9k_get("f1u_W_both")
f9k_set_rows = [
    # (setting, made in, value text, Step 9 does what, used in cell)
    ("Season length", "7a", f9k_show(f9k_get("F1_SEASON_LENGTH"), str), "KEPT", "9b"),
    ("Horizon (weeks)", "7a / Step 0", f9k_show(f9k_get("F1_H"), lambda v: f"{v} -> weeks 67-79"), "KEPT", "9a"),
    ("December version", "7a SPIKE v2",
     f9k_show(f9k_get("F1SP_CLEAN"), lambda v: ("cleaned = OFFICIAL" if globals().get("F1SP_DONE") else
                                                "with spikes = comparison only")
              + (f" ({len(f9k_get('f1sp_sp'))} product-weeks)" if globals().get("F1SP_DONE") else "")),
     "KEPT (same weeks cleaned in memory)", "9a"),
    ("Model list", "7g cell 21", f9k_show(f9k_get("F1Y_RAW"), lambda v: ", ".join(v) + " + log: "
                                          + ", ".join(f9k_get("F1Y_LOG") or []) ), "KEPT", "9b"),
    ("Log bias adjustment z (95%)", "7b #11", f9k_show(f9k_get("F1L_Z95"), lambda v: f"{v:.4f}"), "KEPT", "9b"),
    ("Short-history rule", "7b", f9k_show(f9k_get("F1B_ETS_MIN_WEEKS"),
                                          lambda v: f"AutoETS; SES if fewer than {v} weeks"), "KEPT (weeks counted on 1-66)", "9b"),
    ("Averages per level", "7e cell 16",
     f9k_show(f9k_dec, lambda v: " | ".join(f"{f9k_l}: " + (" + ".join(f9k_b2["Total"]) if f9k_l == "Total"
                                            and v.at[f9k_l, "chosen"] == "(a) top-2 per series"
                                            else f9k_combo.get(f9k_l, "-")) for f9k_l in v.index)),
     "KEPT", "9b"),
    ("TSB setting (intermittent)", "7c cell 13", f9k_show(f9k_get("F1T_BEST"), str), "KEPT", "9b"),
    ("One-sale / new-launch rule", "7c cell 12", "judgement rate = sales since first sale / weeks since first sale",
     "RULE KEPT, rate RECALCULATED on 1-66", "9a, 9b"),
    ("Sparse brands + winners", "7c cell 13b", f9k_show(f9k_win, lambda v: f"{len(v)} brands; winners: "
                                                       + ", ".join(f"{m} x{n}" for m, n in v.value_counts().items())),
     "KEPT (list from weeks 1-53)", "9b"),
    ("Reconciliation", "7g cell 22 v2", f9k_show(f9k_get("F1U_CHOICE"), str), "KEPT", "9c"),
    ("MinT weights W", "7g cell 22 v2", f9k_show(f9k_W, lambda v: f"{v.shape[0]} x {v.shape[1]}, from cv1 + cv2 errors, "
                                                 f"shrinkage {f9k_get('f1u_lam_both'):.2f}"), "KEPT (D1 = A)", "9c"),
    ("Set-negative-to-zero review line", "7g cell 23 v2", f9k_show(f9k_get("F1O_NEG_REVIEW"), lambda v: f"{v} case"),
     "KEPT (+ stopped products set to exactly 0)", "9c"),
    ("Range percentiles per group", "7f cell 18", f9k_show(f9k_q, lambda v: f"{len(v)} groups, from cv1 + cv2 errors "
                                                          "(table below)"), "KEPT (D1 = A)", "9c"),
    ("Range scale", "7f / 7g", "mean |weekly change| of each series", "RECALCULATED on 1-66", "9c"),
    ("Minimum scale for a range", "7f cell 18", f9k_show(f9k_get("F1R_MIN_SCALE"), str), "KEPT", "9c"),
    ("3x safety check", "7g cell 21", f9k_show(f9k_get("F1Y_3X"), lambda v: f"{v} x highest past week"),
     "KEPT, highest week RECALCULATED on 1-66", "9c"),
    ("7f review flags", "7f cell 19", f9k_show(f9k_get("f1z_risk"), lambda v: f"{int((v['flag list'] != '').sum())} "
                                               f"series flagged"), "KEPT", "9c"),
    ("'Tested' flag", "7f cell 18", f9k_show(f9k_get("f1r_cv"), lambda v: f"{v['unique_id'].nunique()} series had a "
                                             f"practice forecast"), "KEPT", "9c")]
f9k_settings = pd.DataFrame(f9k_set_rows, columns=["setting", "made in (Step 7)", "value in this session",
                                                   "Step 9 does", "used in cell"])
f9k_set_missing = f9k_settings["value in this session"].str.startswith("MISSING")
f9k_set_ok = not f9k_set_missing.any()
# December version checked FROM THE DATA, not only from the flag (the flag survives a re-run of cell 1 without a
# kernel restart, and then cell 3 skips the cleaning): is the week of 22 Dec 2025 cleaned in Step 7's training data?
f9k_dec_wk = pd.Timestamp("2025-12-22")                                       # the biggest December spike week
f9k_dec_raw = float(f9k_wide.at["Total", f9k_dec_wk])                         # as recorded in the file
f9k_tr = f9k_get("f1_train")                                                  # 7a (cell 1), cleaned by cell 3
f9k_dec_used = (float(f9k_tr["backtest"].loc[(f9k_tr["backtest"]["unique_id"] == "Total")
                                             & (f9k_tr["backtest"]["ds"] == f9k_dec_wk), "y"].iloc[0])
                if f9k_tr is not None else None)                               # what Step 7 learned from
f9k_cleaned_in_data = f9k_dec_used is not None and f9k_dec_used < f9k_dec_raw - 1
f9k_broken = bool(f9k_get("F1SP_CLEAN")) and not f9k_cleaned_in_data         # switch ON but data NOT cleaned
f9k_official = bool(f9k_get("F1SP_CLEAN")) and f9k_cleaned_in_data
f9k_dec_txt = (f" (Total of 22 Dec 2025 used by Step 7: {f9k_dec_used:,.0f} vs recorded {f9k_dec_raw:,.0f})"
               if f9k_dec_used is not None else "")
f9k_version_txt = ("version unknown (7a SPIKE v2 has not run)" if f9k_get("F1SP_CLEAN") is None
                   else "OFFICIAL version (December cleaned)" + f9k_dec_txt if f9k_official
                   else "NOT CLEANED although the switch is ON - Kernel -> Restart Kernel and Run All Cells" + f9k_dec_txt
                   if f9k_broken else "COMPARISON version (with spikes)" + f9k_dec_txt)
f9k_settings.loc[f9k_settings["setting"] == "December version", "value in this session"] = f9k_version_txt

# ---- Status of each step ----
f9k_level_n = {k: len(v) for k, v in f9k_tags.items()}
f9k_rows_per = f9k_Y.groupby("unique_id").size()
f9k_f1_ok = (len(f9k_weeks) == 66 and (f9k_weeks.dayofweek == 0).all() and f9k_Y["unique_id"].nunique() == 283
             and f9k_gap_week.max() < 1e-6 and f9k_Y["y"].notna().all() and (f9k_Y["y"] >= 0).all())
f9k_f2_ok = f9k_f2_same == 188
f9k_f5_ok = f9k_f5_same == 188 and f9k_fam_same == 188
f9k_cols_by_set = {f9k_s: [f9k_c for f9k_c in f9k_lab.columns if f9k_c.endswith("_" + f9k_s)] for f9k_s in F9K_SETS}
f9k_f4_ok = len(f9k_lab) == 188 and all(f"{f9k_p}_final" in f9k_lab.columns for f9k_p in ["label", "class", "family", "cutoff"])
f9k_status = pd.DataFrame([
    ["F1 weekly data", "DONE" if f9k_f1_ok else "NOT DONE - stop and check", "nothing"],
    ["F2 status (label)", "DONE" if f9k_f2_ok else "PARTLY DONE", "nothing" if f9k_f2_ok else "see F2 section"],
    ["F3 history", "PARTLY DONE (at the seal only)", f"redo on weeks 1-66 -> cell 9a (changes the method of "
     f"{int(f9k_f3_changed['method changes in Step 9?'].str.startswith('YES').sum())} products)"],
    ["F4 saved file", "DONE for label / class / family; PARTLY for history" if f9k_f4_ok else "NOT DONE",
     "no history_final column - cell 9a makes it in memory (no file is changed)"],
    ["F5 type / family", "DONE" if f9k_f5_ok else "PARTLY DONE", "nothing" if f9k_f5_ok else "see F5 section"],
    ["Files reach week 66", "YES (S, tags, labels include the week-66 launch)" if f9k_cov_ok else "NO - stop and check",
     "nothing" if f9k_cov_ok else "see the COVERAGE section; re-run Prep cells 84 and 65"],
    ["Step 7 settings in memory", ("ALL PRESENT" if f9k_set_ok else f"{int(f9k_set_missing.sum())} MISSING")
     + " - " + f9k_version_txt,
     "nothing" if f9k_set_ok and not f9k_broken else "Kernel -> Restart Kernel and Run All Cells (one real restart; "
     "do not run cell 1 or cell 3 on their own afterwards)"]],
    columns=["Prep step", "done on weeks 1-66?", "what still needs to be done"])

# ===================================== REPORT =====================================
print("STEP 9-0 - CHECK BEFORE 9a: are the Prep categories done on weeks 1-66? (nothing written)")
print("SUMMARY:")
display(f9k_status)

# ---- F1 ----
f9k_title("F1 - WEEKLY DATA (the numbers every forecast learns from)", "DONE" if f9k_f1_ok else "NOT DONE")
print("WHAT WAS DONE: daily sales -> Monday-start weeks (Mon-Sat trading week), each series zero-filled from its first "
      "sale, products added up to brands, sub-categories, categories and the Total (S matrix).")
print(f"HOW IT LOOKS ON WEEKS 1-66: {f9k_weeks[0].date()} to {f9k_weeks[-1].date()} = {len(f9k_weeks)} weeks | "
      f"{f9k_Y['unique_id'].nunique()} series | {len(f9k_Y):,} rows | missing values: {int(f9k_Y['y'].isna().sum())} | "
      f"below 0: {int((f9k_Y['y'] < 0).sum())} | largest gap in y = S b over all 66 weeks: {f9k_gap_week.max():.2e} cases")
display(pd.DataFrame({"series": pd.Series(f9k_level_n),
                      "rows (series x weeks on sale)": pd.Series({k: int(f9k_rows_per.reindex(v).sum()) for k, v in f9k_tags.items()}),
                      "cases, weeks 1-66": pd.Series({k: round(float(f9k_wide.loc[v].values.sum()), 0)
                                                      for k, v in f9k_tags.items()})}))
print("(every level has the same cases: each level is the sum of the products under it)")
print("Total cases per quarter (13 weeks) - each level must give the same total:")
display(f9k_wide.loc[["Total"]].T.groupby(np.arange(len(f9k_weeks)) // 13).sum()
        .set_axis([f"weeks {13 * i + 1}-{min(13 * i + 13, 66)}" for i in range(6)]).round(0))

# ---- Coverage to week 66 ----
f9k_title("COVERAGE - DO THE FILES REACH THE LAST WEEK (WEEK 66)?", "YES" if f9k_cov_ok else "NO - stop and check")
print("HOW: files with dates are read directly.  Files WITHOUT dates (S, tags, labels) must contain the series that "
      "first sold after week 53 - a file built from fewer weeks could not know them (like a class register that "
      "includes a student who joined on the last day of term).")
display(f9k_cov)
print(f"Series that first sold after week 53 (week 54 = {(F1_SEAL + pd.Timedelta(weeks=1)).date()}) - the proof:")
display(pd.DataFrame({"first sale": f9k_late.dt.date,
                      "week": [f9k_weeks.get_loc(f9k_d) + 1 for f9k_d in f9k_late],
                      "SKU": [f9k_lab.at[f9k_u, "SKU"] if f9k_u in f9k_lab.index else "(brand)" for f9k_u in f9k_late.index],
                      "label_backtest (weeks 1-53)": [f9k_lab.at[f9k_u, "label_backtest"] if f9k_u in f9k_lab.index
                                                      else "-" for f9k_u in f9k_late.index],
                      "label_final (weeks 1-66)": [f9k_lab.at[f9k_u, "label_final"] if f9k_u in f9k_lab.index
                                                   else "-" for f9k_u in f9k_late.index]}))
print(f"Last-week proof: {[f9k_lab.at[f9k_u, 'SKU'] for f9k_u in f9k_lastwk_prod]} first sold in week 66 "
      f"({f9k_last_wk.date()}) and is in S, tags and labels.")

# ---- F2 ----
f9k_title("F2 - STATUS OF EACH PRODUCT (active / late launch / stopped / unclear)", "DONE" if f9k_f2_ok else "PARTLY DONE")
print("WHAT WAS DONE: Prep cell 28 labelled every product from its 66 weeks of sales (saved as label_final).")
print(f"HOW (rule, first match wins; H = {F9K_F2_H} weeks): late start AND quiet end -> unclear | fewer than "
      f"{F9K_F2_MIN} selling weeks -> unclear | no sale in the last {F9K_F2_H}+ weeks (longer than any earlier pause) "
      f"-> stopped | first sale after week {F9K_F2_H} (longer than any later pause) -> late launch | quiet {F9K_F2_H}+ "
      f"weeks at start/end with earlier pauses -> unclear | otherwise -> active")
print(f"CHECK: re-created here = saved label_final for {f9k_f2_same} of 188 products.")
f9k_f2_tab = f9k_chk.groupby("F2 re-created").agg(products=("unique_id", "size"), cases=("total cases", "sum"))
f9k_f2_tab["share of cases %"] = (f9k_f2_tab["cases"] / f9k_f2_tab["cases"].sum() * 100).round(2)
display(f9k_f2_tab.round({"cases": 0, "share of cases %": 2}))
print("One example per label (with the reason):")
display(f9k_chk.groupby("F2 re-created").head(1)[["SKU", "first sale", "selling weeks", "F2 re-created", "F2 why"]]
        .reset_index(drop=True))
if not f9k_f2_ok:
    print("PARTLY DONE - products where the saved label differs (TO DO: re-run Prep cell 28, then F4):")
    display(f9k_chk[f9k_chk["F2 re-created"] != f9k_chk["F2 saved (label_final)"]])

# ---- F3 ----
f9k_title("F3 - HISTORY OF EACH PRODUCT (full / short / too new)", "PARTLY DONE (at the seal only)")
print("WHAT WAS DONE: Prep cell 30 counted each product's weeks on sale UP TO THE SEAL (6 Apr 2026) and saved "
      "'history' (full = at least 39 weeks = 13 weeks before the first practice round; short = 1-38; too new = 0).")
print("WHAT IS NOT DONE: no weeks 1-66 version exists (there is no history_final column).")
print(f"HOW IT WOULD LOOK ON WEEKS 1-66 (same rule, July cut-off: full = at least {F9K_F3_FULL} weeks on sale):")
display(pd.crosstab(f9k_chk["F3 saved (at the seal)"], f9k_chk["F3 on weeks 1-66"], margins=True,
                    rownames=["F3 at the seal (saved)"], colnames=["F3 on weeks 1-66"]))
print(f"Products whose group changes ({len(f9k_f3_changed)}), and whether their forecasting method changes:")
display(f9k_f3_changed[["SKU", "family saved", "weeks on sale by 6 Apr", "weeks on sale by 6 Jul",
                        "F3 saved (at the seal)", "F3 on weeks 1-66", "method changes in Step 9?"]]
        .sort_values("method changes in Step 9?", key=lambda f9k_s: ~f9k_s.str.startswith("YES")).reset_index(drop=True))
print("TO DO: cell 9a re-does F3 on weeks 1-66 in memory (history_final) and uses it (user decision A); "
      "cell 9a-CHECK confirms it.")

# ---- F4 ----
f9k_title("F4 - THE SAVED LABELS FILE (product_routing_labels.csv)",
          "DONE for label / class / family; PARTLY for history" if f9k_f4_ok else "NOT DONE")
print(f"WHAT WAS DONE: Prep cell 65 saved one row per product ({len(f9k_lab)} rows x {f9k_lab.shape[1] + 1} columns), "
      f"one set of columns per period, each with its cut-off:")
display(pd.DataFrame({"cut-off": {f9k_s: f9k_lab[f"cutoff_{f9k_s}"].iloc[0] for f9k_s in F9K_SETS},
                      "columns saved": {f9k_s: ", ".join(f9k_cols_by_set[f9k_s]) for f9k_s in F9K_SETS},
                      "products launched by the cut-off": {f9k_s: int(f9k_lab[f"label_{f9k_s}"].ne("not known yet").sum())
                                                           for f9k_s in F9K_SETS}}))
print("Columns not tied to one period: unique_id, Product_Code, SKU, first_sale_week, history (AT THE SEAL), "
      "weeks_in_train (by the seal), cv_rounds_usable, master_status_INFO_ONLY (information only, never used)")
print("PARTLY DONE: the 'final' set has label / class / family / cut-off but NO history_final.")
print("TO DO: nothing on disk - cell 9a makes history_final in memory, so the saved file stays exactly as it is.")

# ---- F5 ----
f9k_title("F5 - DEMAND TYPE AND FAMILY OF EACH PRODUCT (from its launch)", "DONE" if f9k_f5_ok else "PARTLY DONE")
print("WHAT WAS DONE: Prep cells 60-61 classified each product from its FIRST SALE to week 66 (saved as class_final, "
      "family_final).")
print("HOW: ADI = weeks since launch / selling weeks (how often it sells); CV2 = (std / mean)^2 of the sale sizes "
      "(how much the size varies).  ADI < 1.32 & CV2 < 0.49 -> smooth | ADI >= 1.32 & CV2 < 0.49 -> intermittent | "
      "ADI < 1.32 & CV2 >= 0.49 -> erratic | otherwise -> lumpy.  smooth / erratic -> family 'standard' (regular "
      "methods); intermittent / lumpy -> family 'intermittent' (TSB).")
print(f"CHECK: re-created here = saved class_final for {f9k_f5_same} of 188 and family_final for {f9k_fam_same} of 188.")
display(pd.crosstab(f9k_chk["F5 re-created"], f9k_chk["family re-created"], margins=True,
                    rownames=["class (weeks 1-66)"], colnames=["family"]))
print("Typical ADI and CV2 per class (median):")
display(f9k_chk.groupby("F5 re-created")[["ADI", "CV2", "selling weeks"]].median().round(2))
print("Products whose family on weeks 1-66 differs from the backtest (weeks 1-53) - their lane changes in Step 9:")
f9k_fam_move = f9k_chk.set_index("unique_id")
f9k_fam_move = f9k_fam_move.assign(**{"family backtest (weeks 1-53)": f9k_lab["family_backtest"]})
display(f9k_fam_move[f9k_fam_move["family saved"] != f9k_fam_move["family backtest (weeks 1-53)"]]
        [["SKU", "family backtest (weeks 1-53)", "family saved", "F5 saved (class_final)"]].reset_index(drop=True))
if not f9k_f5_ok:
    print("PARTLY DONE - products where the saved class differs (TO DO: re-run Prep cells 60-61, then F4):")
    display(f9k_chk[f9k_chk["F5 re-created"] != f9k_chk["F5 saved (class_final)"]])

# ---- Step 7 settings ----
f9k_title("STEP 7 SETTINGS CARRIED INTO STEP 9 (chosen in the practice rounds, kept by decisions D1 = A and D3)",
          ("ALL PRESENT" if f9k_set_ok else "SOME MISSING - run Step 7 first") + " | " + f9k_version_txt)
print("WHAT: the methods and settings Step 7 chose on cv1 / cv2.  Step 9 KEEPS the choices (no re-picking after the "
      "test) and RECALCULATES only facts about each series (sales, labels, history, rates, scale, highest week).")
if f9k_broken:
    print("STOP - WARNING: the December switch is ON (F1SP_CLEAN = True) but Step 7's training data is NOT cleaned "
          f"{f9k_dec_txt}. Cell 1 was re-run without a kernel restart, so cell 3 skipped the cleaning. The settings below "
          "are therefore the WITH-SPIKES ones. Do: Kernel -> Restart Kernel and Run All Cells (one real restart); cell 3 "
          "must print '84 product-weeks replaced'.")
elif f9k_get("F1SP_CLEAN") is not None and not f9k_official:
    print("WARNING: December cleaning is OFF in this session -> the settings below are the WITH-SPIKES version. "
          "Use this run only for the comparison file; the OFFICIAL final forecast needs F1SP_CLEAN = True "
          "and Restart & Run All.")
display(f9k_settings)
if f9k_q is not None:
    print("Range percentiles per group (errors in units of each series' scale; from cv1 + cv2 practice errors):")
    display(f9k_q.round(2))
if f9k_win is not None:
    print("The 15 sparse brands and their own winning method (7c, weeks 1-53):")
    display(f9k_win.rename("method (7c winner)").to_frame())
if not f9k_set_ok:
    print("MISSING settings - TO DO: Kernel -> Restart & Run All, so every Step 7 cell runs before 9-0:")
    display(f9k_settings[f9k_set_missing])

print("\nNEXT: cell 9a (it uses label_final, family_final and F3 re-done on weeks 1-66).")


In [ ]:
# ==========================================================================
# STEP 9 - CELL 9-0b (CHECK BEFORE 9a): ARE WEEKS 54-66 CLEAN ENOUGH TO LEARN FROM?  BOTH VERSIONS   (read-only)
#   Until now weeks 54-66 (13 Apr - 6 Jul 2026) were only used to SCORE Step 8.  In Step 9 the models LEARN from
#   them for the first time - and they are the most recent weeks, which count most.  Every earlier data check
#   ("7a CHECKS", cell 4) and the F8 spike/drop flags (Prep cell 67, saved by F9) cover weeks 1-53 only.
#   This cell checks weeks 54-66 for BOTH learning versions that Step 9 will use:
#     "dec_cleaned" (official)   = the file with the 84 December 2025 spikes replaced (same rule as cell 3)
#     "with_spikes" (comparison) = the file exactly as recorded
#   PART A  data checks (must pass): 13 Monday weeks right after the seal, no duplicates, no missing values,
#           no negative sales, no missing week inside any series, every level adds up (y = S b)
#   PART B  the two versions: built here from the files (does not depend on the cell 3 switch); they must differ
#           ONLY in the 5 December weeks, so weeks 54-66 are identical in both
#   PART C  spike / drop screen on weeks 54-66 with the EXACT F8 rule (Prep cell 67): each product-week is
#           compared with its own 13 earlier weeks only.  Proof first: the same code re-creates the saved F8 flags
#           of weeks 1-53 (with_spikes data) - if not identical, the cell stops.  Then weeks 54-66 are screened in
#           BOTH versions.  (Their 13-week windows are weeks 41-65, which contain no December, so both versions
#           must give the same flags - checked.)  FLAG ONLY, as in F8: nothing is changed or removed.
#   PART D  the Total week by week (weeks 41-66) against its own 13 earlier weeks, with trading days, holidays
#           and Sunday sales from the weekly calendar; then week 66 (the last, highest week) product by product.
#
# WHY (FPP2 2nd ed., Hyndman & Athanasopoulos 2018):
#   12.9 (p.488): "Outliers are observations that are very different from the majority of the observations in the
#     time series. They may be errors, or they may simply be unusual." -> find and show them before learning from them.
#   2.1-2.2: look at the data before forecasting;  10.1: the hierarchy must add up (y = S b).
#   3.1 (Naive, Drift): methods that lean on the LAST week carry an unusual last week forward.
#   Not in FPP2 (named, peer-reviewed): Leys et al. (2013, J. Exp. Soc. Psychology 49:764-766) - median +/- k x 1.4826 x MAD
#     as a robust outlier rule (the one F8 uses).
#
# WHERE EACH INPUT COMES FROM:
#   FILES READ (nothing written):
#     ../eabl parsed csv/Y_product_with_total.csv   -> Prep cell 84: 283 series x 66 weeks (read again: cell 1 keeps
#                                                      only weeks 1-53 in memory)
#     ../eabl parsed csv/product_week_flags.csv     -> Prep cell 68 (F9): the saved F8 flags, weeks 1-53 (used for the
#                                                      proof in Part C and for the December spikes in Part B)
#     ../eabl parsed csv/trading_days_per_week.csv  -> Prep cell 27: trading days, holidays, Sunday sales per week
#   MEMORY: F1_DIR, F1_SEAL, F1_CUTOFF, f1_Smat, f1_train, f1_labels (SKU names only) -> 7a (cell 1; f1_train is
#     cleaned by cell 3 if the switch is on);  f1sp_sp -> 7a SPIKE v2 (cell 3) - optional, only used to
#     cross-check Part B if it exists
#     pd, np -> 7a imports (cell 1);  display -> Jupyter built-in
#
# WHAT IT CHANGES: nothing (f1_train, f1_labels, f1_Smat and every file are only read).  WHAT IT WRITES: nothing.
# New names start with f9q_ / F9Q_ (checked: no other cell uses this prefix).  SAFE TO RUN TWICE, in either version.
# PASTE: a new cell directly BELOW cell 9-0 (cell 38), ABOVE cell 9a.
# ==========================================================================
assert all(f9q_n in globals() for f9q_n in ["F1_DIR", "F1_SEAL", "F1_CUTOFF", "f1_Smat", "f1_train", "f1_labels",
                                             "pd", "np"]), \
    "run 7a (cell 1) first"
F9Q_FIRST, F9Q_LAST = pd.Timestamp("2026-04-13"), pd.Timestamp("2026-07-06")      # weeks 54 and 66
F9Q_DEC = list(pd.date_range("2025-12-01", "2025-12-29", freq="W-MON"))           # the 5 December 2025 weeks
F9Q_CHECK_WEEK = pd.Timestamp("2025-12-22")                                        # biggest spike week (as cell 3)
F9Q_HIST, F9Q_MIN_HIST, F9Q_MIN_NZ = 13, 8, 3     # F8 (Prep cell 67): weeks looked back; minimum weeks; selling weeks
F9Q_SPIKE_X, F9Q_MAD_K = 3.0, 3.0                 # F8: spike > 3 x usual selling week AND > median + 3 x MAD
F9Q_REGULAR, F9Q_DROP_X = 0.75, 0.25              # F8: drop = sells in >= 75% of weeks AND this week < 25% of usual
F9Q_MIN_CASES = 1.0                               # F8: size filter (at least 1 case)
assert F1_SEAL == F9Q_FIRST - pd.Timedelta(weeks=1) and F1_CUTOFF["backtest"] == F1_SEAL, "seal is not week 53 - re-run cell 1 (7a), then this cell"

# ---- Read the files (read-only) ----
f9q_Y = pd.read_csv(F1_DIR + "Y_product_with_total.csv", parse_dates=["ds"])
f9q_fl = pd.read_csv(F1_DIR + "product_week_flags.csv", parse_dates=["ds"])
f9q_cal = pd.read_csv(F1_DIR + "trading_days_per_week.csv", parse_dates=["week_start"])
f9q_ids, f9q_bottom = list(f1_Smat.index), list(f1_Smat.columns)
f9q_weeks = pd.Index(sorted(f9q_Y["ds"].unique()))
f9q_new = f9q_weeks[(f9q_weeks >= F9Q_FIRST) & (f9q_weeks <= F9Q_LAST)]

# ==== PART A - data checks on weeks 54-66 (as recorded; must all pass) ====
f9q_part = f9q_Y[f9q_Y["ds"].isin(f9q_new)]
f9q_first = f9q_Y[f9q_Y["y"] > 0].groupby("unique_id")["ds"].min()
f9q_rows = f9q_Y.groupby("unique_id")["ds"].agg(["min", "max", "size"])
f9q_expect = ((f9q_rows["max"] - f9q_rows["min"]).dt.days // 7 + 1)
f9q_wide = f9q_part.pivot(index="unique_id", columns="ds", values="y").reindex(index=f9q_ids, columns=f9q_new).fillna(0.0)
f9q_gap = float(np.abs(f1_Smat.values @ f9q_wide.loc[f9q_bottom].values - f9q_wide.values).max())
f9q_A = pd.DataFrame([
    ("13 weeks, Monday starts, right after the seal",
     f"{len(f9q_new)} weeks: {f9q_new.min().date()} to {f9q_new.max().date()}",
     len(f9q_new) == 13 and (f9q_new.dayofweek == 0).all() and (np.diff(f9q_new.values) == np.timedelta64(7, "D")).all()
     and f9q_weeks.max() == F9Q_LAST),
    ("no duplicate rows (series + week)", f"{int(f9q_part.duplicated(['unique_id', 'ds']).sum())} duplicates",
     not f9q_part.duplicated(["unique_id", "ds"]).any()),
    ("no missing values", f"{int(f9q_part['y'].isna().sum())} missing", f9q_part["y"].notna().all()),
    ("no negative sales", f"{int((f9q_part['y'] < 0).sum())} below 0", (f9q_part["y"] >= 0).all()),
    ("no missing week inside any series (rows run from first sale to week 66)",
     f"{int((f9q_rows['size'] != f9q_expect).sum())} series with a hole; every series' first row = its first sale: "
     f"{bool((f9q_rows['min'] == f9q_first.reindex(f9q_rows.index)).all())}",
     (f9q_rows["size"] == f9q_expect).all() and (f9q_rows["max"] == F9Q_LAST).all()
     and (f9q_rows["min"] == f9q_first.reindex(f9q_rows.index)).all()),
    ("every level adds up in weeks 54-66 (y = S b)", f"largest gap {f9q_gap:.2e} cases", f9q_gap < 1e-6)],
    columns=["check", "what was found", "passed"])

# ==== PART B - the two learning versions (weeks 1-66), built from the files ====
def f9q_clean_december(f9q_df):
    """Cell 3's rule: the 84 December 2025 product-weeks flagged 'spike' by F8 (flags up to cv2's cut-off) get
    their usual week; every level above is re-added with S.  Returns (cleaned copy, the spike list)."""
    f9q_f = f9q_fl[f9q_fl["ds"] <= F1_CUTOFF["cv2"]]
    f9q_sp = f9q_f[f9q_f["ds"].isin(F9Q_DEC) & (f9q_f["product_flag"] == "spike")][["unique_id", "ds", "y", "usual_week"]]
    f9q_out = f9q_df.copy()
    for f9q_wk in F9Q_DEC:
        f9q_w = f9q_sp[f9q_sp["ds"] == f9q_wk]
        if f9q_w.empty:
            continue
        f9q_delta = pd.Series(0.0, index=f9q_bottom)
        f9q_delta[f9q_w["unique_id"].values] = (f9q_w["usual_week"] - f9q_w["y"]).values
        f9q_up = pd.Series(f1_Smat.values @ f9q_delta.values, index=f9q_ids)
        f9q_up = f9q_up[f9q_up.abs() > 1e-9]
        f9q_r = (f9q_out["ds"] == f9q_wk) & f9q_out["unique_id"].isin(f9q_up.index)
        assert int(f9q_r.sum()) == len(f9q_up), f"{f9q_wk.date()}: a changed series has no row - re-run cell 1, then this cell"
        f9q_out.loc[f9q_r, "y"] = f9q_out.loc[f9q_r, "y"] + f9q_out.loc[f9q_r, "unique_id"].map(f9q_up)
    return f9q_out, f9q_sp

f9q_ver = {"with_spikes": f9q_Y.copy()}
f9q_ver["dec_cleaned"], f9q_sp84 = f9q_clean_december(f9q_Y)
f9q_cmp = f9q_ver["with_spikes"].merge(f9q_ver["dec_cleaned"], on=["unique_id", "ds"], suffixes=("_with", "_clean"))
f9q_diff = f9q_cmp[(f9q_cmp["y_with"] - f9q_cmp["y_clean"]).abs() > 1e-9]
def f9q_tot(f9q_df, f9q_wk):
    return float(f9q_df.loc[(f9q_df["unique_id"] == "Total") & (f9q_df["ds"] == f9q_wk), "y"].iloc[0])
f9q_B = pd.DataFrame([
    ("cleaned version replaces the 84 December spikes", f"{len(f9q_sp84)} product-weeks replaced",
     len(f9q_sp84) == 84 and (f9q_ver["dec_cleaned"]["y"] >= 0).all()),
    ("the versions differ ONLY in the 5 December 2025 weeks",
     f"rows that differ: {len(f9q_diff)}, in weeks {sorted({f9q_dt.date().isoformat() for f9q_dt in f9q_diff['ds']})}",
     set(f9q_diff["ds"]) <= set(F9Q_DEC)),
    ("weeks 54-66 identical in both versions", "largest difference in weeks 54-66: "
     f"{(f9q_cmp.loc[f9q_cmp['ds'].isin(f9q_new), 'y_with'] - f9q_cmp.loc[f9q_cmp['ds'].isin(f9q_new), 'y_clean']).abs().max():.1e}",
     np.allclose(f9q_cmp.loc[f9q_cmp["ds"].isin(f9q_new), "y_with"], f9q_cmp.loc[f9q_cmp["ds"].isin(f9q_new), "y_clean"])),
    ("Total of 22 Dec 2025: with spikes vs cleaned",
     f"{f9q_tot(f9q_ver['with_spikes'], F9Q_CHECK_WEEK):,.0f} vs {f9q_tot(f9q_ver['dec_cleaned'], F9Q_CHECK_WEEK):,.0f}",
     f9q_tot(f9q_ver["dec_cleaned"], F9Q_CHECK_WEEK) < f9q_tot(f9q_ver["with_spikes"], F9Q_CHECK_WEEK) - 1)],
    columns=["check", "what was found", "passed"])
if "f1sp_sp" in globals():                     # cell 3 cleaned in this session: its list must be the same 84
    f9q_same = (f9q_sp84.set_index(["unique_id", "ds"])["usual_week"].sort_index()
                .equals(f1sp_sp.set_index(["unique_id", "ds"])["usual_week"].sort_index()))
    f9q_B.loc[len(f9q_B)] = ("same 84 spikes and values as cell 3 (7a SPIKE v2)", f"identical: {f9q_same}", f9q_same)
# which version is active in THIS session (from the data, as cell 3 and 9-0 do) - information only
f9q_bt = f1_train["backtest"]
f9q_session = ("dec_cleaned (official)"
               if f9q_tot(f9q_bt, F9Q_CHECK_WEEK) < f9q_tot(f9q_ver["with_spikes"], F9Q_CHECK_WEEK) - 1 else "with_spikes")

# ==== PART C - the F8 spike / drop screen (exact Prep cell 67 rule), proof on weeks 1-53, then weeks 54-66 ====
def f9q_screen(f9q_df):
    """F8 rule for every product-week: compare with the product's own 13 earlier weeks (rows start at its first
    sale).  Returns one row per product-week: usual_week, usual_selling_week, flag."""
    f9q_res = []
    for f9q_u, f9q_g in f9q_df[f9q_df["unique_id"].isin(f9q_bottom)].sort_values(["unique_id", "ds"]).groupby("unique_id"):
        f9q_y, f9q_d = f9q_g["y"].to_numpy(), f9q_g["ds"].to_numpy()
        for f9q_t in range(len(f9q_y)):
            f9q_prev = f9q_y[max(0, f9q_t - F9Q_HIST):f9q_t]
            f9q_rec = {"unique_id": f9q_u, "ds": f9q_d[f9q_t], "y": f9q_y[f9q_t], "usual_week": np.nan,
                       "usual_selling_week": np.nan, "flag": "too early"}
            if len(f9q_prev) >= F9Q_MIN_HIST:
                f9q_med = np.median(f9q_prev)
                f9q_mad = 1.4826 * np.median(np.abs(f9q_prev - f9q_med))
                f9q_nz = f9q_prev[f9q_prev > 0]
                f9q_mnz = np.median(f9q_nz) if len(f9q_nz) >= F9Q_MIN_NZ else np.nan
                f9q_spike = (f9q_mnz == f9q_mnz) and f9q_y[f9q_t] > F9Q_SPIKE_X * f9q_mnz \
                    and f9q_y[f9q_t] > f9q_med + F9Q_MAD_K * f9q_mad and f9q_y[f9q_t] - f9q_mnz >= F9Q_MIN_CASES
                f9q_drop = (f9q_prev > 0).mean() >= F9Q_REGULAR and f9q_med >= F9Q_MIN_CASES \
                    and f9q_y[f9q_t] < F9Q_DROP_X * f9q_med
                f9q_rec.update(usual_week=round(f9q_med, 2),
                               usual_selling_week=round(f9q_mnz, 2) if f9q_mnz == f9q_mnz else np.nan,
                               flag="spike" if f9q_spike else ("drop" if f9q_drop else "normal"))
            f9q_res.append(f9q_rec)
    return pd.DataFrame(f9q_res)

# proof: re-create the saved F8 flags of weeks 1-53 (with_spikes data, as Prep did)
f9q_s53 = f9q_screen(f9q_ver["with_spikes"][f9q_ver["with_spikes"]["ds"] <= F1_SEAL])
f9q_pf = f9q_s53.merge(f9q_fl[["unique_id", "ds", "product_flag", "usual_week"]], on=["unique_id", "ds"],
                       how="outer", suffixes=("", "_saved"), indicator=True)
f9q_uw_off = ~np.isclose(f9q_pf["usual_week"].fillna(-1), f9q_pf["usual_week_saved"].fillna(-1), rtol=0, atol=1e-9)
f9q_proof = ((f9q_pf["_merge"] == "both").all() and (f9q_pf["flag"] == f9q_pf["product_flag"]).all()   # flags: exact
             and np.allclose(f9q_pf["usual_week"].fillna(-1), f9q_pf["usual_week_saved"].fillna(-1),
                             rtol=0, atol=0.011))       # usual week: saved rounded to 0.01, so allow 1 rounding step
assert f9q_proof, "the F8 screen does not re-create the saved flags of weeks 1-53 - do not use its results; check that " \
    "Prep cells 67-68 (flags) and 84 (sales file) were not re-run since cell 1, then re-run cells 1-3 and this cell"

f9q_scr = {}
for f9q_v, f9q_df in f9q_ver.items():          # screen weeks 54-66 in both versions (windows reach back to week 41)
    f9q_s = f9q_screen(f9q_df)
    f9q_scr[f9q_v] = f9q_s[f9q_s["ds"].isin(f9q_new)].reset_index(drop=True)
f9q_same_flags = f9q_scr["with_spikes"][["unique_id", "ds", "flag"]].equals(f9q_scr["dec_cleaned"][["unique_id", "ds", "flag"]])
assert f9q_same_flags, "weeks 54-66 flags differ between versions (their windows should not reach December) - re-run cells 1-3, then this cell"
f9q_sku = f1_labels.set_index("unique_id")["SKU"]                               # product names (cell 1)
f9q_res54 = f9q_scr["dec_cleaned"].assign(SKU=lambda f9q_x: f9q_x["unique_id"].map(f9q_sku),
                                          week_no=lambda f9q_x: f9q_x["ds"].map({f9q_w: f9q_i + 1 for f9q_i, f9q_w in enumerate(f9q_weeks)}))
f9q_res54["x_usual"] = np.where(f9q_res54["flag"] == "spike", f9q_res54["y"] / f9q_res54["usual_selling_week"],
                                f9q_res54["y"] / f9q_res54["usual_week"].replace(0, np.nan)).round(2)

# ==== PART D - the Total week by week, and week 66 product by product ====
f9q_T = f9q_Y[f9q_Y["unique_id"] == "Total"].set_index("ds")["y"]
f9q_tw = []
for f9q_wk in f9q_weeks[f9q_weeks >= pd.Timestamp("2026-01-12")]:                 # weeks 41-66
    f9q_prev = f9q_T[(f9q_T.index < f9q_wk)].tail(F9Q_HIST)
    f9q_med, f9q_mad = f9q_prev.median(), 1.4826 * (f9q_prev - f9q_prev.median()).abs().median()
    f9q_tw.append({"ds": f9q_wk, "week_no": f9q_weeks.get_loc(f9q_wk) + 1, "Total": f9q_T[f9q_wk],
                   "median of 13 earlier weeks": f9q_med, "vs median %": 100 * (f9q_T[f9q_wk] / f9q_med - 1),
                   "robust z (MADs above)": (f9q_T[f9q_wk] - f9q_med) / f9q_mad if f9q_mad > 0 else np.nan,
                   "products spiking": int(((f9q_res54["ds"] == f9q_wk) & (f9q_res54["flag"] == "spike")).sum())
                   if f9q_wk in f9q_new else np.nan})
f9q_tw = (pd.DataFrame(f9q_tw).merge(f9q_cal[["week_start", "trading_days", "near_empty_days", "holiday_names",
                                              "sunday_sales"]].rename(columns={"week_start": "ds"}), on="ds", how="left"))
f9q_w66 = f9q_res54[f9q_res54["ds"] == F9Q_LAST].copy()
f9q_w66["change vs usual week"] = f9q_w66["y"] - f9q_w66["usual_week"]
f9q_rise = f9q_T[F9Q_LAST] - f9q_tw.loc[f9q_tw["ds"] == F9Q_LAST, "median of 13 earlier weeks"].iloc[0]
f9q_top66 = f9q_w66.sort_values("change vs usual week", ascending=False).head(10)

# ---- Report ----
print("STEP 9-0b - WEEKS 54-66 BEFORE THE MODELS LEARN FROM THEM: data checks, both versions, spike screen (nothing written)\n")
print(f"This session's Step 7 data (from the DATA, 22 Dec 2025 Total): {f9q_session} | "
      f"this cell checks BOTH versions regardless of the switch\n")
print("PART A - data checks, weeks 54-66 as recorded (must all pass):"); display(f9q_A)
print("PART B - the two learning versions Step 9 will use (weeks 1-66):"); display(f9q_B)
print(f"PART C - spike / drop screen, exact F8 rule (each product-week vs its own 13 earlier weeks; FLAG ONLY)")
print(f"  proof: re-creates the saved F8 flags of weeks 1-53 - {f9q_proof}: all {len(f9q_s53):,} flags identical; "
      f"usual week identical except {int(f9q_uw_off.sum())} values that differ by 0.01 case (rounding of a median)")
print(f"  weeks 54-66: identical flags in both versions - {f9q_same_flags}")
display(f9q_res54["flag"].value_counts().rename("product-weeks, weeks 54-66").to_frame())
display(pd.crosstab(f9q_res54["week_no"], f9q_res54["flag"]).reindex(columns=["normal", "spike", "drop", "too early"],
                                                                    fill_value=0))
print("Spikes in weeks 54-66 (x_usual = that week / usual selling week; to ask EABL: bulk order, promotion, error?):")
display(f9q_res54[f9q_res54["flag"] == "spike"].sort_values("x_usual", ascending=False)
        [["week_no", "ds", "SKU", "y", "usual_selling_week", "x_usual"]].reset_index(drop=True))
print("Drops in weeks 54-66 (regular sellers far below usual - possible stock-outs, to ask EABL):")
display(f9q_res54[f9q_res54["flag"] == "drop"].groupby("SKU").agg(drop_weeks=("ds", "count"), first=("ds", "min"),
        last=("ds", "max"), usual_week=("usual_week", "median")).sort_values("drop_weeks", ascending=False))
print("PART D - the Total week by week (weeks 41-66) against its own 13 earlier weeks:")
display(f9q_tw.round({"Total": 0, "median of 13 earlier weeks": 0, "vs median %": 1, "robust z (MADs above)": 2}))
print(f"Week 66 ({F9Q_LAST.date()}): Total {f9q_T[F9Q_LAST]:,.0f} = {f9q_rise:+,.0f} cases vs the median of its 13 earlier "
      f"weeks.  The 10 products that rose most (vs their own usual week):")
display(f9q_top66[["SKU", "y", "usual_week", "change vs usual week", "flag"]].round(2).reset_index(drop=True))
print(f"  together these 10 products rose {f9q_top66['change vs usual week'].sum():,.0f} cases above their usual weeks; "
      f"products flagged 'spike' in week 66: {int((f9q_w66['flag'] == 'spike').sum())}")
assert f9q_A["passed"].all() and f9q_B["passed"].all(), "a data check failed (see PART A / PART B above) - do NOT run 9a; re-run cells 1-3 and this cell, " \
    "and if it still fails check the sales file made by Prep cell 84"
print("\nPARTS A and B PASSED (both versions).  Parts C and D are FLAGS ONLY - review them; nothing was changed.")


In [ ]:
# ==========================================================================
# STEP 9 - CELL 9a: LOAD ALL 66 WEEKS + THE FINAL LABELS, CLEAN DECEMBER (memory only), BUILD THE FINAL ROUTER
#   Step 9 = the final forecast: the SAME methods chosen in Step 7 (not re-picked after Step 8 - Step 0 rule (d)),
#   now learning from ALL 66 weeks (7 Apr 2025 - week of 6 Jul 2026) and forecasting weeks 67-79 (13 Jul - 5 Oct 2026).
#   This cell only prepares; no model is fitted here (cell 9b fits, 9c reconciles, 9d saves).
#   WHAT IS NEW COMPARED WITH STEP 7:
#     - learning weeks 1-66 instead of 1-53 (the test is over, so weeks 54-66 may now be learned from)
#     - FINAL labels (label_final / family_final, made from weeks 1-66) instead of the backtest labels
#     - products with FEWER THAN 13 WEEKS of sales (F1_MIN_FIT, Step 7's minimum before a series can be modelled) get a
#       judgement rate (sales since launch / weeks since launch, the 7c decision-1 rule), flagged "new launch - please
#       review" - today these are the 7 products first sold after 6 Apr 2026 (1-12 weeks).  Brands etc. with fewer than
#       13 weeks get the same treatment.  (Fix of 14 Nov 2026: the gate used to ask "first sale after 6 Apr?" - a date
#       left over from the test; it now asks about each product's OWN weeks 1-66 data.  Same products, same forecasts.)
#       Evidence: a launch test on 12 past EABL launches (30 cases, each pretended 4 / 7 / 9 weeks old, next 13 weeks
#       compared with real sales) - the judgement rate was more accurate (mean MASE 1.235, bias 1.16) than the lane
#       models (1.299, bias 1.58); decision rule fixed before the test; recorded as a weak spot (small sample).
#   WHAT STAYS THE SAME (user decisions D1-D3, 12 Nov 2026): the Step 7 lanes and rules; December cleaning follows the
#     F1SP_CLEAN switch of "7a SPIKE v2" (True = official "dec_cleaned", False = "with_spikes", comparison only).
#   WHICH VERSION (fix of 13 Nov 2026): decided from the DATA, not from a note left in memory - the Total of the week of
#     22 Dec 2025 in Step 7's learning data (f1_train["backtest"]) is compared with the file: lower = cleaned.  If the
#     cell 3 switch and the data disagree (e.g. the switch was changed without Restart & Run All), the cell STOPS.
#     Cells 9b and 9c add their name to f9_stamp; cell 9d saves only if 9a, 9b and 9c all ran on the same version.
#   THE FINAL ROUTER (one lane per product, in this order):
#     label_final "stopped"                     -> "zero"
#     fewer than 13 weeks of sales (F1_MIN_FIT) -> "judgement (new launch)"
#     family_final "intermittent"               -> "intermittent" (TSB); if it sold in only 1 week -> "judgement (one-sale)"
#     family_final "standard"                   -> "standard"; Step 7 average if F3 history "full", AutoETS (AICc)
#                                                  if "short".  F3 is RE-DONE ON WEEKS 1-66 (user decision A,
#                                                  12 Nov 2026): the same Prep cell 30 rule, moved to the July cut-off:
#                                                  full = at least 39 weeks on sale (13 weeks before the first practice
#                                                  cut-off, which is 2 x 13 weeks before the end).  Cell 9-0 showed it
#                                                  changes the method of 2 products (Captain Morgan Muckpit 250ml, 750ml).
#   F2 (label_final) and F5 (family_final) were already made from weeks 1-66 by Prep - cell 9-0 re-created them.
#
# WHY (FPP2 2nd ed., Hyndman & Athanasopoulos 2018):
#   7.2 (p.253): after choosing a method on cross-validation, "apply it to the whole data set to get forecasts for
#     future years";  9.6 (p.385): "re-estimate that model, but using all the available data".
#   4.6: "Judgmental forecasting is usually the only available method for new product forecasting".
#   12.9 (p.488): outliers may be replaced if "they won't occur in the forecasting period" - Jul-Oct has no December.
#
# WHERE EACH INPUT COMES FROM:
#   FILES READ (nothing written):
#     ../eabl parsed csv/Y_product_with_total.csv   -> Prep cell 84: 283 series x 66 weeks (7a dropped weeks 54-66
#                                                      from memory, so it is read again here)
#     ../eabl parsed csv/product_routing_labels.csv -> Prep cell 65: the FULL file (cell "7a SEAL FOR LABELS" removed
#                                                      the *_final columns from f1_labels in memory, so it is read again)
#   MEMORY:
#     F1_DIR, F1_SEAL, F1_H, F1_MIN_FIT, f1_Smat, f1_tags, F1_BOTTOM, f1_routes, f1_train -> 7a (cell 1)
#     F1SP_CLEAN (the switch), F1SP_WEEKS -> "7a SPIKE v2" (cell 3);  f1sp_sp (the 84 December spikes + usual week)
#       -> cell 3, only if cleaning ran
#     pd, np -> 7a imports;  hashlib -> NEW import (Python standard library), used only for file fingerprints
#     display -> Jupyter built-in
#
# WHAT IT CHANGES: nothing on disk; nothing that Steps 7-8 made is changed in memory.
# WHAT IT WRITES: nothing.  New names start with f9_ / F9_ (checked: no other cell uses this prefix).
# PASTE: a new cell directly BELOW cell 9-0b (cell 39); then 9a-CHECK, 9b, 9c, 9d below it, run in that order.
# ==========================================================================
import hashlib

F9_FILES = ["Y_product_with_total.csv", "S_product_with_total.csv", "tags_product_with_total.json",
            "product_routing_labels.csv", "product_week_flags.csv"]               # the inputs of Steps 7-9
F9_F3_FULL = F1_MIN_FIT + 2 * F1_H                                                 # 39 weeks on sale = F3 "full"

def f9_fingerprint():
    """SHA-1 of each input file - 9d checks that they are unchanged at the end of Step 9."""
    return {f9_f: hashlib.sha1(open(F1_DIR + f9_f, "rb").read()).hexdigest() for f9_f in F9_FILES}

f9_fp_start = f9_fingerprint()

# ---- 1. All 66 weeks and the full labels file ----
f9_Y = pd.read_csv(F1_DIR + "Y_product_with_total.csv", parse_dates=["ds"])
f9_labels = pd.read_csv(F1_DIR + "product_routing_labels.csv", dtype=str)
f9_weeks = pd.Index(sorted(f9_Y["ds"].unique()))

# ---- 2. Which version? From the DATA (22 Dec 2025 Total), checked against the cell 3 switch ----
F9_CHECK_WEEK = pd.Timestamp("2025-12-22")                                         # biggest December spike week (cell 3)
def f9_total(f9_df, f9_wk):
    """Total of one week in a weekly table (unique_id, ds, y)."""
    return float(f9_df.loc[(f9_df["unique_id"] == "Total") & (f9_df["ds"] == f9_wk), "y"].iloc[0])
f9_dec_recorded = f9_total(f9_Y, F9_CHECK_WEEK)                                    # as recorded in the file
f9_s7_cleaned = f9_total(f1_train["backtest"], F9_CHECK_WEEK) < f9_dec_recorded - 1  # what Step 7 learned from
assert "F1SP_CLEAN" in globals() and "F1SP_WEEKS" in globals(), "run 7a SPIKE v2 (cell 3) first - Kernel -> Restart & Run All"
assert bool(F1SP_CLEAN) == f9_s7_cleaned, (
    f"the cell 3 switch says F1SP_CLEAN = {F1SP_CLEAN} but Step 7's learning data is "
    f"{'cleaned' if f9_s7_cleaned else 'as recorded'} - Kernel -> Restart & Run All, then run Step 9 again")
assert (not f9_s7_cleaned) or "f1sp_sp" in globals(), "Step 7's data is cleaned but cell 3's spike list is missing - Restart & Run All"
F9_VERSION = "dec_cleaned" if f9_s7_cleaned else "with_spikes"
f9_stamp = {"9a": F9_VERSION}                       # 9b and 9c add themselves; 9d saves only if all three match
F9_CUTOFF = pd.Timestamp(f9_labels["cutoff_final"].iloc[0])                       # 6 Jul 2026 = week 66
assert len(f9_weeks) == 66 and f9_weeks[-1] == F9_CUTOFF and f9_labels["cutoff_final"].nunique() == 1, \
    "the sales file is not weeks 1-66 to the final cut-off - check the file made by Prep cell 84, then re-run cell 1"
assert f9_Y["unique_id"].nunique() == 283 and len(f9_labels) == 188, \
    "expected 283 series and 188 products - check the files made by Prep cells 84 and 65"
assert set(f9_labels["unique_id"]) == set(f1_tags[F1_BOTTOM]), "labels file does not match the 188 products - check Prep cell 65, then re-run cell 1"
F9_FUT = pd.date_range(F9_CUTOFF + pd.Timedelta(weeks=1), periods=F1_H, freq="W-MON")   # weeks 67-79

# y = S b holds on all 66 weeks (FPP2 10.1)
f9_wide = f9_Y.pivot(index="unique_id", columns="ds", values="y").reindex(f1_Smat.index).fillna(0.0)
assert np.allclose(f1_Smat.values @ f9_wide.loc[list(f1_Smat.columns)].values, f9_wide.values), "y = S b broken in the sales file - check Prep cell 84"

# ---- 3. Training data = weeks 1-66; December cleaned in memory if the switch is on (same 84 product-weeks) ----
f9_train = f9_Y.copy()
f9_tot_dec_before = f9_train[f9_train["unique_id"] == "Total"].set_index("ds")["y"].reindex(F1SP_WEEKS)
if F9_VERSION == "dec_cleaned":
    f9_n_cleaned = 0
    for f9_wk in F1SP_WEEKS:
        f9_sp = f1sp_sp[f1sp_sp["ds"] == f9_wk]
        if f9_sp.empty:
            continue
        f9_delta = pd.Series(0.0, index=f1_Smat.columns)                         # change per product
        f9_delta[f9_sp["unique_id"].values] = (f9_sp["usual_week"] - f9_sp["y"]).values
        f9_up = pd.Series(f1_Smat.values @ f9_delta.values, index=f1_Smat.index)  # change per series (S)
        f9_up = f9_up[f9_up.abs() > 1e-9]
        f9_rows = (f9_train["ds"] == f9_wk) & f9_train["unique_id"].isin(f9_up.index)
        assert int(f9_rows.sum()) == len(f9_up), f"{f9_wk.date()}: a changed series has no row - Kernel -> Restart & Run All"
        f9_train.loc[f9_rows, "y"] = f9_train.loc[f9_rows, "y"] + f9_train.loc[f9_rows, "unique_id"].map(f9_up)
        f9_n_cleaned += len(f9_sp)
    assert f9_n_cleaned == len(f1sp_sp) and (f9_train["y"] >= 0).all(), \
        "cleaning did not match 7a SPIKE v2 - Kernel -> Restart & Run All"
# weeks 1-53 must equal Step 7's backtest learning data exactly - in BOTH versions
f9_chk = f9_train[f9_train["ds"] <= F1_SEAL].merge(f1_train["backtest"], on=["unique_id", "ds"], suffixes=("", "_s7"))
assert len(f9_chk) == len(f1_train["backtest"]) and np.allclose(f9_chk["y"], f9_chk["y_s7"]), \
    "weeks 1-53 differ from Step 7's learning data - Kernel -> Restart & Run All"
assert (f9_total(f9_train, F9_CHECK_WEEK) < f9_dec_recorded - 1) == (F9_VERSION == "dec_cleaned"), \
    "the learning data does not match the version label - Kernel -> Restart & Run All"
f9_wide_tr = f9_train.pivot(index="unique_id", columns="ds", values="y").reindex(f1_Smat.index).fillna(0.0)
assert np.allclose(f1_Smat.values @ f9_wide_tr.loc[list(f1_Smat.columns)].values, f9_wide_tr.values), \
    "the learning data no longer adds up - Kernel -> Restart & Run All"

# ---- 4. The final router (one lane per product) ----
f9_first = f9_train[f9_train["y"] > 0].groupby("unique_id")["ds"].min()          # first selling week, every series
f9_sell = f9_train.groupby("unique_id")["y"].agg(lambda f9_v: int((f9_v > 0).sum()))
f9_route = f9_labels[["unique_id", "SKU", "label_final", "family_final", "history"]].copy()
f9_route["first_sale"] = f9_route["unique_id"].map(f9_first)
f9_route["weeks_on_sale"] = ((F9_CUTOFF - f9_route["first_sale"]).dt.days // 7 + 1).fillna(0).astype(int)
f9_route["selling_weeks"] = f9_route["unique_id"].map(f9_sell).fillna(0).astype(int)
f9_route["lane"] = np.select(
    [f9_route["label_final"] == "stopped",
     f9_route["weeks_on_sale"] < F1_MIN_FIT,                                   # too little data to model (13 weeks)
     (f9_route["family_final"] == "intermittent") & (f9_route["selling_weeks"] <= 1),
     f9_route["family_final"] == "intermittent",
     f9_route["family_final"] == "standard"],
    ["zero", "judgement (new launch)", "judgement (one-sale)", "intermittent", "standard"], default="CHECK")
assert (f9_route["lane"] != "CHECK").all(), "a product has no lane - check the labels file (Prep cell 65), then re-run cells 1-3 and 9a"
f9_route["history_final"] = np.where(f9_route["weeks_on_sale"] >= F9_F3_FULL, "full", "short")   # F3 on weeks 1-66
f9_route["choose_by"] = np.where(f9_route["lane"] == "standard",
                                 np.where(f9_route["history_final"] == "full", "CV", "AICc"), "-")
f9_route["Step 7 choose_by (backtest)"] = f9_route["unique_id"].map(
    f1_routes[f1_routes["set"] == "backtest"].set_index("unique_id")["choose_by"])
f9_route["Step 7 lane (backtest)"] = f9_route["unique_id"].map(
    f1_routes[f1_routes["set"] == "backtest"].set_index("unique_id")["lane"])
f9_upper_weeks = ((F9_CUTOFF - f9_first.reindex([f9_u for f9_u in f1_Smat.index if f9_u not in f1_Smat.columns]))
                  .dt.days // 7 + 1)                                              # weeks on sale of every upper series
f9_new_upper = sorted(f9_upper_weeks.index[f9_upper_weeks < F1_MIN_FIT])          # fewer than 13 weeks -> judgement rate

# ---- Report ----
print(f"STEP 9a - final inputs ready, version '{F9_VERSION}' (nothing written)\n")
print(f"Learning weeks: 1-{len(f9_weeks)} ({f9_weeks[0].date()} to {F9_CUTOFF.date()}) | forecast weeks 67-79: "
      f"{F9_FUT[0].date()} to {F9_FUT[-1].date()} | y = S b holds on all 66 weeks - OK")
if F9_VERSION == "dec_cleaned":
    print(f"December cleaned in memory: {f9_n_cleaned} product-weeks (same as Step 7); weeks 1-53 identical to Step 7's "
          f"backtest training data - OK")
    display(pd.DataFrame({"Total as recorded": f9_tot_dec_before,
                          "Total used for learning": f9_train[f9_train["unique_id"] == "Total"].set_index("ds")["y"]
                          .reindex(F1SP_WEEKS)}).round(0))
else:
    print("December NOT cleaned (switch off): this run makes the 'with_spikes' comparison version")
print("FINAL labels (weeks 1-66):")
display(f9_route["label_final"].value_counts().rename("products").to_frame())
print("Products per lane - Step 7 backtest lane (columns) vs final lane (rows):")
display(pd.crosstab(f9_route["lane"], f9_route["Step 7 lane (backtest)"], margins=True))
print(f"Products whose lane changed: {int((f9_route['lane'] != f9_route['Step 7 lane (backtest)']).sum())} | "
      f"standard by CV / AICc: {int((f9_route['choose_by'] == 'CV').sum())} / {int((f9_route['choose_by'] == 'AICc').sum())}")
f9_f3_moved = f9_route[(f9_route["lane"] == "standard") & (f9_route["Step 7 choose_by (backtest)"] == "AICc")
                       & (f9_route["choose_by"] == "CV")]
print(f"F3 re-done on weeks 1-66 (full = at least {F9_F3_FULL} weeks on sale): standard products moved from AutoETS "
      f"(AICc) to the Step 7 average: {len(f9_f3_moved)}")
display(f9_f3_moved[["unique_id", "SKU", "history", "history_final", "weeks_on_sale"]]
        .rename(columns={"history": "F3 at the seal", "history_final": "F3 on weeks 1-66"}).reset_index(drop=True))
print(f"Upper-level series with fewer than {F1_MIN_FIT} weeks of sales (judgement rate): {f9_new_upper}")
display(f9_route[f9_route["lane"].str.startswith("judgement")]
        [["unique_id", "SKU", "lane", "first_sale", "weeks_on_sale", "selling_weeks"]].reset_index(drop=True))
print("Input files fingerprinted (9d checks they are unchanged).  NEXT: cell 9a-CHECK, then cell 9b (base forecasts).")


In [ ]:
# ==========================================================================
# STEP 9 - CELL 9a-CHECK (CHECK AFTER 9a): DID 9a USE THE WEEKS 1-66 CATEGORIES CORRECTLY?   (read-only)
#   Confirms, product by product, that the final router built in cell 9a follows the categories checked in cell 9-0:
#     F1  learning data = all 66 weeks (to 6 Jul 2026); December cleaned only in the official version - checked in the
#         DATA: the 22 Dec 2025 Total learned is lower than recorded AND 84 spikes were replaced (official), or equal
#         to the recorded Total (comparison)
#     F2  every product labelled "stopped" (label_final) is in lane "zero", and no other product is
#     F5  every modelled product's lane matches its family_final (standard / intermittent); the only exceptions are
#         the judgement lanes (new launches = fewer than 13 weeks of sales, one-sale products), listed
#     LANES RE-COMPUTED INDEPENDENTLY (fix of 14 Nov 2026): every product's lane and method choice is rebuilt here from
#         cell 9-0's OWN re-created F2 / family / F3 / weeks on sale / selling weeks - no date, no 9a number - and must
#         equal 9a's.  (9-0 counts selling weeks on the raw file, 9a on the learning data; they differ only for 4 products
#         with 19+ selling weeks, so the one-sale rule (1 week) gives the same answer.)
#     F3  every "standard" product uses F3 RE-DONE ON WEEKS 1-66 (Step 7 average if full, AutoETS if short)
#     data files unchanged so far (fingerprints taken at the start of 9a)
#
# WHY: FPP2 4.2 "Document and justify" - a rule is only useful if we can show it was applied as written.
#
# WHERE EACH INPUT COMES FROM (memory): f9_route, f9_train, f9_fp_start, f9_fingerprint, F9_F3_FULL, F9_CUTOFF,
#   F9_VERSION, f9_stamp, f9_total, f9_dec_recorded, F9_CHECK_WEEK -> cell 9a;  f9k_chk (re-created F2 / F5 / F3)
#   -> cell 9-0 (cell 38);  f1sp_sp -> "7a SPIKE v2" (cell 3, only if cleaning ran)
#   F1_MIN_FIT (13 weeks, the minimum before a series can be modelled) -> 7a (cell 1)
#   pd, np -> 7a imports;  display -> Jupyter built-in
#
# WHAT IT CHANGES: nothing.  WHAT IT WRITES: nothing.  New names start with f9k_ / F9K_ (check cells of Step 9).
# ==========================================================================
assert "f9_route" in globals() and "f9k_chk" in globals() and "f9_stamp" in globals(), "run cell 9-0 (cell 38) and cell 9a first"
f9k_dec_learned = f9_total(f9_train, F9_CHECK_WEEK)                    # 22 Dec 2025 Total in the learning data
f9k_dec_ok = ((f9k_dec_learned < f9_dec_recorded - 1) and len(f1sp_sp) == 84) if F9_VERSION == "dec_cleaned" \
    else abs(f9k_dec_learned - f9_dec_recorded) < 1e-6
f9k_r = f9_route.set_index("unique_id")
f9k_recreated = f9k_chk.set_index("unique_id").reindex(f9k_r.index)
f9k_model_lane = f9k_r["lane"].isin(["standard", "intermittent"])
f9k_judge = f9k_r["lane"].str.startswith("judgement")
f9k_lane_again = pd.Series(np.select(
    [f9k_recreated["F2 re-created"] == "stopped",
     f9k_recreated["weeks on sale by 6 Jul"] < F1_MIN_FIT,
     (f9k_recreated["family re-created"] == "intermittent") & (f9k_recreated["selling weeks"] <= 1),
     f9k_recreated["family re-created"] == "intermittent",
     f9k_recreated["family re-created"] == "standard"],
    ["zero", "judgement (new launch)", "judgement (one-sale)", "intermittent", "standard"], default="CHECK"),
    index=f9k_r.index)
f9k_choose_again = pd.Series(np.where(f9k_lane_again == "standard",
                                      np.where(f9k_recreated["F3 on weeks 1-66"] == "full", "CV", "AICc"), "-"),
                             index=f9k_r.index)

f9k_tests = [
    ("F1 learning weeks", "weeks 1-66, last week " + str(f9_train["ds"].max().date()),
     f9_train["ds"].nunique() == 66 and f9_train["ds"].max() == F9_CUTOFF),
    ("F1 December cleaning", f"version '{F9_VERSION}': 22 Dec 2025 Total learned {f9k_dec_learned:,.0f} vs recorded "
     f"{f9_dec_recorded:,.0f}" + (f"; {len(f1sp_sp)} product-weeks replaced" if F9_VERSION == "dec_cleaned" else "; nothing replaced"),
     f9k_dec_ok),
    ("F2 stopped -> zero", f"{int((f9k_r['label_final'] == 'stopped').sum())} stopped products, "
     f"{int((f9k_r['lane'] == 'zero').sum())} in lane zero",
     ((f9k_r["label_final"] == "stopped") == (f9k_r["lane"] == "zero")).all()),
    ("F2 = re-created on 1-66", "label_final used by 9a = cell 9-0's re-created F2",
     (f9k_r["label_final"] == f9k_recreated["F2 re-created"]).all()),
    ("F5 family -> lane", f"{int(f9k_model_lane.sum())} modelled products; lane = family_final for all of them",
     (f9k_r.loc[f9k_model_lane, "lane"] == f9k_r.loc[f9k_model_lane, "family_final"]).all()),
    ("F5 = re-created on 1-66", "family_final used by 9a = cell 9-0's re-created family",
     (f9k_r["family_final"] == f9k_recreated["family re-created"]).all()),
    ("F3 re-done on 1-66", f"full = at least {F9_F3_FULL} weeks; 9a's groups = cell 9-0's 'F3 on weeks 1-66'",
     (f9k_r["history_final"] == f9k_recreated["F3 on weeks 1-66"]).all()),
    ("F3 -> method", "standard products: full -> Step 7 average (CV), short -> AutoETS (AICc)",
     (f9k_r.loc[f9k_r["lane"] == "standard", "choose_by"]
      == np.where(f9k_r.loc[f9k_r["lane"] == "standard", "history_final"] == "full", "CV", "AICc")).all()),
    ("Lanes re-computed independently", "lane + method choice rebuilt from 9-0's F2 / F5 / F3 / weeks alone (no date) = 9a's",
     bool((f9k_lane_again == f9k_r["lane"]).all() and (f9k_choose_again == f9k_r["choose_by"]).all())),
    ("Data files unchanged", "fingerprints now = fingerprints at the start of 9a", f9_fingerprint() == f9_fp_start)]
f9k_after = pd.DataFrame(f9k_tests, columns=["check", "what was checked", "passed"])

# ---- Report ----
print("STEP 9a-CHECK - CHECK AFTER 9a: the final router follows the weeks 1-66 categories (nothing written)\n")
display(f9k_after)
print("Products NOT in their family's lane (judgement lanes - expected, please review):")
display(f9k_r.loc[f9k_judge, ["SKU", "label_final", "family_final", "lane", "weeks_on_sale", "selling_weeks"]])
print("Standard products whose method changed because F3 was re-done on weeks 1-66:")
display(f9k_r[(f9k_r["lane"] == "standard") & (f9k_r["Step 7 choose_by (backtest)"] == "AICc") & (f9k_r["choose_by"] == "CV")]
        [["SKU", "history", "history_final", "weeks_on_sale"]]
        .rename(columns={"history": "F3 at the seal", "history_final": "F3 on weeks 1-66"}))
assert f9k_after["passed"].all(), "a check failed (see the table above) - do NOT run 9b; Kernel -> Restart & Run All, then 9a and this cell"
print("ALL CHECKS PASSED.  NEXT: cell 9b - base forecasts for all 283 series.")

In [ ]:
# ==========================================================================
# STEP 9 - CELL 9b: FINAL BASE FORECASTS - ALL 283 SERIES LEARN WEEKS 1-66, FORECAST WEEKS 67-79
#   The same plan rules as Step 7 cell 21 (7g), applied to the final router from cell 9a:
#     Total                 -> 7e (a): average of its 2 best models (f1m_best2, chosen on cv1)
#     Category2, Sub_Cat.   -> 7e (f): the level's best pair/trio (f1m_best_combo)
#     Brand                 -> 7e (f); the 15 sparse brands -> their own 7c winner (f1q_win)
#     Product "standard"    -> 7e (f) if chosen by CV; AICc lane: AutoETS, or SES if fewer than 7 weeks (7b rule)
#     Product "intermittent"-> TSB (setting F1T_BEST)
#     judgement (one-sale products; products AND brands with fewer than 13 weeks of sales - lane / list set in
#                              cell 9a) -> sales since first sale / weeks since first sale (7c decision 1), flat 13 weeks
#     Product "zero" (stopped in the FINAL labels) -> 0
#   No method is re-chosen here: every list of models comes from Step 7 (user decision D3, 12 Nov 2026).
#   Log models (_log) are fitted on log(1 + sales) and turned back with the 7b #11 bias adjustment, as in Step 7.
#   A model that refuses a very short series falls back to the average of past weeks (statsforecast fallback_model).
#
# WHY (FPP2 2nd ed., Hyndman & Athanasopoulos 2018):
#   10.6 (p.414): base forecasts - every series forecast on its own, reconciled afterwards (cell 9c).
#   7.2 (p.253) / 9.6 (p.385): the chosen methods are re-estimated on all the data for the future forecast.
#   12.4 forecast combinations;  12.2 time series of counts (Croston-type methods);  3.2 (bias adjustments) + 12.3
#   log scale keeps forecasts positive;  Ch 4 (p.101) judgement "when a new product is being launched".
#   Not in FPP2 (named, peer-reviewed): Teunter, Syntetos & Babai (2011, European Journal of Operational Research
#   214:606-615) - the TSB method used for the intermittent lane.
#
# WHERE EACH INPUT COMES FROM (all already in memory; nothing read from disk):
#   f9_train, f9_route, f9_first, f9_new_upper (series with < 13 weeks), F9_CUTOFF, F9_FUT, F9_VERSION, f9_stamp -> cell 9a;  f1_tags, F1_H -> 7a
#   F1B_LEVELS, F1B_ETS_MIN_WEEKS, StatsForecast, HistoricAverage -> 7b (cell 6);  F1L_Z95 -> 7b #11 (cell 8)
#   F1T_BEST -> 7c cell 13 (cell 15);  f1q_win (sparse-brand winners) -> 7c cell 13b (cells 17 and 18, identical)
#   f1m_std_dec, f1m_best_combo, f1m_best2 -> 7e cell 16 (cell 22);  F1Y_RAW, F1Y_LOG (model builders) -> 7g cell 21 (cell 28)
#   pd, np -> 7a imports;  display -> Jupyter built-in
#
# WHAT IT CHANGES: nothing.  WHAT IT WRITES: nothing (memory only).  New names start with f9_ / F9_.
# PASTE: a new cell directly BELOW cell 9a-CHECK; run it once, straight after 9a and 9a-CHECK (takes a few minutes).
# ==========================================================================
assert "f9_stamp" in globals() and set(f9_stamp) == {"9a"} and f9_stamp["9a"] == F9_VERSION and "f1m_best_combo" in globals(), \
    "run Step 7 (Restart & Run All), then cell 9a, then this cell (9b runs once, straight after 9a)"

# ---- 1. The plan: method for each of the 283 series ----
f9_rt = f9_route.set_index("unique_id")
f9_weeks_seen = f9_train.groupby("unique_id").size()
f9_plan_rows = []
for f9_key, f9_lev in F1B_LEVELS.items():
    for f9_u in f1_tags[f9_key]:
        if f9_lev == "Product":
            f9_r = f9_rt.loc[f9_u]
            if f9_r["lane"] == "zero":
                f9_plan_rows.append((f9_u, f9_lev, "zero (stopped)", []))
            elif f9_r["lane"].startswith("judgement"):
                f9_plan_rows.append((f9_u, f9_lev, f"judgement rate ({f9_r['lane'][11:-1]})", []))
            elif f9_r["lane"] == "intermittent":
                f9_plan_rows.append((f9_u, f9_lev, f"TSB ({F1T_BEST})", ["TSB"]))
            elif f9_r["choose_by"] == "AICc":
                f9_m = "SES" if f9_weeks_seen[f9_u] < F1B_ETS_MIN_WEEKS else "AutoETS"
                f9_plan_rows.append((f9_u, f9_lev, f"{f9_m} (short history, AICc)", [f9_m]))
            else:
                f9_ms = f1m_best_combo["Product"].split(" + ")
                f9_plan_rows.append((f9_u, f9_lev, " + ".join(f9_ms), f9_ms))
        elif f9_u in f9_new_upper:                                         # fewer than 13 weeks of sales (cell 9a)
            f9_plan_rows.append((f9_u, f9_lev, "judgement rate (new launch)", []))
        elif f9_u in f1q_win.index:                                        # sparse brand: its own 7c winner
            f9_plan_rows.append((f9_u, f9_lev, f"{f1q_win[f9_u]} (sparse brand)", [f1q_win[f9_u]]))
        elif f1m_std_dec.at[f9_lev, "chosen"] == "(a) top-2 per series":
            f9_ms = list(f1m_best2[f9_u])
            f9_plan_rows.append((f9_u, f9_lev, " + ".join(f9_ms), f9_ms))
        else:
            f9_ms = f1m_best_combo[f9_lev].split(" + ")
            f9_plan_rows.append((f9_u, f9_lev, " + ".join(f9_ms), f9_ms))
f9_plan = pd.DataFrame(f9_plan_rows, columns=["unique_id", "level", "method", "models"])
assert len(f9_plan) == 283 and f9_plan["unique_id"].is_unique, "the plan must cover all 283 series once - re-run cell 9a, then this cell"

# ---- 2. Fit the needed models on weeks 1-66 and forecast weeks 67-79 ----
f9_need = f9_plan[f9_plan["models"].str.len() > 0]
f9_raw_m = sorted({f9_m for f9_ms in f9_need["models"] for f9_m in f9_ms if not f9_m.endswith("_log")})
f9_log_m = sorted({f9_m[:-4] for f9_ms in f9_need["models"] for f9_m in f9_ms if f9_m.endswith("_log")})
f9_fc = pd.MultiIndex.from_product([f9_plan["unique_id"], F9_FUT], names=["unique_id", "ds"]).to_frame(index=False)
for f9_ms, f9_build, f9_is_log in [(f9_raw_m, F1Y_RAW, False), (f9_log_m, F1Y_LOG, True)]:
    for f9_m in f9_ms:                                                    # one model at a time, only where needed
        f9_col = f9_m + ("_log" if f9_is_log else "")
        f9_ids = f9_need.loc[f9_need["models"].apply(lambda f9_l: f9_col in f9_l), "unique_id"]
        f9_d = f9_train[f9_train["unique_id"].isin(f9_ids)][["unique_id", "ds", "y"]]
        assert (f9_d.groupby("unique_id")["ds"].max() == F9_CUTOFF).all(), "training must end at week 66 - re-run cell 9a, then this cell"
        f9_sf = StatsForecast(models=[f9_build[f9_m]()], freq="W-MON", n_jobs=-1,
                              fallback_model=HistoricAverage(alias=f9_col))
        if f9_is_log:
            f9_model_out = f9_sf.forecast(df=f9_d.assign(y=np.log1p(f9_d["y"])), h=F1_H, level=[95])
            f9_sig = (f9_model_out[f"{f9_col}-hi-95"] - f9_model_out[f"{f9_col}-lo-95"]) / (2 * F1L_Z95)
            f9_model_out[f9_col] = np.exp(f9_model_out[f9_col]) * (1 + f9_sig ** 2 / 2) - 1        # 7b #11 bias adjustment
        else:
            f9_model_out = f9_sf.forecast(df=f9_d, h=F1_H)
        assert set(f9_model_out["ds"]) == set(F9_FUT), f"{f9_col}: forecast weeks are not 67-79 - re-run cell 9a, then this cell"
        f9_fc = f9_fc.merge(f9_model_out[["unique_id", "ds", f9_col]], on=["unique_id", "ds"], how="left")

# ---- 3. The base forecast per series and week ----
f9_fc = f9_fc.merge(f9_plan, on="unique_id", how="left")
f9_fc["base"] = [f9_r[f9_ms].mean() if f9_ms else 0.0 for f9_ms, (_, f9_r) in zip(f9_fc["models"], f9_fc.iterrows())]
f9_judge_ids = f9_plan.loc[f9_plan["method"].str.startswith("judgement"), "unique_id"]
f9_rate = (f9_train[f9_train["unique_id"].isin(f9_judge_ids)].merge(f9_first.rename("first"), left_on="unique_id",
           right_index=True).query("ds >= first").groupby("unique_id")["y"].mean())   # sales since first sale / weeks
f9_j = f9_fc["unique_id"].isin(f9_judge_ids)
f9_fc.loc[f9_j, "base"] = f9_fc.loc[f9_j, "unique_id"].map(f9_rate)
assert f9_fc["base"].notna().all() and len(f9_fc) == 283 * F1_H, "a base forecast is missing - re-run cell 9a, then this cell"
f9_short = f9_need[f9_need["unique_id"].map(f9_weeks_seen) < F1B_ETS_MIN_WEEKS]
f9_stamp["9b"] = F9_VERSION                                               # 9b ran on this version

# ---- Report ----
print(f"STEP 9b - final BASE forecasts, weeks 67-79, learned from weeks 1-66, version '{F9_VERSION}' (nothing written)\n")
print(f"f9_fc: {len(f9_fc):,} rows = 283 series x {F1_H} weeks ({F9_FUT[0].date()} to {F9_FUT[-1].date()}) | "
      f"models fitted: {f9_raw_m} + log {f9_log_m}")
display(f9_plan.groupby(["level", "method"]).size().rename("series").to_frame()
        .reindex(list(F1B_LEVELS.values()), level=0))
print("Judgement rates (cases per week, flat for 13 weeks) - please review:")
display(f9_plan[f9_plan["method"].str.startswith("judgement")].assign(
    rate_per_week=lambda f9_x: f9_x["unique_id"].map(f9_rate)).drop(columns="models").round(2).reset_index(drop=True))
f9_tot = f9_fc.groupby("level")[["base"]].sum().reindex(list(F1B_LEVELS.values()))
print("13-week base forecast per level (levels do NOT add up yet - that is cell 9c):")
display(f9_tot.round(0))
print(f"Series with fewer than {F1B_ETS_MIN_WEEKS} weeks among the modelled ones (fallback possible): "
      f"{list(f9_short['unique_id'])} | below 0: {int((f9_fc['base'] < 0).sum())} rows (lowest {f9_fc['base'].min():.2f})")
print("NEXT: cell 9c - MinT shrink (W from the practice rounds, decision D1 = A), set-negative-to-zero, ranges, flags.")


In [ ]:
# ==========================================================================
# STEP 9 - CELL 9c: MAKE THE FINAL FORECASTS ADD UP (MinT), SET NEGATIVES TO 0, ADD RANGES AND FLAGS
#   1. Reconcile: the method chosen in Step 7 (F1U_CHOICE = MinT shrink, cell 22 v2) with the SAME W as the backtest:
#      the shrunk covariance of the cv1 + cv2 practice errors (f1u_W_both) - user decision D1 = A, 12 Nov 2026
#      (exactly the version tested in Step 8: 80% range held 84%, 95% held 96%; Step 8 errors are NOT added).
#   2. Set-negative-to-zero (as cell 23 v2): products below 0 -> 0, and products STOPPED in the final labels -> exactly 0
#      (MinT can leave them a small amount: at most 0.0067 cases a week in Step 7's backtest, 0.44 in the final dry run); then every brand, sub-category, category and the
#      Total is re-added from the products with S, so all levels add up exactly.  Products that went to -0.1 case or
#      lower get the flag "set to 0 by reconciliation - review" (F1O_NEG_REVIEW).
#   3. Ranges (7f, as cell 23 v2): forecast + error percentile x scale per group (f1r_q, from cv1 + cv2 errors);
#      scale = mean |weekly change| over the learning weeks 1-66; lower end never below 0.
#      Judgement, stopped and near-flat series (scale < F1R_MIN_SCALE) get no range (blank), as in Step 7.
#   4. Flags: lane flags (stopped / new launch / one-sale - please review), 3x the highest past week, the Step 7 (7f)
#      review flags, and the reconciliation review flag.
#
# WHY (FPP2 2nd ed., Hyndman & Athanasopoulos 2018):
#   Ch 10 introduction (pp.392-393): coherent forecasts "add up in a manner that is consistent with the aggregation
#     structure";  10.7 (equation 10.8): G = (S'W^-1 S)^-1 S'W^-1, MinT.
#   12.3: forecasts constrained to be positive;  3.5 (p.88): 80% / 95% ranges;  p.467: no formula for Croston-type
#     intervals -> ranges from past errors.
#   Not in FPP2 (named, peer-reviewed): Wickramasuriya, Athanasopoulos & Hyndman (2019, JASA) - MinT;
#     Di Fonzo & Girolimetto (2023); Girolimetto (2025, Forecasting) - set-negative-to-zero (the same sources as Step 7 cell 23 v2, cell 30).
#
# WHERE EACH INPUT COMES FROM (all already in memory; nothing read from disk):
#   f9_fc, f9_plan -> cell 9b;  f9_train, f9_route, F9_FUT, F9_VERSION, f9_stamp -> cell 9a;  F1_H -> 7a
#   F1U_IDS, F1U_BOTTOM, F1U_S, F1U_CHOICE, f1u_reconcile, f1u_W_both -> 7g cell 22 v2
#   f1r_q, F1R_MIN_SCALE, f1r_cv -> 7f cell 18;  f1z_risk ("flag list") -> 7f cell 19
#   F1O_NEG_REVIEW -> 7g cell 23 v2;  f1b_level_of -> 7b;  F1E_LEVEL_ORDER -> 7b #6
#   pd, np -> 7a imports;  display -> Jupyter built-in
#
# WHAT IT CHANGES: nothing.  WHAT IT WRITES: nothing (memory only - cell 9d saves).  New names start with f9_ / F9_.
# PASTE: a new cell directly BELOW cell 9b; run it once, straight after 9b.
# ==========================================================================
assert "f9_stamp" in globals() and f9_stamp == {"9a": F9_VERSION, "9b": F9_VERSION}, \
    "run cells 9a -> 9b -> 9c in order (9c straight after 9b)"
F9_3X = 3.0                                                              # 3x safety check (as cell 21)

# ---- 1. Reconcile (D1 = A: W from the cv1 + cv2 practice errors, exactly as the backtest) ----
f9_base = f9_fc.pivot(index="unique_id", columns="ds", values="base").reindex(index=F1U_IDS, columns=F9_FUT).values
f9_all = f1u_reconcile(f9_base, f1u_W_both)                            # every method, 283 x 13 each
f9_rec, f9_bu = f9_all[F1U_CHOICE], f9_all["Bottom-up"]                  # chosen; bottom-up shown for comparison

# ---- 2. Set-negative-to-zero on the products, then re-add every level with S ----
f9_bidx = [F1U_IDS.index(f9_p) for f9_p in F1U_BOTTOM]
f9_prod_rec = pd.DataFrame(f9_rec[f9_bidx, :], index=F1U_BOTTOM, columns=F9_FUT)
f9_stopped = f9_route.loc[f9_route["lane"] == "zero", "unique_id"].tolist()   # stopped in the FINAL labels
f9_stop_max = float(f9_prod_rec.loc[f9_stopped].abs().max().max())        # what MinT gave them (reported)
f9_prod_use = f9_prod_rec.clip(lower=0)                                   # sales cannot be negative
f9_prod_use.loc[f9_stopped] = 0.0                                         # stopped products: exactly 0 (router rule)
f9_final = pd.DataFrame(F1U_S @ f9_prod_use.values, index=F1U_IDS, columns=F9_FUT)
f9_review = sorted(f9_prod_rec.index[f9_prod_rec.min(axis=1) <= F1O_NEG_REVIEW])
f9_gap = float(np.abs(F1U_S @ f9_final.loc[F1U_BOTTOM].values - f9_final.values).max())
assert f9_gap < 1e-6 and (f9_final.values >= 0).all(), "the final forecast must add up exactly and be >= 0 - re-run cells 9a -> 9b -> 9c"

# ---- 3. One long table: base, bottom-up, reconciled, forecast to use ----
def f9_long(f9_mat, f9_name):
    f9_long_df = pd.DataFrame(f9_mat, index=F1U_IDS, columns=F9_FUT).stack().rename(f9_name).reset_index()
    f9_long_df.columns = ["unique_id", "ds", f9_name]
    return f9_long_df

f9_table = f9_fc[["level", "unique_id", "ds", "method", "base"]].copy()
for f9_mat, f9_name in [(f9_bu, "Bottom-up"), (f9_rec, F1U_CHOICE), (f9_final.values, "forecast")]:
    f9_table = f9_table.merge(f9_long(f9_mat, f9_name), on=["unique_id", "ds"], how="left", validate="one_to_one")
f9_table["set to 0 (was below 0)"] = f9_table["unique_id"].isin(F1U_BOTTOM) & (f9_table[F1U_CHOICE] < 0)
f9_table["week"] = f9_table["ds"].map({f9_week_ds: 67 + f9_week_i for f9_week_i, f9_week_ds in enumerate(F9_FUT)})

# ---- 4. Ranges around the forecast to use (7f groups and error percentiles, scale from weeks 1-66) ----
f9_scale = f9_train.sort_values(["unique_id", "ds"]).groupby("unique_id")["y"].apply(lambda f9_s: f9_s.diff().abs().mean())
f9_group = f9_plan.set_index("unique_id").apply(
    lambda f9_plan_row: ("Product (intermittent, TSB)" if f9_plan_row["method"].startswith("TSB")
                  else "Brand (sparse)" if f9_plan_row["method"].endswith("(sparse brand)")
                  else None if f9_plan_row["method"].startswith(("zero", "judgement"))
                  else f9_plan_row["level"]), axis=1)
f9_table["group"] = f9_table["unique_id"].map(f9_group)
f9_table["scale"] = f9_table["unique_id"].map(f9_scale)
f9_ok = f9_table["group"].isin(f1r_q.index) & (f9_table["scale"] >= F1R_MIN_SCALE)
for f9_n, f9_lo, f9_hi in [("80", 0.10, 0.90), ("95", 0.025, 0.975)]:
    f9_lower = (f9_table["forecast"] + f9_table["group"].map(f1r_q[f9_lo]) * f9_table["scale"]).clip(lower=0)
    f9_upper = (f9_table["forecast"] + f9_table["group"].map(f1r_q[f9_hi]) * f9_table["scale"]).clip(lower=f9_lower)
    f9_table[f"lo{f9_n}"], f9_table[f"hi{f9_n}"] = f9_lower.where(f9_ok), f9_upper.where(f9_ok)

# ---- 5. Flags ----
f9_max = f9_train.groupby("unique_id")["y"].max()
f9_table["over 3x past max"] = f9_table["forecast"] > F9_3X * f9_table["unique_id"].map(f9_max)
f9_table["tested"] = f9_table["unique_id"].isin(set(f1r_cv["unique_id"]))          # had a practice forecast in Step 7
f9_table = (f9_table.merge(f9_route[["unique_id", "SKU", "lane"]], on="unique_id", how="left")
          .merge(f1z_risk[["unique_id", "flag list"]], on="unique_id", how="left"))
f9_table["flag list"] = f9_table["flag list"].fillna("")
f9_lane_flag = {"zero": "stopped (final label) - forecast 0", "judgement (new launch)": "new launch - judgement, please review",
                "judgement (one-sale)": "one-sale - judgement, please review"}
for f9_mask, f9_text in [(f9_table["lane"].map(f9_lane_flag).notna(), f9_table["lane"].map(f9_lane_flag)),
                         (f9_table["method"] == "judgement rate (new launch)", "sells only new launches - please review"),
                         (f9_table["unique_id"].isin(f9_review), "set to 0 by reconciliation - review")]:
    f9_table.loc[f9_mask, "flag list"] = (f9_table.loc[f9_mask, "flag list"] + "; " +
                                    (f9_text[f9_mask] if isinstance(f9_text, pd.Series) else f9_text)).str.strip("; ")
f9_table = f9_table[["level", "unique_id", "SKU", "ds", "week", "method", "lane", "tested", "base", "Bottom-up", F1U_CHOICE,
           "forecast", "lo80", "hi80", "lo95", "hi95", "set to 0 (was below 0)", "over 3x past max", "flag list"]]

# ---- 6. Checks ----
assert len(f9_table) == 283 * F1_H and f9_table["unique_id"].nunique() == 283, "rows must be 283 series x 13 weeks - re-run cells 9a -> 9b -> 9c"
assert sorted(f9_table["week"].unique()) == list(range(67, 80)), "weeks must be 67-79 - re-run cells 9a -> 9b -> 9c"
assert f9_table["forecast"].notna().all() and (f9_table["forecast"] >= 0).all(), "a forecast is missing or below 0 - re-run cells 9a -> 9b -> 9c"
assert (f9_table.loc[f9_table["unique_id"].isin(f9_stopped), "forecast"] == 0).all(), "stopped products must be 0 - re-run cells 9a -> 9b -> 9c"
assert ((f9_table["lo80"] <= f9_table["forecast"] + 1e-9) | f9_table["lo80"].isna()).all() and \
       ((f9_table["hi80"] >= f9_table["forecast"] - 1e-9) | f9_table["hi80"].isna()).all(), "ranges must contain the forecast - re-run cells 9a -> 9b -> 9c"

# ---- Report ----
print(f"STEP 9c - final forecasts reconciled ({F1U_CHOICE}, W from cv1 + cv2 errors), version '{F9_VERSION}' (nothing written)\n")
print(f"f9_table: {len(f9_table):,} rows = 283 series x 13 weeks (weeks 67-79) | largest gap between levels: {f9_gap:.2e} cases "
      f"(must be 0) | below 0: {int((f9_table['forecast'] < 0).sum())}")
print(f"Set-negative-to-zero: {int(f9_table['set to 0 (was below 0)'].sum())} product-weeks in "
      f"{f9_table.loc[f9_table['set to 0 (was below 0)'], 'unique_id'].nunique()} products were below 0 (lowest "
      f"{f9_table[F1U_CHOICE].min():.2f}) | flagged 'set to 0 by reconciliation - review': {len(f9_review)} products")
print(f"Stopped products forecast exactly 0: {len(f9_stopped)} of {len(f9_stopped)} - OK (MinT had given them at most "
      f"{f9_stop_max:.4f} cases a week; set to 0 by the router rule, then every level re-added)")
f9_sum = f9_table.groupby("level")[["base", "Bottom-up", F1U_CHOICE, "forecast"]].sum().reindex(F1E_LEVEL_ORDER)
f9_sum["with a range (share of rows)"] = f9_table.groupby("level")["lo80"].apply(lambda f9_s: f9_s.notna().mean()).round(2)
print("13-week totals per level (forecast = the numbers to use; they add up at every level):")
display(f9_sum.round({"base": 0, "Bottom-up": 0, F1U_CHOICE: 0, "forecast": 0}))
print(f"3x check: {int(f9_table['over 3x past max'].sum())} weeks flagged | series with any flag: "
      f"{int((f9_table.drop_duplicates('unique_id')['flag list'] != '').sum())}")
print("Example - the 3 categories in week 67 (ranges are per week; they are not added over weeks):")
display(f9_table[(f9_table["level"] == "Category2") & (f9_table["week"] == 67)]
        [["unique_id", "ds", "forecast", "lo80", "hi80", "lo95", "hi95"]].round(0).reset_index(drop=True))
f9_stamp["9c"] = F9_VERSION                                               # 9c ran on this version
print("NEXT: cell 9d - save the final table to its own file (never overwrites) and read it back.")


In [ ]:
# ==========================================================================
# STEP 9 - CELL 9d: SAVE THE FINAL FORECAST (weeks 67-79) + ITS ROUTING TABLE + ITS RANGE RECIPE - NEVER OVERWRITES
#   Run once per version, like the 7g save cell.  The version comes from cell 9a, which decided it from the DATA;
#   before saving, this cell checks the data AGAIN and checks that 9a, 9b and 9c all ran on that same version.
#   It saves 3 files per version into "../F1 outputs/" (F9_VERSION = dec_cleaned -> OFFICIAL; with_spikes -> comparison only, D2):
#     1. F1_final_forecast_{F9_VERSION}_weeks67-79.csv   - 283 series x 13 weeks = 3,679 rows (from cell 9c)
#     2. F1_final_routing_{F9_VERSION}_weeks1-66.csv     - 188 products: labels, lane and method (Option B, 13 Nov 2026)
#     3. F1_final_range_percentiles_{F9_VERSION}.csv     - the error percentiles per group that built the ranges (14 Nov 2026)
#   SAFETY: writes only into "F1 outputs" (never into "eabl parsed csv"); a file that already exists is NOT saved again
#   and nothing is overwritten.  After saving, each file is read back and compared with memory.
#   CHECKS: the input data files are unchanged since cell 9a (fingerprints), and the 13-week Total is compared with
#   the real sales of the last 13 weeks (weeks 54-66) as a sense check (not a score: weeks 67-79 have no sales here).
#
# WHY: FPP2 1.6 Step 5 ("using and evaluating a forecasting model") - the final forecast is the deliverable;
#   it is kept in versioned files (git + DVC) so it can be compared with EABL's real sales of weeks 67-79 later
#   (Step 10).  The routing table says which method each product got; the percentile table says how the 80% / 95%
#   ranges were built (FPP2 3.5: ranges from the distribution of past forecast errors).
#
# WHERE EACH INPUT COMES FROM (memory): f9_table, f9_group -> cell 9c;  f9_plan -> cell 9b;  f9_route, F9_VERSION, f9_stamp,
#   f9_fp_start, f9_fingerprint, f9_Y, f9_train, f9_total, f9_dec_recorded, F9_CHECK_WEEK, F9_CUTOFF -> cell 9a
#   f1r_q, F1R_Q, F1R_MIN_SCALE -> 7f ranges cell;  F1U_CHOICE -> 7g cell 22 v2;  F1SP_CLEAN -> 7a SPIKE v2 (cell 3);
#   f1_train -> 7a (cell 1, cleaned by cell 3);  F1_SEAL -> 7a;  pd -> 7a import;  os -> Python standard library
#
# WHAT IT CHANGES: nothing in memory.  WHAT IT WRITES: 3 CSVs per version.  New names start with f9_ / F9_.
# ==========================================================================
import os

F9_DIR = "../F1 outputs/"                                                 # the outputs folder (outside the dataset)
F9_FILE = F9_DIR + f"F1_final_forecast_{F9_VERSION}_weeks67-79.csv"
F9_ROUTE_FILE = F9_DIR + f"F1_final_routing_{F9_VERSION}_weeks1-66.csv"
F9_PCT_FILE = F9_DIR + f"F1_final_range_percentiles_{F9_VERSION}.csv"
F9_ROLE = "OFFICIAL" if F9_VERSION == "dec_cleaned" else "comparison only"

assert "f9_table" in globals() and len(f9_table) == 283 * 13 and f9_table["week"].between(67, 79).all(), "run cells 9a -> 9b -> 9c first"
# the version label must match the DATA and the switch, and 9a, 9b, 9c must all have run on it (fix of 13 Nov 2026)
assert f9_stamp == {"9a": F9_VERSION, "9b": F9_VERSION, "9c": F9_VERSION}, \
    f"cells 9a, 9b, 9c did not all run on version '{F9_VERSION}' ({f9_stamp}) - run 9a -> 9b -> 9c -> 9d in order"
f9_data_cleaned = f9_total(f9_train, F9_CHECK_WEEK) < f9_dec_recorded - 1
f9_s7_still = f9_total(f1_train["backtest"], F9_CHECK_WEEK) < f9_dec_recorded - 1
assert f9_data_cleaned == f9_s7_still == bool(F1SP_CLEAN) == (F9_VERSION == "dec_cleaned"), \
    "version label, learning data, Step 7 data and the cell 3 switch disagree - Kernel -> Restart & Run All; nothing saved"
assert all("eabl parsed csv" not in f9_fn for f9_fn in [F9_FILE, F9_ROUTE_FILE, F9_PCT_FILE]), "never write into the dataset folder"
assert f9_fingerprint() == f9_fp_start, "an input data file changed during Step 9 - Kernel -> Restart & Run All; nothing saved"

# ---- 1. Build the 3 tables (in memory only) ----
f9_out = f9_table.assign(version=F9_VERSION, role=F9_ROLE, reconciliation=F1U_CHOICE,
                         learned_weeks=f"1-66 (to {F9_CUTOFF.date()})")
f9_route_out = (f9_route.merge(f9_plan[["unique_id", "method"]], on="unique_id", how="left", validate="one_to_one")
                [["unique_id", "SKU", "label_final", "family_final", "history", "history_final", "first_sale",
                  "weeks_on_sale", "selling_weeks", "lane", "choose_by", "method",
                  "Step 7 lane (backtest)", "Step 7 choose_by (backtest)"]]
                .rename(columns={"label_final": "F2 label (weeks 1-66)", "family_final": "F5 family (weeks 1-66)",
                                 "history": "F3 at the seal", "history_final": "F3 on weeks 1-66"})
                .assign(version=F9_VERSION, role=F9_ROLE))
assert len(f9_route_out) == 188 and f9_route_out["unique_id"].is_unique and f9_route_out["method"].notna().all(), \
    "the routing table must have 188 products, each once, each with a method - re-run cells 9a -> 9b -> 9c"
f9_pct_names = {f9_p: f"{f9_rng} range {'lower' if f9_i == 0 else 'upper'} percentile ({f9_p:g})"
                for f9_rng, f9_lh in F1R_Q.items() for f9_i, f9_p in enumerate(f9_lh)}
f9_pct_out = (f1r_q.rename(columns=f9_pct_names).rename_axis("group").reset_index()
              .assign(errors_from="cv1 + cv2 (Step 7, decision D1 = A)", min_scale=F1R_MIN_SCALE,
                      how_to_use="range end = forecast + percentile x scale (scale = mean |weekly change| on weeks 1-66)",
                      version=F9_VERSION, role=F9_ROLE))
assert len(f9_pct_out) == len(f1r_q) and set(f9_pct_names.values()) <= set(f9_pct_out.columns) \
    and set(f9_group.dropna()) <= set(f1r_q.index), "the percentile table does not match f1r_q - re-run the 7f ranges cell and 9c"

# ---- 2. Save each file once (never overwrite), read it back and compare ----
os.makedirs(F9_DIR, exist_ok=True)
for f9_df, f9_path, f9_what, f9_key in [(f9_out, F9_FILE, "final forecast", "forecast"),
                                         (f9_route_out, F9_ROUTE_FILE, "routing table", "weeks_on_sale"),
                                         (f9_pct_out, F9_PCT_FILE, "range percentiles", "errors used")]:
    if os.path.exists(f9_path):
        print(f"NOT saved ({f9_what}): {f9_path} already exists (nothing was overwritten). Rename or delete it yourself to re-save.")
        continue
    f9_df.to_csv(f9_path, index=False)
    f9_back = pd.read_csv(f9_path)
    assert len(f9_back) == len(f9_df) and abs(f9_back[f9_key].sum() - f9_df[f9_key].sum()) < 1e-6, \
        f"the saved {f9_what} does not read back the same - delete {f9_path} yourself and run 9d again"
    print(f"SAVED {f9_what} - version '{F9_VERSION}' ({F9_ROLE}): {f9_path}  | {len(f9_back):,} rows, read back and identical")

# ---- 3. Sense check: 13-week Total vs the real sales of the last 13 weeks (weeks 54-66, as recorded) ----
f9_real_last13 = f9_Y[(f9_Y["unique_id"] == "Total") & (f9_Y["ds"] > F1_SEAL)]["y"].sum()
f9_fc_total = f9_table.loc[f9_table["level"] == "Total", "forecast"].sum()
print(f"\n13-week Total: forecast weeks 67-79 = {f9_fc_total:,.0f} cases | real sales weeks 54-66 = {f9_real_last13:,.0f} "
      f"| difference {f9_fc_total / f9_real_last13 - 1:+.1%}  (a sense check, not a score)")
print(f"Version check (from the DATA): 22 Dec 2025 Total learned = {f9_total(f9_train, F9_CHECK_WEEK):,.0f} vs recorded "
      f"{f9_dec_recorded:,.0f} -> '{F9_VERSION}', same as the cell 3 switch and cells 9a / 9b / 9c - OK")
print("Input data files unchanged since cell 9a (fingerprints match) - OK")
print("Routing table lanes:", f9_route_out["lane"].value_counts().to_dict())
print("  Files in the folder now:", sorted(f9_fn for f9_fn in os.listdir(F9_DIR) if f9_fn.startswith("F1_final_")))
print("NEXT: run the notebook once more with F1SP_CLEAN = False for the comparison files (if not saved yet); "
      "then switch back to True + Restart & Run All; then dvc add / dvc push / git commit / git push.")


In [ ]:
# ==========================================================================
# STEP 9 - CELL 9e: SAVE THE PRACTICE-ROUND EVIDENCE (cv1 + cv2) FOR THE APP - NEVER OVERWRITES
#   The models were chosen on two practice rounds inside weeks 1-53 (FPP2 3.4, time-series cross-validation):
#     cv1: learned weeks 1-27, forecast weeks 28-40      cv2: learned weeks 1-40, forecast weeks 41-53
#   Those results exist only in memory (Step 7).  This cell writes them to 6 files per version so the Shiny app can
#   show, for every series, how each candidate model did against real sales.  It computes NO new forecast or score:
#   every number is copied from Step 7, except the cv2 ranges, which are rebuilt with Step 7's own rule (7f) and
#   then checked against Step 7's coverage table (they must match exactly, or nothing is saved).
#   Files, in "../F1 outputs/" (F9_VERSION = dec_cleaned -> OFFICIAL; with_spikes -> comparison only, D2):
#     1. F1_practice_forecasts_{v}_cv1-cv2.csv     every candidate model's forecast + real sales, per series and week
#     2. F1_practice_scores_{v}_cv1-cv2.csv        every candidate's score per series and round (MAE/MASE/WAPE; RMSSE for 7c)
#     3. F1_practice_chosen_{v}_cv1-cv2.csv        the chosen forecast per series + cv2 80% / 95% ranges (from cv1 errors)
#     4. F1_practice_winners_{v}.csv               the 7b winner per series (picked on MASE over both rounds) vs Naive
#     5. F1_practice_reconciliation_{v}_cv2.csv    Bottom-up / OLS / WLS / MinT scores on cv2 (why MinT was chosen)
#     6. F1_practice_range_coverage_{v}_cv2.csv    share of real cv2 weeks inside the 80% / 95% ranges, per group
#   SAFETY: same rules as 9d - writes only into "F1 outputs", a file that already exists is NOT saved again,
#   each file is read back, the version comes from the DATA, and the input files must be unchanged since 9a.
#
# WHY: FPP2 3.4 - accuracy must be judged on data the model did not learn from; the app shows that evidence.
#
# WHERE EACH INPUT COMES FROM (memory): f1l_all, F1L_ALL_MODELS -> 7b cell 11 (cell 8);  f1e_scores, f1e_win -> 7b cell 6
#   (cell 9);  f1k_cv, f1k_ms, f1k_rm, F1K_INT -> 7c cell 13a;  F1I_GUARD -> 7c cell 12;  f1r_cv, f1r_q1, f1r_cov, F1R_Q -> 7f
#   ranges cell;  f1u_score, F1U_CHOICE -> 7g cell 22 v2;  f1_labels, f1_train, F1SP_CLEAN -> 7a cells 1-3;
#   F9_VERSION, f9_stamp, f9_Y, f9_train, f9_total, f9_dec_recorded, F9_CHECK_WEEK, f9_fingerprint, f9_fp_start -> 9a;
#   pd, np -> 7a imports;  os -> Python standard library
#
# WHAT IT CHANGES: nothing in memory.  WHAT IT WRITES: 6 CSVs per version.  New names start with f9e_ / F9E_.
# ==========================================================================
import os

F9E_DIR = "../F1 outputs/"
F9E_ROLE = "OFFICIAL" if F9_VERSION == "dec_cleaned" else "comparison only"
F9E_FILES = {"forecasts": f"F1_practice_forecasts_{F9_VERSION}_cv1-cv2.csv",
             "scores": f"F1_practice_scores_{F9_VERSION}_cv1-cv2.csv",
             "chosen": f"F1_practice_chosen_{F9_VERSION}_cv1-cv2.csv",
             "winners": f"F1_practice_winners_{F9_VERSION}.csv",
             "reconciliation": f"F1_practice_reconciliation_{F9_VERSION}_cv2.csv",
             "coverage": f"F1_practice_range_coverage_{F9_VERSION}_cv2.csv"}
F9E_7B = "7b standard models (12)"
F9E_7C = "7c intermittent models (7 + Zero for comparison)"

# ---- 0. Safety checks (same as 9d) - nothing is saved if one fails ----
assert f9_stamp == {"9a": F9_VERSION, "9b": F9_VERSION, "9c": F9_VERSION}, \
    f"cells 9a, 9b, 9c did not all run on version '{F9_VERSION}' ({f9_stamp}) - run 9a -> 9b -> 9c -> 9d -> 9e in order"
f9e_data_cleaned = f9_total(f9_train, F9_CHECK_WEEK) < f9_dec_recorded - 1
f9e_s7_cleaned = f9_total(f1_train["backtest"], F9_CHECK_WEEK) < f9_dec_recorded - 1
assert f9e_data_cleaned == f9e_s7_cleaned == bool(F1SP_CLEAN) == (F9_VERSION == "dec_cleaned"), \
    "version label, learning data, Step 7 data and the cell 3 switch disagree - Kernel -> Restart & Run All; nothing saved"
assert all("eabl parsed csv" not in F9E_DIR + f9e_fn for f9e_fn in F9E_FILES.values()), "never write into the dataset folder"
assert f9_fingerprint() == f9_fp_start, "an input data file changed during Step 9 - Kernel -> Restart & Run All; nothing saved"
assert set(f1l_all["set"]) == set(f1r_cv["set"]) == set(f1k_cv["set"]) == {"cv1", "cv2"}, \
    "the practice tables are not the cv1 / cv2 rounds - Kernel -> Restart & Run All"

f9e_week1 = f9_Y["ds"].min()                                                # Monday of week 1 (7 Apr 2025)
f9e_week = lambda f9e_ds: (f9e_ds - f9e_week1).dt.days // 7 + 1           # date -> week number 1-79
f9e_sku = f1_labels.set_index("unique_id")["SKU"]

# ---- 1. Every candidate model's forecast, long format (one row = round x series x week x model) ----
f9e_fc_7b = (f1l_all.melt(id_vars=["set", "level", "unique_id", "ds", "y"], value_vars=F1L_ALL_MODELS,
                          var_name="model", value_name="forecast").assign(candidate_set=F9E_7B))
f9e_fc_7c = (f1k_cv.melt(id_vars=["set", "level", "unique_id", "ds", "y", "lane_this_round"],
                         value_vars=F1K_INT + [F1I_GUARD], var_name="model", value_name="forecast")
             .assign(candidate_set=F9E_7C))
f9e_forecasts = pd.concat([f9e_fc_7b, f9e_fc_7c], ignore_index=True)
f9e_forecasts["week"] = f9e_week(f9e_forecasts["ds"])
f9e_forecasts["SKU"] = f9e_forecasts["unique_id"].map(f9e_sku)
f9e_forecasts = f9e_forecasts[["set", "level", "unique_id", "SKU", "ds", "week", "y", "candidate_set", "model", "forecast",
                               "lane_this_round"]]
assert len(f9e_forecasts) == len(f1l_all) * len(F1L_ALL_MODELS) + len(f1k_cv) * (len(F1K_INT) + 1), "forecast rows lost"

# ---- 2. Every candidate's score per round (copied from Step 7, not recomputed) ----
f9e_sc_7b = f1e_scores.assign(candidate_set=F9E_7B, RMSSE=np.nan)
f9e_lvl_7c = f1k_cv.drop_duplicates(["set", "unique_id"]).set_index(["set", "unique_id"])["level"]
f9e_sc_7c = (pd.concat({"MASE": f1k_ms.stack(), "RMSSE": f1k_rm.stack()}, axis=1)
             .rename_axis(["set", "unique_id", "model"]).reset_index())
f9e_sc_7c["level"] = pd.MultiIndex.from_frame(f9e_sc_7c[["set", "unique_id"]]).map(f9e_lvl_7c)
f9e_sc_7c = f9e_sc_7c.assign(candidate_set=F9E_7C, MAE=np.nan, WAPE=np.nan)
f9e_scores = pd.concat([f9e_sc_7b, f9e_sc_7c], ignore_index=True)
f9e_scores["SKU"] = f9e_scores["unique_id"].map(f9e_sku)
f9e_scores = f9e_scores[["set", "level", "unique_id", "SKU", "candidate_set", "model", "MAE", "MASE", "WAPE", "RMSSE"]]
assert f9e_scores["level"].notna().all(), "a 7c score has no level - stop and check"

# ---- 3. The chosen forecast + cv2 ranges, rebuilt with Step 7's rule (7f: cv1 percentiles x scale) ----
f9e_chosen = f1r_cv.copy()
f9e_chosen["week"] = f9e_week(f9e_chosen["ds"])
f9e_chosen["SKU"] = f9e_chosen["unique_id"].map(f9e_sku)
f9e_test = (f9e_chosen["set"] == "cv2") & f9e_chosen["r"].notna() & f9e_chosen["group"].isin(f1r_q1.index)
for f9e_name, (f9e_lo, f9e_hi) in F1R_Q.items():                        # "80%": (0.10, 0.90), "95%": (0.025, 0.975)
    f9e_n = f9e_name.rstrip("%")
    f9e_low = (f9e_chosen["chosen"] + f9e_chosen["group"].map(f1r_q1[f9e_lo]) * f9e_chosen["scale"]).clip(lower=0)
    f9e_high = (f9e_chosen["chosen"] + f9e_chosen["group"].map(f1r_q1[f9e_hi]) * f9e_chosen["scale"]).clip(lower=f9e_low)
    f9e_chosen[f"lo{f9e_n}"], f9e_chosen[f"hi{f9e_n}"] = f9e_low.where(f9e_test), f9e_high.where(f9e_test)
    f9e_chosen[f"inside {f9e_name}"] = ((f9e_chosen["y"] >= f9e_low) & (f9e_chosen["y"] <= f9e_high)).where(f9e_test)
    f9e_cov = f9e_chosen[f9e_test].groupby("group")[f"inside {f9e_name}"].mean()
    assert np.allclose(f9e_cov.reindex(f1r_cov.index).values, f1r_cov[f"{f9e_name} coverage"].values, atol=1e-12), \
        f"the rebuilt cv2 {f9e_name} ranges do not match Step 7's coverage table - nothing saved; tell Claude"
f9e_chosen = f9e_chosen[["set", "level", "unique_id", "SKU", "ds", "week", "y", "chosen", "group", "rule", "scale", "h", "r",
                         "lo80", "hi80", "lo95", "hi95", "inside 80%", "inside 95%"]]

# ---- 4-6. Winners, reconciliation scores, range coverage (copied) ----
f9e_winners = f1e_win.copy()
f9e_recon = f1u_score.rename_axis("method").reset_index()
f9e_recon["chosen"] = f9e_recon["method"] == F1U_CHOICE
f9e_coverage = f1r_cov.rename_axis("group").reset_index()

f9e_tables = {"forecasts": (f9e_forecasts, "forecast"), "scores": (f9e_scores, "MASE"), "chosen": (f9e_chosen, "chosen"),
              "winners": (f9e_winners, "winner_avg_MASE"), "reconciliation": (f9e_recon, "mean of the 5 levels"),
              "coverage": (f9e_coverage, "80% coverage")}

# ---- 7. Save each file once (never overwrite), read it back and compare ----
os.makedirs(F9E_DIR, exist_ok=True)
for f9e_key, (f9e_df, f9e_col) in f9e_tables.items():
    f9e_path = F9E_DIR + F9E_FILES[f9e_key]
    if os.path.exists(f9e_path):
        print(f"NOT saved ({f9e_key}): {f9e_path} already exists (nothing was overwritten). Rename or delete it yourself to re-save.")
        continue
    f9e_out = f9e_df.assign(version=F9_VERSION, role=F9E_ROLE)
    f9e_out.to_csv(f9e_path, index=False)
    f9e_back = pd.read_csv(f9e_path)
    assert len(f9e_back) == len(f9e_out) and np.isclose(f9e_back[f9e_col].sum(), f9e_out[f9e_col].sum(), rtol=1e-12, atol=1e-6), \
        f"the saved {f9e_key} file does not read back the same - delete {f9e_path} yourself and run 9e again"
    print(f"SAVED {f9e_key:<15} - version '{F9_VERSION}' ({F9E_ROLE}): {f9e_path}  | {len(f9e_back):,} rows, read back and identical")

# ---- 8. Report ----
print(f"\nPractice evidence, version '{F9_VERSION}':")
print(f"  candidate forecasts: {f9e_forecasts['unique_id'].nunique()} series | 7b: {f1l_all['unique_id'].nunique()} series x "
      f"{len(F1L_ALL_MODELS)} models | 7c: {f1k_cv['unique_id'].nunique()} series x {len(F1K_INT) + 1} models | weeks "
      f"{int(f9e_forecasts['week'].min())}-{int(f9e_forecasts['week'].max())}")
print(f"  chosen forecasts: {f9e_chosen['unique_id'].nunique()} series | cv2 rows with a range: {int(f9e_test.sum()):,}")
print("  cv2 range coverage rebuilt = Step 7's table exactly (80%: "
      f"{f9e_chosen.loc[f9e_test, 'inside 80%'].mean():.3f}, 95%: {f9e_chosen.loc[f9e_test, 'inside 95%'].mean():.3f} overall) - OK")
print(f"  reconciliation chosen on cv2: {F1U_CHOICE}")
print("Input data files unchanged since cell 9a (fingerprints match) - OK")
print("  Practice files in the folder now:", sorted(f9e_fn for f9e_fn in os.listdir(F9E_DIR) if f9e_fn.startswith("F1_practice_")))
print("NEXT: run once more with F1SP_CLEAN = False for the comparison files (if not saved yet); then back to True + "
      "Restart & Run All; then dvc add / dvc push / git commit / git push.")
